# Step 6.1

In [ ]:
import os
import sys
import platform
import shutil

try:
    import torch
except ImportError:
    torch = None

GiB = 1024 ** 3
ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
disk = shutil.disk_usage("/content" if os.path.isdir("/content") else ".")

print("=== SYSTEM ENVIRONMENT ===")
print(f"Python             : {sys.version.split()[0]}")
print(f"Operating system   : {platform.platform()}")
print(f"PyTorch            : {torch.__version__ if torch else 'NOT INSTALLED'}")
print(f"PyTorch build CUDA : {torch.version.cuda if torch else 'N/A'}")
print(f"CPU cores (logical): {os.cpu_count()}")
print(f"System RAM         : {ram_bytes / GiB:.2f} GiB")
print(f"Disk free / total  : {disk.free / GiB:.2f} / {disk.total / GiB:.2f} GiB")

=== SYSTEM ENVIRONMENT ===
Python             : 3.13.16
Operating system   : Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch            : 2.11.0+cu130
PyTorch build CUDA : 13.0
CPU cores (logical): 12
System RAM         : 83.47 GiB
Disk free / total  : 193.17 / 235.68 GiB


In [ ]:
import subprocess

print("=== NVIDIA-SMI ===")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=20,
        check=False,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        print(f"nvidia-smi exit code: {result.returncode}")
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print(f"Không chạy được nvidia-smi: {exc}")

print("\n=== PYTORCH GPU CHECK ===")
try:
    import torch

    available = torch.cuda.is_available()
    print(f"CUDA available: {available}")
    print(f"GPU count     : {torch.cuda.device_count()}")

    if available:
        current = torch.cuda.current_device()
        print(f"GPU đang chọn : cuda:{current}")

        for index in range(torch.cuda.device_count()):
            with torch.cuda.device(index):
                props = torch.cuda.get_device_properties(index)
                capability = torch.cuda.get_device_capability(index)
                free_bytes, total_bytes = torch.cuda.mem_get_info()
                # Yêu cầu hỗ trợ BF16 native, tránh coi emulation là hỗ trợ native.
                bf16 = capability[0] >= 8 and torch.cuda.is_bf16_supported()
                is_a100 = "A100" in props.name.upper()

            print(f"\nGPU {index}: {props.name}")
            print(f"  Total VRAM         : {total_bytes / 1024**3:.2f} GiB")
            print(f"  Free VRAM          : {free_bytes / 1024**3:.2f} GiB")
            print(f"  Compute capability : {capability[0]}.{capability[1]}")
            print(f"  Native BF16        : {bf16}")
            print(f"  NVIDIA A100        : {is_a100}")

            if not is_a100:
                print(f"  GPU được cấp là {props.name}, không phải A100.")
    else:
        print(
            "\nChưa có CUDA GPU khả dụng.\n"
            "Trong Colab: Runtime → Change runtime type → chọn GPU.\n"
            "Chọn A100 nếu tùy chọn này khả dụng, lưu và chạy lại các cell.\n"
            "Kiểm tra tên GPU thực tế sau khi runtime kết nối lại."
        )
except Exception as exc:
    print(f"Không hoàn tất kiểm tra PyTorch GPU: {type(exc).__name__}: {exc}")

=== NVIDIA-SMI ===
Sun Oct 25 15:25:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [ ]:
# Reset kết quả mỗi lần chạy để không dùng nhầm trạng thái PASS cũ.
_colab_gpu_test = {
    "status": "FAIL",
    "device_identity": None,
    "fp32_pass": False,
    "bf16_pass": None,
    "error": None,
}

a = b = c = a_bf16 = b_bf16 = c_bf16 = None
cuda_device = None

try:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA không khả dụng; hãy chọn GPU runtime trong Colab.")

    cuda_device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(cuda_device)
    capability = torch.cuda.get_device_capability(cuda_device)
    bf16_supported = capability[0] >= 8 and torch.cuda.is_bf16_supported()

    _colab_gpu_test["device_identity"] = (
        cuda_device, props.name, capability, props.total_memory
    )

    with torch.inference_mode():
        # Chỉ dùng hai ma trận nhỏ; không benchmark hoặc stress test.
        a = torch.randn(256, 256, device=cuda_device, dtype=torch.float32)
        b = torch.randn(256, 256, device=cuda_device, dtype=torch.float32)

        # Tắt autocast để phép thử đầu tiên có dtype FP32.
        with torch.autocast("cuda", enabled=False):
            c = a @ b

        torch.cuda.synchronize(cuda_device)
        if not torch.isfinite(c).all().item():
            raise RuntimeError("Matmul FP32 sinh NaN hoặc Inf.")
        _colab_gpu_test["fp32_pass"] = True
        print(f"FP32 matmul: PASS — shape={tuple(c.shape)}, dtype={c.dtype}")

        if bf16_supported:
            _colab_gpu_test["bf16_pass"] = False
            a_bf16 = a.to(torch.bfloat16)
            b_bf16 = b.to(torch.bfloat16)

            with torch.autocast("cuda", enabled=False):
                c_bf16 = a_bf16 @ b_bf16

            torch.cuda.synchronize(cuda_device)
            if not torch.isfinite(c_bf16).all().item():
                raise RuntimeError("Matmul BF16 sinh NaN hoặc Inf.")
            _colab_gpu_test["bf16_pass"] = True
            print(
                f"BF16 matmul: PASS — "
                f"shape={tuple(c_bf16.shape)}, dtype={c_bf16.dtype}"
            )
        else:
            print("BF16 matmul: SKIPPED — GPU không hỗ trợ BF16 native.")

    torch.cuda.synchronize(cuda_device)
    _colab_gpu_test["status"] = "PASS"

except Exception as exc:
    _colab_gpu_test["error"] = f"{type(exc).__name__}: {exc}"
    print(f"GPU computation: FAIL — {_colab_gpu_test['error']}")

finally:
    # Bỏ mọi tham chiếu tới tensors dùng trong phép thử.
    a = b = c = a_bf16 = b_bf16 = c_bf16 = None

    if cuda_device is not None:
        try:
            torch.cuda.synchronize(cuda_device)
            torch.cuda.empty_cache()
        except Exception as exc:
            _colab_gpu_test["status"] = "FAIL"
            _colab_gpu_test["error"] = f"CUDA cleanup failed: {exc}"

    print(f"\nGPU computation test: {_colab_gpu_test['status']}")
    if _colab_gpu_test["error"]:
        print(f"Chi tiết: {_colab_gpu_test['error']}")

FP32 matmul: PASS — shape=(256, 256), dtype=torch.float32
BF16 matmul: PASS — shape=(256, 256), dtype=torch.bfloat16

GPU computation test: PASS


In [ ]:
report = {
    "gpu_name": "N/A",
    "total_vram": "N/A",
    "cuda_available": False,
    "bf16_supported": False,
    "pytorch_version": "N/A",
    "is_a100": False,
    "gpu_test": "FAIL",
}
reason = "Cell 3 chưa chạy hoặc chưa hoàn tất thành công."

try:
    import torch

    report["pytorch_version"] = torch.__version__
    report["cuda_available"] = torch.cuda.is_available()

    if report["cuda_available"]:
        index = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(index)
        capability = torch.cuda.get_device_capability(index)
        _, total_bytes = torch.cuda.mem_get_info(index)

        report["gpu_name"] = props.name
        report["total_vram"] = f"{total_bytes / 1024**3:.2f} GiB"
        report["bf16_supported"] = (
            capability[0] >= 8 and torch.cuda.is_bf16_supported()
        )
        report["is_a100"] = "A100" in props.name.upper()

        identity = (index, props.name, capability, props.total_memory)
        test = globals().get("_colab_gpu_test", {})

        required_tests_passed = (
            test.get("status") == "PASS"
            and test.get("fp32_pass") is True
            and test.get("device_identity") == identity
            and (
                not report["bf16_supported"]
                or test.get("bf16_pass") is True
            )
        )
        if required_tests_passed:
            report["gpu_test"] = "PASS"
            reason = ""
        else:
            reason = test.get("error") or (
                "Hãy chạy lại Cell 3 trên GPU hiện tại."
            )
    else:
        reason = "CUDA không khả dụng; hãy chuyển Colab sang GPU runtime."

except Exception as exc:
    reason = f"{type(exc).__name__}: {exc}"

ready = (
    report["cuda_available"]
    and report["is_a100"]
    and report["bf16_supported"]
    and report["gpu_test"] == "PASS"
)

print("=== STEP 6.1 ENVIRONMENT REPORT ===")
print(f"GPU name             : {report['gpu_name']}")
print(f"Total VRAM           : {report['total_vram']}")
print(f"CUDA available       : {report['cuda_available']}")
print(f"Native BF16 supported: {report['bf16_supported']}")
print(f"PyTorch version      : {report['pytorch_version']}")
print(f"GPU computation test : {report['gpu_test']}")
print(f"Environment readiness: {'READY' if ready else 'NOT READY'}")

if not report["is_a100"] and report["cuda_available"]:
    print("Chưa đáp ứng GPU A100 mục tiêu; GPU thực tế được ghi ở trên.")
if reason:
    print(f"Chi tiết              : {reason}")
if ready:
    print("Đã đạt kiểm tra GPU cơ bản; chưa xác minh data/model/training pipeline.")

=== STEP 6.1 ENVIRONMENT REPORT ===
GPU name             : NVIDIA A100-SXM4-40GB
Total VRAM           : 39.49 GiB
CUDA available       : True
Native BF16 supported: True
PyTorch version      : 2.11.0+cu130
GPU computation test : PASS
Environment readiness: READY
Đã đạt kiểm tra GPU cơ bản; chưa xác minh data/model/training pipeline.


# Step 6.2: KẾT NỐI GITHUB VÀ KIỂM TRA MÔ HÌNH TRÊN GOOGLE COLAB

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import re

REPO_URL = input("GitHub HTTPS repository URL: ").strip()
REF_KIND = input("Loại ref — branch hoặc commit: ").strip().lower()
GIT_REF = input("Tên branch hoặc full commit SHA: ").strip()

PROJECT_DIR = Path("/content/moe-erc")
USE_COLAB_SECRET = False  # Đổi thành True nếu repo private.
SECRET_NAME = "GITHUB_TOKEN"

parsed = urlparse(REPO_URL)
if (
    parsed.scheme != "https"
    or parsed.netloc.lower() != "github.com"
    or parsed.username
    or parsed.password
    or parsed.query
    or parsed.fragment
    or len(parsed.path.strip("/").split("/")) != 2
):
    raise ValueError("Dùng HTTPS URL dạng https://github.com/OWNER/REPO.git, không kèm credentials.")

if REF_KIND not in {"branch", "commit"} or not GIT_REF or GIT_REF.startswith("-"):
    raise ValueError("Phải chọn rõ branch hoặc commit và cung cấp ref hợp lệ.")

if REF_KIND == "commit" and not re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF):
    raise ValueError("Commit phải là full SHA gồm 40 ký tự hex.")

print("Repository :", REPO_URL)
print("Ref        :", REF_KIND, GIT_REF)
print("Directory  :", PROJECT_DIR)
print("Private auth:", USE_COLAB_SECRET)

GitHub HTTPS repository URL: https://github.com/kptrong/MoE-ERC-Loss.git
Loại ref — branch hoặc commit: branch
Tên branch hoặc full commit SHA: main
Repository : https://github.com/kptrong/MoE-ERC-Loss.git
Ref        : branch main
Directory  : /content/moe-erc
Private auth: False


In [ ]:
import os
import base64
import subprocess

def run_git(arguments, *, cwd=None, authenticate=False):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    for key in list(env):
        if key.startswith("GIT_TRACE"):
            env.pop(key)

    if authenticate and USE_COLAB_SECRET:
        from google.colab import userdata
        token = userdata.get(SECRET_NAME)
        if not token:
            raise RuntimeError("Colab secret đang rỗng.")
        authorization = base64.b64encode(
            f"x-access-token:{token}".encode()
        ).decode()
        env.update({
            "GIT_CONFIG_COUNT": "2",
            "GIT_CONFIG_KEY_0": "credential.helper",
            "GIT_CONFIG_VALUE_0": "",
            "GIT_CONFIG_KEY_1": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_1": f"AUTHORIZATION: basic {authorization}",
        })

    result = subprocess.run(
        ["git", *arguments], cwd=cwd, env=env,
        text=True, capture_output=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

def repository_key(url):
    return url.rstrip("/").removesuffix(".git")

if REF_KIND == "branch":
    run_git(["check-ref-format", "--branch", GIT_REF])

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError("Thư mục đã tồn tại nhưng không phải Git repository; không ghi đè.")

    origin = run_git(["remote", "get-url", "origin"], cwd=PROJECT_DIR)
    if repository_key(origin) != repository_key(REPO_URL):
        raise RuntimeError("Repository hiện có khác REPO_URL; không thay đổi thư mục.")

    if run_git(["status", "--porcelain"], cwd=PROJECT_DIR):
        raise RuntimeError("Repository có thay đổi local; hãy kiểm tra trước khi tiếp tục.")

    head = run_git(["rev-parse", "HEAD"], cwd=PROJECT_DIR)
    if REF_KIND == "commit":
        if head.lower() != GIT_REF.lower():
            raise RuntimeError("Commit hiện tại khác commit yêu cầu; không tự checkout.")
    else:
        branch = run_git(["branch", "--show-current"], cwd=PROJECT_DIR)
        if branch != GIT_REF:
            raise RuntimeError(f"Branch hiện tại là {branch!r}; không tự chuyển branch.")

    print("Dùng repository hiện có; không pull hoặc ghi đè.")
else:
    if REF_KIND == "branch":
        run_git(
            ["clone", "--single-branch", "--branch", GIT_REF,
             REPO_URL, str(PROJECT_DIR)],
            authenticate=True,
        )
    else:
        run_git(
            ["clone", "--no-checkout", REPO_URL, str(PROJECT_DIR)],
            authenticate=True,
        )
        run_git(
            ["fetch", "--no-tags", "origin", GIT_REF],
            cwd=PROJECT_DIR, authenticate=True,
        )
        run_git(["checkout", "--detach", GIT_REF], cwd=PROJECT_DIR)

required_files = [
    "pyproject.toml",
    "moe_erc/config.py",
    "moe_erc/model.py",
    "moe_erc/model_factory.py",
    "moe_erc/aoe.py",
    "scripts/smoke_test_aoe.py",
    "tests/test_aoe_model.py",
]
missing = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError(f"Checkout thiếu thành phần STEP 5A: {missing}")

os.chdir(PROJECT_DIR)
PROJECT_COMMIT = run_git(["rev-parse", "HEAD"], cwd=PROJECT_DIR)
print("Project directory:", Path.cwd())
print("Git commit       :", PROJECT_COMMIT)

for folder in ("moe_erc", "scripts", "tests", "references"):
    path = PROJECT_DIR / folder
    print(f"\n{folder}/")
    if path.is_dir():
        for item in sorted(path.iterdir()):
            if item.name != "__pycache__":
                print("  ", item.name)

Project directory: /content/moe-erc
Git commit       : 665cb7b1cfbca4c6ed3507345b21008321129a2a

moe_erc/
   __init__.py
   __main__.py
   aoe.py
   aoe_expert.py
   attention.py
   config.py
   data_processing.py
   data_smoke.py
   dataset.py
   decoder.py
   dolma_reader.py
   experts.py
   model.py
   model_factory.py
   moe.py
   parameter_count.py
   rmsnorm.py
   rope.py
   router.py
   smoke_test.py
   tokenizer_utils.py

scripts/
   __init__.py
   _common.py
   inspect_batch.py
   inspect_dolma.py
   prepare_dataset.py
   smoke_test_aoe.py
   train_tokenizer.py

tests/
   __init__.py
   data_fixtures.py
   test_aoe.py
   test_aoe_expert.py
   test_aoe_model.py
   test_config.py
   test_data_cli.py
   test_data_processing.py
   test_dataset.py
   test_decoder.py
   test_dolma_reader.py
   test_experts.py
   test_model.py
   test_model_factory.py
   test_model_initialization.py
   test_moe.py
   test_parameter_count.py
   test_router.py
   test_tokenizer.py
   test_transformer_c

In [ ]:
import sys
import tempfile
import subprocess
import importlib
import importlib.metadata as metadata
import tomllib
import torch

from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.version import Version

manifest = tomllib.loads((PROJECT_DIR / "pyproject.toml").read_text())
project = manifest["project"]

if Version(sys.version.split()[0]) not in SpecifierSet(project["requires-python"]):
    raise RuntimeError(f"Python không đáp ứng {project['requires-python']}.")

torch_before = metadata.version("torch")
build_cuda_before = torch.version.cuda
print("Python giữ nguyên :", sys.version.split()[0])
print("PyTorch giữ nguyên:", torch_before)
print("PyTorch build CUDA:", build_cuda_before)

other_dependencies = []
for text in project["dependencies"]:
    requirement = Requirement(text)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    if requirement.name.lower().replace("_", "-") == "torch":
        if Version(torch_before) not in requirement.specifier:
            raise RuntimeError(
                f"{text} không tương thích với PyTorch {torch_before}. "
                "Dừng cài đặt; cần xem xét compatibility trước khi chỉ cài "
                "dependencies không phải torch và package bằng --no-deps."
            )
    else:
        other_dependencies.append(text)

requirements_file = PROJECT_DIR / "requirements.txt"
if requirements_file.exists():
    print("Ưu tiên pyproject.toml; không chạy pip -r requirements.txt.")
    for line in requirements_file.read_text().splitlines():
        line = line.split("#", 1)[0].strip()
        if not line or line.startswith("-"):
            continue
        try:
            requirement = Requirement(line)
        except Exception:
            continue
        if (
            requirement.name.lower() == "torch"
            and Version(torch_before) not in requirement.specifier
        ):
            print(
                f"CẢNH BÁO: requirements.txt có {line}, không tương thích. "
                "Chỉ cài dependencies không phải torch từ pyproject.toml."
            )

with tempfile.TemporaryDirectory(prefix="moe-colab-install-") as temporary:
    constraint = Path(temporary) / "constraints.txt"
    constraint.write_text(f"torch=={torch_before}\n")

    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--constraint", str(constraint),
         *other_dependencies, "pytest>=9"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-deps", "--editable", str(PROJECT_DIR)],
        check=True,
    )

if metadata.version("torch") != torch_before:
    raise RuntimeError("PyTorch đã thay đổi ngoài dự kiến; dừng kiểm tra.")
if torch.version.cuda != build_cuda_before:
    raise RuntimeError("PyTorch CUDA build đã thay đổi ngoài dự kiến.")

# Nếu một thư viện đang được import đã đổi phiên bản, cần restart runtime.
for name in ("numpy", "tokenizers", "pytest"):
    loaded = sys.modules.get(name)
    if loaded is not None and hasattr(loaded, "__version__"):
        if Version(loaded.__version__) != Version(metadata.version(name)):
            raise RuntimeError(
                f"{name} đã đổi phiên bản trên đĩa. Restart runtime rồi chạy lại các cell."
            )

importlib.invalidate_caches()
import moe_erc

if PROJECT_DIR.resolve() not in Path(moe_erc.__file__).resolve().parents:
    raise RuntimeError("Đang import moe_erc từ vị trí khác; hãy restart runtime.")

print("\nPackage:", moe_erc.__file__)
for distribution in ("moe-erc-research", "torch", "numpy", "tokenizers", "pytest"):
    print(f"{distribution}: {metadata.version(distribution)}")

Python giữ nguyên : 3.13.16
PyTorch giữ nguyên: 2.11.0+cu130
PyTorch build CUDA: 13.0

Package: /content/moe-erc/moe_erc/__init__.py
moe-erc-research: 0.1.0
torch: 2.11.0+cu130
numpy: 2.1.3
tokenizers: 0.23.3
pytest: 9.1.1


In [ ]:
import gc
import torch

from moe_erc import ModelConfig, AoEConfig, count_parameters
from moe_erc.model import MoELanguageModel, AoELanguageModel
from moe_erc.model_factory import create_model

if not torch.cuda.is_available():
    raise RuntimeError("CUDA không khả dụng. Kiểm tra lại GPU runtime.")

device = torch.device("cuda", torch.cuda.current_device())
gpu_name = torch.cuda.get_device_name(device)
print("GPU thực tế:", gpu_name)
if "A100" not in gpu_name.upper():
    print("LƯU Ý: GPU hiện tại không phải A100.")

MODEL_SPECS = [
    ("vanilla_moe", ModelConfig(), MoELanguageModel, 52_288_000),
    ("aoe", AoEConfig(), AoELanguageModel, 52_263_424),
]

def check_parameters(kind, config, expected_class, expected_count):
    model = None
    try:
        model = create_model(kind, config)
        if type(model) is not expected_class:
            raise TypeError(f"{kind}: factory trả về sai model class.")

        actual = sum(p.numel() for p in model.parameters() if p.requires_grad)
        analytical = count_parameters(config, model_type=kind).total
        print(f"{kind}: actual={actual:,}; analytical={analytical:,}; expected={expected_count:,}")

        if actual != expected_count or actual != analytical:
            raise AssertionError(f"{kind}: parameter count không khớp; không sửa kiến trúc.")

        model = model.to(device)
        if model.lm_head.weight is not model.token_embedding.weight:
            raise AssertionError(f"{kind}: input/output embeddings không cùng Parameter.")

        torch.cuda.synchronize(device)
        print(f"  CUDA placement: PASS; tied embeddings: PASS")
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

for specification in MODEL_SPECS:
    check_parameters(*specification)

GPU thực tế: NVIDIA A100-SXM4-40GB
vanilla_moe: actual=52,288,000; analytical=52,288,000; expected=52,288,000
  CUDA placement: PASS; tied embeddings: PASS
aoe: actual=52,263,424; analytical=52,263,424; expected=52,263,424
  CUDA placement: PASS; tied embeddings: PASS


In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA không khả dụng.")

device = torch.device("cuda", torch.cuda.current_device())
bf16 = (
    torch.cuda.get_device_capability(device)[0] >= 8
    and torch.cuda.is_bf16_supported()
)

torch.manual_seed(123)
cpu_ids = torch.randint(0, 16000, (1, 16), dtype=torch.long)

def check_forward(kind, config):
    model = input_ids = logits = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

        model = create_model(kind, config).to(device).eval()
        input_ids = cpu_ids.to(device)

        with torch.inference_mode():
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bf16):
                logits = model(input_ids)

        torch.cuda.synchronize(device)

        if tuple(logits.shape) != (1, 16, 16000):
            raise AssertionError(f"{kind}: logits shape sai: {tuple(logits.shape)}")
        if not torch.isfinite(logits).all().item():
            raise AssertionError(f"{kind}: logits chứa NaN hoặc Inf.")

        GiB = 1024 ** 3
        print(f"\n{kind}: FORWARD PASS")
        print("  Logits shape:", tuple(logits.shape))
        print("  Logits dtype:", logits.dtype)
        print("  Finite      : True")
        print(f"  Allocated   : {torch.cuda.memory_allocated(device) / GiB:.3f} GiB")
        print(f"  Reserved    : {torch.cuda.memory_reserved(device) / GiB:.3f} GiB")
        print(f"  Peak alloc. : {torch.cuda.max_memory_allocated(device) / GiB:.3f} GiB")
    finally:
        logits = input_ids = model = None
        gc.collect()
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()

try:
    for kind, config, _, _ in MODEL_SPECS:
        check_forward(kind, config)
finally:
    cpu_ids = None
    gc.collect()

print("\nĐã giải phóng models và tensors dùng cho forward test.")


vanilla_moe: FORWARD PASS
  Logits shape: (1, 16, 16000)
  Logits dtype: torch.bfloat16
  Finite      : True
  Allocated   : 0.204 GiB
  Reserved    : 0.232 GiB
  Peak alloc. : 0.220 GiB

aoe: FORWARD PASS
  Logits shape: (1, 16, 16000)
  Logits dtype: torch.bfloat16
  Finite      : True
  Allocated   : 0.204 GiB
  Reserved    : 0.246 GiB
  Peak alloc. : 0.220 GiB

Đã giải phóng models và tensors dùng cho forward test.


In [ ]:
import os
import sys
import subprocess

ignored_modules = [
    "tests/test_tokenizer.py",
    "tests/test_data_processing.py",
    "tests/test_dataset.py",
]
deselected_test = (
    "tests/test_model_factory.py::ModelFactoryTests::"
    "test_existing_data_pipeline_batch_into_aoe"
)

print("Project root:", PROJECT_DIR)
print("Commit:", PROJECT_COMMIT)
print("Không chạy các test train tokenizer fixture:")
for name in [*ignored_modules, deselected_test]:
    print("  ", name)

pytest_arguments = [
    "tests", "-q", "-ra", "--tb=short", "-p", "no:cacheprovider",
    *[f"--ignore={name}" for name in ignored_modules],
    f"--deselect={deselected_test}",
]

# Chạy trong process riêng để tránh giữ model/CUDA tensors trong notebook.
runner = r'''
import sys
import torch
import pytest

torch.set_num_threads(2)

class Summary:
    def pytest_terminal_summary(self, terminalreporter, exitstatus, config):
        stats = terminalreporter.stats
        print("\n=== PYTEST REPORT ===")
        for category in ("passed", "failed", "skipped", "error", "deselected"):
            print(f"{category}: {len(stats.get(category, []))}")
        print(f"exit code: {exitstatus}")

raise SystemExit(pytest.main(sys.argv[1:], plugins=[Summary()]))
'''

env = os.environ.copy()
env.update({
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "2",
    "MKL_NUM_THREADS": "2",
})
env.pop("PYTEST_ADDOPTS", None)

result = subprocess.run(
    [sys.executable, "-c", runner, *pytest_arguments],
    cwd=PROJECT_DIR,
    env=env,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Pytest thất bại với exit code {result.returncode}. "
        "Xem traceback phía trên; không có source code nào được tự sửa."
    )

print("\nSTEP 6.2: các test được chọn đã PASS; nhóm train-tokenizer chưa được chạy.")

Project root: /content/moe-erc
Commit: 665cb7b1cfbca4c6ed3507345b21008321129a2a
Không chạy các test train tokenizer fixture:
   tests/test_tokenizer.py
   tests/test_data_processing.py
   tests/test_dataset.py
   tests/test_model_factory.py::ModelFactoryTests::test_existing_data_pipeline_batch_into_aoe

STEP 6.2: các test được chọn đã PASS; nhóm train-tokenizer chưa được chạy.


In [ ]:
%cd /content/moe-erc

!python -m pytest -q \
    tests/test_tokenizer.py \
    tests/test_data_processing.py \
    tests/test_dataset.py \
    tests/test_model_factory.py::ModelFactoryTests::test_existing_data_pipeline_batch_into_aoe

/content
..................                                                       [100%]
18 passed in 3.31s


# STEP 6.3.1 — KIỂM TRA VÀ LỰA CHỌN DOLMA SUBSET TRÊN GOOGLE COLAB

In [ ]:
from pathlib import Path
from itertools import islice
import shutil
import subprocess

PROJECT_DIR = Path("/content/moe-erc")
DATA_ROOT = PROJECT_DIR / "data"
EXPECTED_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
GiB = 1024 ** 3

if not (PROJECT_DIR / "moe_erc/dolma_reader.py").is_file():
    raise RuntimeError(f"Không tìm thấy Dolma Reader tại {PROJECT_DIR}.")

result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
    check=True,
)
PROJECT_COMMIT = result.stdout.strip()

disk = shutil.disk_usage(PROJECT_DIR)
print("=== STORAGE CHECK ===")
print("Project directory :", PROJECT_DIR)
print("Git commit        :", PROJECT_COMMIT)
print("Planned data root :", DATA_ROOT)
print(f"Disk total        : {disk.total / GiB:.2f} GiB")
print(f"Disk used         : {disk.used / GiB:.2f} GiB")
print(f"Disk free         : {disk.free / GiB:.2f} GiB")

if PROJECT_COMMIT != EXPECTED_COMMIT:
    print("LƯU Ý: commit hiện tại khác commit đã báo; không tự checkout.")

print("\n=== DATA DIRECTORIES ===")
for path in [
    DATA_ROOT,
    DATA_ROOT / "raw",
    DATA_ROOT / "processed",
    DATA_ROOT / "tokenizer",
    DATA_ROOT / "selections",
]:
    if not path.exists():
        print(f"{path}: NOT CREATED")
    elif not path.is_dir():
        print(f"{path}: EXISTS, nhưng không phải thư mục")
    else:
        preview = [entry.name for entry in islice(path.iterdir(), 5)]
        print(f"{path}: EXISTS; tối đa 5 mục đầu: {preview}")

print("\nChưa tải manifest hoặc shard; chưa tạo thư mục dữ liệu.")

=== STORAGE CHECK ===
Project directory : /content/moe-erc
Git commit        : 2fd3478e478d3a20381e95b06890f8dcdf2ae013
Planned data root : /content/moe-erc/data
Disk total        : 235.68 GiB
Disk used         : 42.51 GiB
Disk free         : 193.15 GiB

=== DATA DIRECTORIES ===
/content/moe-erc/data: NOT CREATED
/content/moe-erc/data/raw: NOT CREATED
/content/moe-erc/data/processed: NOT CREATED
/content/moe-erc/data/tokenizer: NOT CREATED
/content/moe-erc/data/selections: NOT CREATED

Chưa tải manifest hoặc shard; chưa tạo thư mục dữ liệu.


In [ ]:
from collections import Counter
from pathlib import PurePosixPath
from urllib.parse import urlparse
import moe_erc.dolma_reader as dolma_reader

if PROJECT_DIR.resolve() not in Path(dolma_reader.__file__).resolve().parents:
    raise RuntimeError("Đang import Dolma Reader từ project khác.")

MANIFEST_TIMEOUT = 20
MANIFEST_RETRIES = 2

def source_group(url):
    return PurePosixPath(urlparse(url).path).parent.name or "UNKNOWN"

manifest = None
try:
    manifest = dolma_reader.read_manifest(
        dolma_reader.MANIFEST_URL,
        timeout=MANIFEST_TIMEOUT,
        retries=MANIFEST_RETRIES,
    )
except Exception as exc:
    raise RuntimeError(
        f"Không thể đọc/parse manifest {dolma_reader.MANIFEST_URL}: "
        f"{type(exc).__name__}: {exc}"
    ) from exc

urls = manifest["urls"]

invalid_urls = []
for url in urls:
    parsed = urlparse(url)
    if (
        parsed.scheme not in {"https", "http"}
        or not parsed.hostname
        or parsed.username
        or parsed.password
        or not parsed.path.endswith((".json.gz", ".jsonl.gz"))
    ):
        invalid_urls.append(url)

if invalid_urls:
    raise ValueError(f"Manifest có URL không đúng định dạng: {invalid_urls[:5]}")
if manifest["version"] != "v1_5-sample":
    raise ValueError(f"Sai dataset version: {manifest['version']}")

print("=== DOLMA MANIFEST ===")
print("Repository       :", manifest["repository"])
print("Version          :", manifest["version"])
print("Manifest URL     :", manifest["manifest_source"])
print("Manifest SHA-256 :", manifest["manifest_sha256"])
print("Unique shard URLs:", len(urls))

print("\nURLs mẫu — chỉ để xem cấu trúc, không đại diện toàn bộ dataset:")
for url in urls[:5]:
    print(" ", url)

SOURCE_GROUP_COUNTS = Counter(source_group(url) for url in urls)
print("\nNhóm suy ra từ đường dẫn URL, không phải kiểm chứng nội dung:")
for group, count in sorted(SOURCE_GROUP_COUNTS.items()):
    print(f"  {group}: {count} shards")

print("\nChỉ manifest đã được đọc; chưa tải shard.")

=== DOLMA MANIFEST ===
Repository       : allenai/dolma
Version          : v1_5-sample
Manifest URL     : https://huggingface.co/datasets/allenai/dolma/resolve/main/urls/v1_5-sample.txt
Manifest SHA-256 : d8b00c66388a160637a71f76672a1648a5b50ff3670f561065b946de3da72d6f
Unique shard URLs: 2418

URLs mẫu — chỉ để xem cấu trúc, không đại diện toàn bộ dataset:
  https://olmo-data.org/dolma-v1_5r1/books/books-0000.json.gz
  https://olmo-data.org/dolma-v1_5r1/books/books-0001.json.gz
  https://olmo-data.org/dolma-v1_5r1/books/books-0002.json.gz
  https://olmo-data.org/dolma-v1_5r1/c4/c4-0000.json.gz
  https://olmo-data.org/dolma-v1_5r1/c4/c4-0001.json.gz

Nhóm suy ra từ đường dẫn URL, không phải kiểm chứng nội dung:
  books: 3 shards
  c4: 86 shards
  cc_en_head: 571 shards
  cc_en_middle: 722 shards
  cc_en_tail: 781 shards
  pes2o: 26 shards
  reddit: 78 shards
  stack: 149 shards
  wiki: 2 shards

Chỉ manifest đã được đọc; chưa tải shard.


In [ ]:
import hashlib
import json

SEED = 42
MAX_CANDIDATE_SHARDS = 4
SOURCE_CONTAINS = None
# Ví dụ: SOURCE_CONTAINS = "/books/"
# Chỉ chọn giá trị sau khi xem nhóm nguồn ở Cell 2.

if manifest is None:
    raise RuntimeError("Hãy chạy thành công Cell 2 trước.")

selected_urls = dolma_reader.select_shards(
    manifest["urls"],
    max_shards=MAX_CANDIDATE_SHARDS,
    seed=SEED,
    source_contains=SOURCE_CONTAINS,
)

selection_plan = {
    "repository": manifest["repository"],
    "version": manifest["version"],
    "manifest_source": manifest["manifest_source"],
    "manifest_sha256": manifest["manifest_sha256"],
    "manifest_shard_count": len(manifest["urls"]),
    "selection_seed": SEED,
    "source_contains": SOURCE_CONTAINS,
    "requested_max_shards": MAX_CANDIDATE_SHARDS,
    "selection_algorithm": "sorted-unique, seeded-random-shuffle-v1",
    "selected_urls": selected_urls,
    "project_commit": PROJECT_COMMIT,
}

selection_fingerprint = hashlib.sha256(
    json.dumps(selection_plan, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

selection_dir = DATA_ROOT / "selections"
selection_dir.mkdir(parents=True, exist_ok=True)
SELECTION_FILE = selection_dir / (
    f"dolma-v1_5-sample-seed{SEED}-{selection_fingerprint[:16]}.json"
)

if SELECTION_FILE.exists():
    existing = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
    if existing != selection_plan:
        raise RuntimeError("Selection file đã tồn tại với nội dung khác; không ghi đè.")
    print("Dùng lại selection file giống hệt đã lưu.")
else:
    dolma_reader.atomic_json(SELECTION_FILE, selection_plan)
    print("Đã lưu selection mới.")

print("Selection file     :", SELECTION_FILE)
print("Selection plan SHA :", selection_fingerprint)
print("Seed               :", SEED)
print("Path filter        :", SOURCE_CONTAINS)
print("Requested / actual :", MAX_CANDIDATE_SHARDS, "/", len(selected_urls))

if len(selected_urls) < MAX_CANDIDATE_SHARDS:
    print("LƯU Ý: số shard phù hợp ít hơn số lượng yêu cầu.")

for index, url in enumerate(selected_urls, start=1):
    print(f"{index}. [{source_group(url)}] {url}")

print("\nDanh sách này là ứng viên, chưa được phê duyệt tải.")
print("Không giả định subset này đại diện toàn bộ phân phối Dolma.")

Đã lưu selection mới.
Selection file     : /content/moe-erc/data/selections/dolma-v1_5-sample-seed42-d37a10898c4709bf.json
Selection plan SHA : d37a10898c4709bf1b85b8bf56f30f9be8bb7c3cae3a5a9ef6f62cca8f2a6616
Seed               : 42
Path filter        : None
Requested / actual : 4 / 4
1. [cc_en_head] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
2. [stack] https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0144.json.gz
3. [cc_en_middle] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0653.json.gz
4. [cc_en_tail] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0587.json.gz

Danh sách này là ứng viên, chưa được phê duyệt tải.
Không giả định subset này đại diện toàn bộ phân phối Dolma.


In [ ]:
import shutil

HEAD_TIMEOUT_SECONDS = 10
DISK_RESERVE_GIB = 10  # Dự phòng do người dùng cấu hình, không phải ước tính preprocessing.

if not selected_urls:
    raise RuntimeError("Danh sách ứng viên rỗng; hãy chạy Cell 3.")

shard_sizes = {}
size_check_fingerprint = None

print("=== COMPRESSED SIZE CHECK — HEAD ONLY ===")
for index, url in enumerate(selected_urls, start=1):
    size = dolma_reader.compressed_size(
        url,
        timeout=HEAD_TIMEOUT_SECONDS,
    )
    if size is not None and size <= 0:
        size = None

    shard_sizes[url] = size
    description = (
        f"{size:,} bytes ({size / GiB:.3f} GiB)"
        if size is not None else "UNKNOWN"
    )
    print(f"\n{index}. {url}")
    print("   Content-Length:", description)

size_check_fingerprint = selection_fingerprint
known_total = sum(size for size in shard_sizes.values() if size is not None)
unknown_count = sum(size is None for size in shard_sizes.values())

disk = shutil.disk_usage(DATA_ROOT)
reserve_bytes = int(DISK_RESERVE_GIB * GiB)
usable_bytes = max(0, disk.free - reserve_bytes)

print("\n=== STORAGE ASSESSMENT ===")
print(f"Tổng dung lượng đã xác định : {known_total:,} bytes ({known_total / GiB:.3f} GiB)")
print(f"Số shard UNKNOWN           : {unknown_count}")
print(f"Ổ đĩa còn trống            : {disk.free / GiB:.2f} GiB")
print(f"Dự phòng cấu hình          : {DISK_RESERVE_GIB:.2f} GiB")
print(f"Còn lại sau dự phòng       : {usable_bytes / GiB:.2f} GiB")

if unknown_count:
    print("CẢNH BÁO: tổng ở trên chỉ là phần đã biết; tổng toàn bộ vẫn UNKNOWN.")

if known_total > disk.free:
    print("CẢNH BÁO: riêng phần đã biết đã vượt dung lượng còn trống.")
elif known_total > usable_bytes:
    print("CẢNH BÁO: phần đã biết vượt ngân sách ổ đĩa sau dự phòng.")
elif not unknown_count:
    print("Các bytes nén được server công bố nằm trong ngân sách hiện tại.")
    print("Điều này chưa chứng minh đủ chỗ cho tokenizer hoặc dữ liệu đã preprocess.")

print("\nContent-Length là thông tin server công bố, chưa kiểm chứng bằng nội dung file.")
print("Không tải shard; không suy ra số tokens hoặc dung lượng giải nén.")

=== COMPRESSED SIZE CHECK — HEAD ONLY ===

1. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
   Content-Length: UNKNOWN

2. https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0144.json.gz
   Content-Length: UNKNOWN

3. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0653.json.gz
   Content-Length: UNKNOWN

4. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0587.json.gz
   Content-Length: UNKNOWN

=== STORAGE ASSESSMENT ===
Tổng dung lượng đã xác định : 0 bytes (0.000 GiB)
Số shard UNKNOWN           : 4
Ổ đĩa còn trống            : 193.15 GiB
Dự phòng cấu hình          : 10.00 GiB
Còn lại sau dự phòng       : 183.15 GiB
CẢNH BÁO: tổng ở trên chỉ là phần đã biết; tổng toàn bộ vẫn UNKNOWN.

Content-Length là thông tin server công bố, chưa kiểm chứng bằng nội dung file.
Không tải shard; không suy ra số tokens hoặc dung lượng giải nén.


In [ ]:
import shutil

FIRST_TRIAL_MAX_BYTES = 1_000_000_000  # Giới hạn đề xuất 1 GB, không phải kích thước shard.

if size_check_fingerprint != selection_fingerprint:
    raise RuntimeError("Selection đã thay đổi; hãy chạy lại Cell 4.")

if json.loads(SELECTION_FILE.read_text(encoding="utf-8")) != selection_plan:
    raise RuntimeError("Selection file khác kế hoạch đang dùng; hãy kiểm tra lại.")

disk = shutil.disk_usage(DATA_ROOT)
known_total = sum(size for size in shard_sizes.values() if size is not None)
unknown_count = sum(size is None for size in shard_sizes.values())

print("=== STEP 6.3.1 SUMMARY ===")
print("Dataset version       :", manifest["version"])
print("Manifest shard count  :", len(manifest["urls"]))
print("Candidate shard count :", len(selected_urls))
print("Random seed           :", SEED)
print("Source path filter    :", SOURCE_CONTAINS)
print("Manifest SHA-256      :", manifest["manifest_sha256"])
print("Selection file        :", SELECTION_FILE)
print(f"Current disk free     : {disk.free / GiB:.2f} GiB")

if unknown_count == 0:
    print(
        f"Declared compressed total: {known_total:,} bytes "
        f"({known_total / GiB:.3f} GiB)"
    )
else:
    print("Compressed total      : UNKNOWN")
    print(f"Known portion         : {known_total:,} bytes ({known_total / GiB:.3f} GiB)")
    print(f"Unknown-size shards   : {unknown_count}")

print("\nCandidate URLs:")
for index, url in enumerate(selected_urls, start=1):
    size = shard_sizes[url]
    label = f"{size:,} bytes" if size is not None else "UNKNOWN"
    print(f"{index}. [{source_group(url)}] {url}\n   {label}")

print("\nChưa xác minh:")
print("- Nội dung gzip/JSONL, checksum và tính hợp lệ của documents.")
print("- Số documents, số tokens và kích thước dữ liệu sau preprocessing.")
print("- Mức độ đại diện của subset và phân phối giữa các nguồn.")
print("- Các shard này có trùng chính xác dữ liệu tác giả paper sử dụng hay không.")
print("- UNKNOWN không phân biệt được thiếu header, lỗi HEAD hay lỗi mạng.")

usable_bytes = max(0, disk.free - int(DISK_RESERVE_GIB * GiB))
trial_limit = min(FIRST_TRIAL_MAX_BYTES, usable_bytes)
eligible = [
    (url, size)
    for url, size in shard_sizes.items()
    if size is not None and size <= trial_limit
]

print("\n=== ĐỀ XUẤT LẦN TẢI THỬ ĐẦU TIÊN — CHƯA THỰC HIỆN ===")
if eligible:
    trial_url, trial_size = min(eligible, key=lambda item: (item[1], item[0]))
    print("Chỉ thử 1 shard có kích thước đã biết:")
    print("URL                    :", trial_url)
    print(f"Declared compressed size: {trial_size:,} bytes")
    print(f"Proposed download cap   : {trial_limit:,} bytes")
    print("Kiểm tra lại HEAD và dung lượng trống trước lần tải thực tế.")
    print("Lưu selection riêng chỉ chứa 1 URL khi bước download được yêu cầu.")
    print("Đây là lựa chọn cho smoke test kỹ thuật, không đại diện toàn bộ Dolma.")
else:
    print("Chưa có ứng viên kích thước đã biết phù hợp giới hạn tải thử.")
    print("Cần kiểm tra UNKNOWN hoặc điều chỉnh tập ứng viên/ngân sách thủ công.")
    print("Không tự chọn shard UNKNOWN để tải.")

print("\nĐã dừng ở STEP 6.3.1. Không tải shard, train tokenizer hoặc preprocess.")

=== STEP 6.3.1 SUMMARY ===
Dataset version       : v1_5-sample
Manifest shard count  : 2418
Candidate shard count : 4
Random seed           : 42
Source path filter    : None
Manifest SHA-256      : d8b00c66388a160637a71f76672a1648a5b50ff3670f561065b946de3da72d6f
Selection file        : /content/moe-erc/data/selections/dolma-v1_5-sample-seed42-d37a10898c4709bf.json
Current disk free     : 193.15 GiB
Compressed total      : UNKNOWN
Known portion         : 0 bytes (0.000 GiB)
Unknown-size shards   : 4

Candidate URLs:
1. [cc_en_head] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
   UNKNOWN
2. [stack] https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0144.json.gz
   UNKNOWN
3. [cc_en_middle] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0653.json.gz
   UNKNOWN
4. [cc_en_tail] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0587.json.gz
   UNKNOWN

Chưa xác minh:
- Nội dung gzip/JSONL, checksum và tính hợ

# Step 6.3.2

In [ ]:
from pathlib import Path, PurePosixPath
from urllib.parse import urlparse, urljoin
from collections import Counter
import hashlib
import json

from moe_erc.dolma_reader import (
    canonical_text, file_sha256, atomic_json, iter_documents,
)
from moe_erc.tokenizer_utils import SplitConfig, SPLIT_ALGORITHM

PROJECT_DIR = Path("/content/moe-erc")
SELECTION_FILE = PROJECT_DIR / (
    "data/selections/dolma-v1_5-sample-seed42-d37a10898c4709bf.json"
)

MiB = 1024 ** 2
LIMITS = {
    "max_shards": 4,
    "compressed_per_shard": 32 * MiB,
    "compressed_total": 128 * MiB,
    "decompressed_per_shard": 128 * MiB,
    "documents_per_shard": 1000,
    # Giới hạn RAM bổ sung: bỏ qua dòng quá lớn, có thống kê riêng.
    "line_bytes": 4 * MiB,
}
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 15
MAX_RETRIES_PER_SHARD = 2
MAX_REDIRECTS = 3
CHUNK_BYTES = 64 * 1024

if not SELECTION_FILE.is_file():
    raise FileNotFoundError(SELECTION_FILE)

selection = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
selected_urls = selection.get("selected_urls", [])

if (
    selection.get("repository") != "allenai/dolma"
    or selection.get("version") != "v1_5-sample"
    or selection.get("selection_seed") != 42
):
    raise ValueError("Selection không đúng repository/version/seed dự kiến.")

if len(selected_urls) != 4 or len(set(selected_urls)) != 4:
    raise ValueError("Selection phải chứa đúng 4 URLs khác nhau.")

def source_category(url):
    return PurePosixPath(urlparse(url).path).parent.name

for url in selected_urls:
    p = urlparse(url)
    if (
        p.scheme != "https"
        or p.netloc != "olmo-data.org"
        or p.username or p.password or p.query or p.fragment
        or not p.path.startswith("/dolma-v1_5")
        or not p.path.endswith(".json.gz")
    ):
        raise ValueError(f"URL không thuộc nguồn Dolma dự kiến: {url}")

selection_sha = file_sha256(SELECTION_FILE)
signature = hashlib.sha256(
    json.dumps(
        {"selection_sha256": selection_sha, "limits": LIMITS},
        sort_keys=True,
    ).encode()
).hexdigest()

if "_dolma_partial_state" in globals():
    if _dolma_partial_state["signature"] != signature:
        raise RuntimeError("Ledger hiện có thuộc selection/config khác; không tự reset.")
    print("Giữ lại ledger và số bytes đã đọc từ lần chạy trước.")
else:
    def new_record(url):
        return {
            "url": url,
            "source_category": source_category(url),
            "compressed_bytes": 0,
            "charged_bytes": 0,
            "uncertain_read_bytes": 0,
            "decompressed_bytes": 0,
            "complete_line_bytes": 0,
            "complete_lines": 0,
            "json_parsed": 0,
            "valid_documents": 0,
            "invalid_records": 0,
            "invalid_reasons": {},
            "missing_fields": {},
            "gzip_members_completed": 0,
            "probe_gzip_magic": None,
            "sample_gzip_magic": None,
            "probe_done": False,
            "sampling_finished": False,
            "retries_used": 0,
            "http_events": [],
            "errors": [],
            "activated_limits": [],
            "stop_reason": "not_started",
            "documents": [],
        }

    _dolma_partial_state = {
        "signature": signature,
        "selection_sha256": selection_sha,
        "records": {url: new_record(url) for url in selected_urls},
        "sample_dir": None,
    }

STATE = _dolma_partial_state
SPLIT_CONFIG = SplitConfig(seed=42, validation_fraction=0.01)

print("Selection:", SELECTION_FILE)
print("SHA-256 :", selection_sha)
print("Shards  :", len(selected_urls))
print("Seed    :", selection["selection_seed"])
print("Nhóm nguồn theo đường dẫn:", dict(Counter(map(source_category, selected_urls))))
for index, url in enumerate(selected_urls, 1):
    print(f"{index}. {url}")

print("\nGiới hạn:", LIMITS)
print("Chưa gửi HTTP GET hoặc tạo selection mới.")

Selection: /content/moe-erc/data/selections/dolma-v1_5-sample-seed42-d37a10898c4709bf.json
SHA-256 : 95ab822db32187928bd3fac802de6214e12728781ae84d2e448586115566ca6e
Shards  : 4
Seed    : 42
Nhóm nguồn theo đường dẫn: {'cc_en_head': 1, 'stack': 1, 'cc_en_middle': 1, 'cc_en_tail': 1}
1. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
2. https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0144.json.gz
3. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0653.json.gz
4. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0587.json.gz

Giới hạn: {'max_shards': 4, 'compressed_per_shard': 33554432, 'compressed_total': 134217728, 'decompressed_per_shard': 134217728, 'documents_per_shard': 1000, 'line_bytes': 4194304}
Chưa gửi HTTP GET hoặc tạo selection mới.


In [ ]:
import http.client
import re
import time

class BudgetStop(Exception):
    pass

def remaining_budget(record):
    total_charged = sum(r["charged_bytes"] for r in STATE["records"].values())
    return min(
        LIMITS["compressed_per_shard"] - record["charged_bytes"],
        LIMITS["compressed_total"] - total_charged,
    )

def bounded_read(response, record, requested):
    count = min(requested, remaining_budget(record))
    if count <= 0:
        raise BudgetStop("compressed_byte_budget")

    # Reserve trước khi đọc. Nếu lỗi đọc, không giải phóng phần không chắc chắn.
    record["charged_bytes"] += count
    try:
        data = response.read1(count)
    except BaseException as exc:
        partial = getattr(exc, "partial", b"")
        partial = partial if isinstance(partial, bytes) else b""
        record["compressed_bytes"] += len(partial)
        record["uncertain_read_bytes"] += max(0, count - len(partial))
        raise
    else:
        record["charged_bytes"] -= count - len(data)
        record["compressed_bytes"] += len(data)
        return data

def open_http(record, phase, requested_bytes):
    """GET có Range, timeout riêng, redirects thủ công; không đọc body ở đây."""
    while True:
        current_url = record["url"]
        connection = response = None
        try:
            span = min(requested_bytes, remaining_budget(record))
            if span <= 0:
                raise BudgetStop("compressed_byte_budget")

            for redirect_index in range(MAX_REDIRECTS + 1):
                p = urlparse(current_url)
                if (
                    p.scheme != "https" or not p.hostname
                    or p.username or p.password
                    or p.port not in (None, 443)
                ):
                    raise RuntimeError("Redirect không phải HTTPS hợp lệ.")

                connection = http.client.HTTPSConnection(
                    p.hostname, timeout=CONNECT_TIMEOUT,
                )
                connection.connect()
                connection.sock.settimeout(READ_TIMEOUT)

                target = p.path or "/"
                if p.query:
                    target += "?" + p.query

                connection.request("GET", target, headers={
                    "User-Agent": "moe-erc-bounded-probe",
                    "Accept-Encoding": "identity",
                    "Range": f"bytes=0-{span - 1}",
                    "Connection": "close",
                })
                response = connection.getresponse()

                event = {
                    "phase": phase,
                    "host": p.hostname,
                    "status": response.status,
                    "content_length": response.getheader("Content-Length"),
                    "content_range": response.getheader("Content-Range"),
                    "content_encoding": response.getheader("Content-Encoding"),
                }
                record["http_events"].append(event)

                if response.status in (301, 302, 303, 307, 308):
                    location = response.getheader("Location")
                    response.close()
                    connection.close()
                    if not location or redirect_index == MAX_REDIRECTS:
                        raise RuntimeError("Redirect thiếu Location hoặc vượt giới hạn.")
                    current_url = urljoin(current_url, location)
                    continue

                if response.status == 429 or response.status >= 500:
                    raise OSError(f"HTTP {response.status}")

                if response.status not in (200, 206):
                    raise RuntimeError(f"HTTP {response.status}")

                if response.status == 206:
                    value = response.getheader("Content-Range") or ""
                    if not re.fullmatch(r"bytes 0-\d+/(?:\d+|\*)", value):
                        raise RuntimeError("Content-Range không bắt đầu từ byte 0.")

                # HTTP 200 được chấp nhận; bounded_read vẫn áp giới hạn client.
                return connection, response

            raise RuntimeError("Vượt giới hạn redirects.")

        except (OSError, http.client.HTTPException) as exc:
            if response is not None:
                response.close()
            if connection is not None:
                connection.close()
            if record["retries_used"] >= MAX_RETRIES_PER_SHARD:
                raise RuntimeError(f"HTTP connection/header failed: {type(exc).__name__}: {exc}") from exc
            record["retries_used"] += 1
            time.sleep(1)

        except BaseException:
            if response is not None:
                response.close()
            if connection is not None:
                connection.close()
            raise

for url in selected_urls:
    record = STATE["records"][url]
    if record["probe_done"]:
        print("Đã probe, không đọc lại:", url)
        continue

    connection = response = None
    print("\nPROBE:", url)
    try:
        connection, response = open_http(record, "probe", 2)
        print("HTTP status    :", response.status)
        print("Content-Length :", response.getheader("Content-Length") or "UNKNOWN")
        print("Content-Range  :", response.getheader("Content-Range") or "NONE")
        print("Content-Encoding:", response.getheader("Content-Encoding") or "NONE")

        prefix = b""
        while len(prefix) < 2:
            chunk = bounded_read(response, record, 2 - len(prefix))
            if not chunk:
                break
            prefix += chunk

        record["probe_gzip_magic"] = prefix == b"\x1f\x8b"
        print("Gzip signature :", record["probe_gzip_magic"])
        print("Đây chỉ là chữ ký gzip, chưa xác nhận file gzip hoàn chỉnh.")

    except Exception as exc:
        record["errors"].append(f"probe: {type(exc).__name__}: {exc}")
        print(record["errors"][-1])
    finally:
        record["probe_done"] = True
        if response is not None:
            response.close()
        if connection is not None:
            connection.close()

print("\nRaw compressed bytes đã nhận:",
      sum(r["compressed_bytes"] for r in STATE["records"].values()))
print("Budget charged/reserved:",
      sum(r["charged_bytes"] for r in STATE["records"].values()))


PROBE: https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
HTTP status    : 200
Content-Length : 1225179157
Content-Range  : NONE
Content-Encoding: NONE
Gzip signature : True
Đây chỉ là chữ ký gzip, chưa xác nhận file gzip hoàn chỉnh.

PROBE: https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0144.json.gz
HTTP status    : 200
Content-Length : 1565847867
Content-Range  : NONE
Content-Encoding: NONE
Gzip signature : True
Đây chỉ là chữ ký gzip, chưa xác nhận file gzip hoàn chỉnh.

PROBE: https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0653.json.gz
HTTP status    : 200
Content-Length : 1267095203
Content-Range  : NONE
Content-Encoding: NONE
Gzip signature : True
Đây chỉ là chữ ký gzip, chưa xác nhận file gzip hoàn chỉnh.

PROBE: https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0587.json.gz
HTTP status    : 200
Content-Length : 1645640561
Content-Range  : NONE
Content-Encoding: NONE
Gzip signature : True
Đây chỉ 

In [ ]:
import zlib

def increment(mapping, key):
    mapping[key] = mapping.get(key, 0) + 1

def mark_limit(record, name):
    if name not in record["activated_limits"]:
        record["activated_limits"].append(name)

class SampleStop(Exception):
    pass

def sample_one(record):
    connection = response = None
    decoder = None
    line_buffer = bytearray()
    line_size = 0
    dropping_line = False
    magic = b""
    record["stop_reason"] = "in_progress"

    def consume_output(block):
        nonlocal line_size, dropping_line
        parts = block.split(b"\n")
        for index, fragment in enumerate(parts):
            line_size += len(fragment)
            if not dropping_line:
                if len(line_buffer) + len(fragment) > LIMITS["line_bytes"]:
                    dropping_line = True
                    line_buffer.clear()
                    mark_limit(record, "line_bytes")
                else:
                    line_buffer.extend(fragment)

            if index == len(parts) - 1:
                continue  # Chưa có newline: chưa parse.

            record["complete_lines"] += 1
            record["complete_line_bytes"] += line_size + 1

            if dropping_line:
                record["invalid_records"] += 1
                increment(record["invalid_reasons"], "line_too_large")
            else:
                try:
                    document = json.loads(bytes(line_buffer).decode("utf-8"))
                except (UnicodeError, ValueError):
                    record["invalid_records"] += 1
                    increment(record["invalid_reasons"], "invalid_utf8_or_json")
                else:
                    record["json_parsed"] += 1
                    missing = (
                        [key for key in ("id", "text", "source") if key not in document]
                        if isinstance(document, dict) else ["not_object"]
                    )
                    for field in missing:
                        increment(record["missing_fields"], field)

                    valid = (
                        isinstance(document, dict)
                        and not missing
                        and isinstance(document["id"], (str, int))
                        and not isinstance(document["id"], bool)
                        and str(document["id"]).strip() != ""
                        and isinstance(document["text"], str)
                        and bool(canonical_text(document["text"]))
                        and isinstance(document["source"], str)
                        and bool(document["source"].strip())
                    )
                    if not valid:
                        record["invalid_records"] += 1
                        increment(record["invalid_reasons"], "missing_or_invalid_fields")
                    else:
                        record["documents"].append({
                            "id": document["id"],
                            "text": canonical_text(document["text"]),
                            "source": document["source"],
                            "source_category": record["source_category"],
                            "origin_shard_url": record["url"],
                            "origin_line_number": record["complete_lines"],
                        })
                        record["valid_documents"] += 1

            line_buffer.clear()
            line_size = 0
            dropping_line = False

            if record["valid_documents"] >= LIMITS["documents_per_shard"]:
                mark_limit(record, "documents_per_shard")
                raise SampleStop("document_limit")

    try:
        connection, response = open_http(
            record, "sample", remaining_budget(record),
        )

        while True:
            if record["decompressed_bytes"] >= LIMITS["decompressed_per_shard"]:
                mark_limit(record, "decompressed_per_shard")
                raise SampleStop("decompressed_byte_limit")

            chunk = bounded_read(response, record, CHUNK_BYTES)
            if not chunk:
                record["stop_reason"] = (
                    "http_eof_at_gzip_member_boundary"
                    if decoder is None and record["gzip_members_completed"] > 0
                    else "http_eof_before_complete_gzip_member"
                )
                break

            magic = (magic + chunk)[:2]
            if len(magic) == 2:
                record["sample_gzip_magic"] = magic == b"\x1f\x8b"
                if not record["sample_gzip_magic"]:
                    raise RuntimeError("Body không bắt đầu bằng chữ ký gzip.")

            pending = chunk
            while True:
                if decoder is None:
                    decoder = zlib.decompressobj(16 + zlib.MAX_WBITS)

                remaining_output = (
                    LIMITS["decompressed_per_shard"] - record["decompressed_bytes"]
                )
                if remaining_output <= 0:
                    mark_limit(record, "decompressed_per_shard")
                    raise SampleStop("decompressed_byte_limit")

                output = decoder.decompress(
                    pending, min(CHUNK_BYTES, remaining_output),
                )
                record["decompressed_bytes"] += len(output)
                consume_output(output)

                if decoder.eof:
                    # Hỗ trợ gzip nhiều members; không đồng nhất member với shard.
                    pending = decoder.unused_data
                    record["gzip_members_completed"] += 1
                    decoder = None
                    if pending:
                        continue
                    break

                pending = decoder.unconsumed_tail
                if pending:
                    continue
                if len(output) == min(CHUNK_BYTES, remaining_output):
                    pending = b""  # Drain output nội bộ bằng max_length hữu hạn.
                    continue
                break

    except BudgetStop:
        mark_limit(record, "compressed_byte_budget")
        record["stop_reason"] = "compressed_byte_limit"
    except SampleStop as exc:
        record["stop_reason"] = str(exc)
    except Exception as exc:
        record["stop_reason"] = "stream_error"
        record["errors"].append(f"sample: {type(exc).__name__}: {exc}")
    finally:
        if record["stop_reason"] == "in_progress":
            record["stop_reason"] = "interrupted"
        record["sampling_finished"] = True
        record["unparsed_decompressed_bytes"] = (
            record["decompressed_bytes"] - record["complete_line_bytes"]
        )
        record["full_shard_verified"] = False
        if response is not None:
            response.close()
        if connection is not None:
            connection.close()

for url in selected_urls:
    record = STATE["records"][url]
    if record["sampling_finished"]:
        print("Không đọc lại shard đã thử:", url)
        continue

    sample_one(record)
    print({
        "source_category": record["source_category"],
        "compressed_bytes_including_probe": record["compressed_bytes"],
        "decompressed_bytes": record["decompressed_bytes"],
        "valid_documents": record["valid_documents"],
        "invalid_records": record["invalid_records"],
        "missing_fields": record["missing_fields"],
        "stop_reason": record["stop_reason"],
        "gzip_members_completed": record["gzip_members_completed"],
        "errors": record["errors"],
    })

assert sum(r["charged_bytes"] for r in STATE["records"].values()) <= LIMITS["compressed_total"]
assert all(r["charged_bytes"] <= LIMITS["compressed_per_shard"] for r in STATE["records"].values())
assert all(r["decompressed_bytes"] <= LIMITS["decompressed_per_shard"] for r in STATE["records"].values())
print("\nKhông lưu gzip bị cắt; các dòng thiếu newline không được parse.")

{'source_category': 'cc_en_head', 'compressed_bytes_including_probe': 1654750, 'decompressed_bytes': 4609545, 'valid_documents': 1000, 'invalid_records': 102, 'missing_fields': {}, 'stop_reason': 'document_limit', 'gzip_members_completed': 0, 'errors': []}
{'source_category': 'stack', 'compressed_bytes_including_probe': 1114114, 'decompressed_bytes': 7204896, 'valid_documents': 1000, 'invalid_records': 0, 'missing_fields': {}, 'stop_reason': 'document_limit', 'gzip_members_completed': 0, 'errors': []}
{'source_category': 'cc_en_middle', 'compressed_bytes_including_probe': 1703938, 'decompressed_bytes': 4863793, 'valid_documents': 1000, 'invalid_records': 90, 'missing_fields': {}, 'stop_reason': 'document_limit', 'gzip_members_completed': 0, 'errors': []}
{'source_category': 'cc_en_tail', 'compressed_bytes_including_probe': 1130489, 'decompressed_bytes': 3165095, 'valid_documents': 1000, 'invalid_records': 122, 'missing_fields': {}, 'stop_reason': 'document_limit', 'gzip_members_complet

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import subprocess

if not all(r["sampling_finished"] for r in STATE["records"].values()):
    raise RuntimeError("Hãy chạy Cell 3 cho tất cả shards trước.")
if STATE["sample_dir"] is not None:
    raise RuntimeError(f"Sample đã được lưu tại {STATE['sample_dir']}; không ghi đè.")

run_name = (
    "dolma-partial-"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-" + uuid4().hex[:8]
)
SAMPLE_DIR = PROJECT_DIR / "data/samples" / run_name
relative_directory = SAMPLE_DIR.relative_to(PROJECT_DIR)

ignored = subprocess.run(
    ["git", "check-ignore", "-q", "--", str(relative_directory / "train.jsonl")],
    cwd=PROJECT_DIR,
)
if ignored.returncode != 0:
    raise RuntimeError("Sample chưa được Git ignore; dừng để không sửa source/.gitignore.")

SAMPLE_DIR.mkdir(parents=True, exist_ok=False)
metadata_path = SAMPLE_DIR / "metadata.json"
atomic_json(metadata_path, {"complete": False, "sampling_mode": "partial_http_streaming"})

seen_hashes = set()
duplicate_count = 0
split_counts = Counter()
category_counts = Counter()

with (SAMPLE_DIR / "train.jsonl").open("x", encoding="utf-8") as train_file, \
     (SAMPLE_DIR / "validation.jsonl").open("x", encoding="utf-8") as validation_file:

    streams = {"train": train_file, "validation": validation_file}

    for url in selected_urls:
        for original in STATE["records"][url]["documents"]:
            document = dict(original)
            text = canonical_text(document["text"])
            content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()

            if content_hash in seen_hashes:
                duplicate_count += 1
                continue

            seen_hashes.add(content_hash)
            split_name = SPLIT_CONFIG.assign(content_hash)
            document.update(
                text=text,
                content_hash=content_hash,
                split=split_name,
                partial_streaming=True,
            )
            streams[split_name].write(
                json.dumps(document, ensure_ascii=False) + "\n"
            )
            split_counts[split_name] += 1
            category_counts[document["source_category"]] += 1

reports = [
    {key: value for key, value in STATE["records"][url].items() if key != "documents"}
    for url in selected_urls
]
sample_metadata = {
    "complete": True,
    "repository": selection["repository"],
    "version": selection["version"],
    "sampling_mode": "bounded_partial_http_streaming_not_complete_shards",
    "full_shard_integrity_verified": False,
    "selection_file": str(SELECTION_FILE),
    "selection_sha256": STATE["selection_sha256"],
    "selection_seed": selection["selection_seed"],
    "selected_urls": selected_urls,
    "limits": LIMITS,
    "split": {"seed": SPLIT_CONFIG.seed, "validation_fraction": SPLIT_CONFIG.validation_fraction},
    "split_algorithm": SPLIT_ALGORITHM,
    "duplicates_removed": duplicate_count,
    "split_counts": {name: split_counts[name] for name in ("train", "validation")},
    "documents_by_source_category": dict(category_counts),
    "tokenizer_training_input": "train.jsonl",
    "validation_input": "validation.jsonl",
    "files": {
        name: {"sha256": file_sha256(SAMPLE_DIR / name)}
        for name in ("train.jsonl", "validation.jsonl")
    },
    "shard_reports": reports,
}
atomic_json(metadata_path, sample_metadata)
STATE["sample_dir"] = str(SAMPLE_DIR)

# File đã lưu; giải phóng phần văn bản giữ trong RAM.
for record in STATE["records"].values():
    record["documents"].clear()

print("Sample directory:", SAMPLE_DIR)
print("Train documents :", split_counts["train"])
print("Validation docs :", split_counts["validation"])
print("Duplicates removed:", duplicate_count)
print("Source categories:", dict(category_counts))
if split_counts["validation"] == 0:
    print("CHƯA CÓ validation; không chuyển tài liệu train sang validation.")
print("Chưa train tokenizer hoặc preprocess token sequences.")

Sample directory: /content/moe-erc/data/samples/dolma-partial-20261024T033301Z-520c5300
Train documents : 3963
Validation docs : 37
Duplicates removed: 0
Source categories: {'cc_en_head': 1000, 'stack': 1000, 'cc_en_middle': 1000, 'cc_en_tail': 1000}
Chưa train tokenizer hoặc preprocess token sequences.


In [ ]:
from itertools import zip_longest

SAMPLE_DIR = Path(STATE["sample_dir"])
saved_metadata = json.loads(
    (SAMPLE_DIR / "metadata.json").read_text(encoding="utf-8")
)
if not saved_metadata.get("complete"):
    raise RuntimeError("Sample chưa được ghi hoàn tất.")
if file_sha256(SELECTION_FILE) != saved_metadata["selection_sha256"]:
    raise RuntimeError("Selection file đã thay đổi.")

verification_split = SplitConfig(**saved_metadata["split"])
hashes_by_split = {"train": set(), "validation": set()}
counts_by_category = Counter()
lengths_by_split = {"train": [], "validation": []}

for split_name in ("train", "validation"):
    path = SAMPLE_DIR / f"{split_name}.jsonl"
    if file_sha256(path) != saved_metadata["files"][path.name]["sha256"]:
        raise RuntimeError(f"Checksum sample không khớp: {path.name}")

    reader_stats = {}
    document_iterator = iter_documents(path, stats=reader_stats)
    sentinel = object()

    with path.open(encoding="utf-8") as stream:
        for raw_line, parsed_document in zip_longest(
            stream, document_iterator, fillvalue=sentinel
        ):
            if raw_line is sentinel or parsed_document is sentinel:
                raise RuntimeError("Reader STEP 5 bỏ qua record hoặc số dòng không khớp.")

            record = json.loads(raw_line)
            if not all(key in record for key in ("id", "text", "source", "content_hash")):
                raise ValueError("Sample thiếu trường bắt buộc.")
            if not isinstance(record["source"], str) or not record["source"].strip():
                raise ValueError("Sample có source không hợp lệ.")

            expected_hash = hashlib.sha256(
                canonical_text(record["text"]).encode("utf-8")
            ).hexdigest()
            if expected_hash != record["content_hash"] or expected_hash != parsed_document.content_hash:
                raise ValueError("Content hash không khớp.")
            if verification_split.assign(expected_hash) != split_name:
                raise ValueError("Document nằm sai split.")
            if record["split"] != split_name or record["partial_streaming"] is not True:
                raise ValueError("Metadata document không hợp lệ.")
            if record["origin_shard_url"] not in saved_metadata["selected_urls"]:
                raise ValueError("Document có shard provenance không thuộc selection.")
            if record["source_category"] != source_category(record["origin_shard_url"]):
                raise ValueError("Source category không khớp URL.")
            if expected_hash in hashes_by_split[split_name]:
                raise ValueError("Có duplicate content hash trong cùng split.")

            hashes_by_split[split_name].add(expected_hash)
            lengths_by_split[split_name].append(len(parsed_document.text))
            counts_by_category[record["source_category"]] += 1

    if reader_stats.get("invalid_documents", 0):
        raise ValueError(f"STEP 5 reader phát hiện document không hợp lệ: {split_name}")

overlap = hashes_by_split["train"] & hashes_by_split["validation"]
if overlap:
    raise ValueError(f"Train/validation overlap: {len(overlap)} hashes")

for split_name, lengths in lengths_by_split.items():
    if len(lengths) != saved_metadata["split_counts"][split_name]:
        raise ValueError("Document count khác metadata.")
    print(
        split_name,
        {
            "documents": len(lengths),
            "min_characters": min(lengths) if lengths else None,
            "mean_characters": round(sum(lengths) / len(lengths), 2) if lengths else None,
            "max_characters": max(lengths) if lengths else None,
        },
    )

SAMPLE_AUDIT = {
    "passed": True,
    "train_documents": len(hashes_by_split["train"]),
    "validation_documents": len(hashes_by_split["validation"]),
    "overlap": len(overlap),
    "documents_by_source_category": dict(counts_by_category),
}
print("\nSample audit:", SAMPLE_AUDIT)
print("Train-tokenizer input sau này:", SAMPLE_DIR / "train.jsonl")
print("Validation file phải giữ riêng:", SAMPLE_DIR / "validation.jsonl")
print("Số ký tự không phải số tokens. Chưa train tokenizer.")

train {'documents': 3963, 'min_characters': 1, 'mean_characters': 3617.2, 'max_characters': 721864}
validation {'documents': 37, 'min_characters': 35, 'mean_characters': 1751.59, 'max_characters': 6773}

Sample audit: {'passed': True, 'train_documents': 3963, 'validation_documents': 37, 'overlap': 0, 'documents_by_source_category': {'cc_en_head': 1000, 'stack': 1000, 'cc_en_middle': 1000, 'cc_en_tail': 1000}}
Train-tokenizer input sau này: /content/moe-erc/data/samples/dolma-partial-20261024T033301Z-520c5300/train.jsonl
Validation file phải giữ riêng: /content/moe-erc/data/samples/dolma-partial-20261024T033301Z-520c5300/validation.jsonl
Số ký tự không phải số tokens. Chưa train tokenizer.


In [ ]:
if not SAMPLE_AUDIT.get("passed"):
    raise RuntimeError("Sample audit chưa PASS.")

reports = saved_metadata["shard_reports"]
total_compressed = sum(r["compressed_bytes"] for r in reports)
total_charged = sum(r["charged_bytes"] for r in reports)
total_uncertain = sum(r["uncertain_read_bytes"] for r in reports)
total_decompressed = sum(r["decompressed_bytes"] for r in reports)

access_success = sum(
    any(event["status"] in (200, 206) for event in r["http_events"])
    for r in reports
)
valid_records = sum(r["valid_documents"] for r in reports)
invalid_records = sum(r["invalid_records"] for r in reports)

print("=== STEP 6.3.2 REPORT ===")
print("Dataset version             :", saved_metadata["version"])
print("Shards thử GET               :", sum(bool(r["http_events"] or r["errors"]) for r in reports))
print("Shards có HTTP 200/206       :", access_success)
print("Compressed payload nhận được:", f"{total_compressed:,} bytes ({total_compressed / MiB:.3f} MiB)")
print("Budget charged/reserved      :", f"{total_charged:,} bytes")
print("Read bytes chưa chắc do lỗi  :", f"{total_uncertain:,} bytes tối đa đã reserve")
print("Decompressed bytes trả về    :", f"{total_decompressed:,}")
print("JSON parse thành công        :", sum(r["json_parsed"] for r in reports))
print("Records đủ id/text/source    :", valid_records)
print("Records không hợp lệ         :", invalid_records)
print("Duplicate documents loại bỏ  :", saved_metadata["duplicates_removed"])
print("Train documents              :", SAMPLE_AUDIT["train_documents"])
print("Validation documents         :", SAMPLE_AUDIT["validation_documents"])
print("Content-hash overlap         :", SAMPLE_AUDIT["overlap"])
print("Documents/source category    :", SAMPLE_AUDIT["documents_by_source_category"])
print("Sample directory             :", SAMPLE_DIR)

for record in reports:
    statuses = [
        f"{event['phase']}:{event['status']}"
        for event in record["http_events"]
    ]
    print(f"\n[{record['source_category']}] {record['url']}")
    print("  HTTP events       :", statuses)
    print("  Stop reason       :", record["stop_reason"])
    print("  Activated limits  :", record["activated_limits"])
    print("  Complete gzip members:", record["gzip_members_completed"])
    print("  Unparsed decompressed bytes:", record["unparsed_decompressed_bytes"])
    print("  Missing fields    :", record["missing_fields"])
    print("  Invalid reasons   :", record["invalid_reasons"])
    print("  Errors            :", record["errors"])

assert total_charged <= 128 * MiB
assert all(r["charged_bytes"] <= 32 * MiB for r in reports)
assert all(r["decompressed_bytes"] <= 128 * MiB for r in reports)
assert all(r["valid_documents"] <= 1000 for r in reports)

print("\nLưu ý về kiểm chứng:")
print("- Bộ đếm payload bao gồm cả probe Cell 2 và streaming Cell 3; không tính HTTP/TLS framing.")
print("- Nếu lỗi đọc khiến số bytes nhận không xác định hoàn toàn, phần reserve không được tái sử dụng.")
print("- Dòng quá 4 MiB bị loại và có thống kê; sample có thể chịu thiên lệch prefix/nguồn/kích thước.")
print("- Các dòng cuối chưa hoàn chỉnh không được parse.")
print("- Không lưu gzip bị cắt thành shard hợp lệ.")
print("- Đọc được prefix hoặc một gzip member không xác nhận checksum/toàn vẹn toàn bộ shard.")
print("- Chưa xác định số tokens hoặc tính đại diện của sample.")

if SAMPLE_AUDIT["train_documents"] == 0:
    print("CHƯA có dữ liệu train hợp lệ cho bước tokenizer sau này.")
if SAMPLE_AUDIT["validation_documents"] == 0:
    print("CHƯA có validation hợp lệ; không lấy tài liệu train chuyển sang validation.")

print("\nDừng ở STEP 6.3.2: không train tokenizer, preprocess corpus hoặc huấn luyện model.")

=== STEP 6.3.2 REPORT ===
Dataset version             : v1_5-sample
Shards thử GET               : 4
Shards có HTTP 200/206       : 4
Compressed payload nhận được: 5,603,291 bytes (5.344 MiB)
Budget charged/reserved      : 5,603,291 bytes
Read bytes chưa chắc do lỗi  : 0 bytes tối đa đã reserve
Decompressed bytes trả về    : 19,843,329
JSON parse thành công        : 4314
Records đủ id/text/source    : 4000
Records không hợp lệ         : 314
Duplicate documents loại bỏ  : 0
Train documents              : 3963
Validation documents         : 37
Content-hash overlap         : 0
Documents/source category    : {'cc_en_head': 1000, 'stack': 1000, 'cc_en_middle': 1000, 'cc_en_tail': 1000}
Sample directory             : /content/moe-erc/data/samples/dolma-partial-20261024T033301Z-520c5300

[cc_en_head] https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0194.json.gz
  HTTP events       : ['probe:200', 'sample:206']
  Stop reason       : document_limit
  Activated limits  : ['doc

# STEP 6.3.3B — BACKUP DOLMA CORPUS LÊN GOOGLE DRIVE

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import sys

PROJECT_DIR = Path("/content/moe-erc").resolve()
SAMPLE_DIR = PROJECT_DIR / "data/samples/dolma-partial-20261023T164249Z-b43ef915"
OLD_SELECTION_FILE = (
    PROJECT_DIR / "data/selections/"
    "dolma-v1_5-sample-seed42-d37a10898c4709bf.json"
)

assert PROJECT_DIR.is_dir(), f"Không tìm thấy project: {PROJECT_DIR}"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from moe_erc.dolma_reader import (
    canonical_text, compressed_size, file_sha256,
    parse_manifest, read_manifest, select_shards, VERSION,
)
from moe_erc.tokenizer_utils import SplitConfig, SPLIT_ALGORITHM

metadata_path = SAMPLE_DIR / "metadata.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy metadata sample: {metadata_path}")

old_meta = json.loads(metadata_path.read_text(encoding="utf-8"))
old_selection = (
    json.loads(OLD_SELECTION_FILE.read_text(encoding="utf-8"))
    if OLD_SELECTION_FILE.is_file() else {}
)

reports = old_meta.get("shard_reports", [])
if isinstance(reports, dict):
    reports = list(reports.values())
if not isinstance(reports, list):
    raise ValueError("shard_reports có cấu trúc chưa được hỗ trợ.")

by_category = defaultdict(Counter)
reason_details = defaultdict(Counter)
field_details = defaultdict(Counter)

for report in reports:
    category = report.get("source_category", "UNKNOWN")
    for key in ("json_parsed", "valid_documents", "invalid_records"):
        by_category[category][key] += report.get(key, 0)
    reason_details[category].update(report.get("invalid_reasons", {}))
    field_details[category].update(report.get("missing_fields", {}))

print("Sample:", SAMPLE_DIR)
print("Các trường metadata:", sorted(old_meta))
for category in sorted(by_category):
    print(f"\nNguồn theo đường dẫn: {category}")
    print("  Số đếm:", dict(by_category[category]))
    print("  Lý do đã lưu:", dict(reason_details[category]))
    print("  Trường thiếu đã lưu:", dict(field_details[category]))

schema_rejected = sum(
    r.get("invalid_reasons", {}).get("missing_or_invalid_fields", 0)
    for r in reports
)
print("\nSchema rejects có nhãn rõ ràng:", schema_rejected)
print("Kết quả trước đó do bạn cung cấp: 314 schema rejects.")

print(
    "\nGiới hạn: metadata tổng hợp không chứa raw rejected records. "
    "Không thể khôi phục chi tiết từng trường thiếu/sai kiểu cho 314 records."
)
print(
    "missing_fields là histogram có thể chồng lặp, không phải các nhóm loại trừ nhau. "
    "Nhãn invalid_utf8_or_json gộp lỗi UTF-8 và JSON; không thể tách lại."
)
print(
    "Cell 5 sẽ giữ chính sách schema id/text/source và ghi riêng lỗi UTF-8, "
    "JSON, trường thiếu, sai kiểu và văn bản rỗng cho corpus mới."
)

for name in ("train.jsonl", "validation.jsonl"):
    path = SAMPLE_DIR / name
    entry = old_meta.get("files", {}).get(name, {})
    expected = entry.get("sha256") if isinstance(entry, dict) else None
    if path.is_file():
        actual = file_sha256(path)
        if expected and actual != expected:
            raise RuntimeError(f"Checksum sample không khớp: {name}")
        print(f"{name}: tồn tại; checksum {'PASS' if expected else 'chưa có để đối chiếu'}")
    else:
        print(f"{name}: KHÔNG TÌM THẤY")

FileNotFoundError: Không tìm thấy metadata sample: /content/moe-erc/data/samples/dolma-partial-20261023T164249Z-b43ef915/metadata.json

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

backup_root = Path(
    "/content/drive/MyDrive/MoE_ERC/data_snapshots"
)

snapshot = backup_root / (
    "dolma-20261023T173258Z-"
    "2412fe19bc1a0d22-98e15a0106d6"
)

print("\n=== GOOGLE DRIVE CHECK ===")
print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Backup root exists:", backup_root.is_dir())
print("Snapshot exists:", snapshot.is_dir())
print("COMPLETE.json exists:", (snapshot / "COMPLETE.json").is_file())
print("INCOMPLETE.json exists:", (snapshot / "INCOMPLETE.json").exists())

if snapshot.is_dir():
    print("\nSnapshot files:")
    for p in sorted(snapshot.iterdir()):
        print(" -", p.name)
else:
    print("\nKhông tìm thấy snapshot.")
    print("Hãy kiểm tra tài khoản Google Drive đang kết nối.")

Mounted at /content/drive

=== GOOGLE DRIVE CHECK ===
Drive mounted: True
Backup root exists: True
Snapshot exists: True
COMPLETE.json exists: True
INCOMPLETE.json exists: False

Snapshot files:
 - COMPLETE.json
 - backup_manifest.json
 - backup_manifest.sha256
 - data


In [ ]:
RUN_BACKUP = True  # Chủ động đổi thành True để cho phép Cell 4 chạy.

from datetime import datetime, timezone
import time

BACKUP_RESULT = {
    "status": "FAIL",
    "reason": "Chưa thực hiện backup.",
    "drive_snapshot": None,
    "local_snapshot": None,
    "verified_files": 0,
    "sqlite_integrity": "NOT RUN",
    "manifest_verification": "NOT RUN",
}
CORPUS_REL = CORPUS.relative_to(DATA_ROOT)

def write_json_new(path, value):
    encoded = (
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
    ).encode("utf-8")
    with Path(path).open("xb") as stream:
        stream.write(encoded)
        stream.flush()
        os.fsync(stream.fileno())

def copy_new(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Path(source).open("rb") as src, destination.open("xb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024)
        dst.flush()
        os.fsync(dst.fileno())

def validate_snapshot(root, expected_manifest_sha=None, require_complete=False):
    root = Path(root)
    manifest_path = root / "backup_manifest.json"
    manifest_sha = sha256_file(manifest_path)
    detached_sha = (root / "backup_manifest.sha256").read_text().strip()

    if manifest_sha != detached_sha:
        raise RuntimeError("Checksum của backup manifest không khớp.")
    if expected_manifest_sha and manifest_sha != expected_manifest_sha:
        raise RuntimeError("Manifest khác bản đã xác minh ở local.")

    backup = read_json(manifest_path)
    if backup["plan_sha256"] != PLAN_SHA or backup["selection_sha256"] != SELECTION_SHA:
        raise RuntimeError("Fingerprints trong backup manifest không khớp.")

    entries = backup["files"]
    names = [entry["path"] for entry in entries]
    if len(names) != len(set(names)):
        raise RuntimeError("Backup manifest có đường dẫn trùng.")

    for name in names:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "data":
            raise RuntimeError(f"Đường dẫn không an toàn: {name}")

    actual_names = set()
    for path in (root / "data").rglob("*"):
        if path.is_symlink():
            raise RuntimeError(f"Không chấp nhận symlink trong snapshot: {path}")
        if path.is_file():
            actual_names.add(path.relative_to(root).as_posix())
    if actual_names != set(names):
        raise RuntimeError("Danh sách file thực tế khác backup manifest.")

    for entry in entries:
        path = root / entry["path"]
        if path.stat().st_size != entry["bytes"]:
            raise RuntimeError(f"Size FAIL: {entry['path']}")
        if sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"SHA-256 FAIL: {entry['path']}")

        if path.suffix.lower() == ".json":
            read_json(path)
        elif path.suffix.lower() == ".jsonl":
            with path.open(encoding="utf-8") as stream:
                for number, line in enumerate(stream, 1):
                    item = json.loads(line)
                    if not isinstance(item, dict) or not isinstance(item.get("text"), str):
                        raise RuntimeError(f"JSONL không hợp lệ: {path.name}:{number}")
                    if not item["text"].strip():
                        raise RuntimeError(f"Văn bản rỗng: {path.name}:{number}")

    integrity = {}
    for entry in entries:
        if entry["method"] == "sqlite_backup_api":
            path = root / entry["path"]
            with closing(readonly_db(path, sealed=True)) as connection:
                result = connection.execute("PRAGMA integrity_check").fetchall()
            if result != [("ok",)]:
                raise RuntimeError(f"SQLite integrity FAIL: {entry['path']}: {result}")
            integrity[entry["path"]] = "ok"

    corpus = root / "data" / CORPUS_REL
    exported = corpus / EXPORT_REL
    saved_summary = read_json(exported / "summary.json")
    saved_plan = read_json(root / "data" / PLAN_PATH.relative_to(DATA_ROOT))
    saved_manifest = read_json(root / "data" / MANIFEST_PATH.relative_to(DATA_ROOT))

    assert fingerprint(saved_plan) == PLAN_SHA, "Plan checksum FAIL."
    assert fingerprint(saved_plan["selected_urls"]) == SELECTION_SHA
    assert saved_summary["plan_sha256"] == PLAN_SHA
    assert saved_summary["selection_sha256"] == SELECTION_SHA
    assert fingerprint(saved_summary["plan"]) == PLAN_SHA
    assert saved_manifest["manifest_sha256"] == saved_plan["manifest_sha256"]

    counts = {"train": 0, "validation": 0}
    train_bytes = 0
    seen = set()

    with closing(readonly_db(corpus / "corpus.sqlite", sealed=True)) as connection:
        row = connection.execute(
            "SELECT payload FROM checkpoint WHERE id=1"
        ).fetchone()
        if row is None:
            raise RuntimeError("SQLite thiếu checkpoint.")
        state = json.loads(row[0])
        if fingerprint(state) != saved_summary["state_sha256"]:
            raise RuntimeError("SQLite checkpoint và export summary không cùng trạng thái.")
        assert state["plan_sha256"] == PLAN_SHA

        seed = saved_plan["config"]["seed"]
        fraction = saved_plan["config"]["validation_fraction"]

        for split, name in (("train", "train.jsonl"), ("validation", "validation.jsonl")):
            path = exported / name
            assert sha256_file(path) == saved_summary["files"][name]["sha256"]

            with path.open(encoding="utf-8") as stream:
                for line in stream:
                    item = json.loads(line)
                    text = item["text"].replace("\r\n", "\n").replace("\r", "\n").strip()
                    encoded = text.encode("utf-8")
                    content_hash = hashlib.sha256(encoded).hexdigest()

                    if content_hash != item["content_hash"] or content_hash in seen:
                        raise RuntimeError("Content hash sai hoặc trùng giữa/trong các splits.")
                    seen.add(content_hash)

                    value = int.from_bytes(
                        hashlib.sha256(f"{seed}:{content_hash}".encode()).digest(), "big"
                    )
                    assigned = "validation" if value < int(fraction * (1 << 256)) else "train"
                    if assigned != split or item["split"] != split:
                        raise RuntimeError("Deterministic split không khớp.")

                    stored = connection.execute(
                        "SELECT split, payload FROM documents WHERE content_hash=?",
                        (content_hash,),
                    ).fetchone()
                    if stored is None or stored[0] != split or json.loads(stored[1]) != item:
                        raise RuntimeError("JSONL và SQLite documents không khớp.")

                    counts[split] += 1
                    if split == "train":
                        train_bytes += len(encoded)

        assert counts["train"] > 0 and counts["validation"] > 0
        assert counts["train"] == state["train_documents"]
        assert counts["validation"] == state["validation_documents"]
        assert train_bytes == state["train_text_bytes"]
        assert connection.execute("SELECT COUNT(*) FROM documents").fetchone()[0] == len(seen)

    if require_complete:
        if (root / "INCOMPLETE.json").exists():
            raise RuntimeError("Snapshot vẫn mang marker INCOMPLETE.")
        complete = read_json(root / "COMPLETE.json")
        if complete["status"] != "COMPLETE" or complete["manifest_sha256"] != manifest_sha:
            raise RuntimeError("COMPLETE marker không hợp lệ.")

    return {
        "manifest_sha256": manifest_sha,
        "payload_files": len(entries),
        "integrity": integrity,
        "counts": counts,
        "train_text_bytes": train_bytes,
    }

print("RUN_BACKUP =", RUN_BACKUP)
print("Các hàm đã sẵn sàng; chưa tạo local snapshot hoặc backup trên Drive.")

In [ ]:
import tempfile
import traceback

def execute_backup():
    result = {
        "status": "FAIL",
        "reason": "Backup chưa hoàn thành.",
        "drive_snapshot": None,
        "local_snapshot": None,
        "verified_files": 0,
        "sqlite_integrity": "NOT RUN",
        "manifest_verification": "NOT RUN",
    }
    monitors = {}
    pending_drive = None

    try:
        if not DRIVE_WRITE_OK:
            raise RuntimeError("Drive chưa vượt write/read probe.")

        current_files, current_dbs, current_sidecars = collect_sources()
        if set(current_files) != set(SOURCE_FILES) or set(current_dbs) != set(SQLITE_FILES):
            raise RuntimeError("Cây file nguồn đã đổi; chạy lại Cell 1 trước.")

        estimate = sum(path.stat().st_size for path in current_files)
        estimate += sum(path.stat().st_size for path in current_sidecars)
        if shutil.disk_usage("/content").free < 3 * estimate + 256 * 1024**2:
            raise RuntimeError("Không đủ local disk cho staging và khoảng trống dự phòng.")

        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        snapshot_name = f"dolma-{stamp}-{PLAN_SHA[:16]}-{uuid.uuid4().hex[:12]}"
        local = Path(tempfile.mkdtemp(prefix=snapshot_name + "-", dir="/content"))
        result["local_snapshot"] = str(local)

        # Theo dõi commits từ connection khác trong toàn thời gian tạo local snapshot.
        for path in current_dbs:
            connection = readonly_db(path)
            version = connection.execute("PRAGMA data_version").fetchone()[0]
            monitors[path] = (connection, version)

        entries = []
        for source in current_files:
            relative = Path("data") / source.relative_to(DATA_ROOT)
            destination = local / relative
            destination.parent.mkdir(parents=True, exist_ok=True)

            if source in current_dbs:
                source_connection = monitors[source][0]
                deadline = time.monotonic() + 180

                def progress(status, remaining, total):
                    if time.monotonic() > deadline:
                        raise TimeoutError("SQLite backup vượt timeout 180 giây.")

                source_connection.execute("BEGIN")
                try:
                    # Thiết lập read snapshot; không ghi SQL vào database nguồn.
                    source_connection.execute("SELECT COUNT(*) FROM sqlite_master").fetchone()
                    with closing(sqlite3.connect(destination)) as target:
                        source_connection.backup(
                            target, pages=256, progress=progress, sleep=0.05
                        )
                        # Chỉ thay đổi bản sao: tạo database standalone không cần WAL.
                        mode = target.execute("PRAGMA journal_mode=DELETE").fetchone()[0]
                        if mode.lower() != "delete":
                            raise RuntimeError("Không chuyển được SQLite snapshot sang DELETE.")
                        target.commit()
                finally:
                    source_connection.rollback()
                method = "sqlite_backup_api"
            else:
                before = sha256_file(source)
                copy_new(source, destination)
                if sha256_file(destination) != before or sha256_file(source) != before:
                    raise RuntimeError(f"File nguồn thay đổi trong khi copy: {source}")
                method = "byte_copy"

            entries.append({
                "path": relative.as_posix(),
                "source_path": str(source),
                "method": method,
                "bytes": destination.stat().st_size,
                "sha256": sha256_file(destination),
            })

        # Phát hiện metadata/checkpoint mới xuất hiện trong lúc tạo snapshot.
        final_files, final_dbs, _ = collect_sources()
        if set(final_files) != set(current_files) or set(final_dbs) != set(current_dbs):
            raise RuntimeError("Cây file nguồn thay đổi trong lúc tạo snapshot.")

        for path, (connection, original_version) in monitors.items():
            current_version = connection.execute("PRAGMA data_version").fetchone()[0]
            if current_version != original_version:
                raise RuntimeError(f"Database có commit mới trong lúc backup: {path}")

        # Kiểm tra lại tất cả file thường sau khi tạo xong local snapshot.
        for entry in entries:
            if entry["method"] == "byte_copy":
                if sha256_file(entry["source_path"]) != entry["sha256"]:
                    raise RuntimeError(f"Nguồn đã thay đổi: {entry['source_path']}")

        backup_manifest = {
            "format_version": 1,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "dataset": "allenai/dolma",
            "dataset_version": DATASET_VERSION,
            "dataset_seed": plan["config"]["seed"],
            "plan_sha256": PLAN_SHA,
            "selection_sha256": SELECTION_SHA,
            "git_commit": GIT_COMMIT,
            "project_source": str(PROJECT),
            "corpus_source": str(CORPUS),
            "active_export_relative_to_corpus": EXPORT_REL.as_posix(),
            "sqlite_policy": "read-only source; SQLite backup API; standalone destination",
            "sidecars_handled_by_sqlite_api": [
                {
                    "source_path": str(path),
                    "database": str(owner),
                    "bytes_observed": path.stat().st_size if path.exists() else None,
                }
                for path, owner in current_sidecars.items()
            ],
            "files": entries,
        }
        write_json_new(local / "backup_manifest.json", backup_manifest)
        manifest_sha = sha256_file(local / "backup_manifest.json")
        with (local / "backup_manifest.sha256").open("x", encoding="ascii") as stream:
            stream.write(manifest_sha + "\n")

        print("Xác minh local snapshot...")
        validate_snapshot(local, expected_manifest_sha=manifest_sha)

        # Chỉ bắt đầu tạo snapshot trên Drive sau khi local audit PASS.
        pending_drive = DRIVE_ROOT / (snapshot_name + ".INCOMPLETE")
        pending_drive.mkdir(exist_ok=False)
        result["drive_snapshot"] = str(pending_drive)
        write_json_new(pending_drive / "INCOMPLETE.json", {
            "status": "INCOMPLETE",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "local_snapshot": str(local),
        })

        copied = 0
        for source in sorted(local.rglob("*")):
            if not source.is_file():
                continue
            destination = pending_drive / source.relative_to(local)
            copy_new(source, destination)
            if sha256_file(destination) != sha256_file(source):
                raise RuntimeError(f"Drive copy SHA-256 FAIL: {destination}")
            copied += 1

        print("Xác minh snapshot đọc lại từ Drive...")
        checked = validate_snapshot(
            pending_drive, expected_manifest_sha=manifest_sha
        )

        final_drive = DRIVE_ROOT / snapshot_name
        if final_drive.exists():
            raise FileExistsError(f"Không ghi đè snapshot cũ: {final_drive}")
        pending_drive.rename(final_drive)
        result["drive_snapshot"] = str(final_drive)

        # Chưa có COMPLETE thì snapshot luôn được xem là chưa hoàn thành.
        # Chỉ xóa marker do chính lần chạy này tạo, không xóa dữ liệu nguồn.
        (final_drive / "INCOMPLETE.json").unlink()
        complete = {
            "status": "COMPLETE",
            "verified_at_utc": datetime.now(timezone.utc).isoformat(),
            "manifest_sha256": manifest_sha,
            "payload_files": checked["payload_files"],
            "copied_files_verified_sha256": copied,
            "sqlite_integrity": checked["integrity"],
        }
        write_json_new(final_drive / "COMPLETE.json", complete)
        if read_json(final_drive / "COMPLETE.json") != complete:
            raise RuntimeError("Không đọc lại được COMPLETE marker chính xác.")

        # Kiểm tra hash của marker vừa tạo qua nội dung canonical đã ghi.
        marker_bytes = (
            json.dumps(complete, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
        ).encode("utf-8")
        if sha256_file(final_drive / "COMPLETE.json") != hashlib.sha256(marker_bytes).hexdigest():
            raise RuntimeError("COMPLETE marker SHA-256 FAIL.")

        result.update({
            "status": "PASS",
            "reason": "Local snapshot và bản đọc lại từ Drive vượt toàn bộ kiểm tra.",
            "manifest_sha256": manifest_sha,
            "verified_files": copied + 1,  # Gồm manifest, detached SHA và COMPLETE.
            "sqlite_integrity": checked["integrity"],
            "manifest_verification": "PASS",
            "total_files": sum(path.is_file() for path in final_drive.rglob("*")),
            "total_bytes": sum(
                path.stat().st_size for path in final_drive.rglob("*") if path.is_file()
            ),
        })
    except BaseException as exc:
        result["reason"] = f"{type(exc).__name__}: {exc}"
        print("BACKUP FAIL — giữ nguyên staging/bản chưa hoàn chỉnh để kiểm tra.")
        traceback.print_exc()
    finally:
        for connection, _ in monitors.values():
            connection.close()

    return result

if not RUN_BACKUP:
    print("RUN_BACKUP=False: không tạo snapshot, không copy corpus lên Drive.")
else:
    BACKUP_RESULT = execute_backup()
    print("Backup status:", BACKUP_RESULT["status"])
    print("Chi tiết:", BACKUP_RESULT["reason"])

In [ ]:
print("=== BACKUP REPORT ===")
print("Snapshot trên Drive:", BACKUP_RESULT.get("drive_snapshot"))
print("Local staging:", BACKUP_RESULT.get("local_snapshot"))
print("Tổng số file:", BACKUP_RESULT.get("total_files", "Chưa xác định"))
size = BACKUP_RESULT.get("total_bytes")
print("Tổng dung lượng:",
      f"{size / 1024**2:.2f} MiB" if size is not None else "Chưa xác định")
print("File đã xác minh SHA-256:", BACKUP_RESULT.get("verified_files", 0))
print("SQLite integrity:", BACKUP_RESULT.get("sqlite_integrity"))
print("Manifest verification:", BACKUP_RESULT.get("manifest_verification"))
print("Backup status:", BACKUP_RESULT["status"])
print("Chi tiết:", BACKUP_RESULT["reason"])

if BACKUP_RESULT["status"] == "PASS":
    print("Backup manifest SHA-256:", BACKUP_RESULT["manifest_sha256"])
else:
    print("Không coi bản backup này là đã hoàn thành.")

print("""
HƯỚNG DẪN RESTORE SAU KHI RESET RUNTIME

1. Mount lại Google Drive tại /content/drive.
   Chọn đúng snapshot có COMPLETE.json hợp lệ, không có INCOMPLETE.json.
   Không dùng bản .INCOMPLETE làm backup đã hoàn thành.

2. Kiểm tra SHA-256 của backup_manifest.json:
   - Phải khớp backup_manifest.sha256.
   - Phải khớp manifest_sha256 trong COMPLETE.json.
   - Đối chiếu thêm checksum đã lưu trong báo cáo/notebook này.

3. Đọc backup_manifest.json. Với MỌI entry:
   - Chỉ chấp nhận đường dẫn tương đối bắt đầu bằng data/.
   - Từ chối đường dẫn tuyệt đối, '..', symlink hoặc entry trùng.
   - Kiểm tra file tồn tại, đúng dung lượng và SHA-256.
   - Kiểm tra danh sách file trong snapshot/data/ khớp manifest.

4. Copy snapshot/data/ vào một thư mục staging local MỚI.
   Xác minh lại toàn bộ SHA-256 sau khi copy.
   Mở các database snapshot ở chế độ read-only và chạy:
       PRAGMA integrity_check;
   Kết quả phải chỉ gồm 'ok'.
   Đọc kiểm tra JSON/JSONL; đối chiếu fingerprints và checksum trong summary.

5. Phục hồi vào /content/moe-erc/data/ bằng cách giữ nguyên đường dẫn
   bên dưới snapshot/data/. Ví dụ:
       snapshot/data/tokenizer_corpora/... -> /content/moe-erc/data/tokenizer_corpora/...
       snapshot/data/selections/...       -> /content/moe-erc/data/selections/...
       snapshot/data/manifests/...        -> /content/moe-erc/data/manifests/...

   Trước khi ghi, kiểm tra TOÀN BỘ đường dẫn đích:
   nếu bất kỳ file nào đã tồn tại, DỪNG và yêu cầu xác nhận.
   Không dùng copy có chế độ tự overwrite. Khi thực hiện restore,
   mở file đích bằng chế độ 'xb' để từ chối ghi đè.
   Nếu restore bị ngắt, giữ phần đã copy và kiểm tra lại; không tự ghi đè.

6. Đối chiếu SHA-256 tại các đường dẫn đã phục hồi trước khi sử dụng corpus.
   Khôi phục/checkout source project theo git_commit trong backup manifest;
   backup corpus này không phải bản sao đầy đủ của source repository.

LƯU Ý:
- Checksum trên Drive mount xác nhận dữ liệu đọc lại đúng ở thời điểm kiểm tra;
  không bảo đảm tuyệt đối Google Drive đã hoàn tất đồng bộ phía dịch vụ.
- Hãy lưu notebook Colab vào Google Drive.
  Backup corpus không đồng nghĩa notebook cũng đã được lưu.
- Không cần tải lại Dolma, preprocess lại hoặc train tokenizer để restore.
""")

# STEP 6.3.3C — RESTORE CORPUS.

In [ ]:
from pathlib import Path, PurePosixPath
from contextlib import closing
import hashlib
import json
import os
import re
import shutil
import sqlite3
import subprocess
import sys

PROJECT = Path("/content/moe-erc")
SNAPSHOT = Path(
    "/content/drive/MyDrive/MoE_ERC/data_snapshots/"
    "dolma-20261023T173258Z-2412fe19bc1a0d22-98e15a0106d6"
)
EXPECTED_MANIFEST_SHA = "45fc72f9935eba13afa5693036c97bdca72cee6d8d46cc7e891c20ebd18bc148"
EXPECTED_PLAN_SHA = "2412fe19bc1a0d22ccfcab4cf4631478e08f471954a18a0bb06e1c840e0d4569"
EXPECTED_SELECTION_SHA = "c777494f3fd7c0acfd7d5c39097fe762410c10f894c0b77dc7a88b469f52d4aa"
CONTROL_FILES = {"backup_manifest.json", "backup_manifest.sha256", "COMPLETE.json"}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

def validate_relative_name(name):
    require(isinstance(name, str) and bool(name), "Tên file không hợp lệ.")
    require("\\" not in name and ":" not in name, f"Đường dẫn không an toàn: {name!r}")
    require(not any(ord(c) < 32 for c in name), "Đường dẫn chứa ký tự điều khiển.")
    relative = PurePosixPath(name)
    require(
        not relative.is_absolute()
        and ".." not in relative.parts
        and len(relative.parts) >= 2
        and relative.parts[0] == "data"
        and relative.as_posix() == name,
        f"Đường dẫn không an toàn: {name!r}",
    )
    return relative

def checked_file(root, name):
    """Từ chối symlink ở cả file và các thư mục cha bên trong root."""
    root = Path(root)
    require(not root.is_symlink(), f"Root là symlink: {root}")
    relative = PurePosixPath(name)
    require(not relative.is_absolute() and ".." not in relative.parts, "Path escape.")
    current = root
    for part in relative.parts:
        current = current / part
        require(not current.is_symlink(), f"Không chấp nhận symlink: {current}")
    require(current.resolve().is_relative_to(root.resolve()), "Path escape.")
    require(current.is_file(), f"Thiếu file: {current}")
    return current

def verify_backup(root):
    root = Path(root)
    require(root.is_dir() and not root.is_symlink(), f"Snapshot không hợp lệ: {root}")
    require(
        not os.path.lexists(root / "INCOMPLETE.json"),
        "Từ chối snapshot có INCOMPLETE.json.",
    )

    for name in CONTROL_FILES:
        checked_file(root, name)

    manifest_sha = sha256_file(root / "backup_manifest.json")
    detached_sha = (root / "backup_manifest.sha256").read_text().strip()
    complete = read_json(root / "COMPLETE.json")
    require(
        manifest_sha == detached_sha == EXPECTED_MANIFEST_SHA,
        "SHA-256 backup manifest không khớp checksum đã cung cấp.",
    )
    require(
        complete.get("status") == "COMPLETE"
        and complete.get("manifest_sha256") == manifest_sha,
        "COMPLETE marker không hợp lệ.",
    )

    manifest = read_json(root / "backup_manifest.json")
    require(manifest.get("format_version") == 1, "Backup format chưa được hỗ trợ.")
    require(manifest.get("plan_sha256") == EXPECTED_PLAN_SHA, "Plan fingerprint sai.")
    require(
        manifest.get("selection_sha256") == EXPECTED_SELECTION_SHA,
        "Selection fingerprint sai.",
    )

    entries = manifest.get("files")
    require(isinstance(entries, list) and bool(entries), "Manifest thiếu files.")
    names = set()
    for entry in entries:
        require(isinstance(entry, dict), "Manifest entry không phải object.")
        name = entry["path"]
        validate_relative_name(name)
        require(name not in names, f"Entry trùng: {name}")
        names.add(name)
        require(
            type(entry["bytes"]) is int and entry["bytes"] >= 0,
            f"Size không hợp lệ: {name}",
        )
        require(
            isinstance(entry["sha256"], str)
            and re.fullmatch(r"[0-9a-f]{64}", entry["sha256"]),
            f"SHA-256 không hợp lệ: {name}",
        )
        require(
            entry["method"] in {"byte_copy", "sqlite_backup_api"},
            f"Phương thức backup chưa hỗ trợ: {name}",
        )

    for name in names:
        for parent in PurePosixPath(name).parents:
            require(parent.as_posix() not in names, f"Xung đột file/thư mục: {name}")

    actual = set()
    for path in root.rglob("*"):
        require(not path.is_symlink(), f"Symlink trong snapshot: {path}")
        if path.is_file():
            actual.add(path.relative_to(root).as_posix())
        else:
            require(path.is_dir(), f"Loại file không hỗ trợ: {path}")

    require(actual == names | CONTROL_FILES, "Danh sách file thực tế khác manifest.")

    for entry in entries:
        path = checked_file(root, entry["path"])
        require(path.stat().st_size == entry["bytes"], f"Size FAIL: {entry['path']}")
        require(
            sha256_file(path) == entry["sha256"],
            f"SHA-256 FAIL: {entry['path']}",
        )
    return manifest, entries, actual

require(
    Path("/content/drive/MyDrive").is_dir(),
    "Drive chưa mount. Chạy google.colab.drive.mount('/content/drive') rồi chạy lại cell.",
)
BACKUP_MANIFEST, ENTRIES, BACKUP_FILES = verify_backup(SNAPSHOT)
require(len(BACKUP_FILES) == 9, "Số file snapshot khác thông tin backup đã cung cấp.")

PAYLOAD_BYTES = sum(entry["bytes"] for entry in ENTRIES)
SNAPSHOT_BYTES = sum((SNAPSHOT / name).stat().st_size for name in BACKUP_FILES)

print("Backup verification: PASS")
print("Snapshot:", SNAPSHOT)
print("Manifest SHA-256:", EXPECTED_MANIFEST_SHA)
print("Tổng file backup:", len(BACKUP_FILES))
print("File dữ liệu cần restore:", len(ENTRIES))
print(f"Tổng snapshot: {SNAPSHOT_BYTES / 1024**2:.2f} MiB")
print("Mọi payload file đã vượt size và SHA-256 verification.")
print("Không sửa bất kỳ file nào trên Drive.")

Backup verification: PASS
Snapshot: /content/drive/MyDrive/MoE_ERC/data_snapshots/dolma-20261023T173258Z-2412fe19bc1a0d22-98e15a0106d6
Manifest SHA-256: 45fc72f9935eba13afa5693036c97bdca72cee6d8d46cc7e891c20ebd18bc148
Tổng file backup: 9
File dữ liệu cần restore: 6
Tổng snapshot: 304.51 MiB
Mọi payload file đã vượt size và SHA-256 verification.
Không sửa bất kỳ file nào trên Drive.


In [ ]:
require(PROJECT.is_dir(), f"Project chưa tồn tại: {PROJECT}")
require(PROJECT.resolve() == PROJECT, "Project root đi qua symlink.")

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
print("Git commit hiện tại:", GIT_COMMIT)
print("Git commit trong backup:", BACKUP_MANIFEST["git_commit"])
if GIT_COMMIT != BACKUP_MANIFEST["git_commit"]:
    print("LƯU Ý: commit khác backup; không tự checkout hoặc thay đổi source.")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    from moe_erc.dolma_reader import canonical_text, iter_documents, parse_manifest
    from moe_erc.tokenizer_utils import SplitConfig, SPLIT_ALGORITHM, chain_hash
except ImportError as exc:
    raise RuntimeError(
        "Chưa import được pipeline STEP 5. Kiểm tra dependencies của project; "
        "cell này không tự cài đặt."
    ) from exc

def sealed_db(path):
    # Bản backup standalone đã đóng; immutable tránh tạo WAL/SHM trên Drive.
    return sqlite3.connect(
        Path(path).resolve().as_uri() + "?mode=ro&immutable=1", uri=True
    )

source_project = PurePosixPath(BACKUP_MANIFEST["project_source"])
source_corpus = PurePosixPath(BACKUP_MANIFEST["corpus_source"])
corpus_under_data = source_corpus.relative_to(source_project / "data")
active_export = PurePosixPath(BACKUP_MANIFEST["active_export_relative_to_corpus"])
require(not active_export.is_absolute() and ".." not in active_export.parts, "Export path sai.")

CORPUS_REL = PurePosixPath("data") / corpus_under_data
EXPORT_REL = CORPUS_REL / active_export
CORPUS_LOCAL = PROJECT / CORPUS_REL.as_posix()
TRAIN_LOCAL = PROJECT / (EXPORT_REL / "train.jsonl").as_posix()
VALIDATION_LOCAL = PROJECT / (EXPORT_REL / "validation.jsonl").as_posix()

for relative in (
    CORPUS_REL / "corpus.sqlite",
    EXPORT_REL / "train.jsonl",
    EXPORT_REL / "validation.jsonl",
    EXPORT_REL / "summary.json",
):
    validate_relative_name(relative.as_posix())
    require(
        relative.as_posix() in {entry["path"] for entry in ENTRIES},
        f"Manifest thiếu file bắt buộc: {relative}",
    )

def verify_metadata(root):
    root = Path(root)
    json_objects = {}
    integrity = {}

    for entry in ENTRIES:
        path = checked_file(root, entry["path"])
        if path.suffix.lower() == ".json":
            json_objects[entry["path"]] = read_json(path)
        if entry["method"] == "sqlite_backup_api":
            # Không chấp nhận SQLite sidecars cho snapshot standalone.
            for suffix in ("-wal", "-shm", "-journal"):
                require(
                    not os.path.lexists(Path(str(path) + suffix)),
                    f"SQLite snapshot có sidecar bất ngờ: {path}{suffix}",
                )
            with closing(sealed_db(path)) as connection:
                result = connection.execute("PRAGMA integrity_check").fetchall()
            require(result == [("ok",)], f"SQLite integrity FAIL: {entry['path']}: {result}")
            integrity[entry["path"]] = "ok"

    summary = json_objects[(EXPORT_REL / "summary.json").as_posix()]
    plan = summary["plan"]
    require(fingerprint(plan) == EXPECTED_PLAN_SHA, "Plan nội dung không khớp.")
    require(summary["plan_sha256"] == EXPECTED_PLAN_SHA, "Summary plan sai.")
    require(summary["selection_sha256"] == EXPECTED_SELECTION_SHA, "Summary selection sai.")
    require(
        fingerprint(plan["selected_urls"]) == EXPECTED_SELECTION_SHA,
        "Selected URLs fingerprint sai.",
    )
    require(plan["version"] == BACKUP_MANIFEST["dataset_version"] == "v1_5-sample", "Version sai.")
    require(plan["split_algorithm"] == SPLIT_ALGORITHM, "Split algorithm khác STEP 5.")
    require(plan["config"]["seed"] == 42, "Seed khác 42.")

    # Tìm plan/manifest theo nội dung đã xác minh, không đoán thêm file ngoài backup.
    saved_plans = [
        value for value in json_objects.values()
        if isinstance(value, dict) and fingerprint(value) == EXPECTED_PLAN_SHA
    ]
    require(bool(saved_plans), "Thiếu sampling plan riêng trong backup.")
    full_manifests = [
        value for value in json_objects.values()
        if isinstance(value, dict)
        and value.get("manifest_sha256") == plan["manifest_sha256"]
        and isinstance(value.get("urls"), list)
    ]
    require(len(full_manifests) == 1, "Không xác định duy nhất full Dolma manifest.")
    full_urls = full_manifests[0]["urls"]
    require(parse_manifest("\n".join(full_urls)) == full_urls, "URLs manifest không hợp lệ.")
    require(set(plan["selected_urls"]).issubset(full_urls), "Selected URL ngoài manifest.")

    with closing(sealed_db(root / (CORPUS_REL / "corpus.sqlite").as_posix())) as connection:
        row = connection.execute("SELECT payload FROM checkpoint WHERE id=1").fetchone()
        require(row is not None, "SQLite thiếu checkpoint.")
        state = json.loads(row[0])
        require(fingerprint(state) == summary["state_sha256"], "Checkpoint/summary không khớp.")
        require(state == summary["state"], "State trong summary khác SQLite.")
        require(state["plan_sha256"] == EXPECTED_PLAN_SHA, "Checkpoint plan sai.")
        db_counts = dict(connection.execute(
            "SELECT split, COUNT(*) FROM documents GROUP BY split"
        ))

    require(state["train_documents"] == db_counts.get("train") == 28_051, "Train count sai.")
    require(
        state["validation_documents"] == db_counts.get("validation") == 443,
        "Validation count sai.",
    )
    for name in ("train.jsonl", "validation.jsonl"):
        path = root / (EXPORT_REL / name).as_posix()
        require(path.stat().st_size > 0, f"File rỗng: {name}")
        require(
            sha256_file(path) == summary["files"][name]["sha256"],
            f"Summary checksum FAIL: {name}",
        )
    return summary, plan, integrity

SUMMARY, PLAN, SQLITE_INTEGRITY = verify_metadata(SNAPSHOT)

def destination_conflicts():
    conflicts = []
    for entry in ENTRIES:
        target = PROJECT / entry["path"]
        current = PROJECT
        parts = PurePosixPath(entry["path"]).parts
        for index, part in enumerate(parts):
            current = current / part
            require(not current.is_symlink(), f"Đích đi qua symlink: {current}")
            if index < len(parts) - 1:
                require(
                    not current.exists() or current.is_dir(),
                    f"Thư mục cha bị chặn bởi file: {current}",
                )
        require(target.resolve().is_relative_to(PROJECT / "data"), "Destination path escape.")
        if os.path.lexists(target):
            conflicts.append(str(target))
    return conflicts

conflicts = destination_conflicts()
free_bytes = shutil.disk_usage("/content").free
REQUIRED_FREE = SNAPSHOT_BYTES + PAYLOAD_BYTES + 512 * 1024**2

print("\nRESTORE PLAN")
for entry in ENTRIES:
    print(f"{entry['path']} -> {PROJECT / entry['path']} | {entry['bytes']:,} bytes")
print("SQLite integrity:", SQLITE_INTEGRITY)
print(f"Local free: {free_bytes / 1024**3:.2f} GiB")
print(f"Yêu cầu, gồm dự phòng: {REQUIRED_FREE / 1024**3:.2f} GiB")
print("Control files được giữ trong staging; chỉ payload data/ được đưa vào project.")

if conflicts:
    print("Các đích đã tồn tại:\n" + "\n".join(conflicts))
    raise FileExistsError("Dừng: không overwrite dữ liệu local.")
require(free_bytes >= REQUIRED_FREE, "Không đủ dung lượng local.")
print("Restore preflight: PASS")

Git commit hiện tại: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
Git commit trong backup: 2fd3478e478d3a20381e95b06890f8dcdf2ae013

RESTORE PLAN
data/manifests/dolma-v1_5-sample-d8b00c66388a160637a71f76672a1648a5b50ff3670f561065b946de3da72d6f.json -> /content/moe-erc/data/manifests/dolma-v1_5-sample-d8b00c66388a160637a71f76672a1648a5b50ff3670f561065b946de3da72d6f.json | 211,752 bytes
data/selections/dolma-tokenizer-plan-2412fe19bc1a0d22.json -> /content/moe-erc/data/selections/dolma-tokenizer-plan-2412fe19bc1a0d22.json | 4,084 bytes
data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/corpus.sqlite -> /content/moe-erc/data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/corpus.sqlite | 170,274,816 bytes
data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/summary.json -> /content/moe-erc/data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/summary.json | 33,889 bytes
data/tokenizer_corpora/dolma-bpe16k-2412f

In [ ]:
RUN_RESTORE = True

import tempfile
import traceback

RESTORE_STATE = {
    "status": "NOT RUN",
    "staging": None,
    "copied": [],
    "reason": "RUN_RESTORE=False",
}

def copy_exclusive(source, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    with Path(source).open("rb") as src, target.open("xb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024)
        dst.flush()
        os.fsync(dst.fileno())

def perform_restore():
    state = {"status": "FAIL", "staging": None, "copied": [], "reason": ""}
    progress_path = None

    def save_progress():
        if progress_path is not None:
            temporary = progress_path.with_suffix(".tmp")
            with temporary.open("w", encoding="utf-8") as stream:
                json.dump(state, stream, ensure_ascii=False, indent=2)
                stream.flush()
                os.fsync(stream.fileno())
            temporary.replace(progress_path)

    try:
        require(not destination_conflicts(), "Có file đích tồn tại; không restore.")
        require(
            shutil.disk_usage("/content").free >= REQUIRED_FREE,
            "Không còn đủ dung lượng local.",
        )

        # Xác minh lại Drive trước khi copy, không dựa duy nhất vào biến Cell 1.
        _, current_entries, current_files = verify_backup(SNAPSHOT)
        require(current_entries == ENTRIES, "Backup entries đã thay đổi.")

        staging_base = Path(tempfile.mkdtemp(prefix="moe-corpus-restore-", dir="/content"))
        staging = staging_base / "snapshot"
        staging.mkdir()
        progress_path = staging_base / "restore_progress.json"
        state.update(status="STAGING", staging=str(staging))
        save_progress()

        for name in sorted(current_files):
            copy_exclusive(checked_file(SNAPSHOT, name), staging / name)

        verify_backup(staging)
        verify_metadata(staging)
        print("Staging SHA-256, metadata và SQLite integrity: PASS")

        # Kiểm tra lại tất cả đích sau khi hoàn tất staging.
        require(not destination_conflicts(), "Đích xuất hiện sau preflight; dừng restore.")
        state["status"] = "RESTORING"
        save_progress()

        for entry in ENTRIES:
            # Kiểm tra lại symlink/thư mục cha trước từng lần ghi.
            existing = destination_conflicts()
            unexpected = set(existing) - set(state["copied"])
            require(not unexpected, f"Đích mới xuất hiện: {sorted(unexpected)}")

            source = checked_file(staging, entry["path"])
            target = PROJECT / entry["path"]
            copy_exclusive(source, target)
            require(target.stat().st_size == entry["bytes"], f"Size FAIL: {target}")
            require(sha256_file(target) == entry["sha256"], f"SHA-256 FAIL: {target}")
            state["copied"].append(str(target))
            save_progress()
            print("Restored:", entry["path"])

        state["status"] = "RESTORED"
        state["reason"] = "Copy hoàn tất; cần chạy Cell 4 để xác minh cuối."
        save_progress()
    except BaseException as exc:
        state["status"] = "FAIL"
        state["reason"] = f"{type(exc).__name__}: {exc}"
        try:
            save_progress()
        except Exception as log_error:
            print("Không lưu được progress log:", log_error)
        print("RESTORE FAIL/INTERRUPTED:", state["reason"])
        print(
            "Giữ staging và các file đã tạo để kiểm tra. "
            "Không tự rollback, xóa hoặc overwrite khi chạy lại."
        )
        traceback.print_exc()
    return state

if not RUN_RESTORE:
    print("RUN_RESTORE=False: không tạo staging và không phục hồi dữ liệu.")
else:
    RESTORE_STATE = perform_restore()
    print("Copy status:", RESTORE_STATE["status"])
    print("Staging:", RESTORE_STATE["staging"])
    print("Files đã copy:", len(RESTORE_STATE["copied"]))

Staging SHA-256, metadata và SQLite integrity: PASS
Restored: data/manifests/dolma-v1_5-sample-d8b00c66388a160637a71f76672a1648a5b50ff3670f561065b946de3da72d6f.json
Restored: data/selections/dolma-tokenizer-plan-2412fe19bc1a0d22.json
Restored: data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/corpus.sqlite
Restored: data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/summary.json
Restored: data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/train.jsonl
Restored: data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/validation.jsonl
Copy status: RESTORED
Staging: /content/moe-corpus-restore-uwf7jlm7/snapshot
Files đã copy: 6


In [ ]:
from collections import Counter

def audit_restored_corpus():
    result = {
        "status": "FAIL",
        "files_verified": 0,
        "bytes_verified": 0,
        "train_documents": 0,
        "validation_documents": 0,
        "reason": "",
    }
    try:
        require(
            RESTORE_STATE["status"] == "RESTORED",
            f"Restore chưa hoàn tất: {RESTORE_STATE['status']}",
        )

        for entry in ENTRIES:
            path = checked_file(PROJECT, entry["path"])
            require(path.stat().st_size == entry["bytes"], f"Size FAIL: {path}")
            require(sha256_file(path) == entry["sha256"], f"SHA-256 FAIL: {path}")
            result["files_verified"] += 1
            result["bytes_verified"] += entry["bytes"]

        summary, plan, integrity = verify_metadata(PROJECT)
        split_config = SplitConfig(
            seed=plan["config"]["seed"],
            validation_fraction=plan["config"]["validation_fraction"],
        )
        counts = Counter()
        text_bytes = Counter()
        categories = {"train": Counter(), "validation": Counter()}
        seen = set()
        ordered_fingerprints = {}

        with closing(sealed_db(CORPUS_LOCAL / "corpus.sqlite")) as database:
            for split, path in (
                ("train", TRAIN_LOCAL),
                ("validation", VALIDATION_LOCAL),
            ):
                ordered = "00" * 32
                with path.open(encoding="utf-8") as stream:
                    for line_number, line in enumerate(stream, 1):
                        item = json.loads(line)
                        require(isinstance(item, dict), f"Record sai: {path}:{line_number}")
                        require(
                            isinstance(item.get("text"), str),
                            f"Text không hợp lệ: {path}:{line_number}",
                        )
                        text = canonical_text(item["text"])
                        require(text and text == item["text"], "Text rỗng/chưa canonical.")
                        encoded = text.encode("utf-8")
                        digest = hashlib.sha256(encoded).hexdigest()

                        require(item.get("content_hash") == digest, "Content hash sai.")
                        require(digest not in seen, "Duplicate hoặc train/validation overlap.")
                        require(
                            item.get("split") == split_config.assign(digest) == split,
                            "Deterministic split sai.",
                        )
                        seen.add(digest)

                        stored = database.execute(
                            "SELECT split, payload FROM documents WHERE content_hash=?",
                            (digest,),
                        ).fetchone()
                        require(
                            stored is not None
                            and stored[0] == split
                            and json.loads(stored[1]) == item,
                            "JSONL document không khớp SQLite.",
                        )
                        category = item.get("source_category")
                        require(
                            isinstance(category, str) and bool(category),
                            "Thiếu source_category.",
                        )
                        counts[split] += 1
                        text_bytes[split] += len(encoded)
                        categories[split][category] += 1
                        ordered = chain_hash(ordered, digest)
                ordered_fingerprints[split] = ordered

            database_count = database.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
            require(database_count == len(seen), "SQLite có documents không khớp exports.")

        require(counts["train"] == summary["state"]["train_documents"] == 28_051, "Train count sai.")
        require(
            counts["validation"] == summary["state"]["validation_documents"] == 443,
            "Validation count sai.",
        )
        require(
            text_bytes["train"] == summary["state"]["train_text_bytes"],
            "Clean training text bytes không khớp.",
        )
        require(
            len(set(categories["train"]) | set(categories["validation"])) == 9,
            "Số source categories khác 9.",
        )

        # Đối chiếu độc lập bằng chính reader được trainer STEP 5 sử dụng.
        for split, path in (
            ("train", TRAIN_LOCAL),
            ("validation", VALIDATION_LOCAL),
        ):
            stats = {}
            reader_count = reader_bytes = 0
            ordered = "00" * 32
            for document in iter_documents(path, stats=stats):
                require(split_config.assign(document.content_hash) == split, "Reader split sai.")
                reader_count += 1
                reader_bytes += len(document.text.encode("utf-8"))
                ordered = chain_hash(ordered, document.content_hash)

            require(stats.get("invalid_documents", 0) == 0, f"STEP 5 reader skips: {stats}")
            require(reader_count == counts[split], "STEP 5 reader count sai.")
            require(reader_bytes == text_bytes[split], "STEP 5 reader text bytes sai.")
            require(ordered == ordered_fingerprints[split], "STEP 5 reader fingerprint sai.")

        result.update({
            "status": "PASS",
            "train_documents": counts["train"],
            "validation_documents": counts["validation"],
            "clean_train_bytes": text_bytes["train"],
            "source_categories": {
                split: dict(value) for split, value in categories.items()
            },
            "sqlite_integrity": integrity,
            "reason": "Checksum, metadata, SQLite, document audit và STEP 5 reader đều PASS.",
        })
    except BaseException as exc:
        result["reason"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    return result

RESTORE_AUDIT = audit_restored_corpus()

print("\n=== RESTORE REPORT ===")
print("Restore status:", RESTORE_AUDIT["status"])
print("Files dữ liệu đã xác minh:",
      f"{RESTORE_AUDIT['files_verified']}/{len(ENTRIES)}")
print("Snapshot gốc:", len(BACKUP_FILES), "files, gồm 3 control files giữ trong staging")
print(f"Dung lượng payload đã xác minh: {RESTORE_AUDIT['bytes_verified'] / 1024**2:.2f} MiB")
print("Train documents:", RESTORE_AUDIT["train_documents"])
print("Validation documents:", RESTORE_AUDIT["validation_documents"])
print("Corpus directory:", CORPUS_LOCAL)
print("Train JSONL:", TRAIN_LOCAL)
print("Validation JSONL:", VALIDATION_LOCAL)
print("Chi tiết:", RESTORE_AUDIT["reason"])

if RESTORE_AUDIT["status"] == "PASS":
    print(f"Clean train text: {RESTORE_AUDIT['clean_train_bytes'] / 1024**2:.6f} MiB")
    print("SQLite integrity:", RESTORE_AUDIT["sqlite_integrity"])
    print("Sources:", RESTORE_AUDIT["source_categories"])
    print("Content-hash overlap: 0")
    print("Tokenizer Pipeline STEP 5 compatibility: PASS")
    print("Không tải Dolma, không train tokenizer và không chạy model training.")


=== RESTORE REPORT ===
Restore status: PASS
Files dữ liệu đã xác minh: 6/6
Snapshot gốc: 9 files, gồm 3 control files giữ trong staging
Dung lượng payload đã xác minh: 304.51 MiB
Train documents: 28051
Validation documents: 443
Corpus directory: /content/moe-erc/data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22
Train JSONL: /content/moe-erc/data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/train.jsonl
Validation JSONL: /content/moe-erc/data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/exports/f46e687c4faf4aad97007a76dc8431dd/validation.jsonl
Chi tiết: Checksum, metadata, SQLite, document audit và STEP 5 reader đều PASS.
Clean train text: 127.999989 MiB
SQLite integrity: {'data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22/corpus.sqlite': 'ok'}
Sources: {'train': {'cc_en_middle': 6694, 'reddit': 1487, 'stack': 1490, 'cc_en_tail': 5205, 'wiki': 1492, 'cc_en_head': 7689, 'pes2o': 977, 'books': 58, 'c4': 2959}, 'validation': {'cc_en_midd

# Step 6.3.4: HUẤN LUYỆN BYTE-LEVEL BPE TOKENIZER 16K TRÊN GOOGLE COLAB

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import shutil
import sys

PROJECT = Path("/content/moe-erc")
CORPUS = PROJECT / "data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22"
EXPORT = CORPUS / "exports/f46e687c4faf4aad97007a76dc8431dd"
TRAIN = EXPORT / "train.jsonl"
VALIDATION = EXPORT / "validation.jsonl"
SUMMARY_PATH = EXPORT / "summary.json"

CORPUS_BACKUP = Path(
    "/content/drive/MyDrive/MoE_ERC/data_snapshots/"
    "dolma-20261023T173258Z-2412fe19bc1a0d22-98e15a0106d6"
)
PLAN_SHA = "2412fe19bc1a0d22ccfcab4cf4631478e08f471954a18a0bb06e1c840e0d4569"
SELECTION_SHA = "c777494f3fd7c0acfd7d5c39097fe762410c10f894c0b77dc7a88b469f52d4aa"

missing = [str(p) for p in (TRAIN, VALIDATION, SUMMARY_PATH) if not p.is_file()]
if missing:
    print("Thiếu corpus local:\n" + "\n".join(missing))
    print("Restore từ:", CORPUS_BACKUP)
    print(
        "Mount Drive, xác minh COMPLETE và manifest SHA-256, copy qua staging "
        "và kiểm tra checksum trước khi đưa vào /content/moe-erc/data/. "
        "Dừng nếu file đích đã tồn tại; không tự overwrite."
    )
    raise FileNotFoundError("Cần restore corpus đã xác minh trước khi tiếp tục.")

sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
from moe_erc.dolma_reader import canonical_text, file_sha256

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

SUMMARY = read_json(SUMMARY_PATH)
PLAN = SUMMARY["plan"]
assert SUMMARY["plan_sha256"] == PLAN_SHA
assert SUMMARY["selection_sha256"] == SELECTION_SHA
assert fingerprint(PLAN) == PLAN_SHA
assert fingerprint(PLAN["selected_urls"]) == SELECTION_SHA
assert PLAN["version"] == "v1_5-sample"

SOURCE_SHA = {}
CORPUS_COUNTS = Counter()
SOURCE_CATEGORIES = {"train": Counter(), "validation": Counter()}
TEXT_BYTES = Counter()
HASHES = {"train": set(), "validation": set()}

for split, path in (("train", TRAIN), ("validation", VALIDATION)):
    name = path.name
    SOURCE_SHA[name] = file_sha256(path)
    assert SOURCE_SHA[name] == SUMMARY["files"][name]["sha256"], f"Checksum FAIL: {name}"

    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            item = json.loads(line)
            if not isinstance(item, dict) or not isinstance(item.get("text"), str):
                raise ValueError(f"Record không hợp lệ: {name}:{line_number}")
            text = canonical_text(item["text"])
            if not text or text != item["text"]:
                raise ValueError(f"Text rỗng/chưa canonical: {name}:{line_number}")
            content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
            if item.get("content_hash") != content_hash or item.get("split") != split:
                raise ValueError(f"Hash/split sai: {name}:{line_number}")
            if content_hash in HASHES[split]:
                raise ValueError(f"Duplicate trong {name}:{line_number}")
            category = item.get("source_category")
            if not isinstance(category, str) or not category:
                raise ValueError(f"Thiếu source_category: {name}:{line_number}")

            HASHES[split].add(content_hash)
            CORPUS_COUNTS[split] += 1
            TEXT_BYTES[split] += len(text.encode("utf-8"))
            SOURCE_CATEGORIES[split][category] += 1

assert not HASHES["train"].intersection(HASHES["validation"]), "Train/validation overlap."
assert CORPUS_COUNTS["train"] == 28_051 == SUMMARY["state"]["train_documents"]
assert CORPUS_COUNTS["validation"] == 443 == SUMMARY["state"]["validation_documents"]
assert TEXT_BYTES["train"] == SUMMARY["state"]["train_text_bytes"]

SOURCE_SHA["summary.json"] = file_sha256(SUMMARY_PATH)
CORPUS_FINGERPRINT = fingerprint({
    "files": SOURCE_SHA,
    "plan_sha256": PLAN_SHA,
    "selection_sha256": SELECTION_SHA,
})

print("Corpus audit: PASS")
print("Documents:", dict(CORPUS_COUNTS))
print(f"Clean train text: {TEXT_BYTES['train'] / 1024**2:.6f} MiB")
print("Train sources:", dict(SOURCE_CATEGORIES["train"]))
print("Content-hash overlap: 0")
print("Corpus fingerprint:", CORPUS_FINGERPRINT)
print(f"Local disk free: {shutil.disk_usage(PROJECT).free / 1024**3:.2f} GiB")

if Path("/content/drive/MyDrive").is_dir():
    if not CORPUS_BACKUP.is_dir():
        raise FileNotFoundError(f"Drive đã mount nhưng không thấy backup: {CORPUS_BACKUP}")
    if (CORPUS_BACKUP / "INCOMPLETE.json").exists():
        raise RuntimeError("Corpus backup còn marker INCOMPLETE.")
    complete = read_json(CORPUS_BACKUP / "COMPLETE.json")
    backup_manifest = read_json(CORPUS_BACKUP / "backup_manifest.json")
    actual_manifest_sha = file_sha256(CORPUS_BACKUP / "backup_manifest.json")
    detached = (CORPUS_BACKUP / "backup_manifest.sha256").read_text().strip()
    assert complete["status"] == "COMPLETE"
    assert actual_manifest_sha == detached == complete["manifest_sha256"]
    assert backup_manifest["plan_sha256"] == PLAN_SHA
    assert backup_manifest["selection_sha256"] == SELECTION_SHA

    entries = {entry["path"]: entry for entry in backup_manifest["files"]}
    for path in (TRAIN, VALIDATION, SUMMARY_PATH):
        relative = path.relative_to(PROJECT).as_posix()
        entry = entries[relative]
        assert entry["sha256"] == SOURCE_SHA[path.name]
        assert (CORPUS_BACKUP / relative).stat().st_size == entry["bytes"]
    print("Corpus backup: COMPLETE marker, manifest checksum và metadata khớp.")
    print("Không quét lại SHA-256 toàn bộ payload backup trong cell này.")
else:
    print("Drive chưa mount: chưa kiểm tra trực tiếp trạng thái corpus backup.")

Corpus audit: PASS
Documents: {'train': 28051, 'validation': 443}
Clean train text: 127.999989 MiB
Train sources: {'cc_en_middle': 6694, 'reddit': 1487, 'stack': 1490, 'cc_en_tail': 5205, 'wiki': 1492, 'cc_en_head': 7689, 'pes2o': 977, 'books': 58, 'c4': 2959}
Content-hash overlap: 0
Corpus fingerprint: 2df1e251a4fc3d583d8c8396a79590e99f40c5f63574009a9b3009165b192d04
Local disk free: 192.23 GiB
Corpus backup: COMPLETE marker, manifest checksum và metadata khớp.
Không quét lại SHA-256 toàn bộ payload backup trong cell này.


In [ ]:
import os
import inspect
import platform
import subprocess
import importlib.metadata

# Không cài hoặc cập nhật dependencies.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["RAYON_NUM_THREADS"] = "1"

try:
    import numpy as np
    import torch
    import tokenizers
    from tokenizers import Tokenizer
    from moe_erc.config import ModelConfig, AoEConfig
    from moe_erc.tokenizer_utils import (
        FrozenTokenizer, SPECIAL_TOKENS, SplitConfig,
        SPLIT_ALGORITHM, chain_hash, train_tokenizer,
    )
    from moe_erc.dolma_reader import iter_documents, DownloadOptions
    from moe_erc.data_processing import (
        PreprocessingConfig, prepare_dataset, validate_dataset,
    )
    from moe_erc.dataset import PackedTokenDataset
except ImportError as exc:
    raise RuntimeError(
        "Thiếu dependency của STEP 5. Kiểm tra environment/project installation; "
        "cell này không tự cài dependencies hoặc thay thế PyTorch."
    ) from exc

assert sys.version_info >= (3, 10)
version_pair = tuple(int(x) for x in tokenizers.__version__.split(".")[:2])
assert (0, 20) <= version_pair < (1, 0), "Ngoài constraint tokenizers>=0.20,<1."

EXPECTED_SPECIAL_IDS = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}
assert tuple(SPECIAL_TOKENS) == tuple(EXPECTED_SPECIAL_IDS)
assert dict(zip(SPECIAL_TOKENS, range(4))) == EXPECTED_SPECIAL_IDS
assert ModelConfig().vocab_size == AoEConfig().vocab_size == 16_000

SPLIT = SplitConfig(
    seed=PLAN["config"]["seed"],
    validation_fraction=PLAN["config"]["validation_fraction"],
)
assert SPLIT.seed == 42
assert PLAN["split_algorithm"] == SPLIT_ALGORITHM

signature = inspect.signature(train_tokenizer)
for name in ("sources", "output", "split", "vocab_size", "max_documents", "provenance"):
    assert name in signature.parameters, f"API thiếu tham số {name}"

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "tokenizers": tokenizers.__version__,
}
PIPELINE_SOURCE_SHA = {
    name: file_sha256(PROJECT / "moe_erc" / name)
    for name in (
        "tokenizer_utils.py", "dolma_reader.py",
        "data_processing.py", "dataset.py", "config.py",
    )
}

OUTDIR = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22"
TOKENIZER_PATH = OUTDIR / "tokenizer.json"
TARGET_VOCAB = 16_000

print("Versions:", VERSIONS)
print("Git commit:", GIT_COMMIT)
print("API:", signature)
print("Special IDs:", EXPECTED_SPECIAL_IDS)
print("EOS: <eos>, ID=2")
print("STEP 5: BPE; min_frequency=2; initial alphabet=256 bytes.")
print("ByteLevel(add_prefix_space=False); ByteLevel decoder; không normalizer mới.")
print("Tokenizer encode không tự thêm BOS/EOS.")
print("prepare_dataset thêm tokenizer.eos_id sau mỗi document.")
print("Pipeline packing: uint16, sequence length=512; labels chưa shift.")
print("FrozenTokenizer.validate_model kiểm tra vocab với configuration của model.")
print("Output:", OUTDIR)
print("BPE chạy trên CPU; không cần cấp phát mô hình hoặc tensor CUDA.")
print(
    "Giới hạn reproducibility: BPE trainer không có seed argument. "
    "Đã yêu cầu không parallel; nếu runtime từng khởi tạo thread pool, "
    "cấu hình thread trước đó có thể còn ảnh hưởng."
)

Versions: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'tokenizers': '0.23.3'}
Git commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
API: (sources: list[str], output: str | pathlib._local.Path, *, split: moe_erc.tokenizer_utils.SplitConfig = SplitConfig(seed=42, validation_fraction=0.01), vocab_size: int = 16000, max_documents: int = 100000, test_only: bool = False, downloads: moe_erc.dolma_reader.DownloadOptions | None = None, provenance: dict | None = None) -> moe_erc.tokenizer_utils.FrozenTokenizer
Special IDs: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3}
EOS: <eos>, ID=2
STEP 5: BPE; min_frequency=2; initial alphabet=256 bytes.
ByteLevel(add_prefix_space=False); ByteLevel decoder; không normalizer mới.
Tokenizer encode không tự thêm BOS/EOS.
prepare_dataset thêm tokenizer.eos_id sau mỗi document.
Pipeline packing: uint16, sequence length=512; labels chưa shift.
FrozenTokenizer.validate_model kiểm tra vocab với configuration của model.
Output: /content/moe-erc

In [ ]:
def training_text_iterator():
    stats = {}
    for document in iter_documents(TRAIN, stats=stats):
        if document.content_hash not in HASHES["train"]:
            raise RuntimeError("Document không thuộc tập train đã audit.")
        if SPLIT.assign(document.content_hash) != "train":
            raise RuntimeError("Document bị gán sai split.")
        yield document.text
    if stats.get("invalid_documents", 0):
        raise RuntimeError(f"Reader bỏ qua records lỗi: {stats}")

TRAIN_CONTENT_FINGERPRINT = "00" * 32
TRAIN_DOCUMENTS = 0
TRAIN_TEXT_BYTES = 0

for text in training_text_iterator():
    content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    TRAIN_CONTENT_FINGERPRINT = chain_hash(TRAIN_CONTENT_FINGERPRINT, content_hash)
    TRAIN_DOCUMENTS += 1
    TRAIN_TEXT_BYTES += len(text.encode("utf-8"))

# Xác minh toàn bộ train và validation đều theo split đã thống nhất.
for split_name, hashes in HASHES.items():
    if any(SPLIT.assign(content_hash) != split_name for content_hash in hashes):
        raise RuntimeError(f"Deterministic split FAIL: {split_name}")

assert TRAIN_DOCUMENTS == CORPUS_COUNTS["train"]
assert TRAIN_TEXT_BYTES == TEXT_BYTES["train"]
assert file_sha256(TRAIN) == SOURCE_SHA["train.jsonl"]
assert file_sha256(VALIDATION) == SOURCE_SHA["validation.jsonl"]

print("Train documents:", TRAIN_DOCUMENTS)
print(f"Clean text: {TRAIN_TEXT_BYTES / 1024**2:.6f} MiB")
print("Training source categories:", dict(SOURCE_CATEGORIES["train"]))
print("Corpus fingerprint:", CORPUS_FINGERPRINT)
print("Ordered training-content fingerprint:", TRAIN_CONTENT_FINGERPRINT)
print("Training JSONL SHA-256:", SOURCE_SHA["train.jsonl"])
print("Target vocab, bao gồm special tokens:", TARGET_VOCAB)
print("Không truncate documents; chỉ giữ một document text tại mỗi lần yield.")
print("Validation và sample STEP 6.3.2 không nằm trong fitting sources.")

Train documents: 28051
Clean text: 127.999989 MiB
Training source categories: {'cc_en_middle': 6694, 'reddit': 1487, 'stack': 1490, 'cc_en_tail': 5205, 'wiki': 1492, 'cc_en_head': 7689, 'pes2o': 977, 'books': 58, 'c4': 2959}
Corpus fingerprint: 2df1e251a4fc3d583d8c8396a79590e99f40c5f63574009a9b3009165b192d04
Ordered training-content fingerprint: 68d5b7bacc1f0f69d30715e4d7d1eebae83cabff1666db794f9b4118f5dd3060
Training JSONL SHA-256: 8a3c7dc725bbcef1c1ccc755bd4a0d6062ec68d5459a0535072bc4cc965c6ff1
Target vocab, bao gồm special tokens: 16000
Không truncate documents; chỉ giữ một document text tại mỗi lần yield.
Validation và sample STEP 6.3.2 không nằm trong fitting sources.


In [ ]:
RUN_TOKENIZER_TRAINING = True

import random
from datetime import datetime, timezone

def verify_fitting_metadata(frozen):
    metadata = frozen.metadata
    assert frozen.vocab_size == TARGET_VOCAB
    assert frozen.special_ids == EXPECTED_SPECIAL_IDS
    assert not metadata["test_only"]
    assert metadata["algorithm"] == "byte-level-bpe"
    assert metadata["requested_vocab_size"] == TARGET_VOCAB
    assert metadata["training_documents"] == TRAIN_DOCUMENTS
    assert metadata["training_content_fingerprint"] == TRAIN_CONTENT_FINGERPRINT
    assert metadata["source_plan"] == [str(TRAIN)]
    assert metadata["sources_consumed"] == [
        {"source": str(TRAIN), "sha256": SOURCE_SHA["train.jsonl"]}
    ]
    assert metadata["split"] == {
        "seed": SPLIT.seed, "validation_fraction": SPLIT.validation_fraction
    }
    provenance = metadata["provenance"]
    assert provenance["corpus_fingerprint"] == CORPUS_FINGERPRINT
    assert provenance["plan_sha256"] == PLAN_SHA
    assert provenance["selection_sha256"] == SELECTION_SHA

if not RUN_TOKENIZER_TRAINING:
    print("RUN_TOKENIZER_TRAINING=False: chưa huấn luyện.")
    print("Nếu output đã có, Cell 5 có thể kiểm tra lại; không cần retrain.")
else:
    for path in (TRAIN, VALIDATION, SUMMARY_PATH):
        if file_sha256(path) != SOURCE_SHA[path.name]:
            raise RuntimeError(f"Corpus đã thay đổi: {path}")
    for name, expected_sha in PIPELINE_SOURCE_SHA.items():
        assert file_sha256(PROJECT / "moe_erc" / name) == expected_sha

    if OUTDIR.exists():
        raise FileExistsError(
            f"Không ghi đè {OUTDIR}. Để công tắc False nếu muốn kiểm tra tokenizer đã lưu."
        )
    OUTDIR.mkdir(parents=True, exist_ok=False)

    random.seed(42)
    np.random.seed(42)

    provenance = {
        "dataset": "allenai/dolma",
        "dataset_version": PLAN["version"],
        "corpus_fingerprint": CORPUS_FINGERPRINT,
        "plan_sha256": PLAN_SHA,
        "selection_sha256": SELECTION_SHA,
        "corpus_files_sha256": SOURCE_SHA,
        "expected_training_content_fingerprint": TRAIN_CONTENT_FINGERPRINT,
        "training_sources": [str(TRAIN)],
        "validation_used_for_fitting": False,
        "git_commit": GIT_COMMIT,
        "pipeline_source_sha256": PIPELINE_SOURCE_SHA,
        "environment": VERSIONS,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "reproducibility": {
            "split_seed": 42,
            "bpe_seed_argument": None,
            "tokenizers_parallelism": os.environ["TOKENIZERS_PARALLELISM"],
            "rayon_num_threads": os.environ["RAYON_NUM_THREADS"],
            "bitwise_reproducibility_verified": False,
        },
    }

    print("Bắt đầu fitting Byte-level BPE trên CPU, chỉ từ train.jsonl...")
    try:
        fitted = train_tokenizer(
            sources=[str(TRAIN)],
            output=TOKENIZER_PATH,
            split=SPLIT,
            vocab_size=TARGET_VOCAB,
            max_documents=TRAIN_DOCUMENTS,
            test_only=False,
            downloads=DownloadOptions(allow_download=False),
            provenance=provenance,
        )
    except Exception:
        print(
            "Training FAIL. Không thay vocab hoặc dùng validation để bù dữ liệu. "
            "Output directory được giữ lại để kiểm tra; không tự overwrite khi chạy lại."
        )
        raise

    verify_fitting_metadata(fitted)
    for path in (TRAIN, VALIDATION, SUMMARY_PATH):
        assert file_sha256(path) == SOURCE_SHA[path.name], "Corpus thay đổi khi fitting."

    reloaded = FrozenTokenizer(TOKENIZER_PATH)
    assert reloaded.fingerprint == fitted.fingerprint
    print("Training và reload: PASS")
    print("Vocabulary:", reloaded.vocab_size)
    print("Special IDs:", reloaded.special_ids)
    print("Tokenizer:", TOKENIZER_PATH)
    print("Tokenizer SHA-256:", reloaded.fingerprint)

Bắt đầu fitting Byte-level BPE trên CPU, chỉ từ train.jsonl...
Training và reload: PASS
Vocabulary: 16000
Special IDs: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3}
Tokenizer: /content/moe-erc/data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797


In [ ]:
import tempfile
from itertools import islice

TEST_RESULTS = None
RELOAD_PROBE_TEXT = "Tokenizer reload: tiếng Việt 🌏.\n"

def run_tokenizer_tests():
    frozen = FrozenTokenizer(TOKENIZER_PATH)
    verify_fitting_metadata(frozen)
    frozen.validate_model(ModelConfig())
    frozen.validate_model(AoEConfig())

    assert frozen.eos_id == 2
    assert len(set(frozen.special_ids.values())) == 4
    raw_config = read_json(TOKENIZER_PATH)
    assert raw_config["model"]["type"] == "BPE"
    assert raw_config["model"]["unk_token"] == "<unk>"
    assert raw_config.get("normalizer") is None
    assert raw_config.get("post_processor") is None
    assert raw_config["pre_tokenizer"]["type"] == "ByteLevel"
    assert raw_config["pre_tokenizer"]["add_prefix_space"] is False
    assert raw_config["decoder"]["type"] == "ByteLevel"
    assert raw_config["model"].get("dropout") is None

    examples = [
        "Hello, world! This is a tokenizer test.",
        "  Spaces\tand punctuation: (a + b) = 42.\nSecond line.\n",
        "Tiếng Việt: nghiên cứu mô hình hỗn hợp chuyên gia.",
        "Unicode: 🌏 café — Ελληνικά 日本語 العربية.",
        RELOAD_PROBE_TEXT,
    ]

    reload_ids = {}
    for text in examples:
        ids = frozen.encode(text)
        assert ids and all(type(i) is int and 0 <= i < TARGET_VOCAB for i in ids)
        assert EXPECTED_SPECIAL_IDS["<unk>"] not in ids, "Unexpected UNK."
        assert frozen.decode(ids) == text, "Round-trip FAIL."
        assert FrozenTokenizer(TOKENIZER_PATH).encode(text) == ids
        reload_ids[text] = ids
        print(f"Synthetic round-trip PASS: {len(text)} chars -> {len(ids)} tokens")

    # Serialize/load một lần nữa bằng thư viện, rồi dùng wrapper STEP 5.
    with tempfile.TemporaryDirectory(prefix="moe-tokenizer-reload-") as temporary:
        path = Path(temporary) / "tokenizer.json"
        Tokenizer.from_file(str(TOKENIZER_PATH)).save(str(path))
        copied_meta = dict(frozen.metadata)
        copied_meta["sha256"] = file_sha256(path)
        with path.with_suffix(".meta.json").open("w", encoding="utf-8") as stream:
            json.dump(copied_meta, stream, ensure_ascii=False)
        serialized = FrozenTokenizer(path)
        for text, expected in reload_ids.items():
            assert serialized.encode(text) == expected

    # Evaluation sau fitting; không update vocabulary, không in text Dolma.
    validation_stats = Counter()
    for document in iter_documents(VALIDATION, max_documents=16):
        ids = frozen.encode(document.text)
        assert ids and all(0 <= i < TARGET_VOCAB for i in ids)
        assert frozen.decode(ids, skip_special_tokens=False) == document.text
        validation_stats["documents"] += 1
        validation_stats["tokens"] += len(ids)
        validation_stats["utf8_bytes"] += len(document.text.encode("utf-8"))
        validation_stats["unk_id_occurrences"] += ids.count(EXPECTED_SPECIAL_IDS["<unk>"])
    assert validation_stats["documents"] > 0

    # Fixture chỉ phục vụ packing test, không dùng train tokenizer.
    fixture_texts = {"train": [], "validation": []}
    for attempt in range(10_000):
        text = canonical_text(
            f"Fixture {attempt}: " +
            "Pipeline check: tiếng Việt, punctuation, and document boundaries.\n" * 150
        )
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        split_name = SPLIT.assign(digest)
        target_count = 2 if split_name == "train" else 1
        if len(fixture_texts[split_name]) < target_count:
            fixture_texts[split_name].append(text)
        if len(fixture_texts["train"]) == 2 and len(fixture_texts["validation"]) == 1:
            break
    assert len(fixture_texts["train"]) == 2
    assert len(fixture_texts["validation"]) == 1

    expected_tokens = {}
    for split_name, texts in fixture_texts.items():
        stream = []
        for text in texts:
            stream.extend(frozen.encode(text))
            stream.append(frozen.eos_id)
        expected_tokens[split_name] = stream
        assert 512 <= len(stream) <= 200_000

    with tempfile.TemporaryDirectory(prefix="moe-packing-smoke-") as temporary:
        root = Path(temporary)
        fixture_path = root / "synthetic.jsonl"
        with fixture_path.open("w", encoding="utf-8") as stream:
            for split_name in ("train", "validation"):
                for text in fixture_texts[split_name]:
                    stream.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

        packed_dir = root / "packed"
        packed = prepare_dataset(
            [str(fixture_path)], frozen, packed_dir,
            config=PreprocessingConfig(
                train_max_tokens=len(expected_tokens["train"]),
                validation_max_tokens=len(expected_tokens["validation"]),
                sequence_length=512,
                split=SPLIT,
            ),
            downloads=DownloadOptions(allow_download=False),
            provenance={"purpose": "synthetic tokenizer compatibility smoke test"},
        )
        validate_dataset(packed_dir, tokenizer=frozen, checksums=True)

        packed_counts = {}
        for split_name in ("train", "validation"):
            actual = []
            for segment in packed["splits"][split_name]["segments"]:
                actual.extend(
                    np.fromfile(packed_dir / segment["file"], dtype="<u2").tolist()
                )
            expected = expected_tokens[split_name]
            expected = expected[:len(expected) // 512 * 512]
            assert actual == expected, "Packing hoặc EOS convention FAIL."
            packed_counts[split_name] = len(actual) // 512

            dataset = PackedTokenDataset(
                packed_dir, split_name, tokenizer=frozen, verify_checksums=True
            )
            try:
                sample = dataset[0]
                assert tuple(sample["input_ids"].shape) == (512,)
                assert torch.equal(sample["input_ids"], sample["labels"])
                assert int(sample["input_ids"].min()) >= 0
                assert int(sample["input_ids"].max()) < TARGET_VOCAB
            finally:
                dataset.close()

    return {
        "status": "PASS",
        "tokenizer_sha256": frozen.fingerprint,
        "checks": {
            "vocab_16000": True,
            "special_ids_and_eos": True,
            "byte_level_configuration": True,
            "synthetic_round_trip_and_no_unexpected_unk": True,
            "serialize_reload_same_ids": True,
            "validation_round_trip": True,
            "model_config_compatibility": True,
            "step5_eos_and_packing_512": True,
        },
        "validation_evaluation": dict(validation_stats),
        "packing_smoke_sequences": packed_counts,
        "reload_probe": {
            "text": RELOAD_PROBE_TEXT,
            "ids": frozen.encode(RELOAD_PROBE_TEXT),
        },
    }

if not TOKENIZER_PATH.is_file() or not TOKENIZER_PATH.with_suffix(".meta.json").is_file():
    print("Chưa có tokenizer hoàn chỉnh. Bật training ở Cell 4 trước.")
else:
    TEST_RESULTS = run_tokenizer_tests()
    print("Tokenizer tests:", TEST_RESULTS["status"])
    print("Validation evaluation:", TEST_RESULTS["validation_evaluation"])
    print("Packed synthetic sequences:", TEST_RESULTS["packing_smoke_sequences"])
    print(
        "UNK ID xuất hiện trong validation có thể do literal '<unk>' trong văn bản; "
        "kiểm tra không-UNK ở trên áp dụng cho các câu synthetic thông thường."
    )

Synthetic round-trip PASS: 39 chars -> 12 tokens
Synthetic round-trip PASS: 53 chars -> 23 tokens
Synthetic round-trip PASS: 50 chars -> 49 tokens
Synthetic round-trip PASS: 39 chars -> 40 tokens
Synthetic round-trip PASS: 32 chars -> 21 tokens
Tokenizer tests: PASS
Validation evaluation: {'documents': 16, 'tokens': 9456, 'utf8_bytes': 38479, 'unk_id_occurrences': 0}
Packed synthetic sequences: {'train': 15, 'validation': 7}
UNK ID xuất hiện trong validation có thể do literal '<unk>' trong văn bản; kiểm tra không-UNK ở trên áp dụng cho các câu synthetic thông thường.


In [ ]:
RESEARCH_PATH = OUTDIR / "research_metadata.json"
ARTIFACT_READY = False

def verify_saved_tokenizer(directory):
    directory = Path(directory)
    research = read_json(directory / "research_metadata.json")
    assert research["tests"]["status"] == "PASS"
    for name, expected_sha in research["artifact_sha256"].items():
        assert Path(name).name == name
        assert file_sha256(directory / name) == expected_sha, f"Checksum FAIL: {name}"
    loaded = FrozenTokenizer(directory / research["tokenizer_file"])
    assert loaded.vocab_size == research["actual_vocab_size"] == 16_000
    assert loaded.special_ids == research["special_ids"] == EXPECTED_SPECIAL_IDS
    assert loaded.eos_id == research["eos_token_id"] == 2
    assert loaded.fingerprint == research["tests"]["tokenizer_sha256"]
    probe = research["tests"]["reload_probe"]
    assert loaded.encode(probe["text"]) == probe["ids"]
    return loaded, research

if not TEST_RESULTS or TEST_RESULTS.get("status") != "PASS":
    print("Chưa lưu metadata hoàn tất: tokenizer tests chưa PASS.")
else:
    frozen = FrozenTokenizer(TOKENIZER_PATH)
    assert frozen.fingerprint == TEST_RESULTS["tokenizer_sha256"]
    for path in (TRAIN, VALIDATION, SUMMARY_PATH):
        assert file_sha256(path) == SOURCE_SHA[path.name]

    raw_config = read_json(TOKENIZER_PATH)
    research = {
        "format_version": 1,
        "tokenizer_file": "tokenizer.json",
        "wrapper_metadata_file": "tokenizer.meta.json",
        "algorithm": "byte-level-bpe",
        "target_vocab_size": TARGET_VOCAB,
        "actual_vocab_size": frozen.vocab_size,
        "special_tokens": list(SPECIAL_TOKENS),
        "special_ids": frozen.special_ids,
        "eos_token": "<eos>",
        "eos_token_id": frozen.eos_id,
        "dataset_version": PLAN["version"],
        "corpus_fingerprint": CORPUS_FINGERPRINT,
        "plan_sha256": PLAN_SHA,
        "selection_sha256": SELECTION_SHA,
        "corpus_files_sha256": SOURCE_SHA,
        "training_documents": frozen.metadata["training_documents"],
        "training_text_bytes": TRAIN_TEXT_BYTES,
        "training_content_fingerprint": TRAIN_CONTENT_FINGERPRINT,
        "training_source_categories": dict(SOURCE_CATEGORIES["train"]),
        "git_commit": frozen.metadata["provenance"]["git_commit"],
        "training_environment": frozen.metadata["provenance"]["environment"],
        "verification_environment": VERSIONS,
        "pipeline_source_sha256": frozen.metadata["provenance"]["pipeline_source_sha256"],
        "training_configuration": {
            "trainer": frozen.metadata["trainer"],
            "split": frozen.metadata["split"],
            "split_algorithm": frozen.metadata["split_algorithm"],
            "max_training_documents": frozen.metadata["max_training_documents"],
            "validation_used_for_fitting": False,
            "canonicalization": "STEP5 canonical_text: CRLF/CR -> LF; strip outer whitespace",
            "tokenizer_normalizer": raw_config.get("normalizer"),
            "pre_tokenizer": raw_config["pre_tokenizer"],
            "decoder": raw_config["decoder"],
            "post_processor": raw_config.get("post_processor"),
            "reproducibility": frozen.metadata["provenance"]["reproducibility"],
        },
        "artifact_sha256": {
            "tokenizer.json": file_sha256(TOKENIZER_PATH),
            "tokenizer.meta.json": file_sha256(TOKENIZER_PATH.with_suffix(".meta.json")),
        },
        "corpus_summary": SUMMARY,
        "tests": TEST_RESULTS,
    }

    if RESEARCH_PATH.exists():
        existing = read_json(RESEARCH_PATH)
        comparison = dict(existing)
        comparison.pop("created_at_utc", None)
        if comparison != research:
            raise FileExistsError(
                "research_metadata.json đã tồn tại với nội dung khác; không overwrite."
            )
        print("Tái sử dụng research metadata đã khớp; không ghi đè.")
    else:
        research["created_at_utc"] = datetime.now(timezone.utc).isoformat()
        with RESEARCH_PATH.open("x", encoding="utf-8") as stream:
            json.dump(research, stream, ensure_ascii=False, sort_keys=True, indent=2)
            stream.write("\n")

    verify_saved_tokenizer(OUTDIR)

    # Process mới chỉ nhận output directory; không đọc corpus để load tokenizer.
    reload_script = """
import json, sys
from pathlib import Path
from moe_erc.tokenizer_utils import FrozenTokenizer
root = Path(sys.argv[1])
metadata = json.loads((root / "research_metadata.json").read_text(encoding="utf-8"))
tokenizer = FrozenTokenizer(root / metadata["tokenizer_file"])
probe = metadata["tests"]["reload_probe"]
assert tokenizer.vocab_size == 16000
assert tokenizer.eos_id == metadata["eos_token_id"] == 2
assert tokenizer.encode(probe["text"]) == probe["ids"]
print(json.dumps({"fresh_process_reload": "PASS", "vocab_size": tokenizer.vocab_size}))
"""
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT) + os.pathsep + environment.get("PYTHONPATH", "")
    completed = subprocess.run(
        [sys.executable, "-c", reload_script, str(OUTDIR)],
        cwd=PROJECT, env=environment, text=True, capture_output=True, timeout=120,
    )
    if completed.returncode:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError("Fresh-process reload FAIL.")
    print(completed.stdout.strip())

    temporary_files = [
        p for p in OUTDIR.rglob("*")
        if p.is_file() and (p.name.endswith(".tmp") or p.name.endswith(".part"))
    ]
    if temporary_files:
        raise RuntimeError(f"Có temporary files cần kiểm tra: {temporary_files}")

    ARTIFACT_READY = True
    print("Tokenizer artifact: PASS")
    print("Directory:", OUTDIR)
    print("Vocab:", frozen.vocab_size, "| EOS ID:", frozen.eos_id)
    print("SHA-256:", frozen.fingerprint)
    print("Bitwise reproducibility giữa hai lần fitting: CHƯA KIỂM CHỨNG.")
    print(
        "Phiên mới: cài project/dependencies tương thích, import FrozenTokenizer, "
        "rồi load <output_directory>/tokenizer.json. "
        "Không cần corpus hoặc các biến notebook để load."
    )

{"fresh_process_reload": "PASS", "vocab_size": 16000}
Tokenizer artifact: PASS
Directory: /content/moe-erc/data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22
Vocab: 16000 | EOS ID: 2
SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Bitwise reproducibility giữa hai lần fitting: CHƯA KIỂM CHỨNG.
Phiên mới: cài project/dependencies tương thích, import FrozenTokenizer, rồi load <output_directory>/tokenizer.json. Không cần corpus hoặc các biến notebook để load.


In [ ]:
RUN_TOKENIZER_BACKUP = True

import uuid
import traceback

TOKENIZER_BACKUP_ROOT = Path("/content/drive/MyDrive/MoE_ERC/tokenizer_snapshots")
TOKENIZER_BACKUP_RESULT = {"status": "FAIL", "reason": "Chưa chạy backup."}

def write_new_bytes(path, content):
    with Path(path).open("xb") as stream:
        stream.write(content)
        stream.flush()
        os.fsync(stream.fileno())

def encoded_json(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
    ).encode("utf-8")

def backup_tokenizer():
    result = {"status": "FAIL", "snapshot": None}
    try:
        if not ARTIFACT_READY or not TEST_RESULTS or TEST_RESULTS["status"] != "PASS":
            raise RuntimeError("Cell 5–6 chưa PASS.")
        if not Path("/content/drive/MyDrive").is_dir():
            raise RuntimeError(
                "Drive chưa mount. Mount bằng google.colab.drive tại /content/drive trước."
            )

        local_tokenizer, research = verify_saved_tokenizer(OUTDIR)
        assert local_tokenizer.fingerprint == TEST_RESULTS["tokenizer_sha256"]
        TOKENIZER_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

        sources = []
        for path in sorted(OUTDIR.rglob("*")):
            if path.is_symlink():
                raise RuntimeError(f"Không backup symlink: {path}")
            if path.is_file():
                sources.append(path)
        if not sources:
            raise RuntimeError("Output directory rỗng.")

        entries = [{
            "path": path.relative_to(OUTDIR).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        } for path in sources]

        name = (
            "dolma-bpe16k-"
            + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            + f"-{PLAN_SHA[:16]}-{uuid.uuid4().hex[:12]}"
        )
        pending = TOKENIZER_BACKUP_ROOT / (name + ".INCOMPLETE")
        pending.mkdir(exist_ok=False)
        result["snapshot"] = str(pending)
        write_new_bytes(pending / "INCOMPLETE.json", encoded_json({"status": "INCOMPLETE"}))

        for source, entry in zip(sources, entries):
            target = pending / entry["path"]
            target.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, target.open("xb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)
                dst.flush()
                os.fsync(dst.fileno())
            if target.stat().st_size != entry["bytes"]:
                raise RuntimeError(f"Size FAIL: {entry['path']}")
            if file_sha256(target) != entry["sha256"]:
                raise RuntimeError(f"Drive SHA-256 FAIL: {entry['path']}")
            if file_sha256(source) != entry["sha256"]:
                raise RuntimeError(f"Local file thay đổi khi backup: {source}")

        backup_manifest = {
            "format_version": 1,
            "dataset_version": PLAN["version"],
            "plan_sha256": PLAN_SHA,
            "selection_sha256": SELECTION_SHA,
            "corpus_fingerprint": CORPUS_FINGERPRINT,
            "tokenizer_sha256": local_tokenizer.fingerprint,
            "vocab_size": local_tokenizer.vocab_size,
            "eos_token_id": local_tokenizer.eos_id,
            "git_commit": research["git_commit"],
            "source_directory": str(OUTDIR),
            "files": entries,
        }
        manifest_bytes = encoded_json(backup_manifest)
        manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
        write_new_bytes(pending / "backup_manifest.json", manifest_bytes)
        detached_bytes = (manifest_sha + "\n").encode("ascii")
        write_new_bytes(pending / "backup_manifest.sha256", detached_bytes)

        assert file_sha256(pending / "backup_manifest.json") == manifest_sha
        assert (pending / "backup_manifest.sha256").read_bytes() == detached_bytes
        assert read_json(pending / "backup_manifest.json") == backup_manifest

        backup_loaded, _ = verify_saved_tokenizer(pending)
        assert backup_loaded.fingerprint == local_tokenizer.fingerprint
        probe = research["tests"]["reload_probe"]
        assert backup_loaded.encode(probe["text"]) == local_tokenizer.encode(probe["text"])
        assert backup_loaded.encode(probe["text"]) == probe["ids"]

        final = TOKENIZER_BACKUP_ROOT / name
        if final.exists():
            raise FileExistsError(f"Không overwrite snapshot: {final}")
        pending.rename(final)
        result["snapshot"] = str(final)

        complete = {
            "status": "COMPLETE",
            "verified_at_utc": datetime.now(timezone.utc).isoformat(),
            "manifest_sha256": manifest_sha,
            "tokenizer_sha256": backup_loaded.fingerprint,
            "reload_test": "PASS",
            "payload_files_verified": len(entries),
        }
        complete_bytes = encoded_json(complete)
        marker = final / "COMPLETE.pending.json"
        write_new_bytes(marker, complete_bytes)
        assert marker.read_bytes() == complete_bytes

        # Chỉ loại bỏ marker do lần chạy này tạo; giữ mọi artifact và backup cũ.
        (final / "INCOMPLETE.json").unlink()
        marker.rename(final / "COMPLETE.json")

        result.update({
            "status": "PASS",
            "vocab_size": backup_loaded.vocab_size,
            "eos_token_id": backup_loaded.eos_id,
            "local_tokenizer": str(TOKENIZER_PATH),
            "files": len(entries) + 3,  # Payload + manifest + detached SHA + COMPLETE.
            "sha256_verification": "PASS",
            "reload_test": "PASS",
            "manifest_sha256": manifest_sha,
            "reason": "Checksum và tokenizer reload từ Drive đều PASS.",
        })
    except BaseException as exc:
        result["reason"] = f"{type(exc).__name__}: {exc}"
        print("Backup FAIL; giữ bản chưa hoàn chỉnh để kiểm tra.")
        traceback.print_exc()
    return result

if not RUN_TOKENIZER_BACKUP:
    print("RUN_TOKENIZER_BACKUP=False: chưa sao lưu tokenizer.")
else:
    TOKENIZER_BACKUP_RESULT = backup_tokenizer()
    for key, value in TOKENIZER_BACKUP_RESULT.items():
        print(f"{key}: {value}")
    print("Backup status:", TOKENIZER_BACKUP_RESULT["status"])
    print(
        "Checksum trên Drive mount chỉ xác nhận nội dung đọc lại tại lúc kiểm tra; "
        "không bảo đảm tuyệt đối đồng bộ phía dịch vụ đã hoàn tất."
    )
    print("Hãy lưu notebook Colab vào Google Drive.")

status: PASS
snapshot: /content/drive/MyDrive/MoE_ERC/tokenizer_snapshots/dolma-bpe16k-20261024T044101Z-2412fe19bc1a0d22-1a4739a56d17
vocab_size: 16000
eos_token_id: 2
local_tokenizer: /content/moe-erc/data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json
files: 6
sha256_verification: PASS
reload_test: PASS
manifest_sha256: 96d30a579382e21c51ab6a83199ed3f5ffcedc187142c1a4c3d0cf617c56e0a8
reason: Checksum và tokenizer reload từ Drive đều PASS.
Backup status: PASS
Checksum trên Drive mount chỉ xác nhận nội dung đọc lại tại lúc kiểm tra; không bảo đảm tuyệt đối đồng bộ phía dịch vụ đã hoàn tất.
Hãy lưu notebook Colab vào Google Drive.


**STEP 6.4.1 — KIỂM TRA TOKENIZATION VÀ PACKING 512 TOKENS TRÊN GOOGLE COLAB**

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import inspect
import json
import subprocess
import sys

PROJECT = Path("/content/moe-erc")
CORPUS = PROJECT / "data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22"
EXPORT = CORPUS / "exports/f46e687c4faf4aad97007a76dc8431dd"
SOURCES = {
    "train": EXPORT / "train.jsonl",
    "validation": EXPORT / "validation.jsonl",
}
SUMMARY_PATH = EXPORT / "summary.json"
TOKENIZER_PATH = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"
PLAN_SHA = "2412fe19bc1a0d22ccfcab4cf4631478e08f471954a18a0bb06e1c840e0d4569"
SELECTION_SHA = "c777494f3fd7c0acfd7d5c39097fe762410c10f894c0b77dc7a88b469f52d4aa"
SEQUENCE_LENGTH = 512
INPUT_AUDIT_PASS = False

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

for path in [
    *SOURCES.values(), SUMMARY_PATH, TOKENIZER_PATH,
    TOKENIZER_PATH.with_suffix(".meta.json"),
]:
    require(path.is_file(), f"Thiếu file: {path}. Không tự restore hoặc train lại.")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import numpy as np
import torch
import tokenizers
from moe_erc.config import ModelConfig, AoEConfig
from moe_erc.dolma_reader import canonical_text, file_sha256, iter_documents, DownloadOptions
from moe_erc.tokenizer_utils import FrozenTokenizer, SplitConfig, SPLIT_ALGORITHM, chain_hash
from moe_erc.data_processing import (
    PreprocessingConfig, prepare_dataset, validate_dataset, audit_document_splits,
)
from moe_erc.dataset import PackedTokenDataset, make_dataloader
from moe_erc.model import MoELanguageModel, AoELanguageModel

SUMMARY = read_json(SUMMARY_PATH)
PLAN = SUMMARY["plan"]
require(fingerprint(PLAN) == PLAN_SHA == SUMMARY["plan_sha256"], "Plan fingerprint sai.")
require(
    fingerprint(PLAN["selected_urls"]) == SELECTION_SHA == SUMMARY["selection_sha256"],
    "Selection fingerprint sai.",
)
require(PLAN["split_algorithm"] == SPLIT_ALGORITHM, "Split algorithm khác STEP 5.")

SPLIT = SplitConfig(
    seed=PLAN["config"]["seed"],
    validation_fraction=PLAN["config"]["validation_fraction"],
)
require(SPLIT.seed == 42, "Seed khác 42.")

TOKENIZER = FrozenTokenizer(TOKENIZER_PATH)
require(TOKENIZER.vocab_size == 16000, "Vocab không phải 16000.")
require(TOKENIZER.eos_id == 2, "EOS ID không phải 2.")
require(
    TOKENIZER.special_ids == {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3},
    "Special IDs khác STEP 5.",
)
TOKENIZER.validate_model(ModelConfig())
TOKENIZER.validate_model(AoEConfig())

raw_tokenizer = read_json(TOKENIZER_PATH)
require(raw_tokenizer["model"]["type"] == "BPE", "Tokenizer không phải BPE.")
require(raw_tokenizer["pre_tokenizer"]["type"] == "ByteLevel", "Không phải ByteLevel.")
require(raw_tokenizer.get("post_processor") is None, "Có post-processor chưa được dự kiến.")
require(raw_tokenizer.get("normalizer") is None, "Có normalizer khác STEP 5.")

def strict_records(path, split):
    with Path(path).open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            try:
                item = json.loads(line)
                require(isinstance(item, dict), "Record không phải object.")
                require(isinstance(item.get("text"), str), "Text không phải string.")
                text = canonical_text(item["text"])
                require(bool(text) and text == item["text"], "Text rỗng/chưa canonical.")
                digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
                require(item.get("content_hash") == digest, "Content hash sai.")
                require(item.get("split") == SPLIT.assign(digest) == split, "Split sai.")
                category = item.get("source_category")
                require(isinstance(category, str) and bool(category), "Thiếu source category.")
            except Exception as exc:
                raise RuntimeError(
                    f"Record FAIL: split={split}, line={line_number}, "
                    f"error_type={type(exc).__name__}"
                ) from None
            yield line_number, item

SOURCE_SHA = {}
REFERENCES = {"train": {}, "validation": {}}
INPUT_COUNTS = Counter()
INPUT_BYTES = Counter()

for split, path in SOURCES.items():
    SOURCE_SHA[path.name] = file_sha256(path)
    require(
        SOURCE_SHA[path.name] == SUMMARY["files"][path.name]["sha256"],
        f"Checksum FAIL: {path.name}",
    )
    for line_number, item in strict_records(path, split):
        digest = item["content_hash"]
        require(digest not in REFERENCES[split], f"Duplicate: {split}, line={line_number}")
        REFERENCES[split][digest] = {
            "line": line_number,
            "source_category": item["source_category"],
        }
        INPUT_COUNTS[split] += 1
        INPUT_BYTES[split] += len(item["text"].encode("utf-8"))

OVERLAP = len(REFERENCES["train"].keys() & REFERENCES["validation"].keys())
require(OVERLAP == 0, "Train/validation overlap.")
require(INPUT_COUNTS["train"] == SUMMARY["state"]["train_documents"] == 28051, "Train count sai.")
require(
    INPUT_COUNTS["validation"] == SUMMARY["state"]["validation_documents"] == 443,
    "Validation count sai.",
)
require(INPUT_BYTES["train"] == SUMMARY["state"]["train_text_bytes"], "Train text size sai.")

SOURCE_SHA["summary.json"] = file_sha256(SUMMARY_PATH)
CORPUS_FINGERPRINT = fingerprint({
    "files": SOURCE_SHA,
    "plan_sha256": PLAN_SHA,
    "selection_sha256": SELECTION_SHA,
})
provenance = TOKENIZER.metadata["provenance"]
require(provenance["corpus_fingerprint"] == CORPUS_FINGERPRINT, "Tokenizer thuộc corpus khác.")
require(provenance["plan_sha256"] == PLAN_SHA, "Tokenizer plan sai.")
require(provenance["selection_sha256"] == SELECTION_SHA, "Tokenizer selection sai.")
require(TOKENIZER.metadata["training_documents"] == INPUT_COUNTS["train"], "Training count sai.")
require(TOKENIZER.metadata["split"] == {
    "seed": SPLIT.seed, "validation_fraction": SPLIT.validation_fraction
}, "Tokenizer split khác corpus.")

probe_ids = TOKENIZER.encode("Tokenization check: tiếng Việt.\n")
require(
    bool(probe_ids) and all(type(i) is int and 0 <= i < 16000 for i in probe_ids),
    "Probe token IDs không hợp lệ.",
)

# Chỉ đọc implementation; không khởi tạo Language Model.
forward_source = inspect.getsource(MoELanguageModel.forward)
require(
    "logits[:, :-1, :]" in forward_source and "labels[:, 1:]" in forward_source,
    "Causal shift implementation khác convention đã kiểm tra.",
)
require(AoELanguageModel.forward is MoELanguageModel.forward, "AoE forward khác dự kiến.")

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
TRACKED_SHA = {
    **{str(path): SOURCE_SHA[path.name] for path in [*SOURCES.values(), SUMMARY_PATH]},
    str(TOKENIZER_PATH): file_sha256(TOKENIZER_PATH),
    str(TOKENIZER_PATH.with_suffix(".meta.json")):
        file_sha256(TOKENIZER_PATH.with_suffix(".meta.json")),
}
INPUT_AUDIT_PASS = True

print("Input audit: PASS")
print("Vocab:", TOKENIZER.vocab_size, "| EOS ID:", TOKENIZER.eos_id)
print("Tokenizer SHA-256:", TOKENIZER.fingerprint)
print("Documents:", dict(INPUT_COUNTS), "| Overlap:", OVERLAP)
print("Corpus fingerprint:", CORPUS_FINGERPRINT)
print("Git commit:", GIT_COMMIT)
print("Storage: uint16; Dataset: torch.long; labels là clone chưa shift.")
print("Model tự causal shift: 512 input tokens -> 511 next-token targets/sequence.")

Input audit: PASS
Vocab: 16000 | EOS ID: 2
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Documents: {'train': 28051, 'validation': 443} | Overlap: 0
Corpus fingerprint: 2df1e251a4fc3d583d8c8396a79590e99f40c5f63574009a9b3009165b192d04
Git commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
Storage: uint16; Dataset: torch.long; labels là clone chưa shift.
Model tự causal shift: 512 input tokens -> 511 next-token targets/sequence.


In [ ]:
TOKEN_COUNT_RESULT = {"status": "RUNNING", "splits": {}}

def length_statistics(lengths):
    values = np.asarray(lengths, dtype=np.int64)
    require(values.size > 0, "Không có document để thống kê.")
    return {
        "min": int(values.min()),
        "median": float(np.median(values)),
        "mean": float(values.mean()),
        "p90": float(np.percentile(values, 90)),
        "p99": float(np.percentile(values, 99)),
        "max": int(values.max()),
    }

try:
    for split, path in SOURCES.items():
        stats = {}
        lengths = []
        by_source = {}
        seen = set()
        ordered = "00" * 32
        natural_eos = Counter()
        current_context = {"split": split, "file": path.name}

        for document in iter_documents(path, stats=stats):
            reference = REFERENCES[split].get(document.content_hash)
            require(reference is not None, "Reader trả về document ngoài corpus đã audit.")
            current_context = {
                "split": split,
                "line": reference["line"],
                "content_hash": document.content_hash,
                "source_category": reference["source_category"],
            }
            require(document.content_hash not in seen, "Reader trả về duplicate.")
            require(SPLIT.assign(document.content_hash) == split, "Reader split sai.")
            seen.add(document.content_hash)

            # Không nối EOS ở đây: đếm text IDs và EOS bổ sung riêng.
            ids = TOKENIZER.encode(document.text)
            require(
                all(type(i) is int and 0 <= i < 16000 for i in ids),
                "Token ID ngoài [0,15999].",
            )
            lengths.append(len(ids))
            ordered = chain_hash(ordered, document.content_hash)

            natural_eos["in_text"] += ids.count(TOKENIZER.eos_id)
            natural_eos["documents_starting_with_eos"] += bool(ids and ids[0] == TOKENIZER.eos_id)
            natural_eos["documents_ending_with_eos"] += bool(ids and ids[-1] == TOKENIZER.eos_id)

            category = reference["source_category"]
            record = by_source.setdefault(category, {
                "documents": 0, "text_tokens": 0,
                "appended_eos_tokens": 0, "total_tokens": 0,
            })
            record["documents"] += 1
            record["text_tokens"] += len(ids)
            record["appended_eos_tokens"] += 1
            record["total_tokens"] += len(ids) + 1

            if len(lengths) % 5000 == 0:
                print(f"{split}: đã encode {len(lengths):,} documents")

        require(stats.get("invalid_documents", 0) == 0, f"Reader bỏ qua records: {stats}")
        require(len(lengths) == INPUT_COUNTS[split], "Document count không khớp.")
        require(len(seen) == len(REFERENCES[split]), "Thiếu documents.")
        if split == "train":
            require(
                ordered == TOKENIZER.metadata["training_content_fingerprint"],
                "Ordered train fingerprint khác dữ liệu fitting tokenizer.",
            )
        require(file_sha256(path) == SOURCE_SHA[path.name], "Corpus thay đổi khi encode.")

        text_tokens = sum(lengths)
        report = {
            "documents": len(lengths),
            "text_tokens": text_tokens,
            "appended_eos_tokens": len(lengths),
            "total_tokens": text_tokens + len(lengths),
            "text_tokens_per_document": length_statistics(lengths),
            "total_tokens_per_document": length_statistics([n + 1 for n in lengths]),
            "natural_eos": dict(natural_eos),
            "by_source_category": by_source,
            "ordered_content_fingerprint": ordered,
        }
        TOKEN_COUNT_RESULT["splits"][split] = report

        print(f"\n{split.upper()}")
        print("Text tokens:", f"{report['text_tokens']:,}")
        print("EOS tokens nối thêm:", f"{report['appended_eos_tokens']:,}")
        print("Total tokens:", f"{report['total_tokens']:,}")
        print("Text tokens/document:", report["text_tokens_per_document"])
        print("Total tokens/document, gồm EOS:", report["total_tokens_per_document"])
        print("EOS xuất hiện sẵn trong text:", report["natural_eos"])
        print("Tokens theo source category:")
        for category, values in sorted(by_source.items()):
            print(" ", category, values)

    TOKEN_COUNT_RESULT["status"] = "PASS"
except BaseException as exc:
    TOKEN_COUNT_RESULT["status"] = "FAIL"
    TOKEN_COUNT_RESULT["error"] = {
        "context": current_context,
        "error_type": type(exc).__name__,
    }
    print("TOKEN COUNT FAIL:", TOKEN_COUNT_RESULT["error"])
    raise

print("\nToken counting: PASS")
print(
    "Total = text tokens + một EOS bổ sung/document. "
    "EOS literal trong text đã nằm trong text tokens; không bị xóa hoặc nối thêm lần hai."
)

train: đã encode 5,000 documents
train: đã encode 10,000 documents
train: đã encode 15,000 documents
train: đã encode 20,000 documents
train: đã encode 25,000 documents

TRAIN
Text tokens: 35,932,963
EOS tokens nối thêm: 28,051
Total tokens: 35,961,014
Text tokens/document: {'min': 1, 'median': 364.0, 'mean': 1280.9868810381092, 'p90': 1931.0, 'p99': 11536.5, 'max': 570030}
Total tokens/document, gồm EOS: {'min': 2, 'median': 365.0, 'mean': 1281.9868810381092, 'p90': 1932.0, 'p99': 11537.5, 'max': 570031}
EOS xuất hiện sẵn trong text: {'in_text': 2, 'documents_starting_with_eos': 0, 'documents_ending_with_eos': 0}
Tokens theo source category:
  books {'documents': 58, 'text_tokens': 5844720, 'appended_eos_tokens': 58, 'total_tokens': 5844778}
  c4 {'documents': 2959, 'text_tokens': 1619822, 'appended_eos_tokens': 2959, 'total_tokens': 1622781}
  cc_en_head {'documents': 7689, 'text_tokens': 6313284, 'appended_eos_tokens': 7689, 'total_tokens': 6320973}
  cc_en_middle {'documents': 6694

In [ ]:
import tempfile
import traceback

SMOKE_CONFIG = {
    "documents_per_split": 8,
    "max_scan_documents_per_split": 256,
    "max_document_text_bytes": 64 * 1024,
    "max_tokens_per_split": 32_768,
}
PACKING_RESULT = {"status": "FAIL", "dataloader_status": "FAIL"}

def packing_smoke_test():
    require(TOKEN_COUNT_RESULT["status"] == "PASS", "Token count chưa PASS.")
    require(2 <= SMOKE_CONFIG["documents_per_split"] <= 32, "Smoke document limit sai.")
    require(
        1024 <= SMOKE_CONFIG["max_tokens_per_split"] <= 65_536,
        "Smoke token limit phải nằm trong [1024,65536].",
    )
    require(
        0 < SMOKE_CONFIG["max_document_text_bytes"] <= 128 * 1024,
        "Smoke document byte limit quá lớn.",
    )

    selected = {"train": [], "validation": []}
    expected = {"train": [], "validation": []}
    boundaries = {"train": [], "validation": []}
    skipped = {"train": Counter(), "validation": Counter()}

    for split, path in SOURCES.items():
        for document in iter_documents(
            path, max_documents=SMOKE_CONFIG["max_scan_documents_per_split"]
        ):
            require(SPLIT.assign(document.content_hash) == split, "Smoke split sai.")
            if len(document.text.encode("utf-8")) > SMOKE_CONFIG["max_document_text_bytes"]:
                skipped[split]["document_too_large"] += 1
                continue
            ids = TOKENIZER.encode(document.text)

            # Chỉ giới hạn fixture smoke: chọn text không chứa EOS literal,
            # để kiểm tra chính xác một EOS tại mỗi document boundary.
            if TOKENIZER.eos_id in ids:
                skipped[split]["natural_eos_in_text"] += 1
                continue
            if len(expected[split]) + len(ids) + 1 > SMOKE_CONFIG["max_tokens_per_split"]:
                skipped[split]["would_exceed_smoke_token_cap"] += 1
                continue

            expected[split].extend(ids)
            boundaries[split].append(len(expected[split]))
            expected[split].append(TOKENIZER.eos_id)
            selected[split].append(document)

            if len(selected[split]) >= SMOKE_CONFIG["documents_per_split"]:
                break

        require(len(selected[split]) >= 2, f"Không đủ documents smoke cho {split}.")
        require(
            len(expected[split]) >= 1024,
            f"{split} chưa đủ 2 sequences; tăng documents_per_split trong giới hạn.",
        )
        require(
            expected[split].count(TOKENIZER.eos_id) == len(selected[split]),
            "EOS insertion count sai.",
        )

    train_hashes = {d.content_hash for d in selected["train"]}
    validation_hashes = {d.content_hash for d in selected["validation"]}
    require(not train_hashes.intersection(validation_hashes), "Smoke overlap.")

    with tempfile.TemporaryDirectory(prefix="moe-packing-512-audit-") as temporary:
        root = Path(temporary)
        fixture_paths = [root / "part-0.jsonl", root / "part-1.jsonl"]

        # Hai nguồn nhỏ để kiểm tra carry-over phần dư giữa các nguồn.
        for part, path in enumerate(fixture_paths):
            with path.open("x", encoding="utf-8") as stream:
                for split in ("train", "validation"):
                    documents = selected[split]
                    midpoint = len(documents) // 2
                    subset = documents[:midpoint] if part == 0 else documents[midpoint:]
                    for document in subset:
                        stream.write(json.dumps(
                            {"text": document.text}, ensure_ascii=False
                        ) + "\n")

        packed_root = root / "packed"
        state = prepare_dataset(
            [str(path) for path in fixture_paths],
            TOKENIZER,
            packed_root,
            config=PreprocessingConfig(
                train_max_tokens=len(expected["train"]),
                validation_max_tokens=len(expected["validation"]),
                sequence_length=512,
                split=SPLIT,
            ),
            downloads=DownloadOptions(allow_download=False),
            provenance={"purpose": "STEP 6.4.1 bounded packing smoke test"},
        )
        validate_dataset(packed_root, tokenizer=TOKENIZER, checksums=True)
        split_audit = audit_document_splits(packed_root)
        require(split_audit["overlap"] == 0, "Packed dataset overlap.")

        sequences = {}
        for split in ("train", "validation"):
            info = state["splits"][split]
            actual = []
            for segment in info["segments"]:
                actual.extend(np.fromfile(
                    packed_root / segment["file"], dtype="<u2"
                ).tolist())

            packed_length = len(expected[split]) // 512 * 512
            require(actual == expected[split][:packed_length], "Packed token stream sai.")
            require(info["budget_discarded_tokens"] == 0, "Smoke đã truncate theo budget.")
            require(
                info["tail_discarded_tokens"] == len(expected[split]) % 512,
                "Tail accounting sai.",
            )
            require(info["documents"] == len(selected[split]), "Smoke document count sai.")
            require(split_audit[split] == len(selected[split]), "Smoke split ledger sai.")
            require(all(0 <= token < 16000 for token in actual), "Packed IDs sai.")

            retained_boundaries = [
                position for position in boundaries[split][:-1]
                if position + 1 < packed_length
            ]
            require(bool(retained_boundaries), "Chưa có inter-document boundary được lưu.")
            for position in retained_boundaries:
                require(actual[position] == TOKENIZER.eos_id, "Thiếu EOS separator.")
                require(
                    actual[position - 1] != TOKENIZER.eos_id
                    and actual[position + 1] != TOKENIZER.eos_id,
                    "Có EOS separator liên tiếp trong smoke fixture.",
                )

            dataset = PackedTokenDataset(
                packed_root, split, tokenizer=TOKENIZER, verify_checksums=True
            )
            try:
                require(dataset.sequence_length == 512 and len(dataset) >= 2, "Dataset shape sai.")
                sample = dataset[0]
                require(sample["input_ids"].dtype == torch.long, "Input dtype sai.")
                require(sample["labels"].dtype == torch.long, "Label dtype sai.")
                require(tuple(sample["input_ids"].shape) == (512,), "Sample shape sai.")
                require(torch.equal(sample["input_ids"], sample["labels"]), "Labels bị shift.")
                require(
                    sample["input_ids"].data_ptr() != sample["labels"].data_ptr(),
                    "Labels không phải clone độc lập.",
                )

                loader = make_dataloader(
                    dataset, batch_size=2, seed=42, num_workers=0,
                    pin_memory=False, drop_last=False,
                )
                batch = next(iter(loader))
                require(tuple(batch["input_ids"].shape) == (2, 512), "Batch shape sai.")
                require(batch["input_ids"].dtype == batch["labels"].dtype == torch.long, "Batch dtype sai.")
                require(torch.equal(batch["input_ids"], batch["labels"]), "Batch labels sai.")
                require(
                    int(batch["input_ids"].min()) >= 0
                    and int(batch["input_ids"].max()) < 16000,
                    "Batch IDs ngoài vocabulary.",
                )
                sequences[split] = len(dataset)
            finally:
                dataset.close()

    return {
        "status": "PASS",
        "dataloader_status": "PASS",
        "config": SMOKE_CONFIG,
        "documents": {split: len(values) for split, values in selected.items()},
        "packed_sequences": sequences,
        "skipped_for_smoke_only": {
            split: dict(values) for split, values in skipped.items()
        },
        "eos_boundary_check": "PASS",
        "cross_source_tail_carry": "PASS",
        "labels_convention": "unshifted clone; model shifts internally",
        "batch_shape": [2, 512],
        "batch_dtype": "torch.long",
    }

try:
    PACKING_RESULT = packing_smoke_test()
except Exception as exc:
    PACKING_RESULT = {
        "status": "FAIL",
        "dataloader_status": "FAIL",
        "error": f"{type(exc).__name__}: {exc}",
    }
    traceback.print_exc()

print(json.dumps(PACKING_RESULT, ensure_ascii=False, indent=2))
print("Các filter trên chỉ dành cho smoke fixture, không sửa hoặc lọc lại corpus gốc.")

{
  "status": "PASS",
  "dataloader_status": "PASS",
  "config": {
    "documents_per_split": 8,
    "max_scan_documents_per_split": 256,
    "max_document_text_bytes": 65536,
    "max_tokens_per_split": 32768
  },
  "documents": {
    "train": 8,
    "validation": 8
  },
  "packed_sequences": {
    "train": 11,
    "validation": 15
  },
  "skipped_for_smoke_only": {
    "train": {},
    "validation": {}
  },
  "eos_boundary_check": "PASS",
  "cross_source_tail_carry": "PASS",
  "labels_convention": "unshifted clone; model shifts internally",
  "batch_shape": [
    2,
    512
  ],
  "batch_dtype": "torch.long"
}
Các filter trên chỉ dành cho smoke fixture, không sửa hoặc lọc lại corpus gốc.


In [ ]:
BUDGET_REPORT = None

if TOKEN_COUNT_RESULT["status"] != "PASS":
    print("Không tính budget: token counting chưa PASS.")
else:
    packing_estimates = {}
    for split, report in TOKEN_COUNT_RESULT["splits"].items():
        total = report["total_tokens"]
        sequences, tail = divmod(total, SEQUENCE_LENGTH)
        stored = sequences * SEQUENCE_LENGTH
        packing_estimates[split] = {
            "stream_tokens_with_appended_eos": total,
            "packed_sequences": sequences,
            "tail_discarded_tokens": tail,
            "stored_input_tokens": stored,
            "effective_next_token_targets": sequences * (SEQUENCE_LENGTH - 1),
            "boundary_targets_lost_vs_contiguous_stream": (
                sequences - 1 if sequences else 0
            ),
        }

    train = packing_estimates["train"]
    train_counts = TOKEN_COUNT_RESULT["splits"]["train"]
    available = train["stored_input_tokens"]
    mean_document_tokens = train_counts["total_tokens"] / train_counts["documents"]
    budget_rows = []

    print("PACKING TOÀN CORPUS — suy ra từ counts, chưa tạo full dataset")
    for split, values in packing_estimates.items():
        print(split, values)

    print("\nBUDGETS — tính theo packed input tokens, một lượt qua corpus")
    for budget in (1_000_000, 5_000_000, 50_000_000, 200_000_000):
        required_sequences = (budget + 511) // 512
        required_stream = required_sequences * 512
        additional_stream = max(
            0, required_stream - train["stream_tokens_with_appended_eos"]
        )
        estimated_docs = (
            int(np.ceil(additional_stream / mean_document_tokens))
            if additional_stream else 0
        )
        row = {
            "budget_input_tokens": budget,
            "coverage_percent": 100 * available / budget,
            "enough_single_pass": available >= budget,
            "minimum_additional_stream_tokens": additional_stream,
            "estimated_additional_documents": estimated_docs,
        }
        budget_rows.append(row)
        print(
            f"{budget:,}: coverage={row['coverage_percent']:.2f}% | "
            f"enough={row['enough_single_pass']} | "
            f"additional stream tokens>={additional_stream:,} | "
            f"estimated docs≈{estimated_docs:,}"
        )

    BUDGET_REPORT = {
        "basis": "packed input tokens, decimal budgets, single corpus pass",
        "packing": packing_estimates,
        "budgets": budget_rows,
        "document_estimate_basis": "current mean tokens/document including one appended EOS",
    }

    if available < 50_000_000:
        print("\nCorpus hiện tại KHÔNG đủ 50M packed input tokens trong một lượt.")
    print(
        "511 targets/sequence giả định labels không có -100 và không mask thêm EOS. "
        "Token đầu mỗi sequence không là target; logit cuối không được dùng."
    )
    print(
        "EOS là separator, không phải attention barrier: pipeline hiện tại vẫn cho phép "
        "causal attention qua document boundary trong cùng sequence."
    )
    print(
        "Ước lượng documents bổ sung có độ bất định theo nguồn và độ dài documents. "
        "Không suy ra số full Dolma shards từ sample prefix."
    )
    print("Không lặp corpus, tải thêm dữ liệu hoặc quy đổi thành optimizer steps.")

PACKING TOÀN CORPUS — suy ra từ counts, chưa tạo full dataset
train {'stream_tokens_with_appended_eos': 35961014, 'packed_sequences': 70236, 'tail_discarded_tokens': 182, 'stored_input_tokens': 35960832, 'effective_next_token_targets': 35890596, 'boundary_targets_lost_vs_contiguous_stream': 70235}
validation {'stream_tokens_with_appended_eos': 417119, 'packed_sequences': 814, 'tail_discarded_tokens': 351, 'stored_input_tokens': 416768, 'effective_next_token_targets': 415954, 'boundary_targets_lost_vs_contiguous_stream': 813}

BUDGETS — tính theo packed input tokens, một lượt qua corpus
1,000,000: coverage=3596.08% | enough=True | additional stream tokens>=0 | estimated docs≈0
5,000,000: coverage=719.22% | enough=True | additional stream tokens>=0 | estimated docs≈0
50,000,000: coverage=71.92% | enough=False | additional stream tokens>=14,039,370 | estimated docs≈10,952
200,000,000: coverage=17.98% | enough=False | additional stream tokens>=164,038,986 | estimated docs≈127,957

Corpus h

In [ ]:
from datetime import datetime, timezone
import uuid

input_files_unchanged = True
changed_files = []
for path, expected_sha in TRACKED_SHA.items():
    if not Path(path).is_file() or file_sha256(path) != expected_sha:
        input_files_unchanged = False
        changed_files.append(path)

checks = {
    "input_audit": INPUT_AUDIT_PASS,
    "token_count": TOKEN_COUNT_RESULT["status"] == "PASS",
    "packing_smoke": PACKING_RESULT.get("status") == "PASS",
    "dataloader_smoke": PACKING_RESULT.get("dataloader_status") == "PASS",
    "budget_computation": BUDGET_REPORT is not None,
    "input_files_unchanged": input_files_unchanged,
}
status = "PASS" if all(checks.values()) else "FAIL"

readiness = {}
if BUDGET_REPORT is not None:
    readiness = {
        str(row["budget_input_tokens"]): row["enough_single_pass"]
        for row in BUDGET_REPORT["budgets"]
    }

audit = {
    "format_version": 1,
    "step": "6.4.1",
    "status": status,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "checks": checks,
    "dataset_version": PLAN["version"],
    "dataset_seed": SPLIT.seed,
    "corpus_fingerprint": CORPUS_FINGERPRINT,
    "plan_sha256": PLAN_SHA,
    "selection_sha256": SELECTION_SHA,
    "tokenizer_sha256": TOKENIZER.fingerprint,
    "tokenizer_vocab_size": TOKENIZER.vocab_size,
    "eos_token_id": TOKENIZER.eos_id,
    "git_commit": GIT_COMMIT,
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "numpy": np.__version__,
        "tokenizers": tokenizers.__version__,
    },
    "sequence_length": SEQUENCE_LENGTH,
    "eos_convention": {
        "automatic_bos_eos": False,
        "pipeline": "encode(document.text) + [tokenizer.eos_id]",
        "literal_eos_in_text": "preserved and counted as text tokens",
    },
    "packing_convention": {
        "storage_dtype": "<u2",
        "dataset_dtype": "torch.long",
        "carry_tail_between_sources": True,
        "final_tail": "discard incomplete sequence per split",
        "labels": "input_ids.clone(), unshifted",
        "model_shift": "logits[:, :-1, :] predicts labels[:, 1:]",
        "targets_per_full_sequence_without_ignore_mask": 511,
        "full_corpus_packing_performed": False,
    },
    "document_counts": dict(INPUT_COUNTS),
    "train_validation_overlap": OVERLAP,
    "token_counts": TOKEN_COUNT_RESULT,
    "packing_smoke_test": PACKING_RESULT,
    "budget_analysis": BUDGET_REPORT,
    "readiness_single_pass": readiness,
    "input_file_sha256": TRACKED_SHA,
    "changed_files": changed_files,
    "limitations": [
        "Corpus consists of bounded shard prefixes; sampling bias remains.",
        "Source categories do not prove representative token proportions.",
        "Full-corpus packing counts are derived, not materialized.",
        "Budget readiness counts one pass and does not imply optimizer steps.",
        "EOS does not prevent attention across document boundaries.",
    ],
}

audit_dir = (
    PROJECT / "data/metadata/tokenization_packing_512"
    / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:12])
)
audit_dir.mkdir(parents=True, exist_ok=False)
audit_path = audit_dir / "audit.json"
with audit_path.open("x", encoding="utf-8") as stream:
    json.dump(audit, stream, ensure_ascii=False, sort_keys=True, indent=2)
    stream.write("\n")

print("Tokenizer vocab:", TOKENIZER.vocab_size)
print("EOS ID:", TOKENIZER.eos_id)
print("Tokenizer SHA-256:", TOKENIZER.fingerprint)
print("Train documents:", INPUT_COUNTS["train"])
print("Validation documents:", INPUT_COUNTS["validation"])

for split in ("train", "validation"):
    counts = TOKEN_COUNT_RESULT["splits"].get(split, {})
    print(f"{split} total tokens:", counts.get("total_tokens", "UNVERIFIED"))
    if BUDGET_REPORT is not None:
        packed = BUDGET_REPORT["packing"][split]
        print(f"{split} packed sequences:", packed["packed_sequences"])
        print(f"{split} effective next-token targets:", packed["effective_next_token_targets"])

print("Train/validation overlap:", OVERLAP)
print("Packing smoke:", PACKING_RESULT.get("status"))
print("DataLoader smoke:", PACKING_RESULT.get("dataloader_status"))
print("Đủ debug 1M input tokens:", readiness.get("1000000", "UNVERIFIED"))
print("Đủ debug 5M input tokens:", readiness.get("5000000", "UNVERIFIED"))
print("Đủ pilot 50M input tokens:", readiness.get("50000000", "UNVERIFIED"))
print("Đủ main 200M input tokens:", readiness.get("200000000", "UNVERIFIED"))
print("Audit:", audit_path)
print("Audit SHA-256:", file_sha256(audit_path))
print("Hạn chế: prefix sampling; độ phủ nguồn không bảo đảm phân phối đại diện.")
print("PASS chỉ xác nhận tokenization/packing, không bảo đảm đủ token budget.")
print(f"STEP 6.4.1 STATUS: {status}")

Tokenizer vocab: 16000
EOS ID: 2
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Train documents: 28051
Validation documents: 443
train total tokens: 35961014
train packed sequences: 70236
train effective next-token targets: 35890596
validation total tokens: 417119
validation packed sequences: 814
validation effective next-token targets: 415954
Train/validation overlap: 0
Packing smoke: PASS
DataLoader smoke: PASS
Đủ debug 1M input tokens: True
Đủ debug 5M input tokens: True
Đủ pilot 50M input tokens: False
Đủ main 200M input tokens: False
Audit: /content/moe-erc/data/metadata/tokenization_packing_512/20261024T054122Z-21ea9f3dfffc/audit.json
Audit SHA-256: 492f137031ff3a33db7df0784e3818b5d14c439501a607fc89b19687ca4bedb4
Hạn chế: prefix sampling; độ phủ nguồn không bảo đảm phân phối đại diện.
PASS chỉ xác nhận tokenization/packing, không bảo đảm đủ token budget.
STEP 6.4.1 STATUS: PASS


**STEP 6.4.2A — MỞ RỘNG DOLMA DATASET ĐẾN ÍT NHẤT 55M TRAIN TOKENS**

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import closing
from datetime import datetime, timezone
import hashlib
import json
import os
import sqlite3
import subprocess
import sys
import uuid

PROJECT = Path("/content/moe-erc")
BASE = PROJECT / "data/tokenizer_corpora/dolma-bpe16k-2412fe19bc1a0d22"
BASE_EXPORT = BASE / "exports/f46e687c4faf4aad97007a76dc8431dd"
BASE_SOURCES = {
    "train": BASE_EXPORT / "train.jsonl",
    "validation": BASE_EXPORT / "validation.jsonl",
}
SUMMARY_PATH = BASE_EXPORT / "summary.json"
TOKENIZER_PATH = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"
AUDIT_PATH = (
    PROJECT / "data/metadata/tokenization_packing_512/"
    "20261024T054122Z-21ea9f3dfffc/audit.json"
)
EXPECTED_AUDIT_SHA = "492f137031ff3a33db7df0784e3818b5d14c439501a607fc89b19687ca4bedb4"
EXPECTED_TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

def write_json_new(path, value):
    with Path(path).open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, sort_keys=True, indent=2)
        stream.write("\n")

require(PROJECT.is_dir(), "Thiếu project.")
for path in [AUDIT_PATH, SUMMARY_PATH, TOKENIZER_PATH, *BASE_SOURCES.values()]:
    require(path.is_file(), f"Thiếu file: {path}")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import tokenizers
from moe_erc.dolma_reader import (
    canonical_text, file_sha256, iter_documents, parse_manifest, select_shards,
)
from moe_erc.tokenizer_utils import FrozenTokenizer, SplitConfig, SPLIT_ALGORITHM
from moe_erc.data_processing import _Packer  # implementation thực tế của STEP 5

require(file_sha256(AUDIT_PATH) == EXPECTED_AUDIT_SHA, "Audit SHA-256 FAIL.")
require(file_sha256(TOKENIZER_PATH) == EXPECTED_TOKENIZER_SHA, "Tokenizer SHA-256 FAIL.")

AUDIT = read_json(AUDIT_PATH)
SUMMARY = read_json(SUMMARY_PATH)
OLD_PLAN = SUMMARY["plan"]
OLD_PLAN_SHA = SUMMARY["plan_sha256"]
OLD_SELECTION_SHA = SUMMARY["selection_sha256"]

require(AUDIT["status"] == "PASS", "Audit cũ chưa PASS.")
require(fingerprint(OLD_PLAN) == OLD_PLAN_SHA == AUDIT["plan_sha256"], "Plan sai.")
require(
    fingerprint(OLD_PLAN["selected_urls"]) == OLD_SELECTION_SHA == AUDIT["selection_sha256"],
    "Selection sai.",
)

TOKENIZER = FrozenTokenizer(TOKENIZER_PATH)
require(TOKENIZER.fingerprint == EXPECTED_TOKENIZER_SHA, "Tokenizer wrapper checksum sai.")
require(TOKENIZER.vocab_size == 16000 and TOKENIZER.eos_id == 2, "Vocab/EOS sai.")
require(not TOKENIZER.metadata["test_only"], "Không dùng test tokenizer.")

SPLIT = SplitConfig(
    seed=OLD_PLAN["config"]["seed"],
    validation_fraction=OLD_PLAN["config"]["validation_fraction"],
)
require(SPLIT.seed == 42 and OLD_PLAN["split_algorithm"] == SPLIT_ALGORITHM, "Split sai.")
require(TOKENIZER.metadata["split"] == {
    "seed": SPLIT.seed, "validation_fraction": SPLIT.validation_fraction
}, "Tokenizer split khác corpus.")

for name, path in BASE_SOURCES.items():
    require(
        file_sha256(path) == SUMMARY["files"][path.name]["sha256"],
        f"Corpus checksum FAIL: {name}",
    )

for name, expected in AUDIT["input_file_sha256"].items():
    path = Path(name)
    require(path.resolve().is_relative_to(PROJECT.resolve()), "Audit source ngoài project.")
    require(path.is_file() and file_sha256(path) == expected, f"Audit input đổi: {path}")

base_file_sha = {
    path.name: file_sha256(path)
    for path in [*BASE_SOURCES.values(), SUMMARY_PATH]
}
ORIGINAL_CORPUS_FP = fingerprint({
    "files": base_file_sha,
    "plan_sha256": OLD_PLAN_SHA,
    "selection_sha256": OLD_SELECTION_SHA,
})
require(
    ORIGINAL_CORPUS_FP == AUDIT["corpus_fingerprint"]
    == TOKENIZER.metadata["provenance"]["corpus_fingerprint"],
    "Tokenizer/audit không thuộc corpus hiện tại.",
)

OLD_TOKENS = {
    split: AUDIT["token_counts"]["splits"][split]["total_tokens"]
    for split in BASE_SOURCES
}
require(OLD_TOKENS == {"train": 35_961_014, "validation": 417_119}, "Token counts audit sai.")

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
if GIT_COMMIT != AUDIT["git_commit"]:
    print("LƯU Ý: commit hiện tại khác audit:", GIT_COMMIT, AUDIT["git_commit"])

# Guard các file gốc; chỉ đọc, không sửa SQLite hoặc exports cũ.
guard_paths = [
    path for path in BASE.rglob("*") if path.is_file()
] + [AUDIT_PATH, TOKENIZER_PATH, TOKENIZER_PATH.with_suffix(".meta.json")]
require(not any(p.is_symlink() for p in BASE.rglob("*")), "Corpus có symlink.")
ORIGINAL_GUARD = {str(path): file_sha256(path) for path in guard_paths}
CODE_SHA = {
    name: file_sha256(PROJECT / "moe_erc" / name)
    for name in ("dolma_reader.py", "tokenizer_utils.py", "data_processing.py")
}

def verify_originals():
    for name, expected in ORIGINAL_GUARD.items():
        require(Path(name).is_file() and file_sha256(name) == expected, f"Nguồn gốc đổi: {name}")
    for name, expected in CODE_SHA.items():
        require(file_sha256(PROJECT / "moe_erc" / name) == expected, f"Pipeline đổi: {name}")

require(
    Path("/content/drive/MyDrive").is_dir(),
    "Drive chưa mount. Mount /content/drive rồi chạy lại; chưa tiếp tục mở rộng.",
)
audit_backup = (
    Path("/content/drive/MyDrive/MoE_ERC/audit_snapshots")
    / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-" + uuid.uuid4().hex[:12]
    )
)
audit_backup.mkdir(parents=True, exist_ok=False)
with AUDIT_PATH.open("rb") as src, (audit_backup / "audit.json").open("xb") as dst:
    import shutil
    shutil.copyfileobj(src, dst)
require(file_sha256(audit_backup / "audit.json") == EXPECTED_AUDIT_SHA, "Audit backup FAIL.")
write_json_new(audit_backup / "COMPLETE.json", {
    "status": "COMPLETE", "audit_sha256": EXPECTED_AUDIT_SHA,
    "source": str(AUDIT_PATH),
})

print("Environment/artifact checks: PASS")
print("Git commit:", GIT_COMMIT)
print("Tokenizer SHA-256:", TOKENIZER.fingerprint)
print("Original corpus fingerprint:", ORIGINAL_CORPUS_FP)
print("Original token totals:", OLD_TOKENS)
print("Audit backup verified:", audit_backup)

Environment/artifact checks: PASS
Git commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Original corpus fingerprint: 2df1e251a4fc3d583d8c8396a79590e99f40c5f63574009a9b3009165b192d04
Original token totals: {'train': 35961014, 'validation': 417119}
Audit backup verified: /content/drive/MyDrive/MoE_ERC/audit_snapshots/20261024T062826Z-28ad63c20369


In [ ]:
from itertools import zip_longest
from urllib.parse import urlsplit
from pathlib import PurePosixPath

OLD_PLAN_FILE = (
    PROJECT / "data/selections"
    / f"dolma-tokenizer-plan-{OLD_PLAN_SHA[:16]}.json"
)
MANIFEST_FILE = (
    PROJECT / "data/manifests"
    / f"dolma-v1_5-sample-{OLD_PLAN['manifest_sha256']}.json"
)
require(OLD_PLAN_FILE.is_file() and MANIFEST_FILE.is_file(), "Thiếu plan/manifest đã lưu.")
require(fingerprint(read_json(OLD_PLAN_FILE)) == OLD_PLAN_SHA, "Saved plan sai.")
FULL_MANIFEST = read_json(MANIFEST_FILE)
ALL_URLS = FULL_MANIFEST["urls"]
require(parse_manifest("\n".join(ALL_URLS)) == ALL_URLS, "Manifest URLs sai.")
require(
    FULL_MANIFEST["manifest_sha256"] == OLD_PLAN["manifest_sha256"],
    "Manifest fingerprint sai.",
)

OLD_SELECTED = list(OLD_PLAN["selected_urls"])
require(len(OLD_SELECTED) == len(set(OLD_SELECTED)) == 32, "Selection cũ không phải 32 URLs.")
require(set(OLD_SELECTED).issubset(ALL_URLS), "URL cũ ngoài manifest.")

def source_category(url):
    return PurePosixPath(urlsplit(url).path).parent.name or "UNKNOWN"

def id_digest(value):
    require(type(value) in (str, int), "ID phải là string/int, không phải bool.")
    canonical_id = str(value).strip()
    require(bool(canonical_id), "ID rỗng.")
    return hashlib.sha256(canonical_id.encode("utf-8")).hexdigest()

def checked_records(path, split):
    with Path(path).open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            try:
                item = json.loads(line)
                require(isinstance(item, dict), "Record không phải object.")
                require(isinstance(item.get("text"), str), "Text sai kiểu.")
                text = canonical_text(item["text"])
                require(bool(text) and text == item["text"], "Text rỗng/chưa canonical.")
                require(
                    isinstance(item.get("source"), str) and bool(item["source"].strip()),
                    "Source sai.",
                )
                identifier = id_digest(item["id"])
                digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
                require(item["content_hash"] == digest, "Content hash sai.")
                require(item["split"] == SPLIT.assign(digest) == split, "Split sai.")
                require(isinstance(item["source_category"], str), "Category sai.")
                item["source"].encode("utf-8")
            except Exception as exc:
                raise RuntimeError(
                    f"Record lỗi: {Path(path).name}:{line_number}; {type(exc).__name__}"
                ) from None
            yield item, identifier

def verified_documents(path, split):
    """Đối chiếu JSONL metadata với chính reader STEP 5."""
    stats = {}
    for record, document in zip_longest(
        checked_records(path, split), iter_documents(path, stats=stats)
    ):
        require(record is not None and document is not None, "Reader bỏ sót document.")
        item, identifier = record
        require(
            item["content_hash"] == document.content_hash and item["text"] == document.text,
            "Reader và metadata không khớp.",
        )
        yield item, identifier, document
    require(stats.get("invalid_documents", 0) == 0, f"Reader invalid records: {stats}")

OLD_HASHES = {}
OLD_IDS = {}
old_counts = Counter()
old_categories = Counter()
old_used_urls = set()
old_repeated_ids = 0

for split, path in BASE_SOURCES.items():
    for item, identifier, document in verified_documents(path, split):
        require(document.content_hash not in OLD_HASHES, "Duplicate/overlap trong corpus gốc.")
        OLD_HASHES[document.content_hash] = split
        if identifier in OLD_IDS:
            old_repeated_ids += 1
        else:
            OLD_IDS[identifier] = document.content_hash
        require(item["origin_shard_url"] in OLD_SELECTED, "Origin ngoài 32 URL cũ.")
        old_used_urls.add(item["origin_shard_url"])
        old_counts[split] += 1
        old_categories[item["source_category"]] += 1

require(old_counts == {"train": 28051, "validation": 443}, "Original document counts sai.")
require(
    AUDIT["token_counts"]["splits"]["train"]["total_tokens"] == 35_961_014,
    "Original train tokens khác audit.",
)
for path in (OLD_PLAN_FILE, MANIFEST_FILE):
    ORIGINAL_GUARD[str(path)] = file_sha256(path)

print("32 candidates cũ:")
for index, url in enumerate(OLD_SELECTED, 1):
    print(f"{index:02d}. {url}")
print("\nCategories của candidates:", dict(Counter(map(source_category, OLD_SELECTED))))
print("Categories của documents thực tế:", dict(old_categories))
print("URLs thực sự có documents trong corpus:", len(old_used_urls))
print("Content hashes:", len(OLD_HASHES), "| overlap: 0")
print("ID lặp trong corpus cũ:", old_repeated_ids, "(giữ nguyên corpus cũ)")
print("Original train tokens đã được audit:", OLD_TOKENS["train"])
print(
    "Selection mới sẽ loại toàn bộ 32 URL cũ. "
    "New documents bị loại nếu content hash hoặc canonical ID đã có."
)

32 candidates cũ:
01. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0197.json.gz
02. https://olmo-data.org/dolma-v1_5r1/reddit/reddit-v5-dedupe-pii-nsfw-toxic-0047.json.gz
03. https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0130.json.gz
04. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_tail/cc_en_tail-0364.json.gz
05. https://olmo-data.org/dolma-v1_5r1/wiki/en_simple_wiki_v0-0001.json.gz
06. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0288.json.gz
07. https://olmo-data.org/dolma-v1_5r1/pes2o/pes2o_v2-0016.json.gz
08. https://olmo-data.org/dolma-v1_5r1/books/books-0001.json.gz
09. https://olmo-data.org/dolma-v1_5r1/c4/c4-0062.json.gz
10. https://olmo-data.org/dolma-v1_5r1/c4/c4-0013.json.gz
11. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0135.json.gz
12. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_middle/cc_en_middle-0178.json.gz
13. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc

In [ ]:
import random

MiB = 1024**2
GiB = 1024**3

CONFIG = {
    "seed": 42,
    "selection_salt": "STEP-6.4.2A-additional-dolma-55m-v1",
    "target_packed_train_tokens": 55_000_000,
    "max_candidate_shards": 24,
    "max_compressed_per_shard": 8 * MiB,
    "max_compressed_total": 96 * MiB,
    "max_decompressed_per_shard": 32 * MiB,
    "max_decompressed_total": 384 * MiB,
    "max_records_per_shard": 1500,
    "max_jsonl_line_bytes": 4 * MiB,
    "io_chunk_bytes": 64 * 1024,
    "minimum_free_disk_bytes": 8 * GiB,
}
require(1 <= CONFIG["max_candidate_shards"] <= 24, "Candidates phải trong [1,24].")
require(CONFIG["target_packed_train_tokens"] >= 55_000_000, "Target phải ít nhất 55M.")

TARGET_STREAM_TOKENS = (
    (CONFIG["target_packed_train_tokens"] + 511) // 512 * 512
)
selection_seed = int(hashlib.sha256(
    f"{CONFIG['seed']}:{CONFIG['selection_salt']}".encode()
).hexdigest()[:16], 16)

available = sorted(set(ALL_URLS) - set(OLD_SELECTED))
groups = defaultdict(list)
for url in available:
    groups[source_category(url)].append(url)

categories = sorted(groups)
random.Random(selection_seed).shuffle(categories)
CANDIDATES = [
    select_shards(groups[category], max_shards=1, seed=selection_seed)[0]
    for category in categories[:CONFIG["max_candidate_shards"]]
]
remaining_count = CONFIG["max_candidate_shards"] - len(CANDIDATES)
if remaining_count:
    remaining = sorted(set(available) - set(CANDIDATES))
    CANDIDATES.extend(select_shards(
        remaining, max_shards=remaining_count, seed=selection_seed
    ))

require(len(CANDIDATES) == CONFIG["max_candidate_shards"], "Không đủ candidates.")
require(not set(CANDIDATES).intersection(OLD_SELECTED), "Chọn lại URL cũ.")

ADDITIONAL_PLAN = {
    "format_version": 1,
    "dataset": "allenai/dolma",
    "version": "v1_5-sample",
    "config": CONFIG,
    "selection_seed_derived_from_seed_and_salt": selection_seed,
    "selection_algorithm": "category coverage, then seeded global remainder",
    "selected_urls": CANDIDATES,
    "excluded_original_candidates": OLD_SELECTED,
    "original_audit_sha256": EXPECTED_AUDIT_SHA,
    "original_corpus_fingerprint": ORIGINAL_CORPUS_FP,
    "original_train_tokens": OLD_TOKENS["train"],
    "tokenizer_sha256": EXPECTED_TOKENIZER_SHA,
    "manifest_sha256": FULL_MANIFEST["manifest_sha256"],
    "split": {"seed": SPLIT.seed, "validation_fraction": SPLIT.validation_fraction},
    "split_algorithm": SPLIT_ALGORITHM,
    "id_policy": "global SHA256(str(id).strip()); reject new repeated IDs conservatively",
    "record_limit_policy": "all complete JSONL lines, including invalid lines",
    "target_combined_stream_tokens": TARGET_STREAM_TOKENS,
    "git_commit": GIT_COMMIT,
    "pipeline_source_sha256": CODE_SHA,
    "tokenizers_version": tokenizers.__version__,
}
ADDITIONAL_PLAN_SHA = fingerprint(ADDITIONAL_PLAN)
PLAN_FILE = (
    PROJECT / "data/selections"
    / f"dolma-expansion55m-{ADDITIONAL_PLAN_SHA[:16]}.json"
)
PLAN_FILE.parent.mkdir(parents=True, exist_ok=True)
if PLAN_FILE.exists():
    require(read_json(PLAN_FILE) == ADDITIONAL_PLAN, "Existing plan khác; không overwrite.")
else:
    write_json_new(PLAN_FILE, ADDITIONAL_PLAN)

RUN_DIR = PROJECT / "data/corpus_extensions" / f"dolma55m-{ADDITIONAL_PLAN_SHA[:16]}"

print("Additional plan:", PLAN_FILE)
print("Plan SHA-256:", ADDITIONAL_PLAN_SHA)
print("Output riêng:", RUN_DIR)
print("Required combined stream tokens:", f"{TARGET_STREAM_TOKENS:,}")
print("Minimum additional stream tokens:", f"{TARGET_STREAM_TOKENS - OLD_TOKENS['train']:,}")
print("Allocation:", dict(Counter(map(source_category, CANDIDATES))))
for index, url in enumerate(CANDIDATES, 1):
    print(f"{index:02d}. {url}")
print(
    "Đây là bounded prefix sampling, không bảo đảm phân phối đại diện Dolma. "
    "Không tải shard trong cell này."
)

Additional plan: /content/moe-erc/data/selections/dolma-expansion55m-7e884fa883731ef7.json
Plan SHA-256: 7e884fa883731ef797e4345bbfa146262cf0376abbde441ad51b700a789415d3
Output riêng: /content/moe-erc/data/corpus_extensions/dolma55m-7e884fa883731ef7
Required combined stream tokens: 55,000,064
Minimum additional stream tokens: 19,039,050
Allocation: {'c4': 2, 'pes2o': 1, 'wiki': 1, 'books': 1, 'cc_en_head': 6, 'reddit': 1, 'stack': 2, 'cc_en_middle': 6, 'cc_en_tail': 4}
01. https://olmo-data.org/dolma-v1_5r1/c4/c4-0073.json.gz
02. https://olmo-data.org/dolma-v1_5r1/pes2o/pes2o_v2-0019.json.gz
03. https://olmo-data.org/dolma-v1_5r1/wiki/en_simple_wiki_v0-0000.json.gz
04. https://olmo-data.org/dolma-v1_5r1/books/books-0000.json.gz
05. https://olmo-data.org/dolma-v1_5-1913B-sample/cc_en_head/cc_en_head-0089.json.gz
06. https://olmo-data.org/dolma-v1_5r1/reddit/reddit-v5-dedupe-pii-nsfw-toxic-0049.json.gz
07. https://olmo-data.org/dolma-v1_5r1/stack/stack-v4-train-0123.json.gz
08. https://o

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import Request, urlopen
from urllib.error import HTTPError

for key in (
    "max_compressed_per_shard", "max_compressed_total",
    "max_decompressed_per_shard", "max_decompressed_total",
    "max_records_per_shard", "max_jsonl_line_bytes",
    "io_chunk_bytes", "minimum_free_disk_bytes",
):
    require(type(CONFIG[key]) is int and CONFIG[key] > 0, f"Config sai: {key}")
require(CONFIG["io_chunk_bytes"] <= 64 * 1024, "IO chunk tối đa 64 KiB.")
require(fingerprint(ADDITIONAL_PLAN) == ADDITIONAL_PLAN_SHA, "Plan đã thay đổi.")
require(CONFIG == ADDITIONAL_PLAN["config"], "Config thay đổi; chạy lại Cell 3.")

def head_probe(url):
    try:
        request = Request(url, method="HEAD", headers={"User-Agent": "moe-erc-preflight"})
        with urlopen(request, timeout=8) as response:
            value = response.headers.get("Content-Length")
            size = int(value) if value is not None else None
            if size is not None and size < 0:
                size = None
            return {"status": response.status, "bytes": size, "error": None}
    except HTTPError as exc:
        status = exc.code
        exc.close()
        return {"status": status, "bytes": None, "error": "HEAD rejected"}
    except Exception as exc:
        return {"status": None, "bytes": None, "error": type(exc).__name__}

with ThreadPoolExecutor(max_workers=4) as pool:
    probes = list(pool.map(head_probe, CANDIDATES))
HEAD_RESULTS = dict(zip(CANDIDATES, probes))

for index, (url, result) in enumerate(HEAD_RESULTS.items(), 1):
    size = result["bytes"]
    print(
        f"{index:02d}. {source_category(url):30s} "
        f"HTTP={result['status']} "
        f"size={'UNKNOWN' if size is None else f'{size / MiB:.2f} MiB'} "
        f"error={result['error']}"
    )

free = shutil.disk_usage(PROJECT).free
print(f"\nLocal free: {free / GiB:.2f} GiB")
print("Candidates:", len(CANDIDATES))
print("Compressed: per shard", CONFIG["max_compressed_per_shard"] / MiB,
      "MiB; total", CONFIG["max_compressed_total"] / MiB, "MiB")
print("Decompressed: per shard", CONFIG["max_decompressed_per_shard"] / MiB,
      "MiB; total", CONFIG["max_decompressed_total"] / MiB, "MiB")
print("Complete JSONL records/shard:", CONFIG["max_records_per_shard"])
print("UNKNOWN sizes:", sum(p["bytes"] is None for p in probes))
print("HEAD thất bại không chứng minh GET thất bại; GET chưa được thực hiện.")

PREFLIGHT_OK = free >= CONFIG["minimum_free_disk_bytes"]
require(PREFLIGHT_OK, "Không đủ dung lượng trống.")
print("Preflight: PASS")
print(
    "Compressed limits đo HTTP body mà ứng dụng đọc, không phải TLS/header/network "
    "buffering. Reserve được checkpoint trước mỗi read/decompress để giữ budget "
    "khi có gián đoạn."
)

01. c4                             HTTP=200 size=3859.87 MiB error=None
02. pes2o                          HTTP=200 size=4113.14 MiB error=None
03. wiki                           HTTP=200 size=4040.32 MiB error=None
04. books                          HTTP=200 size=3184.10 MiB error=None
05. cc_en_head                     HTTP=200 size=1169.58 MiB error=None
06. reddit                         HTTP=200 size=2025.56 MiB error=None
07. stack                          HTTP=200 size=1886.79 MiB error=None
08. cc_en_middle                   HTTP=200 size=1110.51 MiB error=None
09. cc_en_tail                     HTTP=200 size=1471.36 MiB error=None
10. cc_en_head                     HTTP=200 size=1553.04 MiB error=None
11. cc_en_head                     HTTP=200 size=1096.73 MiB error=None
12. cc_en_middle                   HTTP=200 size=1408.09 MiB error=None
13. cc_en_tail                     HTTP=200 size=1602.50 MiB error=None
14. c4                             HTTP=200 size=3862.69 MiB err

In [ ]:
RUN_DOWNLOAD = True

import fcntl
import http.client
import re
import traceback
import zlib
from urllib.parse import urljoin

class StopShard(Exception):
    pass

def open_prefix(url, limit):
    original_scheme = urlsplit(url).scheme
    for _ in range(4):
        parsed = urlsplit(url)
        require(
            parsed.scheme in ("http", "https") and parsed.hostname
            and not parsed.username and not parsed.password
            and not (original_scheme == "https" and parsed.scheme != "https"),
            "URL/redirect không được phép.",
        )
        cls = http.client.HTTPSConnection if parsed.scheme == "https" else http.client.HTTPConnection
        connection = cls(parsed.hostname, port=parsed.port, timeout=10)
        try:
            connection.connect()
            connection.sock.settimeout(15)
            target = parsed.path or "/"
            if parsed.query:
                target += "?" + parsed.query
            connection.request("GET", target, headers={
                "Range": f"bytes=0-{limit - 1}",
                "Accept-Encoding": "identity",
                "User-Agent": "moe-erc-bounded-expansion",
                "Connection": "close",
            })
            response = connection.getresponse()
            if response.status in (301, 302, 303, 307, 308):
                location = response.getheader("Location")
                require(bool(location), "Redirect thiếu Location.")
                next_url = urljoin(url, location)
                response.close()
                connection.close()
                url = next_url
                continue
            require(response.status in (200, 206), f"HTTP {response.status}")
            require(
                response.getheader("Content-Encoding", "identity").lower() == "identity",
                "HTTP Content-Encoding không được phép.",
            )
            if response.status == 206:
                match = re.fullmatch(
                    r"bytes 0-(\d+)/(\d+|\*)", response.getheader("Content-Range", "")
                )
                require(match is not None and int(match.group(1)) < limit, "Content-Range sai.")
            return connection, response, url
        except BaseException:
            connection.close()
            raise
    raise RuntimeError("Quá nhiều redirects.")

def build_extension():
    require(PREFLIGHT_OK, "Preflight chưa PASS.")
    require(fingerprint(ADDITIONAL_PLAN) == ADDITIONAL_PLAN_SHA, "Plan thay đổi.")
    require(CONFIG == ADDITIONAL_PLAN["config"], "Config thay đổi.")
    verify_originals()
    require(
        shutil.disk_usage(PROJECT).free >= CONFIG["minimum_free_disk_bytes"],
        "Không đủ disk.",
    )

    database_path = RUN_DIR / "extension.sqlite"
    if RUN_DIR.exists() and not database_path.exists():
        require(not any(RUN_DIR.iterdir()), "Output tồn tại nhưng không có checkpoint hợp lệ.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    with (RUN_DIR / "run.lock").open("a+b") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        is_new = not database_path.exists()
        db = sqlite3.connect(database_path)
        db.execute("PRAGMA synchronous=FULL")

        if is_new:
            db.execute("CREATE TABLE checkpoint (id INTEGER PRIMARY KEY, payload TEXT NOT NULL)")
            db.execute("""
                CREATE TABLE documents (
                    hash TEXT PRIMARY KEY,
                    id_hash TEXT UNIQUE NOT NULL,
                    split TEXT NOT NULL,
                    category TEXT NOT NULL,
                    tokens INTEGER NOT NULL,
                    payload TEXT NOT NULL
                )
            """)
            state = {
                "plan_sha256": ADDITIONAL_PLAN_SHA,
                "new_tokens": {"train": 0, "validation": 0},
                "new_documents": {"train": 0, "validation": 0},
                "shards": [],
                "stop_reason": None,
            }
        else:
            row = db.execute("SELECT payload FROM checkpoint WHERE id=1").fetchone()
            require(row is not None, "Checkpoint thiếu; không tự sửa database.")
            state = json.loads(row[0])
            require(state["plan_sha256"] == ADDITIONAL_PLAN_SHA, "Checkpoint thuộc plan khác.")

        def save():
            try:
                db.execute(
                    "INSERT OR REPLACE INTO checkpoint VALUES (1,?)",
                    (json.dumps(state, ensure_ascii=False, sort_keys=True),),
                )
                db.commit()
            except BaseException:
                db.rollback()
                raise

        def total(key):
            return sum(record[key] for record in state["shards"])

        def rejection(record, reason):
            record["invalid"] += 1
            record["invalid_reasons"][reason] = record["invalid_reasons"].get(reason, 0) + 1

        def handle_line(raw, record):
            record["lines"] += 1
            try:
                decoded = raw.decode("utf-8")
            except UnicodeDecodeError:
                rejection(record, "invalid_utf8")
                save()
                return
            try:
                item = json.loads(decoded)
            except ValueError:
                rejection(record, "invalid_json")
                save()
                return
            record["json_parsed"] += 1

            reasons = []
            if not isinstance(item, dict):
                reasons.append("record_not_object")
            else:
                for field in ("id", "text", "source"):
                    if field not in item:
                        reasons.append("missing_" + field)
                    elif field == "id":
                        if type(item[field]) not in (str, int) or not str(item[field]).strip():
                            reasons.append("invalid_id")
                    elif not isinstance(item[field], str) or not item[field].strip():
                        reasons.append("invalid_or_empty_" + field)

            if not reasons:
                try:
                    text = canonical_text(item["text"])
                    require(bool(text), "empty_text")
                    encoded = text.encode("utf-8")
                    identifier = id_digest(item["id"])
                    item["source"].encode("utf-8")
                except Exception:
                    reasons.append("invalid_unicode_or_empty_canonical_text")

            if reasons:
                record["invalid"] += 1
                for reason in reasons:
                    record["invalid_reasons"][reason] = record["invalid_reasons"].get(reason, 0) + 1
                save()
                return

            record["valid"] += 1
            digest = hashlib.sha256(encoded).hexdigest()
            if digest in OLD_HASHES or db.execute(
                "SELECT 1 FROM documents WHERE hash=?", (digest,)
            ).fetchone():
                record["content_duplicates"] += 1
                save()
                return
            if identifier in OLD_IDS or db.execute(
                "SELECT 1 FROM documents WHERE id_hash=?", (identifier,)
            ).fetchone():
                record["id_conflicts_rejected"] += 1
                save()
                return

            split = SPLIT.assign(digest)
            try:
                ids = TOKENIZER.encode(text)
                require(all(type(i) is int and 0 <= i < 16000 for i in ids), "Invalid token ID.")
            except Exception as exc:
                raise RuntimeError(
                    f"Tokenization failed: hash={digest}, category={record['category']}, "
                    f"error_type={type(exc).__name__}"
                ) from None
            tokens = len(ids) + 1  # Chính xác một EOS bổ sung; không sửa EOS trong text.
            output = {
                "id": item["id"], "text": text, "source": item["source"],
                "source_category": record["category"],
                "origin_shard_url": record["url"],
                "origin_line_number": record["lines"],
                "content_hash": digest, "split": split,
                "partial_streaming": True,
            }
            db.execute(
                "INSERT INTO documents VALUES (?,?,?,?,?,?)",
                (digest, identifier, split, record["category"], tokens,
                 json.dumps(output, ensure_ascii=False)),
            )
            state["new_tokens"][split] += tokens
            state["new_documents"][split] += 1
            save()  # Document và counters commit trong cùng transaction.

        for record in state["shards"]:
            if record["status"] == "running":
                record.update(status="interrupted", stop_reason="previous_execution_interrupted")
        save()

        try:
            attempted = {record["url"] for record in state["shards"]}
            state["stop_reason"] = None
            for url in CANDIDATES:
                if url in attempted:
                    continue
                if OLD_TOKENS["train"] + state["new_tokens"]["train"] >= TARGET_STREAM_TOKENS:
                    state["stop_reason"] = "target_combined_stream_tokens"
                    break
                if total("compressed_charged") >= CONFIG["max_compressed_total"]:
                    state["stop_reason"] = "max_compressed_total"
                    break
                if total("decompressed_charged") >= CONFIG["max_decompressed_total"]:
                    state["stop_reason"] = "max_decompressed_total"
                    break
                if shutil.disk_usage(PROJECT).free < CONFIG["minimum_free_disk_bytes"]:
                    state["stop_reason"] = "minimum_free_disk_bytes"
                    break

                record = {
                    "url": url, "category": source_category(url), "status": "running",
                    "stop_reason": None, "lines": 0, "json_parsed": 0, "valid": 0,
                    "invalid": 0, "invalid_reasons": {},
                    "content_duplicates": 0, "id_conflicts_rejected": 0,
                    "compressed_read": 0, "compressed_charged": 0,
                    "decompressed_produced": 0, "decompressed_charged": 0,
                    "gzip_members_completed": 0,
                }
                state["shards"].append(record)
                save()  # Đánh dấu attempted trước GET.
                connection = response = None
                buffer = bytearray()
                oversized = False
                tail_bytes = 0

                def consume(output):
                    nonlocal buffer, oversized, tail_bytes
                    pieces = output.split(b"\n")
                    for index, piece in enumerate(pieces):
                        tail_bytes += len(piece)
                        if not oversized:
                            if len(buffer) + len(piece) > CONFIG["max_jsonl_line_bytes"]:
                                buffer.clear()
                                oversized = True
                            else:
                                buffer.extend(piece)

                        if index < len(pieces) - 1:
                            if oversized:
                                record["lines"] += 1
                                rejection(record, "line_too_large")
                                save()
                            else:
                                handle_line(bytes(buffer), record)
                            buffer.clear()
                            oversized = False
                            tail_bytes = 0

                            if OLD_TOKENS["train"] + state["new_tokens"]["train"] >= TARGET_STREAM_TOKENS:
                                raise StopShard("target_combined_stream_tokens")
                            if record["lines"] >= CONFIG["max_records_per_shard"]:
                                raise StopShard("max_records_per_shard")

                try:
                    limit = min(
                        CONFIG["max_compressed_per_shard"],
                        CONFIG["max_compressed_total"] - total("compressed_charged"),
                    )
                    connection, response, final_url = open_prefix(url, limit)
                    record.update(http_status=response.status, resolved_url=final_url)
                    save()
                    decompressor = zlib.decompressobj(16 + zlib.MAX_WBITS)

                    while True:
                        if shutil.disk_usage(PROJECT).free < CONFIG["minimum_free_disk_bytes"]:
                            raise StopShard("minimum_free_disk_bytes")
                        if record["decompressed_charged"] >= CONFIG["max_decompressed_per_shard"]:
                            raise StopShard("max_decompressed_per_shard")
                        if total("decompressed_charged") >= CONFIG["max_decompressed_total"]:
                            raise StopShard("max_decompressed_total")

                        read_limit = min(
                            CONFIG["io_chunk_bytes"],
                            CONFIG["max_compressed_per_shard"] - record["compressed_charged"],
                            CONFIG["max_compressed_total"] - total("compressed_charged"),
                        )
                        if read_limit <= 0:
                            raise StopShard(
                                "max_compressed_total"
                                if total("compressed_charged") >= CONFIG["max_compressed_total"]
                                else "max_compressed_per_shard"
                            )

                        record["compressed_charged"] += read_limit
                        save()
                        chunk = response.read1(read_limit)
                        record["compressed_charged"] -= read_limit - len(chunk)
                        record["compressed_read"] += len(chunk)
                        save()
                        if not chunk:
                            raise StopShard("http_eof_prefix")

                        pending, drain = chunk, False
                        while pending or drain:
                            output_limit = min(
                                CONFIG["io_chunk_bytes"],
                                CONFIG["max_decompressed_per_shard"] - record["decompressed_charged"],
                                CONFIG["max_decompressed_total"] - total("decompressed_charged"),
                            )
                            if output_limit <= 0:
                                raise StopShard(
                                    "max_decompressed_total"
                                    if total("decompressed_charged") >= CONFIG["max_decompressed_total"]
                                    else "max_decompressed_per_shard"
                                )

                            record["decompressed_charged"] += output_limit
                            save()
                            output = decompressor.decompress(pending, output_limit)
                            record["decompressed_charged"] -= output_limit - len(output)
                            record["decompressed_produced"] += len(output)
                            save()

                            pending = decompressor.unconsumed_tail
                            drain = len(output) == output_limit
                            if decompressor.eof:
                                record["gzip_members_completed"] += 1
                                pending = decompressor.unused_data
                                decompressor = zlib.decompressobj(16 + zlib.MAX_WBITS)
                                drain = False
                            consume(output)

                except StopShard as exc:
                    record.update(status="stopped", stop_reason=str(exc))
                except (OSError, http.client.HTTPException, zlib.error) as exc:
                    record.update(
                        status="error",
                        stop_reason=f"{type(exc).__name__}: {str(exc)[:200]}",
                    )
                finally:
                    if response is not None:
                        response.close()
                    if connection is not None:
                        connection.close()
                    record["trailing_bytes_discarded"] = tail_bytes

                save()
                print(
                    record["category"], "|", record["stop_reason"],
                    "| combined train tokens:",
                    OLD_TOKENS["train"] + state["new_tokens"]["train"],
                )
            else:
                state["stop_reason"] = "candidate_list_exhausted"

            if OLD_TOKENS["train"] + state["new_tokens"]["train"] >= TARGET_STREAM_TOKENS:
                state["stop_reason"] = "target_combined_stream_tokens"
            save()

        except KeyboardInterrupt:
            db.rollback()
            state = json.loads(db.execute(
                "SELECT payload FROM checkpoint WHERE id=1"
            ).fetchone()[0])
            for record in state["shards"]:
                if record["status"] == "running":
                    record.update(status="interrupted", stop_reason="keyboard_interrupt")
            state["stop_reason"] = "keyboard_interrupt"
            save()
            print("Đã giữ checkpoint. Reserve chưa xác nhận vẫn bị tính vào budget.")
        except BaseException:
            db.close()
            raise

        try:
            export = RUN_DIR / "exports" / uuid.uuid4().hex
            export.mkdir(parents=True, exist_ok=False)
            files = {}
            for split in ("train", "validation"):
                name = split + ".jsonl"
                with (export / name).open("x", encoding="utf-8") as stream:
                    for (payload,) in db.execute(
                        "SELECT payload FROM documents WHERE split=? ORDER BY rowid", (split,)
                    ):
                        stream.write(payload + "\n")
                files[name] = {"sha256": file_sha256(export / name)}

            metadata = {
                "additional_plan": ADDITIONAL_PLAN,
                "plan_sha256": ADDITIONAL_PLAN_SHA,
                "state": state,
                "state_sha256": fingerprint(state),
                "files": files,
                "full_shard_integrity_verified": False,
                "sampling": "bounded gzip prefixes; complete JSONL lines only",
            }
            write_json_new(export / "metadata.json", metadata)
            print("Additional export:", export)
            return export
        finally:
            db.close()

NEW_EXPORT = None
if not RUN_DOWNLOAD:
    print("RUN_DOWNLOAD=False: không GET shard, không tạo corpus mở rộng.")
else:
    NEW_EXPORT = build_extension()

c4 | max_records_per_shard | combined train tokens: 36771335
pes2o | max_compressed_per_shard | combined train tokens: 43057286
wiki | max_records_per_shard | combined train tokens: 44628805
books | max_compressed_per_shard | combined train tokens: 50284000
cc_en_head | max_records_per_shard | combined train tokens: 51393314
reddit | max_records_per_shard | combined train tokens: 51733308
stack | target_combined_stream_tokens | combined train tokens: 55029222
Additional export: /content/moe-erc/data/corpus_extensions/dolma55m-7e884fa883731ef7/exports/41fc36fa213d44259a88f47281e07f7d


In [ ]:
import tempfile

def audit_extension():
    exports = sorted(
        (RUN_DIR / "exports").glob("*/metadata.json"),
        key=lambda path: path.stat().st_mtime_ns,
    )
    if not exports:
        print("Chưa có additional export. Cell 5 đang False hoặc chưa hoàn tất export.")
        return

    export = NEW_EXPORT if NEW_EXPORT is not None else exports[-1].parent
    metadata = read_json(export / "metadata.json")
    require(metadata["plan_sha256"] == ADDITIONAL_PLAN_SHA, "Additional plan sai.")
    require(fingerprint(metadata["additional_plan"]) == ADDITIONAL_PLAN_SHA, "Plan nội dung sai.")

    database = sqlite3.connect(
        (RUN_DIR / "extension.sqlite").resolve().as_uri() + "?mode=ro", uri=True
    )
    try:
        require(
            database.execute("PRAGMA integrity_check").fetchall() == [("ok",)],
            "Extension SQLite integrity FAIL.",
        )
        state = json.loads(database.execute(
            "SELECT payload FROM checkpoint WHERE id=1"
        ).fetchone()[0])
        require(fingerprint(state) == metadata["state_sha256"], "Export cũ hơn checkpoint.")
        require(state == metadata["state"], "State/export khác nhau.")
        for split in ("train", "validation"):
            path = export / f"{split}.jsonl"
            require(
                file_sha256(path) == metadata["files"][path.name]["sha256"],
                f"Additional checksum FAIL: {split}",
            )

        counts = {
            origin: {
                split: {"documents": 0, "text_tokens": 0, "eos_tokens": 0, "total_tokens": 0}
                for split in ("train", "validation")
            }
            for origin in ("original", "additional")
        }
        distributions = defaultdict(Counter)
        seen_hashes = set()
        seen_ids = set()
        new_ids = set()
        ordered_sources = [
            ("original", "train", BASE_SOURCES["train"]),
            ("original", "validation", BASE_SOURCES["validation"]),
            ("additional", "train", export / "train.jsonl"),
            ("additional", "validation", export / "validation.jsonl"),
        ]

        with tempfile.TemporaryDirectory(prefix="moe-pack-count-") as temporary:
            writers = {
                split: _Packer(Path(temporary) / f"{split}.scratch", 512, [])
                for split in ("train", "validation")
            }
            try:
                for origin, split, path in ordered_sources:
                    for item, identifier, document in verified_documents(path, split):
                        digest = document.content_hash
                        require(digest not in seen_hashes, "Combined duplicate/overlap.")
                        seen_hashes.add(digest)
                        if origin == "additional":
                            require(identifier not in seen_ids, "New ID trùng dữ liệu đã dùng.")
                            require(identifier not in new_ids, "New ID trùng trong extension.")
                            require(item["origin_shard_url"] in CANDIDATES, "New origin URL sai.")
                            new_ids.add(identifier)
                        seen_ids.add(identifier)

                        ids = TOKENIZER.encode(document.text)
                        require(
                            all(type(i) is int and 0 <= i < 16000 for i in ids),
                            f"Invalid IDs: hash={digest}",
                        )
                        ntokens = len(ids) + 1
                        if origin == "additional":
                            stored = database.execute(
                                "SELECT id_hash, split, tokens, payload FROM documents WHERE hash=?",
                                (digest,),
                            ).fetchone()
                            require(
                                stored is not None and stored[0] == identifier
                                and stored[1] == split and stored[2] == ntokens
                                and json.loads(stored[3]) == item,
                                "Additional JSONL/SQLite/token counts không khớp.",
                            )

                        result = counts[origin][split]
                        result["documents"] += 1
                        result["text_tokens"] += len(ids)
                        result["eos_tokens"] += 1
                        result["total_tokens"] += ntokens
                        category = (origin, item["source_category"], split)
                        distributions[category]["documents"] += 1
                        distributions[category]["tokens"] += ntokens

                        writer = writers[split]
                        writer.feed(ids + [TOKENIZER.eos_id])
                        # Audit-only: giữ counters/pending của packer,
                        # bỏ bytes scratch đã ghi; không tích lũy full dataset.
                        writer.stream.flush()
                        writer.stream.seek(0)
                        writer.stream.truncate(0)

                        if result["documents"] % 5000 == 0:
                            print(origin, split, result["documents"], "documents audited")

                packed = {}
                for split, writer in writers.items():
                    combined = sum(counts[origin][split]["total_tokens"] for origin in counts)
                    require(writer.sequences == combined // 512, "Packer sequence count sai.")
                    require(len(writer.pending) == combined % 512, "Packer tail sai.")
                    packed[split] = {
                        "stream_tokens": combined,
                        "sequences": writer.sequences,
                        "packed_input_tokens": writer.sequences * 512,
                        "tail_discarded_tokens": len(writer.pending),
                        "effective_next_token_targets": writer.sequences * 511,
                    }
            finally:
                for writer in writers.values():
                    writer.close()

        for split in ("train", "validation"):
            require(counts["original"][split]["total_tokens"] == OLD_TOKENS[split], "Audit gốc sai.")
            require(
                counts["additional"][split]["total_tokens"] == state["new_tokens"][split],
                "New token totals sai.",
            )
            require(
                counts["additional"][split]["documents"] == state["new_documents"][split],
                "New document count sai.",
            )
        require(counts["original"]["train"]["documents"] == 28051, "Original train count sai.")
        require(counts["original"]["validation"]["documents"] == 443, "Original validation count sai.")
        require(
            database.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
            == sum(state["new_documents"].values()),
            "Extension database count sai.",
        )

        verify_originals()
        additional_fp = fingerprint({
            "plan_sha256": ADDITIONAL_PLAN_SHA,
            "tokenizer_sha256": EXPECTED_TOKENIZER_SHA,
            "files": {
                name: file_sha256(export / name)
                for name in ("train.jsonl", "validation.jsonl", "metadata.json")
            },
        })
        source_manifest = [{
            "origin": origin, "split": split, "path": str(path),
            "sha256": file_sha256(path),
        } for origin, split, path in ordered_sources]
        combined_fp = fingerprint({
            "original_corpus_fingerprint": ORIGINAL_CORPUS_FP,
            "additional_corpus_fingerprint": additional_fp,
            "tokenizer_sha256": EXPECTED_TOKENIZER_SHA,
            "ordered_sources": source_manifest,
            "split": ADDITIONAL_PLAN["split"],
            "eos_id": 2, "sequence_length": 512,
        })

        records = state["shards"]
        sum_stat = lambda key: sum(record[key] for record in records)
        invalid_reasons = Counter()
        stops = Counter()
        for record in records:
            invalid_reasons.update(record["invalid_reasons"])
            stops[record["stop_reason"]] += 1
        if state["stop_reason"]:
            stops[state["stop_reason"]] += 1

        target_reached = packed["train"]["packed_input_tokens"] >= CONFIG["target_packed_train_tokens"]
        pilot_ready = packed["train"]["packed_input_tokens"] >= 50_000_000
        report = {
            "audit_status": "PASS",
            "target_55m_packed_reached": target_reached,
            "pilot_50m_packed_ready": pilot_ready,
            "counts": counts,
            "packing": packed,
            "tokenizer_sha256": EXPECTED_TOKENIZER_SHA,
            "original_corpus_fingerprint": ORIGINAL_CORPUS_FP,
            "additional_corpus_fingerprint": additional_fp,
            "combined_dataset_fingerprint": combined_fp,
            "ordered_sources_for_future_packing": source_manifest,
            "additional_plan_sha256": ADDITIONAL_PLAN_SHA,
            "content_duplicates_removed": sum_stat("content_duplicates"),
            "id_conflicts_rejected": sum_stat("id_conflicts_rejected"),
            "invalid_documents": sum_stat("invalid"),
            "invalid_reasons": dict(invalid_reasons),
            "source_distribution": [
                {"origin": key[0], "category": key[1], "split": key[2], **dict(value)}
                for key, value in sorted(distributions.items())
            ],
            "compressed_bytes_confirmed_read": sum_stat("compressed_read"),
            "compressed_bytes_charged": sum_stat("compressed_charged"),
            "decompressed_bytes_produced": sum_stat("decompressed_produced"),
            "decompressed_bytes_charged": sum_stat("decompressed_charged"),
            "stop_reasons": dict(stops),
            "train_validation_overlap": 0,
            "additional_export": str(export),
            "full_shard_integrity_verified": False,
            "packing_audit_only": True,
            "official_tokenized_dataset_created": False,
            "git_commit": GIT_COMMIT,
        }

        audit_dir = RUN_DIR / "audits" / uuid.uuid4().hex
        audit_dir.mkdir(parents=True, exist_ok=False)
        write_json_new(audit_dir / "audit.json", report)
        write_json_new(audit_dir / "combined_sources.json", {
            "dataset_fingerprint": combined_fp,
            "ordered_sources": source_manifest,
            "tokenizer_sha256": EXPECTED_TOKENIZER_SHA,
        })

        print("\n=== EXPANSION REPORT ===")
        print("Original train tokens:", OLD_TOKENS["train"])
        print("New train tokens:", state["new_tokens"]["train"])
        print("Combined train tokens:", packed["train"]["stream_tokens"])
        print("Combined packed train input tokens:", packed["train"]["packed_input_tokens"])
        print("Combined validation tokens:", packed["validation"]["stream_tokens"])
        print("New documents:", state["new_documents"])
        print("Content duplicates removed:", report["content_duplicates_removed"])
        print("ID conflicts rejected:", report["id_conflicts_rejected"])
        print("Invalid documents:", report["invalid_documents"], dict(invalid_reasons))
        print("Source distribution:", report["source_distribution"])
        print("Compressed read/charged MiB:",
              sum_stat("compressed_read") / MiB, sum_stat("compressed_charged") / MiB)
        print("Decompressed produced/charged MiB:",
              sum_stat("decompressed_produced") / MiB, sum_stat("decompressed_charged") / MiB)
        print("Stop reasons:", dict(stops))
        print("Tokenizer SHA-256:", EXPECTED_TOKENIZER_SHA)
        print("Original fingerprint:", ORIGINAL_CORPUS_FP)
        print("Additional fingerprint:", additional_fp)
        print("Combined fingerprint:", combined_fp)
        print("Train/validation overlap: 0")
        print("Additional data:", export)
        print("Audit:", audit_dir / "audit.json")
        print("Data audit: PASS")
        print("At least 55M packed train input tokens:", target_reached)
        print("Pilot 50M packed input tokens ready:", pilot_ready)

        if not target_reached:
            remaining = max(0, TARGET_STREAM_TOKENS - packed["train"]["stream_tokens"])
            print("Target chưa đạt. Còn thiếu ít nhất", remaining, "stream tokens.")
            print("Dừng tại ngân sách hiện tại; không tự chọn thêm shards hoặc tăng giới hạn.")
        print(
            "Charged có thể lớn hơn actual khi read/decompress bị gián đoạn. "
            "Sampling là prefix; chính sách ID thận trọng có thể gây thêm bias."
        )
        print("Không thay tokenizer/corpus gốc; chưa tạo dataset tokenized chính thức.")
    finally:
        database.close()

try:
    audit_extension()
except Exception:
    print("EXPANSION AUDIT: FAIL — không tuyên bố đạt 55M hoặc sẵn sàng pilot.")
    traceback.print_exc()

original train 5000 documents audited
original train 10000 documents audited
original train 15000 documents audited
original train 20000 documents audited
original train 25000 documents audited
additional train 5000 documents audited

=== EXPANSION REPORT ===
Original train tokens: 35961014
New train tokens: 19068208
Combined train tokens: 55029222
Combined packed train input tokens: 55028736
Combined validation tokens: 515272
New documents: {'train': 7134, 'validation': 64}
Content duplicates removed: 0
ID conflicts rejected: 0
Invalid documents: 201 {'line_too_large': 1, 'invalid_or_empty_text': 200}
Source distribution: [{'origin': 'additional', 'category': 'books', 'split': 'train', 'documents': 39, 'tokens': 5655195}, {'origin': 'additional', 'category': 'c4', 'split': 'train', 'documents': 1483, 'tokens': 810321}, {'origin': 'additional', 'category': 'c4', 'split': 'validation', 'documents': 17, 'tokens': 6773}, {'origin': 'additional', 'category': 'cc_en_head', 'split': 'train',

**STEP 6.4.2B — TẠO TOKENIZED DATASET 512 TOKENS VÀ BACKUP LÊN GOOGLE DRIVE**

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
from contextlib import closing, ExitStack
from datetime import datetime, timezone
import hashlib
import inspect
import json
import os
import random
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import uuid

PROJECT = Path("/content/moe-erc")
EXTENSION = PROJECT / "data/corpus_extensions/dolma55m-7e884fa883731ef7"
EXT_EXPORT = EXTENSION / "exports/41fc36fa213d44259a88f47281e07f7d"
EXPANSION_AUDIT_PATH = (
    EXTENSION / "audits/807cae60bb8a459ab9060b516a884748/audit.json"
)
TOKENIZER_PATH = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"

TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
ORIGINAL_FP = "2df1e251a4fc3d583d8c8396a79590e99f40c5f63574009a9b3009165b192d04"
ADDITIONAL_FP = "978097e805a034873ab22afab6a28c4e0d0c13d6d14548fd28fe1d1bb923d6b9"
COMBINED_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"
EXPECTED_TOKENS = {"train": 55_029_222, "validation": 515_272}
EXPECTED_DOCS = {"train": 35_185, "validation": 507}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

def json_bytes(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
    ).encode("utf-8")

def write_new(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(content)
        stream.flush()
        os.fsync(stream.fileno())

def write_json_once(path, value):
    path = Path(path)
    if path.exists():
        require(read_json(path) == value, f"Không overwrite file khác nội dung: {path}")
    else:
        write_new(path, json_bytes(value))

def copy_new(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Path(source).open("rb") as src, destination.open("xb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024)
        dst.flush()
        os.fsync(dst.fileno())

def readonly_db(path, sealed=False):
    return sqlite3.connect(
        Path(path).resolve().as_uri()
        + ("?mode=ro&immutable=1" if sealed else "?mode=ro"),
        uri=True, timeout=10,
    )

require(PROJECT.is_dir(), "Thiếu project.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import numpy as np
import torch
import tokenizers
from moe_erc.config import ModelConfig, AoEConfig
from moe_erc.dolma_reader import canonical_text, file_sha256, DownloadOptions
from moe_erc.tokenizer_utils import FrozenTokenizer, SplitConfig, chain_hash
from moe_erc.data_processing import (
    PreprocessingConfig, prepare_dataset, validate_dataset, audit_document_splits,
)
from moe_erc.dataset import PackedTokenDataset, make_dataloader
from moe_erc.model import MoELanguageModel, AoELanguageModel

for path in (EXPANSION_AUDIT_PATH, TOKENIZER_PATH, EXT_EXPORT / "metadata.json"):
    require(path.is_file(), f"Thiếu file: {path}")

TOKENIZER = FrozenTokenizer(TOKENIZER_PATH)
require(TOKENIZER.fingerprint == TOKENIZER_SHA, "Tokenizer SHA-256 sai.")
require(TOKENIZER.vocab_size == 16000 and TOKENIZER.eos_id == 2, "Vocab/EOS sai.")
TOKENIZER.validate_model(ModelConfig())
TOKENIZER.validate_model(AoEConfig())
tokenizer_config = read_json(TOKENIZER_PATH)
require(tokenizer_config.get("post_processor") is None, "Tokenizer tự thêm special tokens.")
require(tokenizer_config.get("normalizer") is None, "Tokenizer normalization khác dự kiến.")

EXPANSION_AUDIT = read_json(EXPANSION_AUDIT_PATH)
EXT_METADATA = read_json(EXT_EXPORT / "metadata.json")
ADDITIONAL_PLAN = EXT_METADATA["additional_plan"]
require(EXPANSION_AUDIT["audit_status"] == "PASS", "Expansion audit chưa PASS.")
for key, expected in (
    ("original_corpus_fingerprint", ORIGINAL_FP),
    ("additional_corpus_fingerprint", ADDITIONAL_FP),
    ("combined_dataset_fingerprint", COMBINED_FP),
):
    require(EXPANSION_AUDIT[key] == expected, f"Fingerprint sai: {key}")
require(
    EXPANSION_AUDIT["tokenizer_sha256"] == TOKENIZER_SHA
    and EXPANSION_AUDIT["train_validation_overlap"] == 0,
    "Tokenizer/overlap audit sai.",
)
require(
    fingerprint(ADDITIONAL_PLAN) == EXT_METADATA["plan_sha256"],
    "Additional plan sai.",
)

actual_additional_fp = fingerprint({
    "plan_sha256": EXT_METADATA["plan_sha256"],
    "tokenizer_sha256": TOKENIZER_SHA,
    "files": {
        name: file_sha256(EXT_EXPORT / name)
        for name in ("train.jsonl", "validation.jsonl", "metadata.json")
    },
})
require(actual_additional_fp == ADDITIONAL_FP, "Additional files fingerprint sai.")

SPLIT = SplitConfig(**ADDITIONAL_PLAN["split"])
require(SPLIT.seed == 42, "Seed khác 42.")
SOURCE_MANIFEST = EXPANSION_AUDIT["ordered_sources_for_future_packing"]
require(len(SOURCE_MANIFEST) == 4, "Cần đúng bốn nguồn original/additional, train/validation.")

require(fingerprint({
    "original_corpus_fingerprint": ORIGINAL_FP,
    "additional_corpus_fingerprint": ADDITIONAL_FP,
    "tokenizer_sha256": TOKENIZER_SHA,
    "ordered_sources": SOURCE_MANIFEST,
    "split": ADDITIONAL_PLAN["split"],
    "eos_id": 2,
    "sequence_length": 512,
}) == COMBINED_FP, "Combined fingerprint không tái tính được.")

POINTERS = []
seen_hashes = set()
document_counts = Counter()
INPUT_GUARD = {}
for source_index, source in enumerate(SOURCE_MANIFEST):
    path = Path(source["path"])
    require(path.is_file() and not path.is_symlink(), f"Nguồn không hợp lệ: {path}")
    require(file_sha256(path) == source["sha256"], f"Source checksum FAIL: {path}")
    INPUT_GUARD[str(path)] = source["sha256"]

    with path.open("rb") as stream:
        line_number = 0
        while True:
            offset = stream.tell()
            raw = stream.readline()
            if not raw:
                break
            line_number += 1
            item = json.loads(raw.decode("utf-8"))
            text = canonical_text(item["text"])
            require(bool(text) and text == item["text"], f"Text sai: {path}:{line_number}")
            digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
            require(digest == item["content_hash"], "Content hash sai.")
            require(digest not in seen_hashes, "Duplicate/overlap giữa các nguồn.")
            require(
                item["split"] == SPLIT.assign(digest) == source["split"],
                "Document split sai.",
            )
            seen_hashes.add(digest)
            document_counts[source["split"]] += 1
            POINTERS.append({
                "source_index": source_index,
                "byte_offset": offset,
                "byte_length": len(raw),
                "content_hash": digest,
                "split": source["split"],
                "origin": source["origin"],
            })

require(dict(document_counts) == EXPECTED_DOCS, "Số documents khác audit.")

original_train = next(
    Path(s["path"]) for s in SOURCE_MANIFEST
    if s["origin"] == "original" and s["split"] == "train"
)
ORIGINAL_SUMMARY_PATH = original_train.parent / "summary.json"
original_summary = read_json(ORIGINAL_SUMMARY_PATH)
original_sources = [s for s in SOURCE_MANIFEST if s["origin"] == "original"]
require(fingerprint({
    "files": {
        **{Path(s["path"]).name: s["sha256"] for s in original_sources},
        "summary.json": file_sha256(ORIGINAL_SUMMARY_PATH),
    },
    "plan_sha256": original_summary["plan_sha256"],
    "selection_sha256": original_summary["selection_sha256"],
}) == ORIGINAL_FP, "Original corpus fingerprint sai.")

with closing(readonly_db(EXTENSION / "extension.sqlite")) as connection:
    saved_state = json.loads(connection.execute(
        "SELECT payload FROM checkpoint WHERE id=1"
    ).fetchone()[0])
    require(
        fingerprint(saved_state) == EXT_METADATA["state_sha256"]
        and saved_state == EXT_METADATA["state"],
        "Extension checkpoint khác export.",
    )

for split, expected in EXPECTED_TOKENS.items():
    require(
        EXPANSION_AUDIT["packing"][split]["stream_tokens"] == expected,
        f"Audit token total sai: {split}",
    )

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
for path in (
    TOKENIZER_PATH, TOKENIZER_PATH.with_suffix(".meta.json"),
    EXPANSION_AUDIT_PATH, EXT_EXPORT / "metadata.json", ORIGINAL_SUMMARY_PATH,
):
    INPUT_GUARD[str(path)] = file_sha256(path)
CODE_SHA = {
    name: file_sha256(PROJECT / "moe_erc" / name)
    for name in ("data_processing.py", "dataset.py", "tokenizer_utils.py", "model.py")
}

def verify_inputs():
    for name, expected in INPUT_GUARD.items():
        require(file_sha256(name) == expected, f"Input đã thay đổi: {name}")
    for name, expected in CODE_SHA.items():
        require(file_sha256(PROJECT / "moe_erc" / name) == expected, f"Code đổi: {name}")

require(shutil.disk_usage(PROJECT).free >= 8 * 1024**3, "Cần tối thiểu 8 GiB local free.")
print("Preflight: PASS")
print("Documents:", dict(document_counts), "| overlap: 0")
print("Combined source fingerprint:", COMBINED_FP)
print("Tokenizer:", TOKENIZER_SHA)
print("Git commit:", GIT_COMMIT)
print("APIs: FrozenTokenizer -> prepare_dataset -> PackedTokenDataset -> make_dataloader")
print("EOS: encode(text)+[2]; storage uint16; labels clone chưa shift.")

Preflight: PASS
Documents: {'train': 35185, 'validation': 507} | overlap: 0
Combined source fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
Tokenizer: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Git commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
APIs: FrozenTokenizer -> prepare_dataset -> PackedTokenDataset -> make_dataloader
EOS: encode(text)+[2]; storage uint16; labels clone chưa shift.


In [ ]:
RUN_EXTENSION_BACKUP = True
EXISTING_EXTENSION_SNAPSHOT = None  # Điền đường dẫn snapshot hợp lệ nếu đã backup.

def safe_relative(name):
    require(isinstance(name, str) and bool(name), "Tên file sai.")
    p = PurePosixPath(name)
    require(
        not p.is_absolute() and ".." not in p.parts
        and p.as_posix() == name and "\\" not in name
        and not any(ord(c) < 32 for c in name),
        f"Path không an toàn: {name!r}",
    )
    return p

def verify_snapshot(root, kind, complete=True):
    root = Path(root)
    require(root.is_dir() and not root.is_symlink(), "Snapshot root sai.")
    for path in root.rglob("*"):
        require(not path.is_symlink(), f"Symlink: {path}")

    manifest = read_json(root / "backup_manifest.json")
    digest = file_sha256(root / "backup_manifest.json")
    require(
        digest == (root / "backup_manifest.sha256").read_text().strip(),
        "Backup manifest SHA sai.",
    )
    require(
        manifest["kind"] == kind
        and manifest["combined_source_fingerprint"] == COMBINED_FP
        and manifest["tokenizer_sha256"] == TOKENIZER_SHA,
        "Snapshot identity sai.",
    )
    names = set()
    for entry in manifest["files"]:
        name = entry["path"]
        safe_relative(name)
        require(name not in names, "Backup entry trùng.")
        names.add(name)
        path = root / name
        require(path.is_file(), f"Thiếu backup file: {name}")
        require(path.stat().st_size == entry["bytes"], f"Size FAIL: {name}")
        require(file_sha256(path) == entry["sha256"], f"SHA FAIL: {name}")
        if entry["sqlite"]:
            with closing(readonly_db(path, sealed=True)) as connection:
                require(
                    connection.execute("PRAGMA integrity_check").fetchall() == [("ok",)],
                    f"SQLite integrity FAIL: {name}",
                )

    actual = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}
    controls = {"backup_manifest.json", "backup_manifest.sha256"}
    if complete:
        require(not (root / "INCOMPLETE.json").exists(), "Snapshot INCOMPLETE.")
        marker = read_json(root / "COMPLETE.json")
        require(
            marker["status"] == "COMPLETE" and marker["manifest_sha256"] == digest,
            "COMPLETE marker sai.",
        )
        controls.add("COMPLETE.json")
    else:
        controls |= {"INCOMPLETE.json"} if (root / "INCOMPLETE.json").exists() else set()
    require(actual == names | controls, "Backup inventory sai.")
    return manifest, digest

def make_snapshot(items, destination_root, kind, validator, sqlite_api):
    require(Path("/content/drive/MyDrive").is_dir(), "Drive chưa mount.")
    verify_inputs()
    items = {safe_relative(k).as_posix(): Path(v) for k, v in items.items()}
    databases = set()
    for source in items.values():
        require(source.is_file() and not source.is_symlink(), f"Source sai: {source}")
        with source.open("rb") as stream:
            if stream.read(16) == b"SQLite format 3\x00":
                databases.add(source)

    # WAL/SHM/journal chỉ được xử lý qua database chủ.
    for name, source in list(items.items()):
        for suffix in ("-wal", "-shm", "-journal"):
            if str(source).endswith(suffix) and Path(str(source)[:-len(suffix)]) in databases:
                del items[name]

    local = Path(tempfile.mkdtemp(prefix=f"moe-{kind}-backup-", dir="/content"))
    monitors = {}
    try:
        for database in databases:
            connection = readonly_db(database)
            monitors[database] = (
                connection, connection.execute("PRAGMA data_version").fetchone()[0]
            )

        entries = []
        for name, source in sorted(items.items()):
            target = local / name
            target.parent.mkdir(parents=True, exist_ok=True)
            is_sqlite = source in databases

            if is_sqlite and sqlite_api:
                connection = monitors[source][0]
                connection.execute("BEGIN")
                try:
                    connection.execute("SELECT COUNT(*) FROM sqlite_master").fetchone()
                    with closing(sqlite3.connect(target)) as out:
                        connection.backup(out)
                        require(
                            out.execute("PRAGMA journal_mode=DELETE").fetchone()[0] == "delete",
                            "SQLite snapshot chưa standalone.",
                        )
                finally:
                    connection.rollback()
                method = "sqlite_backup_api"
            else:
                if is_sqlite:
                    for suffix in ("-wal", "-shm", "-journal"):
                        require(
                            not Path(str(source) + suffix).exists(),
                            "Dataset database phải sealed, không có sidecar.",
                        )
                before = file_sha256(source)
                copy_new(source, target)
                require(
                    file_sha256(target) == before == file_sha256(source),
                    f"Source/copy thay đổi: {source}",
                )
                method = "sealed_byte_copy" if is_sqlite else "byte_copy"

            entries.append({
                "path": name, "source_path": str(source),
                "bytes": target.stat().st_size, "sha256": file_sha256(target),
                "sqlite": is_sqlite, "method": method,
            })

        for database, (connection, version) in monitors.items():
            require(
                connection.execute("PRAGMA data_version").fetchone()[0] == version,
                f"Database có commit mới: {database}",
            )
        verify_inputs()
    finally:
        for connection, _ in monitors.values():
            connection.close()

    manifest = {
        "format_version": 1, "kind": kind,
        "combined_source_fingerprint": COMBINED_FP,
        "tokenizer_sha256": TOKENIZER_SHA,
        "git_commit": GIT_COMMIT, "files": entries,
    }
    write_new(local / "backup_manifest.json", json_bytes(manifest))
    digest = file_sha256(local / "backup_manifest.json")
    write_new(local / "backup_manifest.sha256", (digest + "\n").encode())
    verify_snapshot(local, kind, complete=False)
    validator(local)

    destination_root = Path(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)
    name = (
        kind + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-" + COMBINED_FP[:12] + "-" + uuid.uuid4().hex[:12]
    )
    pending = destination_root / (name + ".INCOMPLETE")
    pending.mkdir(exist_ok=False)
    write_new(pending / "INCOMPLETE.json", json_bytes({"status": "INCOMPLETE"}))
    print("Backup đang tạo:", pending)

    # Lỗi sẽ giữ nguyên pending và local staging; không tạo COMPLETE.
    for source in sorted(local.rglob("*")):
        if source.is_file():
            target = pending / source.relative_to(local)
            copy_new(source, target)
            require(file_sha256(target) == file_sha256(source), f"Drive SHA FAIL: {target}")

    _, drive_digest = verify_snapshot(pending, kind, complete=False)
    require(drive_digest == digest, "Drive manifest khác local.")
    validator(pending)

    final = destination_root / name
    require(not final.exists(), "Không overwrite snapshot.")
    pending.rename(final)
    marker = {
        "status": "COMPLETE", "manifest_sha256": digest,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_new(final / "COMPLETE.pending.json", json_bytes(marker))
    require(read_json(final / "COMPLETE.pending.json") == marker, "Marker readback FAIL.")
    (final / "INCOMPLETE.json").unlink()
    (final / "COMPLETE.pending.json").rename(final / "COMPLETE.json")
    return final

EXT_REL = EXTENSION.relative_to(PROJECT)
EXPORT_REL = EXT_EXPORT.relative_to(PROJECT)

def extension_validator(root):
    export = Path(root) / EXPORT_REL
    metadata = read_json(export / "metadata.json")
    require(fingerprint({
        "plan_sha256": metadata["plan_sha256"],
        "tokenizer_sha256": TOKENIZER_SHA,
        "files": {
            name: file_sha256(export / name)
            for name in ("train.jsonl", "validation.jsonl", "metadata.json")
        },
    }) == ADDITIONAL_FP, "Backup additional fingerprint sai.")
    with closing(readonly_db(
        Path(root) / EXT_REL / "extension.sqlite", sealed=True
    )) as connection:
        state = json.loads(connection.execute(
            "SELECT payload FROM checkpoint WHERE id=1"
        ).fetchone()[0])
    require(fingerprint(state) == metadata["state_sha256"], "Backup SQLite/export khác nhau.")

EXTENSION_BACKUP_OK = False
if RUN_EXTENSION_BACKUP:
    items = {}
    for path in EXTENSION.rglob("*"):
        require(not path.is_symlink(), f"Symlink trong extension: {path}")
        if path.is_file():
            items[path.relative_to(PROJECT).as_posix()] = path

    related = [
        PROJECT / "data/selections"
        / f"dolma-expansion55m-{EXT_METADATA['plan_sha256'][:16]}.json",
        PROJECT / "data/manifests"
        / f"dolma-v1_5-sample-{ADDITIONAL_PLAN['manifest_sha256']}.json",
        PROJECT / "data/selections"
        / f"dolma-tokenizer-plan-{original_summary['plan_sha256'][:16]}.json",
        ORIGINAL_SUMMARY_PATH,
        TOKENIZER_PATH,
        TOKENIZER_PATH.with_suffix(".meta.json"),
    ]
    for path in related:
        require(path.is_file(), f"Thiếu related metadata: {path}")
        items[path.relative_to(PROJECT).as_posix()] = path

    EXTENSION_SNAPSHOT = make_snapshot(
        items,
        "/content/drive/MyDrive/MoE_ERC/corpus_extension_snapshots",
        "extension", extension_validator, sqlite_api=True,
    )
elif EXISTING_EXTENSION_SNAPSHOT:
    EXTENSION_SNAPSHOT = Path(EXISTING_EXTENSION_SNAPSHOT)

if globals().get("EXTENSION_SNAPSHOT") is not None:
    _, EXTENSION_BACKUP_MANIFEST_SHA = verify_snapshot(EXTENSION_SNAPSHOT, "extension")
    extension_validator(EXTENSION_SNAPSHOT)
    EXTENSION_BACKUP_OK = True
    print("Extension backup: PASS")
    print("Snapshot:", EXTENSION_SNAPSHOT)
else:
    print("Chưa backup extension. Packing sẽ bị chặn; không có chế độ override.")

Backup đang tạo: /content/drive/MyDrive/MoE_ERC/corpus_extension_snapshots/extension-20261024T070408Z-92acf38d5017-8f920af4a40c.INCOMPLETE
Extension backup: PASS
Snapshot: /content/drive/MyDrive/MoE_ERC/corpus_extension_snapshots/extension-20261024T070408Z-92acf38d5017-8f920af4a40c


In [ ]:
ORDER_POLICY = "sha256('doc-order-v1:42:<split>:<content_hash>'), then content_hash"

ORDERED_POINTERS = []
for pointer in POINTERS:
    key = hashlib.sha256(
        f"doc-order-v1:42:{pointer['split']}:{pointer['content_hash']}".encode()
    ).hexdigest()
    ORDERED_POINTERS.append({**pointer, "order_key": key})
ORDERED_POINTERS.sort(
    key=lambda row: (row["split"], row["order_key"], row["content_hash"])
)

DESIGN = {
    "combined_source_fingerprint": COMBINED_FP,
    "tokenizer_sha256": TOKENIZER_SHA,
    "source_manifest": SOURCE_MANIFEST,
    "ordering_policy": ORDER_POLICY,
    "ordering_seed": 42,
    "split": {"seed": SPLIT.seed, "validation_fraction": SPLIT.validation_fraction},
    "sequence_length": 512,
    "dtype": "<u2",
    "eos_id": 2,
    "bos_pad_insertion": False,
    "eos_policy": "encode(text) + [2]; preserve literal specials in text",
    "tail_policy": "discard final incomplete sequence independently per split",
    "labels": "input_ids.clone(); causal shift only inside model",
    "expected_stream_tokens": EXPECTED_TOKENS,
    "expected_documents": EXPECTED_DOCS,
    "pipeline_source_sha256": CODE_SHA,
}
DESIGN_SHA = fingerprint(DESIGN)
FINAL_DATASET_DIR = (
    PROJECT / "data/tokenized_datasets"
    / f"dolma55m-mixed512-{DESIGN_SHA[:16]}"
)

print("Design fingerprint:", DESIGN_SHA)
print("Dataset target:", FINAL_DATASET_DIR)
print("Ordering:", ORDER_POLICY)
print("Ordering trộn original/additional theo document; không cân bằng lại tỷ lệ nguồn.")
print("Train expected sequences:", EXPECTED_TOKENS["train"] // 512)
print("Train expected packed tokens:", EXPECTED_TOKENS["train"] // 512 * 512)
print(
    "Validation dự kiến theo phép chia:",
    EXPECTED_TOKENS["validation"] // 512,
    "sequences; Cell 4–5 sẽ xác minh bằng API.",
)
print("Thứ tự thay đổi nội dung tail nhưng không thay số tokens giữ lại khi dùng toàn corpus.")
print("Không lấy 50M tokens từ prefix original-first; pilot chọn sequences riêng ở Cell 6.")

Design fingerprint: 2b7fb5b510874ad9e393ae5fd2972d1e816159685437633eb67ae3af1cea6fc2
Dataset target: /content/moe-erc/data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9
Ordering: sha256('doc-order-v1:42:<split>:<content_hash>'), then content_hash
Ordering trộn original/additional theo document; không cân bằng lại tỷ lệ nguồn.
Train expected sequences: 107478
Train expected packed tokens: 55028736
Validation dự kiến theo phép chia: 1006 sequences; Cell 4–5 sẽ xác minh bằng API.
Thứ tự thay đổi nội dung tail nhưng không thay số tokens giữ lại khi dùng toàn corpus.
Không lấy 50M tokens từ prefix original-first; pilot chọn sequences riêng ở Cell 6.


In [ ]:
RUN_PACKING = True

def read_pointer(handles, row):
    handle = handles[row["source_index"]]
    handle.seek(row["byte_offset"])
    raw = handle.read(row["byte_length"])
    item = json.loads(raw.decode("utf-8"))
    text = canonical_text(item["text"])
    require(
        text == item["text"]
        and hashlib.sha256(text.encode("utf-8")).hexdigest() == row["content_hash"]
        and item["split"] == SPLIT.assign(row["content_hash"]) == row["split"],
        "Document pointer không còn khớp nguồn.",
    )
    return raw, text

DATASET_DIR = FINAL_DATASET_DIR if FINAL_DATASET_DIR.is_dir() else None

if not RUN_PACKING:
    print("RUN_PACKING=False: không tạo dataset.")
    print("Dataset có sẵn:", DATASET_DIR)
else:
    require(EXTENSION_BACKUP_OK, "Phải backup extension trước khi packing.")
    verify_snapshot(EXTENSION_SNAPSHOT, "extension")
    extension_validator(EXTENSION_SNAPSHOT)
    verify_inputs()
    require(fingerprint(DESIGN) == DESIGN_SHA, "Design thay đổi.")
    require(not FINAL_DATASET_DIR.exists(), "Không overwrite dataset đã tồn tại.")

    build = Path(tempfile.mkdtemp(prefix="moe-packed-build-", dir="/content"))
    ordered_input_dir = build / "ordered_inputs"
    ordered_input_dir.mkdir()
    order_path = build / "document_order.jsonl"
    ordered_paths = {
        split: ordered_input_dir / f"{split}.jsonl"
        for split in ("train", "validation")
    }
    DATASET_DIR = None
    try:
        with ExitStack() as stack:
            handles = [
                stack.enter_context(Path(source["path"]).open("rb"))
                for source in SOURCE_MANIFEST
            ]
            outputs = {
                split: stack.enter_context(path.open("xb"))
                for split, path in ordered_paths.items()
            }
            order_stream = stack.enter_context(order_path.open("x", encoding="utf-8"))
            for row in ORDERED_POINTERS:
                raw, _ = read_pointer(handles, row)
                outputs[row["split"]].write(raw)
                if not raw.endswith(b"\n"):
                    outputs[row["split"]].write(b"\n")
                order_stream.write(json.dumps(row, sort_keys=True) + "\n")

        packed_root = build / "dataset"
        state = prepare_dataset(
            [str(ordered_paths["train"]), str(ordered_paths["validation"])],
            TOKENIZER,
            packed_root,
            config=PreprocessingConfig(
                train_max_tokens=EXPECTED_TOKENS["train"],
                validation_max_tokens=EXPECTED_TOKENS["validation"],
                sequence_length=512,
                split=SPLIT,
            ),
            provenance={
                "combined_source_fingerprint": COMBINED_FP,
                "design_sha256": DESIGN_SHA,
                "ordering_policy": ORDER_POLICY,
                "git_commit": GIT_COMMIT,
            },
            downloads=DownloadOptions(allow_download=False),
        )

        for split in ("train", "validation"):
            info = state["splits"][split]
            require(info["documents"] == EXPECTED_DOCS[split], "Documents bị bỏ sót.")
            require(info["encoded_tokens"] == EXPECTED_TOKENS[split], "Token total sai.")
            require(info["budget_discarded_tokens"] == 0, "Có truncation theo budget.")
            require(info["sequences"] == EXPECTED_TOKENS[split] // 512, "Sequence count sai.")
            require(info["tail_discarded_tokens"] == EXPECTED_TOKENS[split] % 512, "Tail sai.")
        require(state["duplicates"] == 0, "Pipeline phát hiện duplicate.")
        require(state["reader_stats"].get("invalid_documents", 0) == 0, "Reader có lỗi.")

        copy_new(order_path, packed_root / "document_order.jsonl")
        write_json_once(packed_root / "packing_design.json", DESIGN)
        write_json_once(packed_root / "source_manifest.json", SOURCE_MANIFEST)
        write_json_once(packed_root / "provenance/expansion_audit.json", EXPANSION_AUDIT)
        write_json_once(packed_root / "provenance/extension_backup_receipt.json", {
            "snapshot": str(EXTENSION_SNAPSHOT),
            "manifest_sha256": EXTENSION_BACKUP_MANIFEST_SHA,
        })
        copy_new(TOKENIZER_PATH, packed_root / "tokenizer/tokenizer.json")
        copy_new(
            TOKENIZER_PATH.with_suffix(".meta.json"),
            packed_root / "tokenizer/tokenizer.meta.json",
        )
        verify_inputs()
        DATASET_DIR = packed_root

        print("Packing hoàn tất, CHƯA được công nhận PASS tests.")
        print("Staging dataset:", DATASET_DIR)
        print("Build directory:", build)
        print("Train sequences:", state["splits"]["train"]["sequences"])
        print("Validation sequences:", state["splits"]["validation"]["sequences"])
        print(
            "ordered_inputs là dữ liệu build trung gian. Metadata STEP 5 giữ "
            "đường dẫn lịch sử; load dataset hoàn chỉnh chỉ cần binary/metadata/tokenizer."
        )
    except BaseException:
        print("PACKING FAIL/INTERRUPTED. Giữ staging:", build)
        print("Không dùng output chưa được Cell 5 xác minh; không tự resume.")
        raise

Packing hoàn tất, CHƯA được công nhận PASS tests.
Staging dataset: /content/moe-packed-build-sjblqbh4/dataset
Build directory: /content/moe-packed-build-sjblqbh4
Train sequences: 107478
Validation sequences: 1006
ordered_inputs là dữ liệu build trung gian. Metadata STEP 5 giữ đường dẫn lịch sử; load dataset hoàn chỉnh chỉ cần binary/metadata/tokenizer.


In [ ]:
DATASET_TESTS_PASS = False

def verify_core(root):
    root = Path(root)
    marker = read_json(root / "COMPLETE.json")
    require(marker["status"] == "COMPLETE", "Dataset chưa COMPLETE.")
    require(
        file_sha256(root / "dataset_manifest.json") == marker["dataset_manifest_sha256"],
        "Dataset manifest SHA sai.",
    )
    manifest = read_json(root / "dataset_manifest.json")
    require(manifest["combined_source_fingerprint"] == COMBINED_FP, "Dataset source identity sai.")
    for entry in manifest["files"]:
        safe_relative(entry["path"])
        path = root / entry["path"]
        require(path.is_file() and not path.is_symlink(), "Core file thiếu/symlink.")
        require(file_sha256(path) == entry["sha256"], f"Core SHA FAIL: {entry['path']}")
        require(path.stat().st_size == entry["bytes"], "Core size FAIL.")
    return manifest

def binary_chunks(root, info):
    for segment in info["segments"]:
        with (Path(root) / segment["file"]).open("rb") as stream:
            while True:
                block = stream.read(65_536 * 2)
                if not block:
                    break
                require(len(block) % 2 == 0, "Binary size không chia hết cho uint16.")
                yield np.frombuffer(block, dtype="<u2")

def test_packed(root):
    root = Path(root)
    saved_tokenizer = FrozenTokenizer(root / "tokenizer/tokenizer.json")
    require(saved_tokenizer.fingerprint == TOKENIZER_SHA, "Bundled tokenizer sai.")
    state = validate_dataset(root, tokenizer=saved_tokenizer, checksums=True)
    require(state["dtype"] == "<u2" and state["sequence_length"] == 512, "Storage config sai.")
    require(read_json(root / "packing_design.json") == DESIGN, "Saved design sai.")
    require(read_json(root / "source_manifest.json") == SOURCE_MANIFEST, "Source manifest sai.")

    ledger = audit_document_splits(root)
    require(ledger["overlap"] == 0, "Document leakage.")
    require(
        ledger["train"] == EXPECTED_DOCS["train"]
        and ledger["validation"] == EXPECTED_DOCS["validation"],
        "Ledger count sai.",
    )

    forward_source = inspect.getsource(MoELanguageModel.forward)
    require(
        "logits[:, :-1, :]" in forward_source and "labels[:, 1:]" in forward_source,
        "Model shift khác dự kiến.",
    )
    require(AoELanguageModel.forward is MoELanguageModel.forward, "AoE shift khác.")

    results = {}
    with ExitStack() as stack:
        handles = [
            stack.enter_context(Path(source["path"]).open("rb"))
            for source in SOURCE_MANIFEST
        ]
        for split in ("train", "validation"):
            info = state["splits"][split]
            require(info["documents"] == EXPECTED_DOCS[split], "Document count sai.")
            require(info["encoded_tokens"] == EXPECTED_TOKENS[split], "Raw tokens sai.")
            require(info["budget_discarded_tokens"] == 0, "Unexpected budget truncation.")
            require(info["sequences"] == EXPECTED_TOKENS[split] // 512, "Sequence count sai.")

            n = info["sequences"]
            indices = sorted(set(
                [0, n - 1] + random.Random(42).sample(range(n), min(32, n))
            ))
            references = {index: [] for index in indices}
            chunks = iter(binary_chunks(root, info))
            current = np.empty(0, dtype="<u2")
            cursor = 0
            raw_position = 0
            document_count = 0
            tail = []
            content_chain = "00" * 32

            with (root / "document_order.jsonl").open(encoding="utf-8") as order_stream:
                previous_key = None
                for line in order_stream:
                    row = json.loads(line)
                    if row["split"] != split:
                        continue
                    expected_key = hashlib.sha256(
                        f"doc-order-v1:42:{split}:{row['content_hash']}".encode()
                    ).hexdigest()
                    require(row["order_key"] == expected_key, "Order key sai.")
                    key = (row["order_key"], row["content_hash"])
                    require(previous_key is None or key > previous_key, "Ordering không tăng.")
                    previous_key = key

                    _, text = read_pointer(handles, row)
                    ids = saved_tokenizer.encode(text) + [saved_tokenizer.eos_id]
                    require(all(0 <= token < 16000 for token in ids), "Invalid token IDs.")
                    kept = min(len(ids), max(0, info["tokens"] - raw_position))

                    # Full reference comparison: không giữ toàn bộ stream trong RAM.
                    offset = 0
                    while offset < kept:
                        if cursor == len(current):
                            current = next(chunks, None)
                            require(current is not None, "Binary kết thúc sớm.")
                            cursor = 0
                        size = min(kept - offset, len(current) - cursor)
                        require(
                            np.array_equal(
                                current[cursor:cursor + size],
                                np.asarray(ids[offset:offset + size], dtype="<u2"),
                            ),
                            f"Reference mismatch: split={split}, hash={row['content_hash']}",
                        )
                        cursor += size
                        offset += size

                    for index in indices:
                        left = max(raw_position, index * 512)
                        right = min(raw_position + kept, (index + 1) * 512)
                        if left < right:
                            references[index].extend(ids[left - raw_position:right - raw_position])

                    tail.extend(ids[kept:])
                    raw_position += len(ids)
                    document_count += 1
                    content_chain = chain_hash(content_chain, row["content_hash"])

            require(cursor == len(current) and next(chunks, None) is None, "Binary có tokens dư.")
            require(raw_position == EXPECTED_TOKENS[split], "Reference raw count sai.")
            require(document_count == EXPECTED_DOCS[split], "Reference document count sai.")
            require(content_chain == info["content_fingerprint"], "Content order fingerprint sai.")
            require(tail == info["pending"], "Final tail nội dung sai.")
            require(len(tail) == info["tail_discarded_tokens"] == raw_position % 512, "Tail count sai.")

            minimum, maximum = 16000, -1
            for block in binary_chunks(root, info):
                if len(block):
                    minimum = min(minimum, int(block.min()))
                    maximum = max(maximum, int(block.max()))
            require(0 <= minimum <= maximum < 16000, "Storage IDs sai.")

            dataset = PackedTokenDataset(root, split, tokenizer=saved_tokenizer)
            try:
                require(len(dataset) == n and dataset.sequence_length == 512, "Dataset shape sai.")
                for index, expected in references.items():
                    sample = dataset[index]
                    require(len(expected) == 512, "Reference window thiếu.")
                    require(sample["input_ids"].tolist() == expected, "Random sample mismatch.")
                    require(torch.equal(sample["input_ids"], sample["labels"]), "Labels shifted.")
                    require(
                        sample["input_ids"].data_ptr() != sample["labels"].data_ptr(),
                        "Labels không clone.",
                    )
                batch = next(iter(make_dataloader(
                    dataset, batch_size=2, seed=42, num_workers=0, pin_memory=False
                )))
                require(tuple(batch["input_ids"].shape) == (2, 512), "Batch shape sai.")
                require(batch["input_ids"].dtype == batch["labels"].dtype == torch.int64, "Dtype sai.")
                require(torch.equal(batch["input_ids"], batch["labels"]), "Batch labels sai.")
            finally:
                dataset.close()

            split_fp = fingerprint({
                "sequence_length": 512, "dtype": "<u2",
                "segments": info["segments"],
            })
            results[split] = {
                "shape": [n, 512],
                "packed_input_tokens": info["tokens"],
                "effective_next_token_targets": n * 511,
                "tail_discarded_tokens": len(tail),
                "token_id_min": minimum,
                "token_id_max": maximum,
                "binary_bytes": sum(s["bytes"] for s in info["segments"]),
                "fingerprint": split_fp,
                "random_reference_indices": indices,
            }

    # Một process Python mới, không dùng notebook variables để mở dataset.
    script = """
import sys
from pathlib import Path
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.dataset import PackedTokenDataset, make_dataloader
r = Path(sys.argv[1])
t = FrozenTokenizer(r / "tokenizer/tokenizer.json")
for split in ("train", "validation"):
    d = PackedTokenDataset(r, split, tokenizer=t)
    try:
        b = next(iter(make_dataloader(d, batch_size=2, num_workers=0)))
        assert tuple(b["input_ids"].shape) == (2,512)
    finally:
        d.close()
print("fresh_process_reload: PASS")
"""
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT) + os.pathsep + environment.get("PYTHONPATH", "")
    run = subprocess.run(
        [sys.executable, "-c", script, str(root)],
        cwd=PROJECT, env=environment, text=True, capture_output=True, timeout=180,
    )
    require(run.returncode == 0, f"Fresh-process reload FAIL: {run.stderr}")
    print(run.stdout.strip())
    return {
        "status": "PASS", "splits": results,
        "full_reference_comparison": "PASS",
        "eos_exactly_one_appended_per_document": "PASS",
        "natural_eos_in_text": "preserved, not removed",
        "document_overlap": 0,
        "dataloader": "PASS",
        "fresh_process_reload": "PASS",
    }

if DATASET_DIR is None:
    print("Chưa có dataset; bật RUN_PACKING trước.")
else:
    verify_inputs()
    TEST_REPORT = test_packed(DATASET_DIR)
    write_json_once(DATASET_DIR / "dataset_tests.json", TEST_REPORT)

    if (DATASET_DIR / "COMPLETE.json").exists():
        verify_core(DATASET_DIR)
    else:
        core_files = [{
            "path": path.relative_to(DATASET_DIR).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        } for path in sorted(DATASET_DIR.rglob("*")) if path.is_file()]
        core = {
            "combined_source_fingerprint": COMBINED_FP,
            "design_sha256": DESIGN_SHA,
            "tokenizer_sha256": TOKENIZER_SHA,
            "split_fingerprints": {
                split: TEST_REPORT["splits"][split]["fingerprint"]
                for split in ("train", "validation")
            },
            "files": core_files,
        }
        write_new(DATASET_DIR / "dataset_manifest.json", json_bytes(core))
        write_new(DATASET_DIR / "COMPLETE.json", json_bytes({
            "status": "COMPLETE",
            "dataset_manifest_sha256": file_sha256(DATASET_DIR / "dataset_manifest.json"),
            "tests_status": "PASS",
        }))
        require(not FINAL_DATASET_DIR.exists(), "Final dataset đã tồn tại; không overwrite.")
        FINAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
        DATASET_DIR.rename(FINAL_DATASET_DIR)
        DATASET_DIR = FINAL_DATASET_DIR
        verify_core(DATASET_DIR)

    DATASET_TESTS_PASS = True
    print(json.dumps(TEST_REPORT, ensure_ascii=False, indent=2))
    print("Dataset tests: PASS")
    print("Published dataset:", DATASET_DIR)
    print("Đây là kiểm thử dữ liệu, không phải bằng chứng mô hình đã train.")

fresh_process_reload: PASS
{
  "status": "PASS",
  "splits": {
    "train": {
      "shape": [
        107478,
        512
      ],
      "packed_input_tokens": 55028736,
      "effective_next_token_targets": 54921258,
      "tail_discarded_tokens": 486,
      "token_id_min": 0,
      "token_id_max": 15999,
      "binary_bytes": 110057472,
      "fingerprint": "5eec3e97e908dd6b8658ffb9340d106c587ee023ac7b12d0cc53ec99f59c3700",
      "random_reference_indices": [
        0,
        3278,
        3478,
        3905,
        4165,
        11395,
        12280,
        13434,
        14592,
        18289,
        26062,
        28657,
        28893,
        29256,
        30495,
        32098,
        36048,
        54987,
        55302,
        66237,
        71426,
        71482,
        73563,
        77397,
        78907,
        83810,
        85181,
        88696,
        91924,
        93850,
        96530,
        97080,
        97196,
        107477
      ]
    },
    "validation"

In [ ]:
PILOT_READY = False

if not DATASET_TESTS_PASS:
    print("Chưa tạo pilot: dataset tests chưa PASS.")
else:
    core = verify_core(DATASET_DIR)
    saved_tests = read_json(DATASET_DIR / "dataset_tests.json")
    n_train = saved_tests["splits"]["train"]["shape"][0]
    selected_sequences = (50_000_000 + 511) // 512
    require(n_train >= selected_sequences, "Không đủ sequences train cho pilot.")

    rng = np.random.Generator(np.random.PCG64(42))
    indices = rng.permutation(n_train)[:selected_sequences].astype("<i8")
    require(len(np.unique(indices)) == len(indices), "Pilot indices bị lặp.")
    require(int(indices.min()) >= 0 and int(indices.max()) < n_train, "Index ngoài train.")

    index_path = DATASET_DIR / "pilot_50m_indices.npy"
    if index_path.exists():
        require(np.array_equal(np.load(index_path, allow_pickle=False), indices), "Pilot indices khác.")
    else:
        with index_path.open("xb") as stream:
            np.save(stream, indices, allow_pickle=False)

    PILOT = {
        "format_version": 1,
        "split": "train",
        "combined_source_fingerprint": COMBINED_FP,
        "dataset_manifest_sha256": file_sha256(DATASET_DIR / "dataset_manifest.json"),
        "train_binary_fingerprint": core["split_fingerprints"]["train"],
        "validation_binary_fingerprint": core["split_fingerprints"]["validation"],
        "tokenizer_sha256": TOKENIZER_SHA,
        "index_file": index_path.name,
        "index_file_sha256": file_sha256(index_path),
        "index_dtype": "<i8",
        "selection": "PCG64(42).permutation(N_train)[:ceil(50M/512)]",
        "seed": 42,
        "numpy_version": np.__version__,
        "sequence_length": 512,
        "selected_sequences": len(indices),
        "selected_input_tokens": len(indices) * 512,
        "requested_input_tokens": 50_000_000,
        "excess_input_tokens": len(indices) * 512 - 50_000_000,
        "effective_next_token_targets": len(indices) * 511,
        "repeat_sequences": False,
        "training_order": "exact order stored in index_file; no additional shuffle",
        "epoch_policy": "one finite pass; no automatic epoch repetition",
        "resume_contract": {
            "initial_next_sequence_ordinal": 0,
            "checkpoint_required": [
                "dataset_manifest_sha256", "pilot_manifest_sha256",
                "next_sequence_ordinal",
            ],
            "cursor_meaning": (
                "number of pilot sequences actually consumed by completed training work, "
                "not DataLoader prefetch position"
            ),
            "note": "Training/optimizer/RNG checkpoints will be implemented in a later step.",
        },
        "shared_by": ["Vanilla MoE", "MoE + ERC", "AoE"],
    }
    pilot_path = DATASET_DIR / "pilot_50m_manifest.json"
    write_json_once(pilot_path, PILOT)
    PILOT_SHA = file_sha256(pilot_path)
    sha_path = DATASET_DIR / "pilot_50m_manifest.sha256"
    if sha_path.exists():
        require(sha_path.read_text().strip() == PILOT_SHA, "Pilot manifest SHA khác.")
    else:
        write_new(sha_path, (PILOT_SHA + "\n").encode())

    PILOT_READY = True
    print("Pilot manifest:", pilot_path)
    print("Pilot manifest SHA-256:", PILOT_SHA)
    print("Sequences:", PILOT["selected_sequences"])
    print("Selected input tokens:", PILOT["selected_input_tokens"])
    print("Excess tokens:", PILOT["excess_input_tokens"])
    print("Validation không nằm trong selection.")
    print("Không chạy training hoặc tạo sampler/training loop ở bước này.")

Pilot manifest: /content/moe-erc/data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9/pilot_50m_manifest.json
Pilot manifest SHA-256: 812ea85c57344598adf4c18268fcca656da8661bdab400cdfd6542b9cc9fbe98
Sequences: 97657
Selected input tokens: 50000384
Excess tokens: 384
Validation không nằm trong selection.
Không chạy training hoặc tạo sampler/training loop ở bước này.


In [ ]:
RUN_DATASET_BACKUP = True

DATASET_BACKUP_RESULT = {
    "status": "FAIL",
    "reason": "Backup chưa chạy.",
    "snapshot": None,
}

def dataset_backup_validator(snapshot_root):
    root = Path(snapshot_root) / "dataset"
    verify_core(root)
    frozen = FrozenTokenizer(root / "tokenizer/tokenizer.json")
    require(frozen.fingerprint == TOKENIZER_SHA, "Backup tokenizer sai.")
    metadata = validate_dataset(root, tokenizer=frozen, checksums=True)
    require(read_json(root / "dataset_tests.json")["status"] == "PASS", "Tests record chưa PASS.")

    pilot = read_json(root / "pilot_50m_manifest.json")
    require(
        file_sha256(root / "pilot_50m_manifest.json")
        == (root / "pilot_50m_manifest.sha256").read_text().strip(),
        "Pilot manifest SHA FAIL.",
    )
    require(
        pilot["dataset_manifest_sha256"] == file_sha256(root / "dataset_manifest.json"),
        "Pilot trỏ tới dataset khác.",
    )
    index_path = root / pilot["index_file"]
    require(file_sha256(index_path) == pilot["index_file_sha256"], "Pilot indices SHA FAIL.")
    indices = np.load(index_path, allow_pickle=False)
    require(len(indices) == pilot["selected_sequences"], "Pilot count sai.")
    require(len(np.unique(indices)) == len(indices), "Pilot duplicate.")
    require(
        int(indices.min()) >= 0
        and int(indices.max()) < metadata["splits"]["train"]["sequences"],
        "Pilot index ngoài training dataset.",
    )

    for split in ("train", "validation"):
        dataset = PackedTokenDataset(root, split, tokenizer=frozen)
        try:
            loader = make_dataloader(
                dataset, batch_size=2, seed=42, num_workers=0, pin_memory=False
            )
            batch = next(iter(loader))
            require(tuple(batch["input_ids"].shape) == (2, 512), "Drive batch shape sai.")
            require(batch["input_ids"].dtype == torch.int64, "Drive dtype sai.")
            require(torch.equal(batch["input_ids"], batch["labels"]), "Drive labels sai.")
        finally:
            dataset.close()

if not RUN_DATASET_BACKUP:
    print("RUN_DATASET_BACKUP=False: chưa backup dataset.")
else:
    require(DATASET_TESTS_PASS and PILOT_READY, "Dataset tests/pilot chưa PASS.")
    verify_core(DATASET_DIR)
    require(file_sha256(DATASET_DIR / "pilot_50m_manifest.json") == PILOT_SHA, "Pilot đã thay đổi.")

    items = {}
    for path in DATASET_DIR.rglob("*"):
        require(not path.is_symlink(), f"Dataset có symlink: {path}")
        if path.is_file():
            require(
                not path.name.endswith((".tmp", ".part")),
                f"Dataset còn temporary file: {path}",
            )
            items["dataset/" + path.relative_to(DATASET_DIR).as_posix()] = path

    snapshot = make_snapshot(
        items,
        "/content/drive/MyDrive/MoE_ERC/tokenized_dataset_snapshots",
        "dataset",
        dataset_backup_validator,
        sqlite_api=False,  # sealed DB + source/destination SHA + integrity check
    )
    DATASET_BACKUP_RESULT = {
        "status": "PASS",
        "reason": "SHA-256, metadata, pilot và Drive DataLoader đều PASS.",
        "snapshot": str(snapshot),
    }

print("\n=== FINAL REPORT ===")
print("Combined source dataset fingerprint:", COMBINED_FP)
print("Tokenizer SHA-256:", TOKENIZER_SHA)

if DATASET_TESTS_PASS:
    saved_tests = read_json(DATASET_DIR / "dataset_tests.json")
    print("Train packed sequences:", saved_tests["splits"]["train"]["shape"][0])
    print("Validation packed sequences:", saved_tests["splits"]["validation"]["shape"][0])
    print("Total train packed tokens:", saved_tests["splits"]["train"]["packed_input_tokens"])
else:
    print("Packed dataset: chưa PASS tests.")

print("Pilot selected tokens:", PILOT["selected_input_tokens"] if PILOT_READY else "NOT READY")
print("Local dataset directory:", DATASET_DIR)
print("Google Drive snapshot:", DATASET_BACKUP_RESULT["snapshot"])
print("Backup verification:", DATASET_BACKUP_RESULT["status"])
print("Local dataset readiness:", "PASS" if DATASET_TESTS_PASS and PILOT_READY else "FAIL")
print(
    "Dataset readiness:",
    "PASS" if DATASET_TESTS_PASS and PILOT_READY
    and DATASET_BACKUP_RESULT["status"] == "PASS" else "FAIL",
)
print("Chi tiết:", DATASET_BACKUP_RESULT["reason"])
print(
    "Checksum trên Drive mount xác nhận nội dung đọc lại tại thời điểm kiểm tra, "
    "không bảo đảm tuyệt đối dịch vụ đã đồng bộ xong."
)
print("Không xóa local data, không train mô hình và không chuyển sang STEP 6.5.")

Backup đang tạo: /content/drive/MyDrive/MoE_ERC/tokenized_dataset_snapshots/dataset-20261024T071437Z-92acf38d5017-201fa9ae45a4.INCOMPLETE

=== FINAL REPORT ===
Combined source dataset fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Train packed sequences: 107478
Validation packed sequences: 1006
Total train packed tokens: 55028736
Pilot selected tokens: 50000384
Local dataset directory: /content/moe-erc/data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9
Google Drive snapshot: /content/drive/MyDrive/MoE_ERC/tokenized_dataset_snapshots/dataset-20261024T071437Z-92acf38d5017-201fa9ae45a4
Backup verification: PASS
Local dataset readiness: PASS
Dataset readiness: PASS
Chi tiết: SHA-256, metadata, pilot và Drive DataLoader đều PASS.
Checksum trên Drive mount xác nhận nội dung đọc lại tại thời điểm kiểm tra, không bảo đảm tuyệt đối dịch vụ đã đồng bộ xong.
Không xóa local d

In [ ]:
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/MoE_ERC/tokenized_dataset_snapshots"
)

snapshot_name = "dataset-20261024T071437Z-92acf38d5017-201fa9ae45a4"

snapshot = root / snapshot_name

print("Snapshot exists:", snapshot.is_dir())
print("COMPLETE.json:", (snapshot / "COMPLETE.json").is_file())
print("INCOMPLETE.json:", (snapshot / "INCOMPLETE.json").exists())
print("Temporary directory:", (root / (snapshot_name + ".INCOMPLETE")).exists())

Snapshot exists: True
COMPLETE.json: True
INCOMPLETE.json: False
Temporary directory: False


# Step 6.5A

In [ ]:
from pathlib import Path, PurePosixPath
from contextlib import closing
import hashlib
import inspect
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import traceback

PROJECT = Path("/content/moe-erc")
TESTED_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
COMBINED_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"

DATASET_SNAPSHOT = Path(
    "/content/drive/MyDrive/MoE_ERC/tokenized_dataset_snapshots/"
    "dataset-20261024T071437Z-92acf38d5017-201fa9ae45a4"
)
TOKENIZER_SNAPSHOT = Path(
    "/content/drive/MyDrive/MoE_ERC/tokenizer_snapshots/"
    "dolma-bpe16k-20261024T044101Z-2412fe19bc1a0d22-1a4739a56d17"
)
DATASET_TARGET = (
    PROJECT / "data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
)
TOKENIZER_TARGET_DIR = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22"
TOKENIZER_TARGET = TOKENIZER_TARGET_DIR / "tokenizer.json"

SNAPSHOTS_VERIFIED = False
SNAPSHOT_INFO = {}
RESTORE_RESULT = {"status": "NOT RUN", "reason": "Chưa chạy restore."}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

require(
    PROJECT.is_dir(),
    "Không tìm thấy /content/moe-erc. Hãy chạy lại phần clone STEP 6.2 "
    "bằng repository URL của bạn; cell này không tự đoán URL.",
)
require(PROJECT.resolve() == PROJECT, "Project root đi qua symlink.")

try:
    from google.colab import drive
    import numpy as np
    import torch
    import tokenizers
except ImportError as exc:
    raise RuntimeError(
        "Cần chạy trên Colab với dependencies của project. "
        "Không tự cài lại hoặc cập nhật PyTorch trong bước này."
    ) from exc

if str(PROJECT) in sys.path:
    sys.path.remove(str(PROJECT))
sys.path.insert(0, str(PROJECT))

try:
    from moe_erc.config import ModelConfig, AoEConfig
    from moe_erc.model import MoELanguageModel, AoELanguageModel
    from moe_erc.model_factory import create_model
    from moe_erc.tokenizer_utils import FrozenTokenizer, SplitConfig
    from moe_erc.data_processing import (
        prepare_dataset, validate_dataset, audit_document_splits,
    )
    from moe_erc.dataset import PackedTokenDataset, make_dataloader
except ImportError as exc:
    raise RuntimeError(
        "Import project thất bại. Kiểm tra dependencies STEP 6.2; "
        "không sửa source hoặc thay thế PyTorch tự động."
    ) from exc

for obj in (FrozenTokenizer, PackedTokenDataset, prepare_dataset, MoELanguageModel):
    require(
        Path(inspect.getfile(obj)).resolve().is_relative_to(PROJECT),
        f"Import nhầm package ngoài project: {obj.__name__}",
    )

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
RUNTIME_CODE_SHA = {
    name: sha256_file(PROJECT / "moe_erc" / name)
    for name in ("data_processing.py", "dataset.py", "tokenizer_utils.py", "model.py")
}

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = "NO CUDA GPU"
BF16_SUPPORTED = False
GPU_IS_A100 = False

if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    GPU_NAME = properties.name
    with torch.cuda.device(0):
        BF16_SUPPORTED = bool(
            properties.major >= 8 and torch.cuda.is_bf16_supported()
        )
    GPU_IS_A100 = "A100" in GPU_NAME.upper()
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    print(f"VRAM total/free: {total_vram / 1024**3:.2f} / {free_vram / 1024**3:.2f} GiB")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch build CUDA:", torch.version.cuda)
print("CUDA available:", CUDA_AVAILABLE)
print("GPU:", GPU_NAME)
print("GPU count:", torch.cuda.device_count())
print("BF16:", BF16_SUPPORTED)
print("A100:", GPU_IS_A100)
if not GPU_IS_A100:
    print("Runtime hiện tại không phải A100; restore vẫn có thể thực hiện.")
    print("Chọn Runtime > Change runtime type > GPU/A100 nếu tài khoản cho phép.")

print("Git HEAD:", GIT_COMMIT)
print("Git giống commit đã kiểm thử:", GIT_COMMIT == TESTED_COMMIT)
print("Model/dataset/tokenizer/packing imports: PASS")
print("Chưa khởi tạo mô hình hoặc chạy forward/backward.")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
require(Path("/content/drive/MyDrive").is_dir(), "Google Drive chưa mount.")
print("Google Drive: mounted")

VRAM total/free: 39.49 / 38.96 GiB
Python: 3.13.16
PyTorch: 2.11.0+cu130
PyTorch build CUDA: 13.0
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU count: 1
BF16: True
A100: True
Git HEAD: 665cb7b1cfbca4c6ed3507345b21008321129a2a
Git giống commit đã kiểm thử: False
Model/dataset/tokenizer/packing imports: PASS
Chưa khởi tạo mô hình hoặc chạy forward/backward.
Mounted at /content/drive
Google Drive: mounted


In [ ]:
def safe_relative(name):
    require(isinstance(name, str) and bool(name), "Tên file không hợp lệ.")
    relative = PurePosixPath(name)
    require(
        not relative.is_absolute()
        and ".." not in relative.parts
        and relative.as_posix() == name
        and "\\" not in name and ":" not in name
        and not any(ord(char) < 32 for char in name),
        f"Path không an toàn: {name!r}",
    )
    return relative

def checked_file(root, name):
    relative = safe_relative(name)
    root = Path(root)
    require(not root.is_symlink(), f"Root là symlink: {root}")
    current = root
    for component in relative.parts:
        current = current / component
        require(not current.is_symlink(), f"Symlink không được phép: {current}")
    require(current.resolve().is_relative_to(root.resolve()), "Path escape.")
    require(current.is_file(), f"Thiếu file: {current}")
    return current

def file_inventory(root):
    result = set()
    for path in Path(root).rglob("*"):
        require(not path.is_symlink(), f"Symlink trong artifact: {path}")
        if path.is_file():
            result.add(path.relative_to(root).as_posix())
        else:
            require(path.is_dir(), f"Loại file không hỗ trợ: {path}")
    return result

def verify_entries(root, entries, exact=False):
    names = set()
    for entry in entries:
        name = safe_relative(entry["path"]).as_posix()
        require(name not in names, f"Entry trùng: {name}")
        names.add(name)
        require(type(entry["bytes"]) is int and entry["bytes"] >= 0, "Size sai.")
        path = checked_file(root, name)
        require(path.stat().st_size == entry["bytes"], f"Size FAIL: {name}")
        require(sha256_file(path) == entry["sha256"], f"SHA-256 FAIL: {name}")
    for name in names:
        require(
            not any(parent.as_posix() in names for parent in PurePosixPath(name).parents),
            f"Xung đột file/thư mục: {name}",
        )
    if exact:
        require(file_inventory(root) == names, f"Local inventory không khớp: {root}")
    return names

def verify_snapshot(root, kind, expected_manifest_sha=None):
    root = Path(root)
    require(root.is_dir() and not root.is_symlink(), f"Thiếu snapshot: {root}")
    for marker in ("INCOMPLETE.json", "COMPLETE.pending.json"):
        require(not os.path.lexists(root / marker), f"Snapshot chưa hoàn thành: {marker}")

    for name in ("backup_manifest.json", "backup_manifest.sha256", "COMPLETE.json"):
        checked_file(root, name)
    digest = sha256_file(root / "backup_manifest.json")
    require(
        digest == (root / "backup_manifest.sha256").read_text().strip(),
        "Backup manifest checksum sai.",
    )
    if expected_manifest_sha:
        require(digest == expected_manifest_sha, "Snapshot thay đổi sau lần xác minh trước.")
    marker = read_json(root / "COMPLETE.json")
    require(
        marker["status"] == "COMPLETE" and marker["manifest_sha256"] == digest,
        "COMPLETE marker sai.",
    )

    manifest = read_json(root / "backup_manifest.json")
    require(manifest["format_version"] == 1, "Backup format chưa hỗ trợ.")
    require(manifest["tokenizer_sha256"] == TOKENIZER_SHA, "Backup tokenizer identity sai.")
    if kind == "dataset":
        require(manifest["kind"] == "dataset", "Không phải dataset snapshot.")
        require(
            manifest["combined_source_fingerprint"] == COMBINED_FP,
            "Combined source fingerprint sai.",
        )
        require(
            all(PurePosixPath(e["path"]).parts[0] == "dataset" for e in manifest["files"]),
            "Dataset payload nằm ngoài dataset/.",
        )
    else:
        # Tokenizer backup STEP 6.3.4 không có trường kind.
        require(manifest["vocab_size"] == 16000 and manifest["eos_token_id"] == 2, "Vocab/EOS sai.")

    names = verify_entries(root, manifest["files"])
    expected_files = names | {
        "backup_manifest.json", "backup_manifest.sha256", "COMPLETE.json"
    }
    require(file_inventory(root) == expected_files, "Snapshot inventory khác manifest.")

    return {
        "root": root,
        "manifest_sha256": digest,
        "manifest": manifest,
        "files": sorted(expected_files),
        "total_bytes": sum((root / name).stat().st_size for name in expected_files),
    }

def verify_tokenizer_dir(root):
    root = Path(root)
    frozen = FrozenTokenizer(root / "tokenizer.json")
    require(frozen.fingerprint == TOKENIZER_SHA, "Tokenizer SHA sai.")
    require(frozen.vocab_size == 16000 and frozen.eos_id == 2, "Tokenizer vocab/EOS sai.")
    require(
        frozen.special_ids == {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3},
        "Special IDs sai.",
    )
    frozen.validate_model(ModelConfig())
    frozen.validate_model(AoEConfig())

    research = read_json(root / "research_metadata.json")
    require(research["tests"]["status"] == "PASS", "Tokenizer tests metadata chưa PASS.")
    for name, expected in research["artifact_sha256"].items():
        require(sha256_file(checked_file(root, name)) == expected, "Tokenizer artifact SHA sai.")
    probe = research["tests"]["reload_probe"]
    require(frozen.encode(probe["text"]) == probe["ids"], "Tokenizer reload IDs sai.")
    return frozen

import ast
import gc
from functools import lru_cache

APPROVED_DATASET_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
APPROVED_FIX_COMMIT = "665cb7b1cfbca4c6ed3507345b21008321129a2a"
APPROVED_OLD_MODEL_SHA = "53d4ad3cade511a70c8c0944561ff9ed8437cbc8cbabc80b1d32a81a2fe9dfbe"
APPROVED_FIX_MODEL_SHA = "2389cc7be64b8f68a85ccbe37769a0786754dcce4b36c9859ddb53f68b8c7166"

COMPAT_REPORT = {
    "approved": False,
    "original_dataset_creation_commit": None,
    "current_fix_commit": None,
    "different_recorded_files": [],
    "initialization": {},
}

def compat_git(*arguments):
    try:
        return subprocess.check_output(
            ["git", "-C", str(PROJECT), *arguments],
            stderr=subprocess.PIPE,
        )
    except subprocess.CalledProcessError as exc:
        detail = exc.stderr.decode("utf-8", errors="replace")
        raise RuntimeError(
            "Không đọc được Git objects cần xác minh. "
            "Không tự fetch/checkout/reset.\n" + detail
        ) from None

@lru_cache(maxsize=None)
def compat_blob(commit, relative_path):
    # Git object bất biến; không cache nội dung working tree.
    return compat_git("show", f"{commit}:{relative_path}")

@lru_cache(maxsize=None)
def compat_package_files(commit):
    names = compat_git(
        "ls-tree", "-r", "--name-only", commit, "--", "moe_erc"
    ).decode("utf-8").splitlines()
    return tuple(sorted(name for name in names if name.endswith(".py")))

def compat_sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

@lru_cache(maxsize=1)
def compat_verify_approved_ast_delta():
    old_bytes = compat_blob(APPROVED_DATASET_COMMIT, "moe_erc/model.py")
    fix_bytes = compat_blob(APPROVED_FIX_COMMIT, "moe_erc/model.py")
    require(compat_sha_bytes(old_bytes) == APPROVED_OLD_MODEL_SHA, "Old model Git blob SHA sai.")
    require(compat_sha_bytes(fix_bytes) == APPROVED_FIX_MODEL_SHA, "FIX model Git blob SHA sai.")

    old_tree = ast.parse(old_bytes.decode("utf-8"))
    fix_tree = ast.parse(fix_bytes.decode("utf-8"))
    model_class = next(
        node for node in fix_tree.body
        if isinstance(node, ast.ClassDef) and node.name == "MoELanguageModel"
    )
    constructor = next(
        node for node in model_class.body
        if isinstance(node, ast.FunctionDef) and node.name == "__init__"
    )
    expected_statement = ast.parse(
        "nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)"
    ).body[0]
    matches = [
        index for index, node in enumerate(constructor.body)
        if ast.dump(node) == ast.dump(expected_statement)
    ]
    require(
        len(matches) == 1 and matches[0] == len(constructor.body) - 1,
        "FIX phải có đúng một explicit initializer ở cuối constructor.",
    )
    index = matches[0]
    tying_statement = ast.parse(
        "self.lm_head.weight = self.token_embedding.weight"
    ).body[0]
    require(
        index > 0
        and ast.dump(constructor.body[index - 1]) == ast.dump(tying_statement),
        "Initializer không nằm ngay sau weight tying đã phê duyệt.",
    )

    # Chỉ bỏ statement đã phê duyệt trong AST RAM, không sửa source.
    constructor.body.pop(index)
    require(
        ast.dump(old_tree) == ast.dump(fix_tree),
        "model.py có thay đổi logic khác ngoài initializer đã phê duyệt.",
    )
    return True

def compat_test_runtime_initialization():
    results = {}
    expected_counts = {"vanilla_moe": 52_288_000, "aoe": 52_263_424}

    for kind, expected_count in expected_counts.items():
        temporary_model = None
        try:
            # Bảo toàn RNG của notebook; không chạy model forward.
            with torch.random.fork_rng(
                devices=list(range(torch.cuda.device_count()))
            ):
                torch.manual_seed(42)
                temporary_model = create_model(kind).float().cpu()

            embedding = temporary_model.token_embedding.weight
            head = temporary_model.lm_head.weight
            measured_std = float(embedding.detach().std(unbiased=False))
            tied = embedding is head
            same_storage = (
                embedding.untyped_storage().data_ptr()
                == head.untyped_storage().data_ptr()
            )
            total = sum(p.numel() for p in temporary_model.parameters())

            require(tied and same_storage, f"{kind}: weight tying/storage sai.")
            require(tuple(embedding.shape) == (16000, 512), f"{kind}: embedding shape sai.")
            require(abs(measured_std - 0.02) < 1e-4, f"{kind}: embedding std sai.")
            require(total == expected_count, f"{kind}: parameter count sai.")

            results[kind] = {
                "embedding_std": measured_std,
                "lm_head_std": float(head.detach().std(unbiased=False)),
                "same_parameter": tied,
                "same_storage": same_storage,
                "parameters": total,
                "status": "PASS",
            }
        finally:
            temporary_model = None
            # embedding/head vẫn là references local nếu đã được gán.
            embedding = head = None
            gc.collect()

    return results

def check_current_code(design):
    """Strict data-code identity, with one pinned model initialization exception."""
    COMPAT_REPORT["approved"] = False

    current_commit = compat_git("rev-parse", "HEAD").decode().strip()
    COMPAT_REPORT["current_fix_commit"] = current_commit
    require(
        current_commit == APPROVED_FIX_COMMIT,
        f"Git HEAD phải đúng FIX commit {APPROVED_FIX_COMMIT}; hiện tại {current_commit}.",
    )

    # Đọc commit tạo dataset từ metadata thật, không thay bằng current HEAD.
    creation_metadata = read_json(
        DATASET_SNAPSHOT / "dataset" / "dataset.json"
    )
    creation_commit = creation_metadata["plan"]["provenance"]["git_commit"]
    COMPAT_REPORT["original_dataset_creation_commit"] = creation_commit
    require(
        creation_commit == APPROVED_DATASET_COMMIT,
        "Dataset creation commit không thuộc ngoại lệ đã phê duyệt.",
    )

    recorded = design["pipeline_source_sha256"]
    require(
        set(recorded) == set(RUNTIME_CODE_SHA)
        == {"data_processing.py", "dataset.py", "tokenizer_utils.py", "model.py"},
        "Danh sách protected source files khác format đã kiểm tra.",
    )
    require(
        recorded["model.py"] == APPROVED_OLD_MODEL_SHA,
        "Stored model.py checksum không thuộc bản tạo dataset đã phê duyệt.",
    )

    old_files = compat_package_files(APPROVED_DATASET_COMMIT)
    fix_files = compat_package_files(APPROVED_FIX_COMMIT)
    require(old_files == fix_files, "Có thêm/bớt runtime Python files trong FIX.")
    disk_files = tuple(sorted(
        path.relative_to(PROJECT).as_posix()
        for path in (PROJECT / "moe_erc").rglob("*.py")
    ))
    require(disk_files == fix_files, "Working tree có runtime Python files ngoài dự kiến.")

    commit_differences = []
    for name in fix_files:
        old_bytes = compat_blob(APPROVED_DATASET_COMMIT, name)
        fix_bytes = compat_blob(APPROVED_FIX_COMMIT, name)
        if old_bytes != fix_bytes:
            commit_differences.append(name)

        path = PROJECT / name
        require(not path.is_symlink(), f"Source symlink: {path}")
        require(
            sha256_file(path) == compat_sha_bytes(fix_bytes),
            f"Working-tree source khác FIX commit: {name}",
        )

    require(
        commit_differences == ["moe_erc/model.py"],
        f"Runtime package có thay đổi ngoài phê duyệt: {commit_differences}",
    )
    compat_verify_approved_ast_delta()

    differences = []
    for name, old_sha in recorded.items():
        current_sha = sha256_file(PROJECT / "moe_erc" / name)
        require(
            RUNTIME_CODE_SHA[name] == current_sha,
            f"Source thay đổi sau import Cell 1: {name}; restart/run lại preflight.",
        )
        if current_sha != old_sha:
            differences.append({
                "file": name,
                "dataset_creation_sha256": old_sha,
                "current_sha256": current_sha,
            })
            require(
                name == "model.py"
                and old_sha == APPROVED_OLD_MODEL_SHA
                and current_sha == APPROVED_FIX_MODEL_SHA,
                f"Không cho phép ngoại lệ cho {name}.",
            )
        else:
            require(
                old_sha == compat_sha_bytes(
                    compat_blob(APPROVED_DATASET_COMMIT, f"moe_erc/{name}")
                ),
                f"Stored protected checksum khác creation commit: {name}",
            )

    require(
        [entry["file"] for entry in differences] == ["model.py"],
        "Chỉ cho phép đúng một khác biệt model.py đã phê duyệt.",
    )
    COMPAT_REPORT["different_recorded_files"] = differences
    COMPAT_REPORT["initialization"] = compat_test_runtime_initialization()
    COMPAT_REPORT.update({
        "approved": True,
        "reason": (
            "Pinned FIX commit/blob; AST chỉ thêm Normal(0,0.02) đúng một lần "
            "sau tying; các runtime files khác giữ nguyên bytes."
        ),
    })

def verify_dataset_bundle(root):
    root = Path(root)
    marker = read_json(checked_file(root, "COMPLETE.json"))
    require(marker["status"] == "COMPLETE" and marker["tests_status"] == "PASS", "Dataset chưa COMPLETE.")
    require(
        sha256_file(checked_file(root, "dataset_manifest.json"))
        == marker["dataset_manifest_sha256"],
        "Dataset manifest SHA sai.",
    )
    core = read_json(root / "dataset_manifest.json")
    require(core["combined_source_fingerprint"] == COMBINED_FP, "Dataset fingerprint sai.")
    require(core["tokenizer_sha256"] == TOKENIZER_SHA, "Dataset tokenizer sai.")
    verify_entries(root, core["files"])

    design = read_json(root / "packing_design.json")
    require(fingerprint(design) == core["design_sha256"], "Packing design SHA sai.")
    require(design["combined_source_fingerprint"] == COMBINED_FP, "Design source identity sai.")
    check_current_code(design)

    frozen = verify_tokenizer_dir(TOKENIZER_SNAPSHOT)
    bundled = FrozenTokenizer(root / "tokenizer/tokenizer.json")
    require(bundled.fingerprint == frozen.fingerprint == TOKENIZER_SHA, "Bundled tokenizer sai.")
    require(
        sha256_file(root / "tokenizer/tokenizer.meta.json")
        == sha256_file(TOKENIZER_SNAPSHOT / "tokenizer.meta.json"),
        "Bundled tokenizer metadata khác backup tokenizer.",
    )

    metadata = validate_dataset(root, tokenizer=bundled, checksums=True)
    require(metadata["dtype"] == "<u2" and metadata["sequence_length"] == 512, "Storage convention sai.")
    require(
        metadata["plan"]["provenance"]["combined_source_fingerprint"] == COMBINED_FP
        and metadata["plan"]["provenance"]["design_sha256"] == core["design_sha256"],
        "Pipeline provenance sai.",
    )

    tests = read_json(root / "dataset_tests.json")
    require(
        tests["status"] == "PASS"
        and tests["full_reference_comparison"] == "PASS"
        and tests["document_overlap"] == 0,
        "Dataset test record sai.",
    )

    expected = {
        "train": (107478, 55028736, 54921258),
        "validation": (1006, 515072, 514066),
    }
    segment_names = set()
    for split, (sequences, tokens, targets) in expected.items():
        info = metadata["splits"][split]
        require(info["sequences"] == sequences and info["tokens"] == tokens, f"Counts sai: {split}")
        require(sequences * 511 == targets, "Effective targets sai.")
        require(tests["splits"][split]["shape"] == [sequences, 512], "Test shape metadata sai.")
        require(info["budget_discarded_tokens"] == 0, "Có budget truncation không dự kiến.")
        split_fp = fingerprint({
            "sequence_length": 512, "dtype": "<u2", "segments": info["segments"],
        })
        require(
            core["split_fingerprints"][split]
            == tests["splits"][split]["fingerprint"] == split_fp,
            "Binary split fingerprint sai.",
        )
        for entry in info["segments"]:
            require(entry["file"] not in segment_names, "Segment bị dùng lại giữa các splits.")
            segment_names.add(entry["file"])

    # Ledger: chỉ đọc SQLite sealed, không tạo WAL/SHM trên Drive.
    split_config = SplitConfig(**metadata["plan"]["config"]["split"])
    ledger = {"train": 0, "validation": 0}
    with closing(sqlite3.connect(
        (root / "documents.sqlite").resolve().as_uri() + "?mode=ro&immutable=1",
        uri=True,
    )) as connection:
        require(
            connection.execute("PRAGMA integrity_check").fetchall() == [("ok",)],
            "Document ledger integrity FAIL.",
        )
        total, unique = connection.execute(
            "SELECT COUNT(*), COUNT(DISTINCT hash) FROM documents"
        ).fetchone()
        require(total == unique, "Ledger có duplicate hashes.")
        for digest, assigned in connection.execute("SELECT hash, split FROM documents"):
            require(assigned == split_config.assign(digest), "Ledger split sai.")
            ledger[assigned] += 1
    require(
        all(ledger[s] == metadata["splits"][s]["documents"] for s in ledger),
        "Ledger document counts sai.",
    )

    pilot = read_json(checked_file(root, "pilot_50m_manifest.json"))
    pilot_sha = sha256_file(root / "pilot_50m_manifest.json")
    require(
        pilot_sha == (root / "pilot_50m_manifest.sha256").read_text().strip(),
        "Pilot manifest SHA sai.",
    )
    require(
        pilot["split"] == "train"
        and pilot["tokenizer_sha256"] == TOKENIZER_SHA
        and pilot["combined_source_fingerprint"] == COMBINED_FP
        and pilot["dataset_manifest_sha256"] == marker["dataset_manifest_sha256"],
        "Pilot trỏ tới dataset khác.",
    )
    require(
        pilot["train_binary_fingerprint"] == core["split_fingerprints"]["train"]
        and pilot["validation_binary_fingerprint"] == core["split_fingerprints"]["validation"],
        "Pilot binary fingerprints sai.",
    )
    index_path = checked_file(root, pilot["index_file"])
    require(sha256_file(index_path) == pilot["index_file_sha256"], "Pilot index SHA sai.")
    indices = np.load(index_path, allow_pickle=False)
    require(indices.ndim == 1 and indices.dtype.str == pilot["index_dtype"] == "<i8", "Index format sai.")
    require(
        len(indices) == pilot["selected_sequences"] == 97657
        and pilot["selected_input_tokens"] == len(indices) * 512 == 50000384,
        "Pilot token/sequence count sai.",
    )
    require(
        int(indices.min()) >= 0 and int(indices.max()) < 107478,
        "Pilot index ngoài training split.",
    )
    require(
        pilot["repeat_sequences"] is False and len(np.unique(indices)) == len(indices),
        "Pilot indices bị lặp.",
    )

    allowed = {entry["path"] for entry in core["files"]} | {
        "dataset_manifest.json", "COMPLETE.json",
        "pilot_50m_manifest.json", "pilot_50m_manifest.sha256", pilot["index_file"],
    }
    require(file_inventory(root) == allowed, "Dataset bundle có file thiếu/không được ghi nhận.")
    return {
        "metadata": metadata, "pilot": pilot,
        "dataset_manifest_sha256": marker["dataset_manifest_sha256"],
        "pilot_manifest_sha256": pilot_sha,
        "ledger": ledger,
    }

SNAPSHOT_INFO["tokenizer"] = verify_snapshot(TOKENIZER_SNAPSHOT, "tokenizer")
SNAPSHOT_INFO["dataset"] = verify_snapshot(DATASET_SNAPSHOT, "dataset")
verify_tokenizer_dir(TOKENIZER_SNAPSHOT)
VERIFIED_BUNDLE = verify_dataset_bundle(DATASET_SNAPSHOT / "dataset")

for kind, info in SNAPSHOT_INFO.items():
    print(
        f"{kind}: PASS | {len(info['files'])} files | "
        f"{info['total_bytes'] / 1024**2:.2f} MiB"
    )
    print("  Manifest SHA-256:", info["manifest_sha256"])
print("Pilot: 97657 indices, không lặp, giữ nguyên thứ tự đã lưu.")
print("Packed arrays, ledger và code hashes: PASS")
if GIT_COMMIT != TESTED_COMMIT:
    print("Git HEAD khác commit cũ, nhưng bốn protected modules khớp snapshot.")
print("Full reference comparison là audit đã lưu; không chạy lại corpus/tokenization.")
SNAPSHOTS_VERIFIED = True

tokenizer: PASS | 6 files | 1.09 MiB
  Manifest SHA-256: 96d30a579382e21c51ab6a83199ed3f5ffcedc187142c1a4c3d0cf617c56e0a8
dataset: PASS | 23 files | 122.61 MiB
  Manifest SHA-256: 53dc7e7ffb5b000eb970ee519071edb5605fe97cd7cb978f6af71a3d1e6855c8
Pilot: 97657 indices, không lặp, giữ nguyên thứ tự đã lưu.
Packed arrays, ledger và code hashes: PASS
Git HEAD khác commit cũ, nhưng bốn protected modules khớp snapshot.
Full reference comparison là audit đã lưu; không chạy lại corpus/tokenization.


In [ ]:
RUN_RESTORE = True

TOKENIZER_ENTRIES = SNAPSHOT_INFO["tokenizer"]["manifest"]["files"]
DATASET_ENTRIES = [
    {**entry, "path": PurePosixPath(entry["path"]).relative_to("dataset").as_posix()}
    for entry in SNAPSHOT_INFO["dataset"]["manifest"]["files"]
]

def check_destination_path(target):
    target = Path(target)
    relative = target.relative_to(PROJECT)
    current = PROJECT
    for index, part in enumerate(relative.parts):
        current = current / part
        require(not current.is_symlink(), f"Destination symlink: {current}")
        if index < len(relative.parts) - 1 and current.exists():
            require(current.is_dir(), f"Thư mục cha bị chặn bởi file: {current}")
    require(target.resolve().is_relative_to(PROJECT / "data"), "Destination path escape.")

def verify_local_artifact(root, entries, kind):
    check_destination_path(root)
    require(Path(root).is_dir(), f"Local artifact không phải directory: {root}")
    verify_entries(root, entries, exact=True)
    return (
        verify_tokenizer_dir(root) if kind == "tokenizer"
        else verify_dataset_bundle(root)
    )

def copy_exclusive(source, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    with Path(source).open("rb") as src, target.open("xb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024)
        dst.flush()
        os.fsync(dst.fileno())

def restore_artifacts():
    require(SNAPSHOTS_VERIFIED, "Cell 2 chưa PASS.")
    result = {"status": "FAIL", "staging": None, "published": [], "kept_existing": []}
    progress_path = None

    def progress():
        if progress_path is not None:
            temporary = progress_path.with_suffix(".tmp")
            with temporary.open("w", encoding="utf-8") as stream:
                json.dump(result, stream, ensure_ascii=False, indent=2)
                stream.flush()
                os.fsync(stream.fileno())
            temporary.replace(progress_path)

    specs = {
        "tokenizer": (TOKENIZER_TARGET_DIR, TOKENIZER_ENTRIES),
        "dataset": (DATASET_TARGET, DATASET_ENTRIES),
    }
    try:
        needed = []
        for kind, (target, entries) in specs.items():
            check_destination_path(target)
            if os.path.lexists(target):
                verify_local_artifact(target, entries, kind)
                result["kept_existing"].append(str(target))
                print("Giữ nguyên local artifact đã xác minh:", target)
            else:
                needed.append(kind)

        if not needed:
            result.update(status="PASS", reason="Cả hai artifact local đã khớp backup.")
            return result

        required = sum(
            SNAPSHOT_INFO[kind]["total_bytes"] * 2 for kind in needed
        ) + 256 * 1024**2
        require(shutil.disk_usage(PROJECT).free >= required, "Không đủ disk cho staging.")

        check_destination_path(PROJECT / "data")
        (PROJECT / "data").mkdir(exist_ok=True)
        stage_base = Path(tempfile.mkdtemp(prefix=".restore65a-", dir=PROJECT / "data"))
        result["staging"] = str(stage_base)
        progress_path = stage_base / "restore_progress.json"
        result["phase"] = "COPYING"
        progress()
        publish_sources = {}

        for kind in needed:
            info = SNAPSHOT_INFO[kind]
            # Không dùng snapshot đã thay đổi kể từ Cell 2.
            verify_snapshot(info["root"], kind, info["manifest_sha256"])
            stage_snapshot = stage_base / f"{kind}_snapshot"
            stage_snapshot.mkdir()

            for name in info["files"]:
                copy_exclusive(
                    checked_file(info["root"], name),
                    stage_snapshot / name,
                )
            verify_snapshot(stage_snapshot, kind, info["manifest_sha256"])

            if kind == "dataset":
                payload = stage_snapshot / "dataset"
            else:
                # Không đưa backup control files vào tokenizer directory.
                payload = stage_base / "tokenizer_payload"
                payload.mkdir()
                for entry in TOKENIZER_ENTRIES:
                    copy_exclusive(
                        checked_file(stage_snapshot, entry["path"]),
                        payload / entry["path"],
                    )

            entries = specs[kind][1]
            verify_entries(payload, entries, exact=True)
            if kind == "dataset":
                verify_dataset_bundle(payload)
            else:
                verify_tokenizer_dir(payload)
            publish_sources[kind] = payload
            print("Staging verified:", kind)

        # Chỉ publish sau khi TẤT CẢ artifact cần copy đã PASS.
        for kind, (target, entries) in specs.items():
            check_destination_path(target)
            if kind in needed:
                require(not os.path.lexists(target), f"Đích mới xuất hiện: {target}")
            else:
                verify_local_artifact(target, entries, kind)

        result["phase"] = "PUBLISHING"
        progress()
        for kind in needed:
            target, _ = specs[kind]
            target.parent.mkdir(parents=True, exist_ok=True)
            check_destination_path(target)
            require(not os.path.lexists(target), f"Không overwrite: {target}")
            publish_sources[kind].rename(target)
            result["published"].append(str(target))
            progress()

        for kind, (target, entries) in specs.items():
            verify_local_artifact(target, entries, kind)

        result.update(
            status="PASS", phase="FINISHED",
            reason="Cả hai local artifacts đã được đối chiếu checksum và metadata.",
        )
        progress()
    except BaseException as exc:
        result.update(status="FAIL", reason=f"{type(exc).__name__}: {exc}")
        try:
            progress()
        except Exception as log_error:
            print("Không lưu được progress log:", log_error)
        print("RESTORE FAIL/INTERRUPTED. Giữ staging và các artifact đã publish.")
        print("Không tự xóa, rollback hoặc overwrite khi chạy lại.")
        traceback.print_exc()
    return result

if not RUN_RESTORE:
    print("RUN_RESTORE=False: chưa copy hoặc publish dữ liệu.")
    print("Cell 4 vẫn có thể xác minh artifact local nếu đã restore từ trước.")
else:
    RESTORE_RESULT = restore_artifacts()
    print(json.dumps(RESTORE_RESULT, ensure_ascii=False, indent=2))

Staging verified: tokenizer
Staging verified: dataset
{
  "status": "PASS",
  "staging": "/content/moe-erc/data/.restore65a-pz7qtdi7",
  "published": [
    "/content/moe-erc/data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22",
    "/content/moe-erc/data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
  ],
  "kept_existing": [],
  "phase": "FINISHED",
  "reason": "Cả hai local artifacts đã được đối chiếu checksum và metadata."
}


In [ ]:
READINESS = {
    "status": "FAIL",
    "dataset_integrity": "NOT RUN",
    "restore_verification": "NOT RUN",
    "dataloader": "NOT RUN",
    "fresh_process_reload": "NOT RUN",
    "reason": "",
}

try:
    require(SNAPSHOTS_VERIFIED, "Cell 2 snapshot verification chưa PASS.")

    # Giữ nguyên strict file inventory, size và SHA-256 của STEP 6.5A.
    frozen = verify_local_artifact(
        TOKENIZER_TARGET_DIR, TOKENIZER_ENTRIES, "tokenizer"
    )
    bundle = verify_local_artifact(
        DATASET_TARGET, DATASET_ENTRIES, "dataset"
    )
    # verify_dataset_bundle gọi check_current_code mới,
    # đồng thời giữ nguyên checksums/fingerprints trong dataset.
    metadata = bundle["metadata"]
    pilot = bundle["pilot"]
    indices = np.load(
        DATASET_TARGET / pilot["index_file"], allow_pickle=False
    )

    require(COMPAT_REPORT["approved"], "Code compatibility chưa được chấp nhận.")
    require(frozen.fingerprint == TOKENIZER_SHA, "Tokenizer checksum sai.")
    require(metadata["dtype"] == "<u2", "Storage không phải uint16.")
    require(metadata["sequence_length"] == 512, "Sequence length sai.")
    require(metadata["splits"]["train"]["sequences"] == 107478, "Train sequences sai.")
    require(metadata["splits"]["validation"]["sequences"] == 1006, "Validation sequences sai.")
    require(pilot["selected_input_tokens"] == 50000384, "Pilot token budget sai.")
    require(
        len(indices) == 97657 and len(np.unique(indices)) == len(indices),
        "Pilot duplicate/count sai.",
    )
    require(pilot["split"] == "train", "Pilot không thuộc training split.")

    ledger = audit_document_splits(DATASET_TARGET)
    require(ledger["overlap"] == 0, "Document leakage.")
    for split in ("train", "validation"):
        require(
            ledger[split] == metadata["splits"][split]["documents"],
            f"Ledger count sai: {split}",
        )

    READINESS["dataset_integrity"] = "PASS"
    READINESS["restore_verification"] = "PASS"

    for split, expected_count in (("train", 107478), ("validation", 1006)):
        dataset = PackedTokenDataset(
            DATASET_TARGET, split, tokenizer=frozen, verify_checksums=True
        )
        try:
            require(len(dataset) == expected_count, "Dataset length sai.")
            require(dataset.sequence_length == 512, "Dataset context sai.")
            sample = dataset[0]
            require(tuple(sample["input_ids"].shape) == (512,), "Sequence shape sai.")
            require(sample["input_ids"].dtype == sample["labels"].dtype == torch.long, "Dtype sai.")
            require(torch.equal(sample["input_ids"], sample["labels"]), "Labels bị shift.")
            require(
                sample["input_ids"].data_ptr() != sample["labels"].data_ptr(),
                "Labels không phải clone.",
            )

            batch = next(iter(make_dataloader(
                dataset, batch_size=2, seed=42, num_workers=0, pin_memory=False
            )))
            require(tuple(batch["input_ids"].shape) == (2, 512), "Batch shape sai.")
            require(batch["input_ids"].dtype == torch.long, "Batch dtype sai.")
            require(
                int(batch["input_ids"].min()) >= 0
                and int(batch["input_ids"].max()) < 16000,
                "Token IDs ngoài vocabulary.",
            )
            require(torch.equal(batch["input_ids"], batch["labels"]), "Batch labels sai.")

            if split == "train":
                # Không regenerate hoặc shuffle lại pilot indices.
                subset = torch.utils.data.Subset(
                    dataset, [int(index) for index in indices[:2]]
                )
                ordered_batch = next(iter(torch.utils.data.DataLoader(
                    subset, batch_size=2, shuffle=False, num_workers=0
                )))
                require(
                    torch.equal(
                        ordered_batch["input_ids"][0],
                        dataset[int(indices[0])]["input_ids"],
                    ),
                    "Pilot order sai.",
                )
        finally:
            dataset.close()
    READINESS["dataloader"] = "PASS"

    # Process độc lập: không dùng model/dataset objects từ notebook.
    script = """
import json, sys
from pathlib import Path
import numpy as np
import torch
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.data_processing import validate_dataset
from moe_erc.dataset import PackedTokenDataset, make_dataloader

root, tokenizer_path = Path(sys.argv[1]), Path(sys.argv[2])
tokenizer = FrozenTokenizer(tokenizer_path)
assert tokenizer.fingerprint == "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
assert tokenizer.vocab_size == 16000 and tokenizer.eos_id == 2
metadata = validate_dataset(root, tokenizer=tokenizer, checksums=True)
pilot = json.loads((root / "pilot_50m_manifest.json").read_text())
indices = np.load(root / pilot["index_file"], allow_pickle=False)
assert pilot["split"] == "train"
assert len(indices) == 97657 and len(indices) * 512 == 50000384
assert len(np.unique(indices)) == len(indices)

for split, expected in (("train",107478), ("validation",1006)):
    dataset = PackedTokenDataset(root, split, tokenizer=tokenizer, verify_checksums=True)
    try:
        assert len(dataset) == expected
        batch = next(iter(make_dataloader(dataset, batch_size=2, num_workers=0)))
        assert tuple(batch["input_ids"].shape) == (2,512)
        assert batch["input_ids"].dtype == torch.long
        assert torch.equal(batch["input_ids"], batch["labels"])
        if split == "train":
            subset = torch.utils.data.Subset(dataset, [int(i) for i in indices[:2]])
            ordered = next(iter(torch.utils.data.DataLoader(
                subset, batch_size=2, shuffle=False, num_workers=0
            )))
            assert torch.equal(
                ordered["input_ids"][0], dataset[int(indices[0])]["input_ids"]
            )
    finally:
        dataset.close()
print("Fresh-process dataset/pilot reload: PASS")
"""
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT) + os.pathsep + environment.get("PYTHONPATH", "")
    process = subprocess.run(
        [sys.executable, "-c", script, str(DATASET_TARGET), str(TOKENIZER_TARGET)],
        cwd=PROJECT, env=environment, text=True, capture_output=True, timeout=300,
    )
    require(process.returncode == 0, f"Fresh-process reload FAIL:\n{process.stderr[-4000:]}")
    print(process.stdout.strip())
    READINESS["fresh_process_reload"] = "PASS"

    require(
        CUDA_AVAILABLE and GPU_IS_A100 and BF16_SUPPORTED,
        "Dataset đã xác minh, nhưng runtime chưa đáp ứng A100/CUDA/BF16.",
    )
    READINESS.update(
        status="PASS",
        reason="Approved model initialization FIX; mọi data-integrity checks được giữ nguyên.",
    )
except Exception as exc:
    READINESS["reason"] = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()

print("\n=== STEP 6.5A APPROVED FIX RESTORE REPORT ===")
print("Original dataset creation commit:",
      COMPAT_REPORT["original_dataset_creation_commit"])
print("Current FIX commit:", COMPAT_REPORT["current_fix_commit"])
print("Files có SHA khác recorded source:")
for difference in COMPAT_REPORT["different_recorded_files"]:
    print(json.dumps(difference, indent=2))
print("Lý do chấp nhận model.py:", COMPAT_REPORT.get("reason", "NOT APPROVED"))
print("Model initialization checks:",
      json.dumps(COMPAT_REPORT["initialization"], indent=2))
print("Dataset fingerprint:", COMBINED_FP)
print("Tokenizer SHA-256:", TOKENIZER_SHA)
print("Dataset integrity:", READINESS["dataset_integrity"])
print("Drive snapshot verification:", "PASS" if SNAPSHOTS_VERIFIED else "FAIL")
print("Restore verification:", READINESS["restore_verification"])
print("DataLoader:", READINESS["dataloader"])
print("Fresh-process reload:", READINESS["fresh_process_reload"])
print("Local dataset:", DATASET_TARGET)
print("Local tokenizer:", TOKENIZER_TARGET)
print("Restore status:", READINESS["status"])
print("Chi tiết:", READINESS["reason"])
print("Không sửa source, snapshot, stored checksums, dataset hoặc pilot manifest.")
print("Không chạy training và không tự chuyển STEP 6.5B.")

Fresh-process dataset/pilot reload: PASS

=== STEP 6.5A APPROVED FIX RESTORE REPORT ===
Original dataset creation commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
Current FIX commit: 665cb7b1cfbca4c6ed3507345b21008321129a2a
Files có SHA khác recorded source:
{
  "file": "model.py",
  "dataset_creation_sha256": "53d4ad3cade511a70c8c0944561ff9ed8437cbc8cbabc80b1d32a81a2fe9dfbe",
  "current_sha256": "2389cc7be64b8f68a85ccbe37769a0786754dcce4b36c9859ddb53f68b8c7166"
}
Lý do chấp nhận model.py: Pinned FIX commit/blob; AST chỉ thêm Normal(0,0.02) đúng một lần sau tying; các runtime files khác giữ nguyên bytes.
Model initialization checks: {
  "vanilla_moe": {
    "embedding_std": 0.020000705495476723,
    "lm_head_std": 0.020000705495476723,
    "same_parameter": true,
    "same_storage": true,
    "parameters": 52288000,
    "status": "PASS"
  },
  "aoe": {
    "embedding_std": 0.0200003981590271,
    "lm_head_std": 0.0200003981590271,
    "same_parameter": true,
    "same_storage": true,
 

# STEP 6.5B — XÂY DỰNG SHARED TRAINING LOOP VÀ KIỂM THỬ BAN ĐẦU

In [ ]:
from pathlib import Path
from contextlib import ExitStack
from collections import Counter
from dataclasses import dataclass, asdict, replace
import gc
import hashlib
import inspect
import json
import math
import os
import random
import subprocess
import sys
import time

PROJECT = Path("/content/moe-erc")
DATA_ROOT = PROJECT / "data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
TOKENIZER_PATH = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"
EXPECTED_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
COMBINED_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"
TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
MODEL_TYPES = ("vanilla_moe", "aoe")
EXPECTED_PARAMETERS = {"vanilla_moe": 52_288_000, "aoe": 52_263_424}
INTEGRITY_RESULTS = {kind: {"status": "NOT RUN"} for kind in MODEL_TYPES}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode()).hexdigest()

def artifact_file(root, name):
    path = Path(name)
    require(
        not path.is_absolute() and ".." not in path.parts and "\\" not in name,
        f"Artifact path sai: {name}",
    )
    current = Path(root)
    for part in path.parts:
        current = current / part
        require(not current.is_symlink(), f"Artifact symlink: {current}")
    require(current.is_file(), f"Thiếu artifact: {current}")
    require(current.resolve().is_relative_to(Path(root).resolve()), "Path escape.")
    return current

require(PROJECT.is_dir(), "Thiếu project; không tự clone hoặc sửa repository.")
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None

import numpy as np
import torch
import torch.nn.functional as F
from moe_erc.config import ModelConfig, AoEConfig
from moe_erc.model import LanguageModelOutput, MoELanguageModel, AoELanguageModel
from moe_erc.model_factory import create_model
from moe_erc.parameter_count import count_parameters
from moe_erc.router import MoERouter
from moe_erc.aoe import SparseAoE
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.data_processing import validate_dataset, audit_document_splits
from moe_erc.dataset import PackedTokenDataset

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
require(GIT_COMMIT == EXPECTED_COMMIT, "Git HEAD khác commit yêu cầu; không tự checkout/reset.")
require(torch.cuda.is_available(), "CUDA unavailable.")
GPU = torch.cuda.get_device_properties(0)
require("A100" in GPU.name.upper(), f"GPU được cấp là {GPU.name}, không phải A100.")
with torch.cuda.device(0):
    require(torch.cuda.is_bf16_supported(), "GPU/runtime không hỗ trợ BF16.")

TOKENIZER = FrozenTokenizer(TOKENIZER_PATH)
require(
    TOKENIZER.fingerprint == TOKENIZER_SHA
    and TOKENIZER.vocab_size == 16000 and TOKENIZER.eos_id == 2,
    "Tokenizer mismatch.",
)

marker = read_json(DATA_ROOT / "COMPLETE.json")
require(marker["status"] == "COMPLETE", "Dataset chưa COMPLETE.")
require(
    sha256_file(DATA_ROOT / "dataset_manifest.json") == marker["dataset_manifest_sha256"],
    "Dataset manifest checksum sai.",
)
CORE = read_json(DATA_ROOT / "dataset_manifest.json")
require(CORE["combined_source_fingerprint"] == COMBINED_FP, "Dataset fingerprint sai.")

ARTIFACT_GUARD = {}
for entry in CORE["files"]:
    path = artifact_file(DATA_ROOT, entry["path"])
    require(
        path.stat().st_size == entry["bytes"] and sha256_file(path) == entry["sha256"],
        f"Core checksum FAIL: {path}",
    )
    ARTIFACT_GUARD[str(path)] = entry["sha256"]

PILOT_PATH = DATA_ROOT / "pilot_50m_manifest.json"
PILOT = read_json(PILOT_PATH)
PILOT_SHA = sha256_file(PILOT_PATH)
require(
    PILOT_SHA == (DATA_ROOT / "pilot_50m_manifest.sha256").read_text().strip(),
    "Pilot manifest checksum sai.",
)
require(
    PILOT["split"] == "train"
    and PILOT["combined_source_fingerprint"] == COMBINED_FP
    and PILOT["tokenizer_sha256"] == TOKENIZER_SHA
    and PILOT["dataset_manifest_sha256"] == marker["dataset_manifest_sha256"],
    "Pilot identity mismatch.",
)
PILOT_INDEX_PATH = artifact_file(DATA_ROOT, PILOT["index_file"])
require(sha256_file(PILOT_INDEX_PATH) == PILOT["index_file_sha256"], "Pilot index checksum sai.")
PILOT_INDICES = np.load(PILOT_INDEX_PATH, allow_pickle=False)
require(
    PILOT_INDICES.ndim == 1 and len(PILOT_INDICES) == 97657
    and len(np.unique(PILOT_INDICES)) == len(PILOT_INDICES)
    and int(PILOT_INDICES.min()) >= 0 and int(PILOT_INDICES.max()) < 107478,
    "Pilot indices sai.",
)

METADATA = validate_dataset(DATA_ROOT, tokenizer=TOKENIZER, checksums=True)
require(METADATA["sequence_length"] == 512 and METADATA["dtype"] == "<u2", "Dataset config sai.")
require(
    METADATA["splits"]["train"]["sequences"] == 107478
    and METADATA["splits"]["validation"]["sequences"] == 1006,
    "Dataset counts sai.",
)
require(audit_document_splits(DATA_ROOT)["overlap"] == 0, "Document leakage.")

CODE_GUARD = {
    name: sha256_file(PROJECT / "moe_erc" / name)
    for name in (
        "config.py", "model.py", "model_factory.py", "decoder.py",
        "moe.py", "router.py", "aoe.py", "aoe_expert.py",
        "experts.py", "dataset.py", "data_processing.py",
    )
}
design = read_json(DATA_ROOT / "packing_design.json")
for name, expected in design["pipeline_source_sha256"].items():
    require(sha256_file(PROJECT / "moe_erc" / name) == expected, f"Dataset code mismatch: {name}")

for path in (
    DATA_ROOT / "dataset_manifest.json", DATA_ROOT / "COMPLETE.json",
    PILOT_PATH, DATA_ROOT / "pilot_50m_manifest.sha256", PILOT_INDEX_PATH,
    TOKENIZER_PATH, TOKENIZER_PATH.with_suffix(".meta.json"),
):
    ARTIFACT_GUARD[str(path)] = sha256_file(path)

def verify_fixed_artifacts():
    for name, expected in ARTIFACT_GUARD.items():
        require(sha256_file(name) == expected, f"Artifact đã thay đổi: {name}")
    for name, expected in CODE_GUARD.items():
        require(sha256_file(PROJECT / "moe_erc" / name) == expected, f"Source đã đổi: {name}")

MODEL_CONFIGS = {"vanilla_moe": ModelConfig(), "aoe": AoEConfig()}
TRAIN_DATA = PackedTokenDataset(DATA_ROOT, "train", tokenizer=TOKENIZER)
VALIDATION_DATA = PackedTokenDataset(DATA_ROOT, "validation", tokenizer=TOKENIZER)

def ordered_loader(dataset, indices, batch_size, seed=42):
    # Không sử dụng shuffle=True mặc định của training DataLoader STEP 5.
    return torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [int(i) for i in indices]),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False,
        generator=torch.Generator().manual_seed(seed),
    )

FIRST_BATCH_CPU = next(iter(ordered_loader(TRAIN_DATA, PILOT_INDICES[:2], 2)))
require(tuple(FIRST_BATCH_CPU["input_ids"].shape) == (2, 512), "Batch shape sai.")
require(FIRST_BATCH_CPU["input_ids"].dtype == torch.long, "Batch dtype sai.")
require(torch.equal(FIRST_BATCH_CPU["input_ids"], FIRST_BATCH_CPU["labels"]), "Labels bị shift.")

MODEL_INFO = {}
for kind in MODEL_TYPES:
    model = None
    try:
        torch.manual_seed(42)
        config = MODEL_CONFIGS[kind]
        expected_architecture = {
            "vocab_size": 16000, "d_model": 512, "n_layers": 6,
            "n_heads": 8, "n_experts": 16, "top_k": 2,
            "expert_hidden_dim": 256, "max_seq_len": 512,
            "tie_embeddings": True,
        }
        for key, value in expected_architecture.items():
            require(getattr(config, key) == value, f"Architecture mismatch: {kind}.{key}")

        model = create_model(kind, config).float()
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        require(total == trainable == EXPECTED_PARAMETERS[kind], f"Parameter mismatch: {kind}")
        require(count_parameters(config, kind).total == total, "Analytical count mismatch.")
        require(model.lm_head.weight is model.token_embedding.weight, "Embeddings không tied.")
        if kind == "aoe":
            require(config.aoe_rank == 171, "AoE rank sai.")
            require(not any(isinstance(m, MoERouter) for m in model.modules()), "AoE có router lạ.")
            require(all(isinstance(layer.moe, SparseAoE) for layer in model.layers), "AoE FFN sai.")

        model = model.cuda().eval()
        batch = {name: value.cuda() for name, value in FIRST_BATCH_CPU.items()}
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            output = model(**batch, return_routing_info=True)
        require(isinstance(output, LanguageModelOutput), "Forward output API sai.")
        require(tuple(output.logits.shape) == (2, 512, 16000), "Logits shape sai.")
        require(bool(torch.isfinite(output.loss)), "Initial CE nonfinite.")
        MODEL_INFO[kind] = {
            "total_parameters": total,
            "trainable_parameters": trainable,
            "activated_parameters": None,  # Project không có công cụ đếm active parameters.
            "config": config.to_dict(),
            "forward_shape": list(output.logits.shape),
        }
        print(kind, MODEL_INFO[kind])
        del output, batch
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

print("GPU:", GPU.name, "| VRAM GiB:", GPU.total_memory / 1024**3)
print("Forward API: labels -> LanguageModelOutput(logits, loss, routing_info).")
print("output.loss = CE only; model tự shift logits[:, :-1] và labels[:, 1:].")
print("routing_info là detached diagnostics; không dùng trực tiếp để tạo LB gradient.")
print("Vanilla: full-softmax Top-2, không renormalize selected weights.")
print("AoE: norm-based Top-2, rank=171, không independent router.")
print("Built-in LB raw/weighted: không có; Cell 3 bổ sung rõ ràng ở notebook.")

vanilla_moe {'total_parameters': 52288000, 'trainable_parameters': 52288000, 'activated_parameters': None, 'config': {'vocab_size': 16000, 'd_model': 512, 'n_layers': 6, 'n_heads': 8, 'n_experts': 16, 'top_k': 2, 'expert_hidden_dim': 256, 'max_seq_len': 512, 'dropout': 0.0, 'linear_bias': False, 'tie_embeddings': True, 'attention_type': 'multi_head_causal', 'positional_embedding': 'rope', 'normalization': 'rmsnorm', 'expert_activation': 'swiglu', 'shared_experts': 0, 'rope_theta': 10000.0, 'rms_norm_eps': 1e-06}, 'forward_shape': [2, 512, 16000]}
aoe {'total_parameters': 52263424, 'trainable_parameters': 52263424, 'activated_parameters': None, 'config': {'vocab_size': 16000, 'd_model': 512, 'n_layers': 6, 'n_heads': 8, 'n_experts': 16, 'top_k': 2, 'expert_hidden_dim': 256, 'max_seq_len': 512, 'dropout': 0.0, 'linear_bias': False, 'tie_embeddings': True, 'attention_type': 'multi_head_causal', 'positional_embedding': 'rope', 'normalization': 'rmsnorm', 'expert_activation': 'swiglu', 'sha

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    purpose: str = "smoke"
    seed: int = 42
    micro_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    total_optimizer_steps: int | None = 3
    warmup_steps: int | None = 1
    peak_learning_rate: float = 4e-4
    min_learning_rate: float = 4e-5
    beta1: float = 0.9
    beta2: float = 0.95
    weight_decay: float = 0.1
    load_balancing_weight: float = 0.01
    gradient_clip_norm: float = 1.0
    validation_batch_size: int = 2
    validation_max_batches: int = 2

    def __post_init__(self):
        require(self.purpose in ("smoke", "pilot"), "Purpose sai.")
        require(self.micro_batch_size > 0 and self.gradient_accumulation_steps > 0, "Batch config sai.")
        require(0 < self.min_learning_rate <= self.peak_learning_rate, "LR config sai.")
        require(0 <= self.beta1 < 1 and 0 <= self.beta2 < 1, "Adam betas sai.")
        require(self.weight_decay >= 0 and self.load_balancing_weight >= 0, "Loss/decay config sai.")
        require(self.gradient_clip_norm > 0, "Clip norm sai.")
        if self.total_optimizer_steps is not None:
            require(
                self.warmup_steps is not None
                and 1 <= self.warmup_steps < self.total_optimizer_steps,
                "Cần 1 <= warmup < total optimizer steps.",
            )
        if self.purpose == "smoke":
            require(
                self.total_optimizer_steps is not None
                and 1 <= self.total_optimizer_steps <= 3,
                "Smoke giới hạn cứng tối đa 3 optimizer steps.",
            )
            require(
                self.micro_batch_size <= 2 and self.gradient_accumulation_steps <= 2,
                "Smoke chỉ cho phép micro-batch/accumulation tối đa 2.",
            )
            require(
                1 <= self.validation_max_batches <= 2 and 1 <= self.validation_batch_size <= 2,
                "Smoke validation tối đa 2 batches, batch size tối đa 2.",
            )

SMOKE_CFG = TrainingConfig()
PILOT_CFG_TEMPLATE = replace(
    SMOKE_CFG, purpose="pilot", total_optimizer_steps=None, warmup_steps=None
)

LB_POLICY = {
    "enabled_for": ["vanilla_moe", "aoe"],
    "formula": "mean_layers(E * sum_e(f_e * mean_token_probability_e))",
    "assignment_normalization": "f_e = count_e / (N_tokens * top_k)",
    "microbatch_reduction": "effective-target-weighted mean across accumulation microbatches",
    "routing_tokens": "all input positions, including the last position",
    "raw_balanced_reference": 1.0,
    "weighted_balanced_reference": 0.01,
    "vanilla_source": "attached MoERouter output probabilities",
    "aoe_source": "norm-softmax reconstructed from actual attached gate-down outputs",
    "extra_gate_projections": False,
    "note": "Explicit project reduction policy; not a claim of exact large-model recipe reproduction.",
}

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

def state_digest(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def fresh_model(kind, config):
    seed_all(config.seed)
    model = create_model(kind, MODEL_CONFIGS[kind]).float()
    initial_sha = state_digest(model)
    model = model.cuda()
    require(all(p.dtype == torch.float32 for p in model.parameters()), "Parameters không FP32.")
    return model, initial_sha

def build_optimizer_scheduler(model, config):
    require(
        config.total_optimizer_steps is not None and config.warmup_steps is not None,
        "Chưa chốt lịch training chính thức.",
    )
    # Chính sách công bố: decay mọi unique trainable parameter, kể cả RMSNorm.
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config.peak_learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=config.weight_decay,
        foreach=False,
        fused=False,
    )

    def multiplier(index):
        # LambdaLR index=0 thiết lập LR dùng cho optimizer update đầu tiên.
        update = index + 1
        if update <= config.warmup_steps:
            return update / config.warmup_steps
        progress = min(1.0, (
            update - config.warmup_steps
        ) / (config.total_optimizer_steps - config.warmup_steps))
        ratio = config.min_learning_rate / config.peak_learning_rate
        return ratio + (1 - ratio) * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
    return optimizer, scheduler

print("Smoke configuration:", asdict(SMOKE_CFG))
print("Pilot template — chưa có schedule:", asdict(PILOT_CFG_TEMPLATE))
print("LB policy:", LB_POLICY)
print("FP32 parameters/Adam moments; BF16 autocast; không dùng GradScaler.")
print("LR smoke: warmup/cosine ngắn, không được dùng làm lịch chính thức 50M.")
print("Seeds được kiểm soát; CUDA sparse reductions không bảo đảm bitwise reproducibility.")
print("Chưa tạo optimizer state hoặc bắt đầu training.")

Smoke configuration: {'purpose': 'smoke', 'seed': 42, 'micro_batch_size': 2, 'gradient_accumulation_steps': 2, 'total_optimizer_steps': 3, 'warmup_steps': 1, 'peak_learning_rate': 0.0004, 'min_learning_rate': 4e-05, 'beta1': 0.9, 'beta2': 0.95, 'weight_decay': 0.1, 'load_balancing_weight': 0.01, 'gradient_clip_norm': 1.0, 'validation_batch_size': 2, 'validation_max_batches': 2}
Pilot template — chưa có schedule: {'purpose': 'pilot', 'seed': 42, 'micro_batch_size': 2, 'gradient_accumulation_steps': 2, 'total_optimizer_steps': None, 'warmup_steps': None, 'peak_learning_rate': 0.0004, 'min_learning_rate': 4e-05, 'beta1': 0.9, 'beta2': 0.95, 'weight_decay': 0.1, 'load_balancing_weight': 0.01, 'gradient_clip_norm': 1.0, 'validation_batch_size': 2, 'validation_max_batches': 2}
LB policy: {'enabled_for': ['vanilla_moe', 'aoe'], 'formula': 'mean_layers(E * sum_e(f_e * mean_token_probability_e))', 'assignment_normalization': 'f_e = count_e / (N_tokens * top_k)', 'microbatch_reduction': 'effecti

In [ ]:
class RoutingLossTap:
    def __init__(self, model, kind):
        self.model, self.kind = model, kind
        self.handles = []
        self.probabilities = {}
        self.scores = {i: {} for i in range(len(model.layers))}

    def __enter__(self):
        try:
            for layer_index, layer in enumerate(self.model.layers):
                if self.kind == "vanilla_moe":
                    def capture_router(module, args, output, index=layer_index):
                        self.probabilities[index] = output.probabilities
                    self.handles.append(
                        layer.moe.router.register_forward_hook(capture_router)
                    )
                elif self.kind == "aoe":
                    for expert_index, expert in enumerate(layer.moe.experts):
                        def capture_gate(module, args, output, li=layer_index, ei=expert_index):
                            with torch.amp.autocast("cuda", enabled=False):
                                self.scores[li][ei] = torch.linalg.vector_norm(
                                    output.float(), ord=2, dim=-1
                                )
                            # Không return tensor: giữ nguyên module output.
                        self.handles.append(
                            expert.gate_down_proj.register_forward_hook(capture_gate)
                        )
                else:
                    raise ValueError("Unsupported architecture.")
        except BaseException:
            self.__exit__(None, None, None)
            raise
        return self

    def loss(self, routing_info, verify=False):
        require(len(routing_info) == len(self.model.layers), "Routing layers thiếu.")
        losses = []
        for index, layer_info in enumerate(routing_info):
            diagnostic = layer_info.diagnostics
            experts = self.model.config.n_experts

            if self.kind == "vanilla_moe":
                probabilities = self.probabilities[index].reshape(-1, experts)
            else:
                require(len(self.scores[index]) == experts, "AoE gate-down captures thiếu.")
                scores = torch.stack(
                    [self.scores[index][e] for e in range(experts)], dim=-1
                )
                with torch.amp.autocast("cuda", enabled=False):
                    probabilities = scores.float().softmax(dim=-1)
                if verify:
                    torch.testing.assert_close(
                        probabilities.detach(),
                        diagnostic.probabilities.reshape_as(probabilities),
                        rtol=3e-6, atol=3e-7,
                    )

            indices = diagnostic.topk_indices.reshape(-1, self.model.config.top_k)
            require(indices.shape[0] == probabilities.shape[0], "Routing token count sai.")
            counts = torch.bincount(indices.reshape(-1), minlength=experts).float()
            fractions = counts / indices.numel()  # Detached discrete assignments.
            mean_probability = probabilities.mean(dim=0)  # Attached.
            if verify:
                torch.testing.assert_close(
                    mean_probability.detach(), diagnostic.mean_probabilities,
                    rtol=3e-6, atol=3e-7,
                )
            losses.append(experts * (fractions * mean_probability).sum())
        return torch.stack(losses).mean()

    def __exit__(self, exc_type, exc, tb):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()
        self.probabilities.clear()
        self.scores.clear()

def compute_loss(model, kind, batch, config, verify_routing=False):
    # CE được lấy trực tiếp từ model; KHÔNG shift labels ở đây.
    with RoutingLossTap(model, kind) as tap:
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            output = model(
                input_ids=batch["input_ids"],
                labels=batch["labels"],
                return_routing_info=True,
            )
        lb_raw = tap.loss(output.routing_info, verify=verify_routing)
    lb_weighted = config.load_balancing_weight * lb_raw
    total = output.loss + lb_weighted
    for name, loss in (("ce", output.loss), ("lb_raw", lb_raw), ("total", total)):
        require(loss.ndim == 0 and bool(torch.isfinite(loss)), f"Nonfinite {name}.")
    if torch.is_grad_enabled():
        require(lb_raw.requires_grad, "LB bị detach.")
    # STEP 7 có thể mở rộng điểm tính objective này; hiện không có ERC.
    return {
        "output": output, "ce": output.loss,
        "lb_raw": lb_raw, "lb_weighted": lb_weighted, "total": total,
    }

def validate_gradients(model):
    present = 0
    for name, parameter in model.named_parameters():
        if parameter.grad is not None:
            require(
                bool(torch.isfinite(parameter.grad).all()),
                f"Gradient NaN/Inf: {name}",
            )
            present += 1
    require(present > 0, "Không có gradients.")
    return present

def collect_metrics(losses, **extra):
    return {
        "ce_loss": float(losses["ce"]),
        "lb_raw": float(losses["lb_raw"]),
        "lb_weighted": float(losses["lb_weighted"]),
        "total_loss": float(losses["total"]),
        **extra,
    }

def train_one_step(model, kind, optimizer, scheduler, microbatches, config, step):
    require(1 <= step <= config.total_optimizer_steps, "Optimizer step ngoài cấu hình.")
    if config.purpose == "smoke":
        require(step <= 3, "Hard smoke limit exceeded.")
    require(len(microbatches) == config.gradient_accumulation_steps, "Accumulation count sai.")

    targets = [int((b["labels"][:, 1:] != -100).sum()) for b in microbatches]
    require(all(n > 0 for n in targets), "Microbatch không có next-token targets.")
    effective_targets = sum(targets)
    input_positions = sum(b["input_ids"].numel() for b in microbatches)

    model.train()
    optimizer.zero_grad(set_to_none=True)
    witness = model.final_norm.weight.detach().clone()
    used_lr = optimizer.param_groups[0]["lr"]
    previous_scheduler_step = scheduler.last_epoch

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    accumulated = {"ce": 0.0, "lb_raw": 0.0, "lb_weighted": 0.0, "total": 0.0}

    for batch_cpu, target_count in zip(microbatches, targets):
        batch = {name: value.cuda() for name, value in batch_cpu.items()}
        losses = compute_loss(model, kind, batch, config)
        weight = target_count / effective_targets
        (losses["total"] * weight).backward()
        for name in accumulated:
            accumulated[name] += float(losses[name].detach()) * weight
        del losses, batch

    gradient_tensors = validate_gradients(model)
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), config.gradient_clip_norm, error_if_nonfinite=True
    )
    require(bool(torch.isfinite(gradient_norm)), "Gradient norm nonfinite.")
    optimizer.step()
    scheduler.step()
    require(
        scheduler.last_epoch == previous_scheduler_step + 1,
        "Scheduler không tăng đúng một lần.",
    )
    require(
        not torch.equal(witness, model.final_norm.weight.detach()),
        "Optimizer không cập nhật witness parameter.",
    )
    for state in optimizer.state.values():
        for name in ("exp_avg", "exp_avg_sq"):
            if name in state:
                require(state[name].dtype == torch.float32, "Adam moments không FP32.")
    require(all(p.dtype == torch.float32 for p in model.parameters()), "Parameters đổi dtype.")

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return collect_metrics(
        accumulated,
        optimizer_step=step,
        lr_used=used_lr,
        lr_next=optimizer.param_groups[0]["lr"],
        gradient_norm_before_clip=float(gradient_norm),
        gradient_tensors_present=gradient_tensors,
        optimizer_parameter_updated=True,
        input_token_positions=input_positions,
        effective_next_token_targets=effective_targets,
        elapsed_seconds=elapsed,
        input_tokens_per_second=input_positions / elapsed,
        peak_allocated_gib=torch.cuda.max_memory_allocated() / 1024**3,
        peak_reserved_gib=torch.cuda.max_memory_reserved() / 1024**3,
        throughput_scope="training step only; excludes validation and setup",
    )

def evaluate(model, validation_dataset, config):
    flags = [(module, module.training) for module in model.modules()]
    rng = (
        random.getstate(), np.random.get_state(),
        torch.get_rng_state(), torch.cuda.get_rng_state_all(),
    )
    count = min(
        len(validation_dataset),
        config.validation_batch_size * config.validation_max_batches,
    )
    indices = list(range(count))
    loader = ordered_loader(
        validation_dataset, indices, config.validation_batch_size, config.seed
    )
    total_nll = targets = input_positions = batches = 0
    try:
        model.eval()
        with torch.no_grad():
            for batch_cpu in loader:
                batch = {name: value.cuda() for name, value in batch_cpu.items()}
                n = int((batch["labels"][:, 1:] != -100).sum())
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    output = model(**batch)
                require(bool(torch.isfinite(output.loss)), "Validation CE nonfinite.")
                total_nll += float(output.loss) * n
                targets += n
                input_positions += batch["input_ids"].numel()
                batches += 1
                del output, batch
        require(targets > 0, "Validation không có targets.")
        ce = total_nll / targets
        overflow = ce > math.log(sys.float_info.max)
        result = {
            "ce_loss": ce,
            "perplexity": None if overflow else math.exp(ce),
            "perplexity_overflow": overflow,
            "log_perplexity": ce,
            "batches": batches,
            "unique_sequences": count,
            "sequence_indices": indices,
            "input_token_positions": input_positions,
            "effective_next_token_targets": targets,
            "scope": "fixed smoke subset, not full validation",
        }
    finally:
        for module, flag in flags:
            module.training = flag
        random.setstate(rng[0])
        np.random.set_state(rng[1])
        torch.set_rng_state(rng[2])
        torch.cuda.set_rng_state_all(rng[3])
    require(all(module.training == flag for module, flag in flags), "Train mode không được restore.")
    return result

print("Shared functions ready; chưa optimizer step.")
print("CE + 0.01*LB, không ERC; raw LB và weighted LB được log riêng.")

Shared functions ready; chưa optimizer step.
CE + 0.01*LB, không ERC; raw LB và weighted LB được log riêng.


In [ ]:
def require_gradient(parameter, name):
    require(parameter.grad is not None, f"Thiếu gradient: {name}")
    require(bool(torch.isfinite(parameter.grad).all()), f"Gradient nonfinite: {name}")

def integrity_test(kind):
    model = None
    try:
        model, initial_sha = fresh_model(kind, SMOKE_CFG)
        model.train()
        batch = {name: value.cuda() for name, value in FIRST_BATCH_CPU.items()}
        torch.cuda.reset_peak_memory_stats()

        losses = compute_loss(model, kind, batch, SMOKE_CFG, verify_routing=True)
        output = losses["output"]
        require(tuple(output.logits.shape) == (2, 512, 16000), "Logits shape sai.")
        require(output.logits.dtype == torch.bfloat16, "BF16 logits chưa hoạt động.")
        require(bool(torch.isfinite(output.logits).all()), "Logits NaN/Inf.")

        with torch.no_grad():
            targets = batch["labels"][:, 1:]
            valid = targets != -100
            safe_targets = targets.masked_fill(~valid, 0)
            log_probabilities = F.log_softmax(output.logits[:, :-1].float(), dim=-1)
            nll = -log_probabilities.gather(
                -1, safe_targets.unsqueeze(-1)
            ).squeeze(-1)
            reference_ce = nll[valid].mean()
        torch.testing.assert_close(
            losses["ce"].detach(), reference_ce, rtol=1e-5, atol=1e-4
        )
        del log_probabilities, nll

        routing_parameter = (
            model.layers[0].moe.router.weight if kind == "vanilla_moe"
            else model.layers[0].moe.experts[0].gate_down_proj.weight
        )
        auxiliary_gradient = torch.autograd.grad(
            losses["lb_raw"], routing_parameter, retain_graph=True
        )[0]
        require(bool(torch.isfinite(auxiliary_gradient).all()), "LB gradient nonfinite.")
        aux_gradient_norm = float(auxiliary_gradient.norm())
        del auxiliary_gradient

        losses["total"].backward()
        gradient_tensors = validate_gradients(model)
        require_gradient(model.token_embedding.weight, "token_embedding")
        require_gradient(model.final_norm.weight, "final_norm")

        active_experts = []
        for layer_index, (layer, routing) in enumerate(zip(model.layers, output.routing_info)):
            for name, parameter in layer.attention.named_parameters():
                require_gradient(parameter, f"layer{layer_index}.attention.{name}")
            require_gradient(layer.norm_1.weight, f"layer{layer_index}.norm_1")
            require_gradient(layer.norm_2.weight, f"layer{layer_index}.norm_2")

            selected = torch.where(routing.diagnostics.token_counts > 0)[0].tolist()
            require(bool(selected), "Không có expert được chọn.")
            active_experts.append(len(selected))
            for expert_index in selected:
                for name, parameter in layer.moe.experts[expert_index].named_parameters():
                    require_gradient(
                        parameter, f"layer{layer_index}.expert{expert_index}.{name}"
                    )
            if kind == "vanilla_moe":
                require_gradient(layer.moe.router.weight, f"layer{layer_index}.router")
            else:
                # Tất cả gate-down projections tham gia norm-based routing.
                for expert_index, expert in enumerate(layer.moe.experts):
                    require_gradient(
                        expert.gate_down_proj.weight,
                        f"layer{layer_index}.expert{expert_index}.gate_down",
                    )

        torch.cuda.synchronize()
        result = {
            "status": "PASS",
            "initial_state_sha256": initial_sha,
            "initial_ce": float(losses["ce"].detach()),
            "ln_vocab_reference": math.log(16000),
            "lb_raw": float(losses["lb_raw"].detach()),
            "lb_weighted": float(losses["lb_weighted"].detach()),
            "total_loss": float(losses["total"].detach()),
            "auxiliary_gradient_graph": "PASS",
            "auxiliary_probe_gradient_norm": aux_gradient_norm,
            "causal_shift_reference": "PASS",
            "bf16_logits": True,
            "gradient_tensors_present": gradient_tensors,
            "selected_experts_per_layer": active_experts,
            "embedding_std": float(model.token_embedding.weight.detach().std()),
            "tied_embeddings": model.lm_head.weight is model.token_embedding.weight,
            "logit_abs_max": float(output.logits.detach().abs().max()),
            "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
            "optimizer_steps": 0,
            "input_token_positions": batch["input_ids"].numel(),
            "effective_next_token_targets": int(valid.sum()),
        }
        print(kind, json.dumps(result, ensure_ascii=False, indent=2))
        print(
            "CE được so với ln(16000) chỉ để tham khảo. Initialization mặc định "
            "và tied embeddings được giữ nguyên; không dùng ngưỡng CE tùy ý để kết luận bug."
        )
        return result
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

for kind in MODEL_TYPES:
    try:
        INTEGRITY_RESULTS[kind] = integrity_test(kind)
    except Exception as exc:
        INTEGRITY_RESULTS[kind] = {
            "status": "FAIL", "error": f"{type(exc).__name__}: {exc}"
        }
        import traceback
        traceback.print_exc()
        print("Dừng integrity tests; Cell 5 sẽ không được phép training.")
        break

INTEGRITY_PASS = all(
    INTEGRITY_RESULTS[kind]["status"] == "PASS" for kind in MODEL_TYPES
)
print("Forward/backward integrity:", "PASS" if INTEGRITY_PASS else "FAIL")
print("Không optimizer step; không checkpoint.")

vanilla_moe {
  "status": "PASS",
  "initial_state_sha256": "ad66a18f9442fdb3cf9892b9f2437efa6f2e9bb546ca5dc41b0f4c5fa2814bac",
  "initial_ce": 499.6509094238281,
  "ln_vocab_reference": 9.680344001221918,
  "lb_raw": 1.014458179473877,
  "lb_weighted": 0.010144582018256187,
  "total_loss": 499.6610412597656,
  "auxiliary_gradient_graph": "PASS",
  "auxiliary_probe_gradient_norm": 0.04595744237303734,
  "causal_shift_reference": "PASS",
  "bf16_logits": true,
  "gradient_tensors_present": 332,
  "selected_experts_per_layer": [
    16,
    16,
    16,
    16,
    16,
    16
  ],
  "embedding_std": 0.9999698996543884,
  "tied_embeddings": true,
  "logit_abs_max": 548.0,
  "peak_allocated_gib": 0.723475456237793,
  "optimizer_steps": 0,
  "input_token_positions": 1024,
  "effective_next_token_targets": 1022
}
CE được so với ln(16000) chỉ để tham khảo. Initialization mặc định và tied embeddings được giữ nguyên; không dùng ngưỡng CE tùy ý để kết luận bug.
aoe {
  "status": "PASS",
  "initia

In [ ]:
RUN_TRAIN_SMOKE = True
RUN_AUDIT_BACKUP = True  # Chỉ backup audit nhỏ nếu chủ động bật.

from datetime import datetime, timezone
import uuid
import traceback

SMOKE_RESULTS = {kind: {"status": "NOT RUN", "steps": []} for kind in MODEL_TYPES}
AUDIT_PATH = None
FIXED_ARTIFACTS_PASS = False

def run_model_smoke(kind, config):
    require(INTEGRITY_PASS, "Integrity chưa PASS.")
    require(config.purpose == "smoke", "Cell này chỉ chạy smoke.")
    require(config.total_optimizer_steps <= 3, "Hard maximum: 3 optimizer steps.")
    required_sequences = (
        config.total_optimizer_steps
        * config.gradient_accumulation_steps
        * config.micro_batch_size
    )
    selected_indices = PILOT_INDICES[:required_sequences].astype(np.int64).tolist()
    require(len(selected_indices) == required_sequences, "Pilot prefix thiếu sequences.")
    require(len(set(selected_indices)) == required_sequences, "Smoke prefix bị lặp.")

    model = optimizer = scheduler = None
    result = SMOKE_RESULTS[kind]
    result.update(
        status="RUNNING",
        total_parameters=MODEL_INFO[kind]["total_parameters"],
        selected_pilot_indices=selected_indices,
        initialization_seed=config.seed,
        steps=[],
    )
    try:
        model, initial_sha = fresh_model(kind, config)
        require(
            initial_sha == INTEGRITY_RESULTS[kind]["initial_state_sha256"],
            "Smoke không bắt đầu từ cùng initialization với integrity test.",
        )
        result["initial_state_sha256"] = initial_sha
        optimizer, scheduler = build_optimizer_scheduler(model, config)
        iterator = iter(ordered_loader(
            TRAIN_DATA, selected_indices, config.micro_batch_size, config.seed
        ))

        cumulative_positions = cumulative_targets = 0
        for step in range(1, min(3, config.total_optimizer_steps) + 1):
            microbatches = [
                next(iterator) for _ in range(config.gradient_accumulation_steps)
            ]
            metrics = train_one_step(
                model, kind, optimizer, scheduler, microbatches, config, step
            )
            cumulative_positions += metrics["input_token_positions"]
            cumulative_targets += metrics["effective_next_token_targets"]
            metrics["cumulative_input_token_positions"] = cumulative_positions
            metrics["cumulative_effective_next_token_targets"] = cumulative_targets
            # Ghi training step trước evaluation để giữ thông tin nếu eval lỗi.
            result["steps"].append(metrics)
            validation = evaluate(model, VALIDATION_DATA, config)
            metrics["validation"] = validation

            ppl = (
                "OVERFLOW; xem log_perplexity"
                if validation["perplexity_overflow"] else validation["perplexity"]
            )
            print({
                "model": kind,
                "parameters": result["total_parameters"],
                "step": step,
                "ce": metrics["ce_loss"],
                "lb_raw": metrics["lb_raw"],
                "lb_weighted": metrics["lb_weighted"],
                "total": metrics["total_loss"],
                "gradient_norm_before_clip": metrics["gradient_norm_before_clip"],
                "lr_used": metrics["lr_used"],
                "effective_targets": metrics["effective_next_token_targets"],
                "validation_ce_subset": validation["ce_loss"],
                "validation_ppl_subset": ppl,
                "peak_allocated_gib": metrics["peak_allocated_gib"],
                "input_tokens_per_second": metrics["input_tokens_per_second"],
            })

        require(
            cumulative_positions == required_sequences * 512
            and cumulative_targets == required_sequences * 511,
            "Consumed token accounting sai.",
        )
        require(
            all(row["optimizer_parameter_updated"] for row in result["steps"]),
            "Optimizer update check chưa PASS.",
        )
        result.update(
            status="PASS",
            consumed_train_sequences=required_sequences,
            consumed_input_token_positions=cumulative_positions,
            consumed_effective_next_token_targets=cumulative_targets,
            validation_unique_sequences=config.validation_batch_size * config.validation_max_batches,
            validation_sequence_visits=sum(
                row["validation"]["unique_sequences"] for row in result["steps"]
            ),
        )
        return result
    finally:
        optimizer = scheduler = model = None
        gc.collect()
        torch.cuda.empty_cache()

if not RUN_TRAIN_SMOKE:
    print("RUN_TRAIN_SMOKE=False: không optimizer steps, không tạo training audit.")
else:
    require(INTEGRITY_PASS, "Cell 4 chưa PASS; không cho phép smoke training.")
    verify_fixed_artifacts()
    require(
        SMOKE_CFG.total_optimizer_steps <= 3
        and SMOKE_CFG.micro_batch_size <= 2
        and SMOKE_CFG.gradient_accumulation_steps <= 2,
        "Smoke limits đã bị thay đổi.",
    )

    run_directory = (
        PROJECT / "data/metadata/training_smoke"
        / (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            + "-" + uuid.uuid4().hex[:12]
        )
    )
    run_directory.mkdir(parents=True, exist_ok=False)
    AUDIT_PATH = run_directory / "audit.json"

    try:
        for kind in MODEL_TYPES:
            try:
                run_model_smoke(kind, SMOKE_CFG)
            except Exception as exc:
                SMOKE_RESULTS[kind].update(
                    status="FAIL",
                    error=f"{type(exc).__name__}: {exc}",
                )
                traceback.print_exc()
                print("SMOKE STOP: không chạy tiếp model còn lại hoặc tăng ngân sách.")
                break
        verify_fixed_artifacts()
        FIXED_ARTIFACTS_PASS = True
    except Exception as exc:
        print("Artifact verification FAIL:", type(exc).__name__, str(exc))
    finally:
        complete = (
            INTEGRITY_PASS and FIXED_ARTIFACTS_PASS
            and all(SMOKE_RESULTS[k]["status"] == "PASS" for k in MODEL_TYPES)
        )
        audit = {
            "step": "6.5B",
            "status": "PASS" if complete else "FAIL",
            "git_commit": GIT_COMMIT,
            "dataset_fingerprint": COMBINED_FP,
            "dataset_manifest_sha256": sha256_file(DATA_ROOT / "dataset_manifest.json"),
            "tokenizer_sha256": TOKENIZER_SHA,
            "pilot_manifest_sha256": PILOT_SHA,
            "pilot_index_sha256": PILOT["index_file_sha256"],
            "model_factory": "moe_erc.model_factory.create_model",
            "models": MODEL_INFO,
            "smoke_configuration": asdict(SMOKE_CFG),
            "load_balancing_policy": LB_POLICY,
            "weight_decay_scope": "all unique trainable parameters",
            "initialization_policy": (
                "fresh independent model per architecture, controlled seed; "
                "integrity/smoke initial-state fingerprints must match"
            ),
            "future_vanilla_erc_requirement": (
                "STEP 8 must reuse exactly the same initial state_dict, "
                "not merely assume equal seed is sufficient."
            ),
            "integrity_tests": INTEGRITY_RESULTS,
            "training_smoke": SMOKE_RESULTS,
            "fixed_artifacts_unchanged": FIXED_ARTIFACTS_PASS,
            "versions": {
                "python": sys.version.split()[0],
                "torch": torch.__version__,
                "numpy": np.__version__,
            },
            "gpu": GPU.name,
            "bf16": True,
            "erc_implemented": False,
            "checkpoint_created": False,
            "limitations": [
                "At most three optimizer steps per model.",
                "Validation is a fixed tiny subset, not full validation.",
                "Throughput is instrumentation only, not a controlled benchmark.",
                "No research conclusion from relative smoke losses.",
                "Bitwise training reproducibility has not been established.",
            ],
        }
        with AUDIT_PATH.open("x", encoding="utf-8") as stream:
            json.dump(audit, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
            stream.write("\n")

if RUN_AUDIT_BACKUP:
    require(AUDIT_PATH is not None and AUDIT_PATH.is_file(), "Chưa có smoke audit.")
    require(Path("/content/drive/MyDrive").is_dir(), "Drive chưa mount.")
    destination = (
        Path("/content/drive/MyDrive/MoE_ERC/training_smoke_audits")
        / (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            + "-" + uuid.uuid4().hex[:12]
        )
    )
    destination.mkdir(parents=True, exist_ok=False)
    import shutil
    with AUDIT_PATH.open("rb") as src, (destination / "audit.json").open("xb") as dst:
        shutil.copyfileobj(src, dst)
    audit_sha = sha256_file(AUDIT_PATH)
    require(sha256_file(destination / "audit.json") == audit_sha, "Audit backup SHA FAIL.")
    with (destination / "COMPLETE.json").open("x", encoding="utf-8") as stream:
        json.dump({"status": "COMPLETE", "audit_sha256": audit_sha}, stream, indent=2)
    print("Audit backup verified:", destination)

overall_pass = (
    RUN_TRAIN_SMOKE and INTEGRITY_PASS and FIXED_ARTIFACTS_PASS
    and all(SMOKE_RESULTS[k]["status"] == "PASS" for k in MODEL_TYPES)
)

print("\n=== STEP 6.5B SHARED TRAINING REPORT ===")
print("Git commit:", GIT_COMMIT)
print("Dataset fingerprint:", COMBINED_FP)
print("Tokenizer SHA-256:", TOKENIZER_SHA)
print("Model factory: moe_erc.model_factory.create_model")
print("Vanilla MoE parameter count:", MODEL_INFO["vanilla_moe"]["total_parameters"])
print("AoE parameter count:", MODEL_INFO["aoe"]["total_parameters"])
print("Causal shift verification:", {
    k: INTEGRITY_RESULTS[k].get("causal_shift_reference", "NOT PASS") for k in MODEL_TYPES
})
print("Load balancing implementation:", LB_POLICY)
for kind in MODEL_TYPES:
    print(kind, "forward/backward:", INTEGRITY_RESULTS[kind]["status"])
    print(kind, "smoke training:", SMOKE_RESULTS[kind]["status"])
print("NaN/Inf checks:", "PASS" if overall_pass else "Chưa đủ kiểm tra để PASS")
print("Optimizer update checks:", "PASS" if overall_pass else "NOT PASS/NOT RUN")
print("GPU memory peak GiB:", {
    kind: max(
        [row["peak_allocated_gib"] for row in SMOKE_RESULTS[kind]["steps"]],
        default=INTEGRITY_RESULTS[kind].get("peak_allocated_gib"),
    )
    for kind in MODEL_TYPES
})
print("Audit path:", AUDIT_PATH)
print("STATUS:", "PASS" if overall_pass else "FAIL")
if not RUN_TRAIN_SMOKE:
    print("Lý do chưa PASS toàn bước: smoke training đang tắt theo mặc định.")
print("Không chạy full training, không checkpoint, không ERC, không chuyển bước tiếp theo.")

{'model': 'vanilla_moe', 'parameters': 52288000, 'step': 1, 'ce': 501.4583435058594, 'lb_raw': 1.0141470432281494, 'lb_weighted': 0.010141470469534397, 'total': 501.4684753417969, 'gradient_norm_before_clip': 32.41044616699219, 'lr_used': 0.0004, 'effective_targets': 2044, 'validation_ce_subset': 490.82957458496094, 'validation_ppl_subset': 1.460749671005244e+213, 'peak_allocated_gib': 0.9171624183654785, 'input_tokens_per_second': 3752.4145381337034}
{'model': 'vanilla_moe', 'parameters': 52288000, 'step': 2, 'ce': 485.7870788574219, 'lb_raw': 1.0303188562393188, 'lb_weighted': 0.01030318858101964, 'total': 485.79737854003906, 'gradient_norm_before_clip': 81.1296615600586, 'lr_used': 0.00022000000000000003, 'effective_targets': 2044, 'validation_ce_subset': 471.4069519042969, 'validation_ppl_subset': 5.363372277578643e+204, 'peak_allocated_gib': 1.3074703216552734, 'input_tokens_per_second': 5111.35099353665}
{'model': 'vanilla_moe', 'parameters': 52288000, 'step': 3, 'ce': 465.847335

# STEP 6.5B-DIAG — CHẨN ĐOÁN CROSS-ENTROPY LOSS BẤT THƯỜNG

In [ ]:
from pathlib import Path
from contextlib import contextmanager
import gc
import hashlib
import inspect
import json
import math
import subprocess
import sys
import traceback

DIAG_PROJECT = Path("/content/moe-erc")
DIAG_DATA_ROOT = (
    DIAG_PROJECT / "data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
)
DIAG_TOKENIZER_PATH = (
    DIAG_PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"
)
DIAG_TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
DIAG_EXPECTED_COUNTS = {"vanilla_moe": 52_288_000, "aoe": 52_263_424}
DIAG_KINDS = tuple(DIAG_EXPECTED_COUNTS)
DIAG_INIT = {}
DIAG_RESULTS = {}
DIAG_CONTROL = {}

def diag_require(condition, message):
    if not condition:
        raise RuntimeError(message)

def diag_read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def diag_sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

diag_require(DIAG_PROJECT.is_dir(), "Project không tồn tại.")
if str(DIAG_PROJECT) not in sys.path:
    sys.path.insert(0, str(DIAG_PROJECT))

import numpy as np
import torch
import torch.nn.functional as F
from moe_erc.config import ModelConfig as DiagModelConfig, AoEConfig as DiagAoEConfig
from moe_erc.model import MoELanguageModel as DiagLanguageModel
from moe_erc.model_factory import create_model as diag_create_model
from moe_erc.rmsnorm import RMSNorm as DiagRMSNorm
from moe_erc.decoder import MoEDecoderBlock as DiagDecoderBlock
from moe_erc.dataset import PackedTokenDataset as DiagPackedDataset
from moe_erc.tokenizer_utils import FrozenTokenizer as DiagFrozenTokenizer

diag_require(torch.cuda.is_available(), "Cần CUDA để so sánh BF16/FP32.")
with torch.cuda.device(0):
    diag_require(torch.cuda.is_bf16_supported(), "BF16 không được hỗ trợ.")

DIAG_CONFIGS = {
    "vanilla_moe": DiagModelConfig(),
    "aoe": DiagAoEConfig(),
}
DIAG_GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(DIAG_PROJECT), "rev-parse", "HEAD"], text=True
).strip()

@contextmanager
def diag_seed_context():
    # Bảo toàn RNG của notebook; chỉ seed cho model chẩn đoán mới.
    devices = list(range(torch.cuda.device_count()))
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        yield

def diag_fresh_cpu(kind):
    with diag_seed_context():
        result = diag_create_model(kind, DIAG_CONFIGS[kind]).float()
    diag_require(
        sum(p.numel() for p in result.parameters()) == DIAG_EXPECTED_COUNTS[kind],
        "Parameter count không khớp.",
    )
    return result

@torch.no_grad()
def diag_stats(tensor):
    value = tensor.detach().float()
    result = {
        "shape": list(value.shape),
        "mean": float(value.mean()),
        "std_population": float(value.std(unbiased=False)),
        "min": float(value.min()),
        "max": float(value.max()),
        "mean_abs": float(value.abs().mean()),
        "rms": float(value.square().mean().sqrt()),
        "l2_norm": float(value.norm()),
    }
    if value.ndim == 2:
        row_norms = value.norm(dim=-1)
        result["row_norm_mean"] = float(row_norms.mean())
        result["row_norm_min"] = float(row_norms.min())
        result["row_norm_max"] = float(row_norms.max())
    return result

def diag_tying(model):
    embedding = model.token_embedding.weight
    head = model.lm_head.weight
    return {
        "same_parameter_object": embedding is head,
        "same_storage": (
            embedding.untyped_storage().data_ptr()
            == head.untyped_storage().data_ptr()
        ),
        "same_data_ptr": embedding.data_ptr() == head.data_ptr(),
    }

print("Git commit:", DIAG_GIT_COMMIT)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

for label, obj in (
    ("Model __init__", DiagLanguageModel.__init__),
    ("Model forward / loss / shift", DiagLanguageModel.forward),
    ("Decoder residual path", DiagDecoderBlock.forward),
    ("Final RMSNorm", DiagRMSNorm.forward),
    ("Model factory", diag_create_model),
    ("PyTorch Embedding.reset_parameters", torch.nn.Embedding.reset_parameters),
    ("PyTorch Linear.reset_parameters", torch.nn.Linear.reset_parameters),
):
    print(f"\n--- {label} ---")
    print(inspect.getsource(obj))

initialization_mentions = []
for source_file in (DIAG_PROJECT / "moe_erc").glob("*.py"):
    for number, line in enumerate(source_file.read_text(encoding="utf-8").splitlines(), 1):
        if any(marker in line for marker in (
            "nn.init.", "reset_parameters", "_init_weights", ".apply("
        )):
            initialization_mentions.append(f"{source_file.name}:{number}: {line.strip()}")
print("\nInitialization mentions trong project:", initialization_mentions or "Không tìm thấy.")

# Xác minh dataset/pilot, lấy đúng hai sequences đầu của pilot đã lưu.
DIAG_CORE = diag_read_json(DIAG_DATA_ROOT / "dataset_manifest.json")
DIAG_COMPLETE = diag_read_json(DIAG_DATA_ROOT / "COMPLETE.json")
diag_require(
    DIAG_COMPLETE["status"] == "COMPLETE"
    and diag_sha_file(DIAG_DATA_ROOT / "dataset_manifest.json")
    == DIAG_COMPLETE["dataset_manifest_sha256"],
    "Dataset completion/manifest sai.",
)
DIAG_FILE_GUARD = {}
for entry in DIAG_CORE["files"]:
    relative = Path(entry["path"])
    diag_require(not relative.is_absolute() and ".." not in relative.parts, "Artifact path sai.")
    path = DIAG_DATA_ROOT / relative
    diag_require(
        path.is_file() and not path.is_symlink()
        and path.stat().st_size == entry["bytes"]
        and diag_sha_file(path) == entry["sha256"],
        f"Artifact checksum FAIL: {path}",
    )
    DIAG_FILE_GUARD[str(path)] = entry["sha256"]

DIAG_PILOT_PATH = DIAG_DATA_ROOT / "pilot_50m_manifest.json"
DIAG_PILOT = diag_read_json(DIAG_PILOT_PATH)
diag_require(
    diag_sha_file(DIAG_PILOT_PATH)
    == (DIAG_DATA_ROOT / "pilot_50m_manifest.sha256").read_text().strip(),
    "Pilot manifest SHA sai.",
)
DIAG_INDEX_PATH = DIAG_DATA_ROOT / DIAG_PILOT["index_file"]
diag_require(
    DIAG_INDEX_PATH.resolve().is_relative_to(DIAG_DATA_ROOT.resolve())
    and diag_sha_file(DIAG_INDEX_PATH) == DIAG_PILOT["index_file_sha256"],
    "Pilot indices SHA/path sai.",
)
DIAG_INDICES = np.load(DIAG_INDEX_PATH, allow_pickle=False)
DIAG_TOKENIZER = DiagFrozenTokenizer(DIAG_TOKENIZER_PATH)
diag_require(DIAG_TOKENIZER.fingerprint == DIAG_TOKENIZER_SHA, "Tokenizer mismatch.")

diag_dataset = DiagPackedDataset(DIAG_DATA_ROOT, "train", tokenizer=DIAG_TOKENIZER)
try:
    diag_samples = [diag_dataset[int(index)] for index in DIAG_INDICES[:2]]
    DIAG_BATCH_CPU = {
        name: torch.stack([sample[name] for sample in diag_samples])
        for name in ("input_ids", "labels")
    }
finally:
    diag_dataset.close()

diag_require(
    DIAG_BATCH_CPU["input_ids"].shape == (2, 512)
    and DIAG_BATCH_CPU["input_ids"].dtype == torch.long
    and torch.equal(DIAG_BATCH_CPU["input_ids"], DIAG_BATCH_CPU["labels"]),
    "Diagnostic batch/labels sai.",
)
diag_require(
    int(DIAG_BATCH_CPU["input_ids"].min()) >= 0
    and int(DIAG_BATCH_CPU["input_ids"].max()) < 16000,
    "Input IDs ngoài vocab.",
)

for path in (
    DIAG_PILOT_PATH, DIAG_INDEX_PATH,
    DIAG_DATA_ROOT / "pilot_50m_manifest.sha256",
    DIAG_DATA_ROOT / "dataset_manifest.json", DIAG_DATA_ROOT / "COMPLETE.json",
    DIAG_TOKENIZER_PATH, DIAG_TOKENIZER_PATH.with_suffix(".meta.json"),
    *(DIAG_PROJECT / "moe_erc").glob("*.py"),
):
    DIAG_FILE_GUARD[str(path)] = diag_sha_file(path)

def diag_verify_unchanged():
    for name, expected in DIAG_FILE_GUARD.items():
        diag_require(diag_sha_file(name) == expected, f"File đã thay đổi: {name}")

def diag_inspect_initialization(kind):
    local_model = diag_fresh_cpu(kind)
    try:
        # Embedding là parameter đầu tiên được tạo trong implementation hiện tại.
        with diag_seed_context():
            reference_embedding = torch.nn.Embedding(16000, 512)
        exact_default_match = torch.equal(
            local_model.token_embedding.weight.detach(),
            reference_embedding.weight.detach(),
        )
        del reference_embedding
        result = {
            "parameters": sum(p.numel() for p in local_model.parameters()),
            "embedding": diag_stats(local_model.token_embedding.weight),
            "lm_head": diag_stats(local_model.lm_head.weight),
            "tying": diag_tying(local_model),
            "exact_match_standard_embedding_seed42": exact_default_match,
            "final_norm_scale": diag_stats(local_model.final_norm.weight),
            "final_norm_eps": local_model.final_norm.eps,
            "first_attention_q_weight": diag_stats(
                local_model.layers[0].attention.q_proj.weight
            ),
        }
        print(f"\n{kind} initialization:")
        print(json.dumps(result, indent=2))
        return result
    finally:
        del local_model
        gc.collect()

for diag_kind in DIAG_KINDS:
    DIAG_INIT[diag_kind] = diag_inspect_initialization(diag_kind)

print("\nSource observations:")
print("- nn.Embedding không nhận explicit initialization trong project.")
print("- LM head được tạo bằng Linear, sau đó weight bị thay bằng embedding parameter.")
print("- Vì vậy effective LM-head distribution là embedding distribution, không phải Linear default.")
print("- Final RMSNorm được gọi trước LM head; không có explicit embedding/logit scaling trong forward.")
print("- Chưa thay đổi bất kỳ parameter nào của model hiện có trong notebook.")

Git commit: 2fd3478e478d3a20381e95b06890f8dcdf2ae013
PyTorch: 2.11.0+cu130
GPU: NVIDIA A100-SXM4-40GB

--- Model __init__ ---
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList(self.decoder_block_cls(config) for _ in range(config.n_layers))
        self.final_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        # Share the actual Parameter, not a one-time copy of its values.
        self.lm_head.weight = self.token_embedding.weight


--- Model forward / loss / shift ---
    def forward(
        self, input_ids: torch.Tensor, labels: torch.Tensor | None = None,
        *, return_routing_info: bool = False,
    ) -> torch.Tensor | LanguageModelOutput:
        self._validate_inputs(input_ids, labels)
        hidden = self.token_embed

In [ ]:
@torch.no_grad()
def diag_measure(local_model, precision):
    captures = {}
    handles = []
    previous_tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False

    def capture_embedding(module, args, output):
        captures["embedding"] = output.detach().float()

    def capture_norm(module, args, output):
        captures["norm_calls"] = captures.get("norm_calls", 0) + 1
        captures["pre_final_norm"] = args[0].detach().float()
        captures["post_final_norm"] = output.detach().float()

    def capture_head(module, args):
        captures["head_input"] = args[0].detach().float()

    handles.append(local_model.token_embedding.register_forward_hook(capture_embedding))
    handles.append(local_model.final_norm.register_forward_hook(capture_norm))
    handles.append(local_model.lm_head.register_forward_pre_hook(capture_head))

    try:
        local_model.eval()
        batch = {name: value.to("cuda:0") for name, value in DIAG_BATCH_CPU.items()}
        torch.cuda.synchronize()
        baseline_allocated = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()

        with torch.amp.autocast(
            "cuda", dtype=torch.bfloat16, enabled=(precision == "bf16")
        ):
            # Model-only CE: không gọi shared trainer, LB hooks hoặc ERC.
            output = local_model(**batch)

        diag_require(output.logits.shape == (2, 512, 16000), "Logits shape sai.")
        diag_require(bool(torch.isfinite(output.logits).all()), "Logits nonfinite.")
        diag_require(captures.get("norm_calls") == 1, "Final RMSNorm không được gọi đúng một lần.")
        diag_require(
            torch.equal(captures["head_input"], captures["post_final_norm"]),
            "LM head không nhận đúng output final RMSNorm.",
        )

        logits = output.logits.float()
        prediction_logits = logits[:, :-1, :]
        current_ids = batch["input_ids"][:, :-1]
        target_ids = batch["labels"][:, 1:]

        per_position_ce = F.cross_entropy(
            prediction_logits.reshape(-1, 16000),
            target_ids.reshape(-1),
            reduction="none",
        ).reshape_as(target_ids)
        independent_ce = per_position_ce.mean()
        torch.testing.assert_close(
            output.loss.float(), independent_ce, rtol=1e-5, atol=1e-4
        )

        current_logits = prediction_logits.gather(
            -1, current_ids.unsqueeze(-1)
        ).squeeze(-1)
        target_logits = prediction_logits.gather(
            -1, target_ids.unsqueeze(-1)
        ).squeeze(-1)
        log_partition = prediction_logits.logsumexp(dim=-1)
        gap = current_logits - target_logits

        # CE = (current_logit - target_logit) + (logsumexp - current_logit).
        decomposition_remainder = log_partition - current_logits
        torch.testing.assert_close(
            per_position_ce, gap + decomposition_remainder,
            rtol=1e-5, atol=1e-4,
        )

        different = current_ids != target_ids
        top1 = prediction_logits.argmax(dim=-1)
        same_position_ce = F.cross_entropy(
            prediction_logits.reshape(-1, 16000), current_ids.reshape(-1)
        )

        segments = []
        for start in range(0, 511, 128):
            stop = min(start + 128, 511)
            segments.append({
                "prediction_position_start": start,
                "prediction_position_stop_exclusive": stop,
                "next_token_ce": float(per_position_ce[:, start:stop].mean()),
                "current_token_top1_fraction": float(
                    (top1[:, start:stop] == current_ids[:, start:stop]).float().mean()
                ),
                "mean_current_minus_target_logit": float(gap[:, start:stop].mean()),
            })

        embedding = captures["embedding"]
        hidden = captures["pre_final_norm"]
        post_norm = captures["post_final_norm"]
        cosine = F.cosine_similarity(post_norm, embedding, dim=-1)
        residual_change_ratio = (
            (hidden - embedding).norm(dim=-1)
            / embedding.norm(dim=-1).clamp_min(1e-12)
        )

        flat_logits = prediction_logits.reshape(-1, 16000)
        sample_positions = torch.linspace(
            0, flat_logits.shape[0] - 1, steps=min(64, flat_logits.shape[0])
        ).long().to(flat_logits.device)
        sampled_logits = flat_logits.index_select(0, sample_positions).cpu()

        ce_value = float(independent_ce)
        result = {
            "precision": precision,
            "logits_dtype": str(output.logits.dtype),
            "logits": diag_stats(logits),
            "model_ce": float(output.loss),
            "independent_next_token_ce": ce_value,
            "same_position_ce_diagnostic_only": float(same_position_ce),
            "current_token_logits": diag_stats(current_logits),
            "correct_next_token_logits": diag_stats(target_logits),
            "current_minus_next_logit_gap": diag_stats(gap),
            "current_token_top1_fraction": float((top1 == current_ids).float().mean()),
            "current_differs_from_next_fraction": float(different.float().mean()),
            "current_top1_when_next_differs": (
                float((top1[different] == current_ids[different]).float().mean())
                if bool(different.any()) else None
            ),
            "ce_when_next_differs": (
                float(per_position_ce[different].mean()) if bool(different.any()) else None
            ),
            "ce_when_next_repeats_current": (
                float(per_position_ce[~different].mean()) if bool((~different).any()) else None
            ),
            "logsumexp_minus_current_mean": float(decomposition_remainder.mean()),
            "mean_gap_divided_by_ce": float(gap.mean()) / max(ce_value, 1e-12),
            "hidden_input_embedding_cosine_mean": float(cosine.mean()),
            "pre_final_residual_change_over_embedding_norm_mean": float(residual_change_ratio.mean()),
            "pre_final_norm_rms": float(hidden.square().mean().sqrt()),
            "post_final_norm_rms": float(post_norm.square().mean().sqrt()),
            "final_norm_applied": True,
            "causal_shift_reference": "PASS",
            "auxiliary_loss_included_in_ce": False,
            "segments": segments,
            "peak_total_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
            "peak_increment_from_forward_gib": (
                torch.cuda.max_memory_allocated() - baseline_allocated
            ) / 1024**3,
            "tf32_enabled_for_measurement": False,
        }
        return result, sampled_logits
    finally:
        for handle in handles:
            handle.remove()
        captures.clear()
        torch.backends.cuda.matmul.allow_tf32 = previous_tf32

def diag_precision_pair(local_model):
    bf16, bf16_samples = diag_measure(local_model, "bf16")
    fp32, fp32_samples = diag_measure(local_model, "fp32")
    difference = bf16_samples - fp32_samples
    comparison = {
        "ce_absolute_difference": abs(
            bf16["independent_next_token_ce"] - fp32["independent_next_token_ce"]
        ),
        "ce_relative_difference": abs(
            bf16["independent_next_token_ce"] - fp32["independent_next_token_ce"]
        ) / max(abs(fp32["independent_next_token_ce"]), 1e-12),
        "sampled_logits_mean_absolute_difference": float(difference.abs().mean()),
        "sampled_logits_max_absolute_difference": float(difference.abs().max()),
        "sampled_logits_relative_l2_difference": float(
            difference.norm() / fp32_samples.norm().clamp_min(1e-12)
        ),
        "sampled_positions_top1_agreement": float(
            (bf16_samples.argmax(-1) == fp32_samples.argmax(-1)).float().mean()
        ),
        "both_shift_references_pass": True,
        "scope": "full-batch CE; logits comparison uses 64 fixed prediction positions",
    }
    return {"bf16": bf16, "fp32": fp32, "precision_comparison": comparison}

def diag_initial_forward(kind):
    local_model = None
    try:
        local_model = diag_fresh_cpu(kind).to("cuda:0")
        return diag_precision_pair(local_model)
    finally:
        local_model = None
        gc.collect()
        torch.cuda.empty_cache()

for diag_kind in DIAG_KINDS:
    try:
        DIAG_RESULTS[diag_kind] = diag_initial_forward(diag_kind)
        print(f"\n=== {diag_kind}: INITIAL MODEL, SEED 42 ===")
        print(json.dumps(DIAG_RESULTS[diag_kind], indent=2))
    except Exception as exc:
        DIAG_RESULTS[diag_kind] = {
            "error": f"{type(exc).__name__}: {exc}"
        }
        traceback.print_exc()
        if isinstance(exc, torch.cuda.OutOfMemoryError):
            print("Dừng do OOM; không xóa hoặc thay đổi model khác trong notebook.")
            break

print("\nInterpretation:")
print("- current_token_top1_fraction đo xu hướng dự đoán lại chính input token.")
print("- CE decomposition tách current-target gap và logsumexp-current.")
print("- Khi gap giải thích phần lớn CE, đó là bằng chứng định lượng về self-token bias.")
print("- same_position_ce chỉ là diagnostic counterfactual, không phải training objective.")
print("- BF16/FP32 được đo tuần tự trên cùng weights và batch; không optimizer step.")


=== vanilla_moe: INITIAL MODEL, SEED 42 ===
{
  "bf16": {
    "precision": "bf16",
    "logits_dtype": "torch.bfloat16",
    "logits": {
      "shape": [
        2,
        512,
        16000
      ],
      "mean": 0.06821047514677048,
      "std_population": 22.985363006591797,
      "min": -114.0,
      "max": 548.0,
      "mean_abs": 18.088605880737305,
      "rms": 22.985464096069336,
      "l2_norm": 93038.609375
    },
    "model_ce": 499.6509094238281,
    "independent_next_token_ce": 499.6509094238281,
    "same_position_ce_diagnostic_only": 0.0,
    "current_token_logits": {
      "shape": [
        2,
        511
      ],
      "mean": 503.90606689453125,
      "std_population": 17.69293975830078,
      "min": 404.0,
      "max": 548.0,
      "mean_abs": 503.90606689453125,
      "rms": 504.2165832519531,
      "l2_norm": 16119.166015625,
      "row_norm_mean": 11397.900390625,
      "row_norm_min": 11357.70703125,
      "row_norm_max": 11438.0947265625
    },
    "correct_n

In [ ]:
DIAG_CONTROL_STD = 0.02

def diag_non_embedding_digest(local_model):
    digest = hashlib.sha256()
    shared_embedding = local_model.token_embedding.weight
    for name, parameter in local_model.named_parameters():
        if parameter is shared_embedding:
            continue
        value = parameter.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def diag_controlled_experiment(kind):
    local_model = None
    try:
        local_model = diag_fresh_cpu(kind)
        count_before = sum(p.numel() for p in local_model.parameters())
        other_parameters_before = diag_non_embedding_digest(local_model)
        local_model = local_model.to("cuda:0")

        # Baseline và treatment cùng model thử nghiệm, không training.
        before = diag_precision_pair(local_model)
        embedding_before = diag_stats(local_model.token_embedding.weight)

        with torch.no_grad():
            measured_std = float(
                local_model.token_embedding.weight.std(unbiased=False)
            )
            diag_require(measured_std > 0, "Embedding std bằng 0.")
            scale_factor = DIAG_CONTROL_STD / measured_std
            # Chỉ thực hiện MỘT lần: LM head là cùng Parameter.
            local_model.token_embedding.weight.mul_(scale_factor)

        other_parameters_after = diag_non_embedding_digest(local_model)
        diag_require(
            other_parameters_after == other_parameters_before,
            "Có parameter khác embedding bị thay đổi.",
        )
        tying_after = diag_tying(local_model)
        diag_require(all(tying_after.values()), "Weight tying bị phá vỡ.")
        diag_require(
            sum(p.numel() for p in local_model.parameters()) == count_before
            == DIAG_EXPECTED_COUNTS[kind],
            "Architecture/parameter count thay đổi.",
        )

        after = diag_precision_pair(local_model)
        diag_require(
            diag_non_embedding_digest(local_model) == other_parameters_before,
            "Forward đã thay đổi non-embedding parameters.",
        )
        result = {
            "status": "MEASURED",
            "intervention": (
                "embedding-only rescaling; same directions; tied head changes automatically"
            ),
            "requested_std": DIAG_CONTROL_STD,
            "scale_factor": scale_factor,
            "embedding_before": embedding_before,
            "embedding_after": diag_stats(local_model.token_embedding.weight),
            "lm_head_after": diag_stats(local_model.lm_head.weight),
            "other_learned_parameters_unchanged": True,
            "tying_after": tying_after,
            "parameter_count_unchanged": count_before,
            "before": before,
            "after": after,
            "fp32_ce_decrease": (
                before["fp32"]["independent_next_token_ce"]
                - after["fp32"]["independent_next_token_ce"]
            ),
            "bf16_ce_decrease": (
                before["bf16"]["independent_next_token_ce"]
                - after["bf16"]["independent_next_token_ce"]
            ),
            "baseline_recheck_ce_difference_vs_cell2": abs(
                before["fp32"]["independent_next_token_ce"]
                - DIAG_RESULTS[kind]["fp32"]["independent_next_token_ce"]
            ),
        }
        return result
    finally:
        local_model = None
        gc.collect()
        torch.cuda.empty_cache()

for diag_kind in DIAG_KINDS:
    if "fp32" not in DIAG_RESULTS.get(diag_kind, {}):
        DIAG_CONTROL[diag_kind] = {"status": "SKIPPED", "reason": "Cell 2 chưa hoàn tất."}
        continue
    try:
        DIAG_CONTROL[diag_kind] = diag_controlled_experiment(diag_kind)
        control = DIAG_CONTROL[diag_kind]
        print(f"\n=== CONTROLLED TEST: {diag_kind} ===")
        print(json.dumps({
            "intervention": control["intervention"],
            "embedding_std_before": control["embedding_before"]["std_population"],
            "embedding_std_after": control["embedding_after"]["std_population"],
            "parameters": control["parameter_count_unchanged"],
            "other_parameters_unchanged": control["other_learned_parameters_unchanged"],
            "tying": control["tying_after"],
            "before_fp32": control["before"]["fp32"],
            "after_fp32": control["after"]["fp32"],
            "before_precision_comparison": control["before"]["precision_comparison"],
            "after_precision_comparison": control["after"]["precision_comparison"],
            "fp32_ce_decrease": control["fp32_ce_decrease"],
            "bf16_ce_decrease": control["bf16_ce_decrease"],
            "baseline_recheck_ce_difference_vs_cell2":
                control["baseline_recheck_ce_difference_vs_cell2"],
        }, indent=2))
    except Exception as exc:
        DIAG_CONTROL[diag_kind] = {
            "status": "ERROR", "error": f"{type(exc).__name__}: {exc}"
        }
        traceback.print_exc()
        if isinstance(exc, torch.cuda.OutOfMemoryError):
            break

DIAG_FILES_UNCHANGED = False
try:
    diag_verify_unchanged()
    DIAG_FILES_UNCHANGED = True
except Exception:
    traceback.print_exc()

# Quy tắc kết luận công bố rõ:
# phải có cả cơ chế self-token dominance và tác động kiểm soát trên FP32.
DIAG_EVIDENCE = {}
for kind in DIAG_KINDS:
    base = DIAG_RESULTS.get(kind, {}).get("fp32")
    control = DIAG_CONTROL.get(kind, {})
    if base is None or control.get("status") != "MEASURED":
        DIAG_EVIDENCE[kind] = {"sufficient_evidence": False}
        continue

    before = control["before"]["fp32"]
    after = control["after"]["fp32"]
    precision_delta = control["before"]["precision_comparison"]["ce_absolute_difference"]
    tests = {
        "matches_standard_pytorch_embedding":
            DIAG_INIT[kind]["exact_match_standard_embedding_seed42"],
        "tied_parameter_and_storage":
            all(DIAG_INIT[kind]["tying"].values()),
        "final_norm_and_shift_verified":
            base["final_norm_applied"] and base["causal_shift_reference"] == "PASS",
        "current_token_is_top1_at_majority_positions":
            base["current_token_top1_fraction"] > 0.5,
        "current_target_gap_explains_majority_of_ce":
            base["mean_gap_divided_by_ce"] > 0.5,
        "controlled_fp32_ce_decreases":
            after["independent_next_token_ce"] < before["independent_next_token_ce"],
        "controlled_gap_decreases":
            after["current_minus_next_logit_gap"]["mean"]
            < before["current_minus_next_logit_gap"]["mean"],
        "control_effect_exceeds_precision_ce_difference":
            control["fp32_ce_decrease"] > precision_delta,
        "other_parameters_unchanged":
            control["other_learned_parameters_unchanged"],
    }
    DIAG_EVIDENCE[kind] = {**tests, "sufficient_evidence": all(tests.values())}

DIAG_DIAGNOSED = (
    DIAG_FILES_UNCHANGED
    and all(DIAG_EVIDENCE.get(k, {}).get("sufficient_evidence", False) for k in DIAG_KINDS)
)

print("\n=== STEP 6.5B LOSS DIAGNOSTIC REPORT ===")
for kind in DIAG_KINDS:
    base = DIAG_RESULTS.get(kind, {}).get("fp32", {})
    init = DIAG_INIT.get(kind, {})
    print(f"\n{kind}:")
    print("  Initial CE FP32:", base.get("independent_next_token_ce", "UNAVAILABLE"))
    print("  Initial CE BF16:",
          DIAG_RESULTS.get(kind, {}).get("bf16", {}).get("independent_next_token_ce", "UNAVAILABLE"))
    print("  Embedding weight std:", init.get("embedding", {}).get("std_population"))
    print("  LM head weight std:", init.get("lm_head", {}).get("std_population"))
    print("  Tied weights:", init.get("tying"))
    print("  Final normalization:", base.get("final_norm_applied"))
    print("  Logits statistics:", base.get("logits"))
    print("  Current-token logits:", base.get("current_token_logits"))
    print("  Current-token top1 fraction:", base.get("current_token_top1_fraction"))
    print("  Correct-next-token logits:", base.get("correct_next_token_logits"))
    print("  Current-target gap:", base.get("current_minus_next_logit_gap"))
    print("  BF16 vs FP32:", DIAG_RESULTS.get(kind, {}).get("precision_comparison"))
    print("  Causal shift:", base.get("causal_shift_reference"))
    print("  Controlled initialization test:", DIAG_CONTROL.get(kind, {}).get("status"))
    print("  Evidence:", DIAG_EVIDENCE[kind])

print("\nTheoretical uniform CE:", math.log(16000))
print("Embedding initialization: xem Embedding.reset_parameters và exact-reference test ở Cell 1.")

if DIAG_DIAGNOSED:
    print(
        "Most likely root cause: scale của default embedding kết hợp residual "
        "self-alignment và tied, unscaled LM head tạo current-token logits cực lớn."
    )
    print(
        "Recommended minimal fix để xem xét: explicit initialization cho shared "
        "embedding/LM-head parameter đúng một lần trong base model dùng chung. "
        "Kết quả std=0.02 là bằng chứng chẩn đoán, chưa chứng minh đây là recipe tối ưu. "
        "Cần rà soát toàn bộ Linear initialization và residual scaling trước khi chốt."
    )
else:
    print(
        "Most likely root cause: INCONCLUSIVE. Không đủ bằng chứng cho toàn bộ "
        "chuỗi giả thuyết; xem các evidence flags và errors phía trên."
    )
    print(
        "Recommended minimal fix: chưa áp dụng. Kiểm tra thêm hidden/residual scales, "
        "normalization, runtime implementation và khác biệt FP32/BF16."
    )

print(
    "Expected effects on architecture/parameter count nếu chỉ đổi initialization: "
    "không đổi architecture hoặc count; Vanilla=52,288,000; AoE=52,263,424."
)
print("Additional regression tests needed:")
print("- Explicit initialization statistics và shared Parameter/storage sau save/load.")
print("- Initial CE, logits scale, self-token bias trên nhiều seeds; không ép CE=ln(vocab).")
print("- Independent shifted CE, FP32/BF16, gradients hữu hạn cho backbone/routing/experts.")
print("- Chạy lại smoke ngắn và validation subset sau khi một bản vá được phê duyệt.")
print("- Giữ tokenizer/dataset/pilot checksums; Vanilla và Vanilla+ERC dùng cùng initial state_dict.")
print("Files unchanged:", DIAG_FILES_UNCHANGED)
print("STATUS:", "DIAGNOSED" if DIAG_DIAGNOSED else "INCONCLUSIVE")
print("Không sửa source, không optimizer step, không lưu checkpoint hoặc áp dụng bản vá.")


=== CONTROLLED TEST: vanilla_moe ===
{
  "intervention": "embedding-only rescaling; same directions; tied head changes automatically",
  "embedding_std_before": 0.9999698400497437,
  "embedding_std_after": 0.019999999552965164,
  "parameters": 52288000,
  "other_parameters_unchanged": true,
  "tying": {
    "same_parameter_object": true,
    "same_storage": true,
    "same_data_ptr": true
  },
  "before_fp32": {
    "precision": "fp32",
    "logits_dtype": "torch.float32",
    "logits": {
      "shape": [
        2,
        512,
        16000
      ],
      "mean": 0.06819261610507965,
      "std_population": 22.985557556152344,
      "min": -113.90780639648438,
      "max": 547.1590576171875,
      "mean_abs": 18.088720321655273,
      "rms": 22.985660552978516,
      "l2_norm": 93039.40625
    },
    "model_ce": 499.7168273925781,
    "independent_next_token_ce": 499.716796875,
    "same_position_ce_diagnostic_only": 0.0,
    "current_token_logits": {
      "shape": [
        2,
   

# STEP 6.5B-RETEST — KIỂM THỬ TRAINING SAU KHI FIX INITIALIZATION

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import traceback

RT_PROJECT = Path("/content/moe-erc")
RT_DATA = RT_PROJECT / "data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
RT_TOKENIZER_PATH = RT_PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22/tokenizer.json"
RT_COMMIT = "665cb7b1cfbca4c6ed3507345b21008321129a2a"
RT_OLD_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
RT_MODEL_SHA = "2389cc7be64b8f68a85ccbe37769a0786754dcce4b36c9859ddb53f68b8c7166"
RT_OLD_MODEL_SHA = "53d4ad3cade511a70c8c0944561ff9ed8437cbc8cbabc80b1d32a81a2fe9dfbe"
RT_TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
RT_DATA_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"
RT_KINDS = ("vanilla_moe", "aoe")
RT_EXPECTED_PARAMETERS = {"vanilla_moe": 52_288_000, "aoe": 52_263_424}
RT_DIAGNOSTICS = {}
RT_DIAGNOSTICS_PASS = False
RT_TRAIN_RESULTS = {kind: {"status": "NOT RUN", "steps": []} for kind in RT_KINDS}

def rt_require(condition, message):
    if not condition:
        raise RuntimeError(message)

def rt_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def rt_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def rt_fingerprint(value):
    return hashlib.sha256(json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode()).hexdigest()

def rt_git(*arguments):
    return subprocess.check_output(
        ["git", "-C", str(RT_PROJECT), *arguments]
    )

def rt_safe_file(root, name):
    relative = Path(name)
    rt_require(
        not relative.is_absolute() and ".." not in relative.parts
        and "\\" not in name,
        f"Artifact path sai: {name}",
    )
    current = Path(root)
    for part in relative.parts:
        current = current / part
        rt_require(not current.is_symlink(), f"Artifact symlink: {current}")
    rt_require(
        current.is_file()
        and current.resolve().is_relative_to(Path(root).resolve()),
        f"Thiếu hoặc sai artifact: {current}",
    )
    return current

rt_require(RT_PROJECT.is_dir(), "Project không tồn tại.")
rt_require(rt_git("rev-parse", "HEAD").decode().strip() == RT_COMMIT, "Git HEAD không phải FIX commit.")
RT_WORKTREE = rt_git("status", "--porcelain").decode()
print("Working tree:", RT_WORKTREE or "CLEAN")
rt_require(not RT_WORKTREE.strip(), "Working tree không sạch; không tự sửa/reset.")

if str(RT_PROJECT) not in sys.path:
    sys.path.insert(0, str(RT_PROJECT))

import numpy as np
import torch
import torch.nn.functional as F
from moe_erc.config import ModelConfig, AoEConfig
from moe_erc.model_factory import create_model
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.data_processing import validate_dataset, audit_document_splits
from moe_erc.dataset import PackedTokenDataset

rt_require(torch.cuda.is_available(), "CUDA unavailable.")
RT_DEVICE = torch.device("cuda:0")
RT_GPU = torch.cuda.get_device_properties(0)
rt_require("A100" in RT_GPU.name.upper(), f"GPU thực tế không phải A100: {RT_GPU.name}")
with torch.cuda.device(0):
    rt_require(torch.cuda.is_bf16_supported(), "BF16 unavailable.")

# Strict runtime source identity theo commit FIX.
RT_CODE_GUARD = {}
package_files = rt_git(
    "ls-tree", "-r", "--name-only", RT_COMMIT, "--", "moe_erc"
).decode().splitlines()
package_files = [name for name in package_files if name.endswith(".py")]
for name in package_files:
    expected = hashlib.sha256(rt_git("show", f"{RT_COMMIT}:{name}")).hexdigest()
    path = RT_PROJECT / name
    rt_require(not path.is_symlink() and rt_sha(path) == expected, f"Source mismatch: {name}")
    RT_CODE_GUARD[str(path)] = expected
rt_require(rt_sha(RT_PROJECT / "moe_erc/model.py") == RT_MODEL_SHA, "FIX model SHA sai.")

RT_TOKENIZER = FrozenTokenizer(RT_TOKENIZER_PATH)
rt_require(
    RT_TOKENIZER.fingerprint == RT_TOKENIZER_SHA
    and RT_TOKENIZER.vocab_size == 16000 and RT_TOKENIZER.eos_id == 2,
    "Tokenizer mismatch.",
)

marker = rt_json(RT_DATA / "COMPLETE.json")
rt_require(
    marker["status"] == "COMPLETE"
    and rt_sha(RT_DATA / "dataset_manifest.json") == marker["dataset_manifest_sha256"],
    "Dataset completion/manifest sai.",
)
core = rt_json(RT_DATA / "dataset_manifest.json")
rt_require(core["combined_source_fingerprint"] == RT_DATA_FP, "Dataset fingerprint sai.")
RT_ARTIFACT_GUARD = {}
for entry in core["files"]:
    path = rt_safe_file(RT_DATA, entry["path"])
    rt_require(
        path.stat().st_size == entry["bytes"] and rt_sha(path) == entry["sha256"],
        f"Dataset checksum FAIL: {path}",
    )
    RT_ARTIFACT_GUARD[str(path)] = entry["sha256"]

# Giữ strict data-pipeline checks, chỉ ngoại lệ model.py cũ -> FIX đã pin.
design = rt_json(RT_DATA / "packing_design.json")
for name, stored_sha in design["pipeline_source_sha256"].items():
    path = RT_PROJECT / "moe_erc" / name
    if name == "model.py":
        rt_require(
            stored_sha == RT_OLD_MODEL_SHA and rt_sha(path) == RT_MODEL_SHA,
            "Model compatibility exception không hợp lệ.",
        )
    else:
        rt_require(
            rt_sha(path) == stored_sha
            == hashlib.sha256(rt_git("show", f"{RT_OLD_COMMIT}:moe_erc/{name}")).hexdigest(),
            f"Data-pipeline source mismatch: {name}",
        )

RT_METADATA = validate_dataset(RT_DATA, tokenizer=RT_TOKENIZER, checksums=True)
rt_require(
    RT_METADATA["sequence_length"] == 512 and RT_METADATA["dtype"] == "<u2"
    and RT_METADATA["splits"]["train"]["sequences"] == 107478
    and RT_METADATA["splits"]["validation"]["sequences"] == 1006,
    "Dataset layout/count mismatch.",
)
rt_require(audit_document_splits(RT_DATA)["overlap"] == 0, "Document leakage.")

RT_PILOT_PATH = RT_DATA / "pilot_50m_manifest.json"
RT_PILOT = rt_json(RT_PILOT_PATH)
RT_PILOT_SHA = rt_sha(RT_PILOT_PATH)
rt_require(
    RT_PILOT_SHA == (RT_DATA / "pilot_50m_manifest.sha256").read_text().strip()
    and RT_PILOT["split"] == "train"
    and RT_PILOT["combined_source_fingerprint"] == RT_DATA_FP
    and RT_PILOT["tokenizer_sha256"] == RT_TOKENIZER_SHA
    and RT_PILOT["dataset_manifest_sha256"] == marker["dataset_manifest_sha256"],
    "Pilot manifest mismatch.",
)
RT_INDEX_PATH = rt_safe_file(RT_DATA, RT_PILOT["index_file"])
rt_require(rt_sha(RT_INDEX_PATH) == RT_PILOT["index_file_sha256"], "Pilot indices SHA sai.")
RT_INDICES = np.load(RT_INDEX_PATH, allow_pickle=False)
rt_require(
    RT_INDICES.ndim == 1 and len(RT_INDICES) == 97657
    and len(np.unique(RT_INDICES)) == len(RT_INDICES)
    and int(RT_INDICES.min()) >= 0 and int(RT_INDICES.max()) < 107478
    and RT_PILOT["selected_input_tokens"] == 50000384,
    "Pilot indices/count sai.",
)

for path in (
    RT_DATA / "COMPLETE.json", RT_DATA / "dataset_manifest.json",
    RT_PILOT_PATH, RT_DATA / "pilot_50m_manifest.sha256", RT_INDEX_PATH,
    RT_TOKENIZER_PATH, RT_TOKENIZER_PATH.with_suffix(".meta.json"),
):
    RT_ARTIFACT_GUARD[str(path)] = rt_sha(path)

def rt_verify_immutable():
    rt_require(rt_git("rev-parse", "HEAD").decode().strip() == RT_COMMIT, "Git HEAD đổi.")
    for name, expected in {**RT_ARTIFACT_GUARD, **RT_CODE_GUARD}.items():
        rt_require(rt_sha(name) == expected, f"Protected file đã thay đổi: {name}")

@dataclass(frozen=True)
class RTConfig:
    seed: int = 42
    micro_batch_size: int = 2
    accumulation_steps: int = 2
    optimizer_steps: int = 3
    warmup_steps: int = 1
    peak_lr: float = 4e-4
    min_lr: float = 4e-5
    beta1: float = 0.9
    beta2: float = 0.95
    weight_decay: float = 0.1
    lb_weight: float = 0.01
    clip_norm: float = 1.0
    validation_batch_size: int = 2
    validation_batches: int = 2
    max_initial_ce: float = 50.0  # Catastrophic-scale guard, không ép CE=ln(V).

    def validate(self):
        rt_require(1 <= self.optimizer_steps <= 3, "Hard limit: tối đa 3 steps.")
        rt_require(
            1 <= self.micro_batch_size <= 2 and 1 <= self.accumulation_steps <= 2,
            "Smoke batch/accumulation tối đa 2.",
        )
        rt_require(1 <= self.warmup_steps <= self.optimizer_steps, "Warmup sai.")
        rt_require(
            1 <= self.validation_batch_size <= 2 and 1 <= self.validation_batches <= 2,
            "Validation smoke tối đa 4 sequences.",
        )
        rt_require(self.lb_weight == 0.01, "Giữ LB coefficient đã kiểm thử.")
        rt_require(
            0 < self.min_lr <= self.peak_lr and self.clip_norm > 0,
            "LR/clip config sai.",
        )

RT_CFG = RTConfig()
RT_CFG.validate()
RT_CONFIGS = {"vanilla_moe": ModelConfig(), "aoe": AoEConfig()}

def rt_seed():
    random.seed(RT_CFG.seed)
    np.random.seed(RT_CFG.seed)
    torch.manual_seed(RT_CFG.seed)
    torch.cuda.manual_seed_all(RT_CFG.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

def rt_state_sha(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def rt_new_cpu(kind):
    rt_seed()
    model = create_model(kind, RT_CONFIGS[kind]).float()
    embedding, head = model.token_embedding.weight, model.lm_head.weight
    rt_require(
        embedding is head
        and embedding.untyped_storage().data_ptr() == head.untyped_storage().data_ptr(),
        "Weight tying/storage sai.",
    )
    rt_require(abs(float(embedding.detach().std(unbiased=False)) - 0.02) < 1e-4, "FIX chưa có hiệu lực.")
    rt_require(
        sum(p.numel() for p in model.parameters()) == RT_EXPECTED_PARAMETERS[kind],
        "Parameter count sai.",
    )
    return model

def rt_loader(dataset, indices, batch_size):
    return torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [int(index) for index in indices]),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False,
        generator=torch.Generator().manual_seed(RT_CFG.seed),
    )

RT_TRAIN_DATA = PackedTokenDataset(RT_DATA, "train", tokenizer=RT_TOKENIZER)
RT_VAL_DATA = PackedTokenDataset(RT_DATA, "validation", tokenizer=RT_TOKENIZER)
RT_FIRST_BATCH = next(iter(rt_loader(RT_TRAIN_DATA, RT_INDICES[:2], 2)))
rt_require(
    RT_FIRST_BATCH["input_ids"].shape == (2, 512)
    and RT_FIRST_BATCH["input_ids"].dtype == torch.long
    and torch.equal(RT_FIRST_BATCH["input_ids"], RT_FIRST_BATCH["labels"]),
    "Real Dolma batch/labels sai.",
)

RT_MODEL_INFO = {}
for kind in RT_KINDS:
    temporary_model = rt_new_cpu(kind)
    RT_MODEL_INFO[kind] = {
        "parameters": sum(p.numel() for p in temporary_model.parameters()),
        "embedding_std": float(temporary_model.token_embedding.weight.detach().std(unbiased=False)),
        "weight_tying": True,
        "initial_state_sha256": rt_state_sha(temporary_model),
    }
    del temporary_model
    gc.collect()

print("Preflight: PASS")
print("Git:", RT_COMMIT)
print("GPU:", RT_GPU.name)
print("Tokenizer:", RT_TOKENIZER_SHA)
print("Dataset:", RT_DATA_FP)
print("Factory: moe_erc.model_factory.create_model")
print(json.dumps(RT_MODEL_INFO, indent=2))
print("Dữ liệu và pilot giữ nguyên; không sử dụng model objects cũ.")

Working tree: CLEAN
Preflight: PASS
Git: 665cb7b1cfbca4c6ed3507345b21008321129a2a
GPU: NVIDIA A100-SXM4-40GB
Tokenizer: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
Dataset: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
Factory: moe_erc.model_factory.create_model
{
  "vanilla_moe": {
    "parameters": 52288000,
    "embedding_std": 0.020000705495476723,
    "weight_tying": true,
    "initial_state_sha256": "5f64c49c3ccbf7988b8fb6c86637e72a9dc4c44fcddd90e9714ca8fab12735a7"
  },
  "aoe": {
    "parameters": 52263424,
    "embedding_std": 0.0200003981590271,
    "weight_tying": true,
    "initial_state_sha256": "3fddb80523354b680b9488cbcbee0ed2569bcc6aa92e3ad9ecf44158138a689e"
  }
}
Dữ liệu và pilot giữ nguyên; không sử dụng model objects cũ.


In [ ]:
RT_LB_POLICY = {
    "formula": "mean_layers(E * sum(f_e * mean_probability_e))",
    "assignment_fraction": "count_e / (N * K)",
    "accumulation": "effective-target-weighted microbatch mean",
    "coefficient": 0.01,
    "routing_weights": "full-softmax Top-K probabilities, not renormalized",
    "aoe": "actual gate-down outputs -> FP32 L2 norm -> softmax",
    "erc": False,
}

class RTLossTap:
    def __init__(self, model, kind):
        self.model, self.kind = model, kind
        self.handles = []
        self.probabilities = {}
        self.scores = {i: {} for i in range(len(model.layers))}

    def __enter__(self):
        try:
            for li, layer in enumerate(self.model.layers):
                if self.kind == "vanilla_moe":
                    def capture(module, args, output, index=li):
                        self.probabilities[index] = output.probabilities
                    self.handles.append(layer.moe.router.register_forward_hook(capture))
                else:
                    for ei, expert in enumerate(layer.moe.experts):
                        def capture(module, args, output, layer_index=li, expert_index=ei):
                            with torch.amp.autocast("cuda", enabled=False):
                                self.scores[layer_index][expert_index] = (
                                    torch.linalg.vector_norm(output.float(), ord=2, dim=-1)
                                )
                            # Không return output mới: không thay đổi routing.
                        self.handles.append(expert.gate_down_proj.register_forward_hook(capture))
        except BaseException:
            self.__exit__(None, None, None)
            raise
        return self

    def raw_loss(self, routing_info, verify=False):
        losses = []
        experts = self.model.config.n_experts
        for li, layer_info in enumerate(routing_info):
            diagnostic = layer_info.diagnostics
            if self.kind == "vanilla_moe":
                probabilities = self.probabilities[li].reshape(-1, experts)
            else:
                rt_require(len(self.scores[li]) == experts, "AoE capture thiếu experts.")
                scores = torch.stack([self.scores[li][e] for e in range(experts)], dim=-1)
                with torch.amp.autocast("cuda", enabled=False):
                    probabilities = scores.float().softmax(-1)
                if verify:
                    torch.testing.assert_close(
                        probabilities.detach(),
                        diagnostic.probabilities.reshape_as(probabilities),
                        rtol=3e-6, atol=3e-7,
                    )
            indices = diagnostic.topk_indices.reshape(-1, self.model.config.top_k)
            rt_require(indices.shape[0] == probabilities.shape[0], "Routing shape sai.")
            fractions = (
                torch.bincount(indices.reshape(-1), minlength=experts).float()
                / indices.numel()
            )
            mean_probability = probabilities.mean(0)
            if verify:
                torch.testing.assert_close(
                    mean_probability.detach(), diagnostic.mean_probabilities,
                    rtol=3e-6, atol=3e-7,
                )
            losses.append(experts * (fractions * mean_probability).sum())
        return torch.stack(losses).mean()

    def __exit__(self, exc_type, exc, tb):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()
        self.probabilities.clear()
        self.scores.clear()

def rt_compute_loss(model, kind, batch, bf16=True, verify_routing=False):
    with RTLossTap(model, kind) as tap:
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=bf16):
            output = model(**batch, return_routing_info=True)
        raw = tap.raw_loss(output.routing_info, verify=verify_routing)
    weighted = RT_CFG.lb_weight * raw
    total = output.loss + weighted
    for name, value in (("ce", output.loss), ("lb_raw", raw), ("total", total)):
        rt_require(value.ndim == 0 and bool(torch.isfinite(value)), f"Nonfinite {name}.")
    if torch.is_grad_enabled():
        rt_require(raw.requires_grad, "LB đã bị detach.")
    return {
        "output": output, "ce": output.loss,
        "lb_raw": raw, "lb_weighted": weighted, "total": total,
    }

def rt_initial_diagnostics(kind):
    model = None
    try:
        model = rt_new_cpu(kind)
        initial_sha = rt_state_sha(model)
        rt_require(initial_sha == RT_MODEL_INFO[kind]["initial_state_sha256"], "Initialization không tái lập.")
        model = model.to(RT_DEVICE).eval()
        batch = {name: value.to(RT_DEVICE) for name, value in RT_FIRST_BATCH.items()}
        measurements = {}

        for bf16 in (False, True):
            torch.cuda.reset_peak_memory_stats(0)
            with torch.no_grad():
                losses = rt_compute_loss(
                    model, kind, batch, bf16=bf16, verify_routing=True
                )
                output = losses["output"]
                rt_require(output.logits.shape == (2, 512, 16000), "Logits shape sai.")
                rt_require(
                    output.logits.dtype == (torch.bfloat16 if bf16 else torch.float32),
                    "Autocast dtype sai.",
                )
                logits = output.logits.float()
                rt_require(bool(torch.isfinite(logits).all()), "Logits NaN/Inf.")
                reference = F.cross_entropy(
                    logits[:, :-1].reshape(-1, 16000),
                    batch["labels"][:, 1:].reshape(-1),
                )
                torch.testing.assert_close(losses["ce"], reference, rtol=1e-5, atol=1e-4)
                torch.testing.assert_close(
                    losses["total"], losses["ce"] + losses["lb_weighted"]
                )
                ce = float(reference)
                rt_require(
                    0 < ce < RT_CFG.max_initial_ce,
                    f"{kind}: initial CE={ce:.4f}; vượt catastrophic-scale guard "
                    f"{RT_CFG.max_initial_ce}. Dừng trước training.",
                )

                measurements["bf16" if bf16 else "fp32"] = {
                    "ce": ce,
                    "independent_shifted_ce": float(reference),
                    "lb_raw": float(losses["lb_raw"]),
                    "lb_weighted": float(losses["lb_weighted"]),
                    "total_loss": float(losses["total"]),
                    "logits_dtype": str(output.logits.dtype),
                    "logits_mean": float(logits.mean()),
                    "logits_std": float(logits.std(unbiased=False)),
                    "logits_min": float(logits.min()),
                    "logits_max": float(logits.max()),
                    "current_token_top1_fraction": float(
                        (logits[:, :-1].argmax(-1) == batch["input_ids"][:, :-1]).float().mean()
                    ),
                    "current_equals_next_fraction": float(
                        (batch["input_ids"][:, :-1] == batch["labels"][:, 1:]).float().mean()
                    ),
                    "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
                    "causal_shift": "PASS",
                    "finite_checks": "PASS",
                }
                del losses, output, logits, reference

        return {
            "status": "PASS",
            "seed": RT_CFG.seed,
            "parameters": RT_MODEL_INFO[kind]["parameters"],
            "initial_state_sha256": initial_sha,
            "measurements": measurements,
            "fp32_bf16_ce_absolute_difference": abs(
                measurements["fp32"]["ce"] - measurements["bf16"]["ce"]
            ),
            "optimizer_steps": 0,
        }
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

for kind in RT_KINDS:
    try:
        RT_DIAGNOSTICS[kind] = rt_initial_diagnostics(kind)
        print(kind, json.dumps(RT_DIAGNOSTICS[kind], indent=2))
    except Exception as exc:
        RT_DIAGNOSTICS[kind] = {
            "status": "FAIL", "error": f"{type(exc).__name__}: {exc}"
        }
        traceback.print_exc()
        print("Dừng diagnostics; không cho phép training.")
        break

RT_DIAGNOSTICS_PASS = all(
    RT_DIAGNOSTICS.get(kind, {}).get("status") == "PASS" for kind in RT_KINDS
)
print("Uniform CE reference:", math.log(16000))
print("Initial diagnostics:", "PASS" if RT_DIAGNOSTICS_PASS else "FAIL")
print(
    "CE guard=50 là chặn scale bất thường, không phải quality target "
    "và không bắt buộc CE bằng ln(vocab). Không optimizer step."
)

vanilla_moe {
  "status": "PASS",
  "seed": 42,
  "parameters": 52288000,
  "initial_state_sha256": "5f64c49c3ccbf7988b8fb6c86637e72a9dc4c44fcddd90e9714ca8fab12735a7",
  "measurements": {
    "fp32": {
      "ce": 9.851984977722168,
      "independent_shifted_ce": 9.851984977722168,
      "lb_raw": 1.5391662120819092,
      "lb_weighted": 0.015391661785542965,
      "total_loss": 9.867376327514648,
      "logits_dtype": "torch.float32",
      "logits_mean": -0.005423651076853275,
      "logits_std": 0.45324191451072693,
      "logits_min": -2.1167056560516357,
      "logits_max": 1.8905314207077026,
      "current_token_top1_fraction": 0.001956947147846222,
      "current_equals_next_fraction": 0.003913894295692444,
      "peak_allocated_gib": 0.3887653350830078,
      "causal_shift": "PASS",
      "finite_checks": "PASS"
    },
    "bf16": {
      "ce": 9.851974487304688,
      "independent_shifted_ce": 9.851974487304688,
      "lb_raw": 1.5392942428588867,
      "lb_weighted": 0.0153

In [ ]:
RUN_TRAIN_SMOKE = True

def rt_optimizer_scheduler(model):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=RT_CFG.peak_lr,
        betas=(RT_CFG.beta1, RT_CFG.beta2),
        weight_decay=RT_CFG.weight_decay,
        foreach=False, fused=False,
    )

    def multiplier(index):
        update = index + 1
        if update <= RT_CFG.warmup_steps:
            return update / RT_CFG.warmup_steps
        progress = min(
            1.0,
            (update - RT_CFG.warmup_steps)
            / max(1, RT_CFG.optimizer_steps - RT_CFG.warmup_steps),
        )
        ratio = RT_CFG.min_lr / RT_CFG.peak_lr
        return ratio + (1 - ratio) * 0.5 * (1 + math.cos(math.pi * progress))

    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)

def rt_check_gradients(model, kind, active_experts):
    present = 0
    for name, parameter in model.named_parameters():
        if parameter.grad is not None:
            rt_require(bool(torch.isfinite(parameter.grad).all()), f"Gradient NaN/Inf: {name}")
            present += 1
    rt_require(present > 0, "Không có gradients.")

    def check(parameter, name):
        rt_require(parameter.grad is not None, f"Thiếu gradient: {name}")

    check(model.token_embedding.weight, "embedding")
    check(model.final_norm.weight, "final_norm")
    for li, layer in enumerate(model.layers):
        for name, parameter in layer.attention.named_parameters():
            check(parameter, f"layer{li}.attention.{name}")
        if kind == "vanilla_moe":
            check(layer.moe.router.weight, f"layer{li}.router")
        else:
            for ei, expert in enumerate(layer.moe.experts):
                check(expert.gate_down_proj.weight, f"layer{li}.expert{ei}.gate_down")
        for ei in active_experts[li]:
            for name, parameter in layer.moe.experts[ei].named_parameters():
                check(parameter, f"layer{li}.expert{ei}.{name}")
    return present

def rt_train_step(model, kind, optimizer, scheduler, batches, step):
    rt_require(1 <= step <= RT_CFG.optimizer_steps <= 3, "Hard step limit exceeded.")
    rt_require(len(batches) == RT_CFG.accumulation_steps, "Accumulation count sai.")
    targets = [int((batch["labels"][:, 1:] != -100).sum()) for batch in batches]
    rt_require(all(n > 0 for n in targets), "Microbatch không có targets.")
    total_targets = sum(targets)
    positions = sum(batch["input_ids"].numel() for batch in batches)
    totals = {name: 0.0 for name in ("ce", "lb_raw", "lb_weighted", "total")}
    active = [set() for _ in model.layers]

    model.train()
    optimizer.zero_grad(set_to_none=True)
    witness = model.final_norm.weight.detach().clone()
    used_lr = optimizer.param_groups[0]["lr"]
    previous_scheduler_step = scheduler.last_epoch
    torch.cuda.synchronize(0)
    torch.cuda.reset_peak_memory_stats(0)
    start = time.perf_counter()

    for batch_cpu, n_targets in zip(batches, targets):
        batch = {name: value.to(RT_DEVICE) for name, value in batch_cpu.items()}
        losses = rt_compute_loss(model, kind, batch, bf16=True)
        for li, routing in enumerate(losses["output"].routing_info):
            active[li].update(
                torch.where(routing.diagnostics.token_counts > 0)[0].tolist()
            )
        weight = n_targets / total_targets
        (losses["total"] * weight).backward()
        for name in totals:
            totals[name] += float(losses[name].detach()) * weight
        del losses, batch

    gradient_tensors = rt_check_gradients(model, kind, active)
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), RT_CFG.clip_norm, error_if_nonfinite=True
    )
    rt_require(bool(torch.isfinite(gradient_norm)), "Nonfinite gradient norm.")
    optimizer.step()
    scheduler.step()
    rt_require(scheduler.last_epoch == previous_scheduler_step + 1, "Scheduler step count sai.")
    updated = not torch.equal(witness, model.final_norm.weight.detach())
    rt_require(updated, "Optimizer không cập nhật witness parameter.")
    for state in optimizer.state.values():
        for name in ("exp_avg", "exp_avg_sq"):
            if name in state:
                rt_require(state[name].dtype == torch.float32, "Adam moments không FP32.")
    rt_require(all(p.dtype == torch.float32 for p in model.parameters()), "Parameters không FP32.")

    torch.cuda.synchronize(0)
    elapsed = time.perf_counter() - start
    return {
        "step": step,
        "ce": totals["ce"],
        "lb_raw": totals["lb_raw"],
        "lb_weighted": totals["lb_weighted"],
        "total_loss": totals["total"],
        "gradient_norm_before_clip": float(gradient_norm),
        "gradient_tensors_present": gradient_tensors,
        "lr_used": used_lr,
        "lr_next": optimizer.param_groups[0]["lr"],
        "optimizer_parameter_updated": updated,
        "gradient_checks": "PASS",
        "finite_checks": "PASS",
        "input_token_positions": positions,
        "effective_next_token_targets": total_targets,
        "elapsed_seconds": elapsed,
        "input_tokens_per_second": positions / elapsed,
        "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
        "throughput_scope": "training optimizer step, excluding validation/setup",
    }

def rt_evaluate(model):
    training_flags = [(module, module.training) for module in model.modules()]
    rng_state = (
        random.getstate(), np.random.get_state(),
        torch.get_rng_state(), torch.cuda.get_rng_state_all(),
    )
    count = min(
        len(RT_VAL_DATA),
        RT_CFG.validation_batch_size * RT_CFG.validation_batches,
    )
    loader = rt_loader(RT_VAL_DATA, range(count), RT_CFG.validation_batch_size)
    nll_sum = target_count = positions = 0
    try:
        model.eval()
        with torch.no_grad():
            for batch_cpu in loader:
                batch = {name: value.to(RT_DEVICE) for name, value in batch_cpu.items()}
                n = int((batch["labels"][:, 1:] != -100).sum())
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    output = model(**batch)  # CE-only evaluation, không LB/PPL từ train loss.
                rt_require(bool(torch.isfinite(output.loss)), "Validation CE nonfinite.")
                nll_sum += float(output.loss) * n
                target_count += n
                positions += batch["input_ids"].numel()
                del output, batch
        rt_require(target_count > 0, "Validation không có targets.")
        ce = nll_sum / target_count
        overflow = ce > math.log(sys.float_info.max)
        result = {
            "ce": ce,
            "perplexity": None if overflow else math.exp(ce),
            "perplexity_overflow": overflow,
            "log_perplexity": ce,
            "sequence_indices": list(range(count)),
            "unique_sequences": count,
            "input_token_positions": positions,
            "effective_next_token_targets": target_count,
            "scope": "fixed smoke subset, not full validation",
        }
    finally:
        for module, flag in training_flags:
            module.training = flag
        random.setstate(rng_state[0])
        np.random.set_state(rng_state[1])
        torch.set_rng_state(rng_state[2])
        torch.cuda.set_rng_state_all(rng_state[3])
    rt_require(
        all(module.training == flag for module, flag in training_flags),
        "Evaluation không restore train modes.",
    )
    return result

def rt_run_smoke(kind):
    model = optimizer = scheduler = None
    result = RT_TRAIN_RESULTS[kind]
    needed = RT_CFG.optimizer_steps * RT_CFG.accumulation_steps * RT_CFG.micro_batch_size
    selected = RT_INDICES[:needed].astype(np.int64).tolist()
    rt_require(len(selected) == needed and len(set(selected)) == needed, "Smoke prefix sai.")
    result.update(status="RUNNING", pilot_sequence_indices=selected, steps=[])

    try:
        model = rt_new_cpu(kind)
        initial_sha = rt_state_sha(model)
        rt_require(
            initial_sha == RT_DIAGNOSTICS[kind]["initial_state_sha256"],
            "Training không bắt đầu từ cùng initial state đã diagnostic.",
        )
        result["initial_state_sha256"] = initial_sha
        model = model.to(RT_DEVICE)
        optimizer, scheduler = rt_optimizer_scheduler(model)
        iterator = iter(rt_loader(RT_TRAIN_DATA, selected, RT_CFG.micro_batch_size))
        consumed_positions = consumed_targets = 0

        for step in range(1, RT_CFG.optimizer_steps + 1):
            batches = [next(iterator) for _ in range(RT_CFG.accumulation_steps)]
            metrics = rt_train_step(model, kind, optimizer, scheduler, batches, step)
            result["steps"].append(metrics)
            consumed_positions += metrics["input_token_positions"]
            consumed_targets += metrics["effective_next_token_targets"]
            metrics["validation"] = rt_evaluate(model)
            metrics["cumulative_input_token_positions"] = consumed_positions
            metrics["cumulative_effective_next_token_targets"] = consumed_targets
            print(kind, json.dumps(metrics, ensure_ascii=False))

        rt_require(
            consumed_positions == needed * 512 and consumed_targets == needed * 511,
            "Consumed token accounting sai.",
        )
        result.update(
            status="PASS",
            consumed_train_sequences=needed,
            consumed_input_token_positions=consumed_positions,
            consumed_effective_next_token_targets=consumed_targets,
            validation_sequence_visits=sum(
                row["validation"]["unique_sequences"] for row in result["steps"]
            ),
        )
    finally:
        optimizer = scheduler = model = None
        gc.collect()
        torch.cuda.empty_cache()

if not RUN_TRAIN_SMOKE:
    print("RUN_TRAIN_SMOKE=False: không optimizer step.")
else:
    rt_require(RT_DIAGNOSTICS_PASS, "Cell 2 chưa PASS; training bị chặn.")
    RT_CFG.validate()
    rt_verify_immutable()
    for kind in RT_KINDS:
        try:
            rt_run_smoke(kind)
        except Exception as exc:
            RT_TRAIN_RESULTS[kind].update(
                status="FAIL", error=f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()
            print("TRAINING STOP: không chạy tiếp hoặc tăng ngân sách.")
            break

print("Training results:", {kind: RT_TRAIN_RESULTS[kind]["status"] for kind in RT_KINDS})
print("Schedule ba bước chỉ phục vụ smoke, không phải schedule pilot 50M.")
print("Không kết luận AoE/Vanilla tốt hơn từ các kết quả này.")

vanilla_moe {"step": 1, "ce": 9.76702070236206, "lb_raw": 1.5448904037475586, "lb_weighted": 0.015448903664946556, "total_loss": 9.782469749450684, "gradient_norm_before_clip": 3.925640344619751, "gradient_tensors_present": 281, "lr_used": 0.0004, "lr_next": 0.00022000000000000003, "optimizer_parameter_updated": true, "gradient_checks": "PASS", "finite_checks": "PASS", "input_token_positions": 2048, "effective_next_token_targets": 2044, "elapsed_seconds": 0.5127910020000854, "input_tokens_per_second": 3993.8298293300763, "peak_allocated_gib": 0.8464431762695312, "throughput_scope": "training optimizer step, excluding validation/setup", "validation": {"ce": 9.501686573028564, "perplexity": 13382.2779963509, "perplexity_overflow": false, "log_perplexity": 9.501686573028564, "sequence_indices": [0, 1, 2, 3], "unique_sequences": 4, "input_token_positions": 2048, "effective_next_token_targets": 2044, "scope": "fixed smoke subset, not full validation"}, "cumulative_input_token_positions": 20

In [ ]:
RUN_AUDIT_BACKUP = True

from datetime import datetime, timezone
import uuid

RT_UNCHANGED = False
RT_GUARD_ERROR = None
try:
    rt_verify_immutable()
    RT_UNCHANGED = True
except Exception as exc:
    RT_GUARD_ERROR = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()

RT_LOCAL_PASS = (
    RT_DIAGNOSTICS_PASS and RT_UNCHANGED
    and all(RT_TRAIN_RESULTS[kind]["status"] == "PASS" for kind in RT_KINDS)
)

RT_AUDIT = {
    "step": "6.5B-RETEST",
    "status": "PASS" if RT_LOCAL_PASS else "FAIL",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": RT_COMMIT,
    "gpu": RT_GPU.name,
    "dataset_fingerprint": RT_DATA_FP,
    "tokenizer_sha256": RT_TOKENIZER_SHA,
    "pilot_manifest_sha256": RT_PILOT_SHA,
    "pilot_index_sha256": RT_PILOT["index_file_sha256"],
    "config": asdict(RT_CFG),
    "load_balancing_policy": RT_LB_POLICY,
    "weight_decay_scope": "all unique trainable parameters, same as STEP 6.5B",
    "models": RT_MODEL_INFO,
    "initial_diagnostics": RT_DIAGNOSTICS,
    "training_retest": RT_TRAIN_RESULTS,
    "historical_before_fix": {
        "vanilla_training_ce_approx": 501,
        "aoe_training_ce_approx": 501,
        "source": "user-reported historical STEP 6.5B",
        "comparison_scope": (
            "initialization diagnosis, not a controlled performance benchmark; "
            "historical training-step CE and fresh initial-batch CE are different measurements"
        ),
    },
    "protected_files_unchanged": RT_UNCHANGED,
    "guard_error": RT_GUARD_ERROR,
    "source_sha256": RT_CODE_GUARD,
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "numpy": np.__version__,
    },
    "erc_implemented": False,
    "checkpoint_created": False,
    "limitations": [
        "At most three optimizer steps per architecture.",
        "Validation is a tiny fixed subset.",
        "No conclusion about comparative model quality.",
        "Three-step LR schedule is not an official pilot schedule.",
        "Seeds controlled; bitwise CUDA training reproducibility not established.",
    ],
}

RT_AUDIT_DIR = (
    RT_PROJECT / "data/metadata/training_smoke_retest"
    / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-" + uuid.uuid4().hex[:12]
    )
)
RT_AUDIT_DIR.mkdir(parents=True, exist_ok=False)
RT_AUDIT_PATH = RT_AUDIT_DIR / "audit.json"
with RT_AUDIT_PATH.open("x", encoding="utf-8") as stream:
    json.dump(RT_AUDIT, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
    stream.write("\n")
RT_AUDIT_SHA = rt_sha(RT_AUDIT_PATH)

RT_BACKUP_PATH = None
RT_BACKUP_STATUS = "NOT REQUESTED"
if RUN_AUDIT_BACKUP:
    RT_BACKUP_STATUS = "FAIL"
    try:
        rt_require(Path("/content/drive/MyDrive").is_dir(), "Drive chưa mount.")
        RT_BACKUP_PATH = (
            Path("/content/drive/MyDrive/MoE_ERC/training_smoke_retest_audits")
            / RT_AUDIT_DIR.name
        )
        RT_BACKUP_PATH.mkdir(parents=True, exist_ok=False)
        with RT_AUDIT_PATH.open("rb") as src, (RT_BACKUP_PATH / "audit.json").open("xb") as dst:
            shutil.copyfileobj(src, dst)
            dst.flush()
            os.fsync(dst.fileno())
        rt_require(rt_sha(RT_BACKUP_PATH / "audit.json") == RT_AUDIT_SHA, "Audit backup SHA FAIL.")
        marker = {
            "status": "COMPLETE",
            "audit_sha256": RT_AUDIT_SHA,
            "training_retest_status": RT_AUDIT["status"],
        }
        with (RT_BACKUP_PATH / "COMPLETE.json").open("x", encoding="utf-8") as stream:
            json.dump(marker, stream, indent=2)
        rt_require(rt_json(RT_BACKUP_PATH / "COMPLETE.json") == marker, "Marker readback FAIL.")
        RT_BACKUP_STATUS = "PASS"
    except Exception:
        traceback.print_exc()
        print("Audit backup FAIL; không coi bản copy là đã xác minh.")

def rt_initial_ce_pair(kind):
    measurements = RT_DIAGNOSTICS.get(kind, {}).get("measurements", {})
    return {
        precision: measurements.get(precision, {}).get("ce", "UNAVAILABLE")
        for precision in ("fp32", "bf16")
    }

def rt_validation_report(kind):
    rows = RT_TRAIN_RESULTS[kind]["steps"]
    return rows[-1].get("validation", "NOT RUN") if rows else "NOT RUN"

RT_FINAL_PASS = RT_LOCAL_PASS and (
    not RUN_AUDIT_BACKUP or RT_BACKUP_STATUS == "PASS"
)

print("\n=== STEP 6.5B-RETEST REPORT ===")
print("Git commit:", RT_COMMIT)
print("GPU:", RT_GPU.name)
print("Dataset fingerprint:", RT_DATA_FP)
print("Tokenizer SHA-256:", RT_TOKENIZER_SHA)
for kind in RT_KINDS:
    print(kind, "parameters:", RT_MODEL_INFO[kind]["parameters"])
    print(kind, "embedding std:", RT_MODEL_INFO[kind]["embedding_std"])
    print(kind, "weight tying:", RT_MODEL_INFO[kind]["weight_tying"])
    print(kind, "initial CE FP32/BF16:", rt_initial_ce_pair(kind))
    print(kind, "causal shift:",
          RT_DIAGNOSTICS.get(kind, {}).get("status", "NOT PASS"))
    print(kind, "training smoke:", RT_TRAIN_RESULTS[kind]["status"])
    print(kind, "validation:", rt_validation_report(kind))

print("Gradient checks:", "PASS" if RT_LOCAL_PASS else "NOT PASS/NOT RUN")
print("Optimizer update checks:", "PASS" if RT_LOCAL_PASS else "NOT PASS/NOT RUN")
print("NaN/Inf checks:", "PASS" if RT_LOCAL_PASS else "Chưa đủ kiểm tra để PASS")
print("Peak training GPU memory GiB:", {
    kind: max(
        [row["peak_allocated_gib"] for row in RT_TRAIN_RESULTS[kind]["steps"]],
        default=None,
    )
    for kind in RT_KINDS
})
print("Protected artifacts unchanged:", RT_UNCHANGED)
print("Before FIX historical training CE: Vanilla≈501; AoE≈501.")
print("Đây là so sánh chẩn đoán initialization, không phải benchmark hai training runs.")
print("Audit path:", RT_AUDIT_PATH)
print("Audit SHA-256:", RT_AUDIT_SHA)
print("Audit backup path:", RT_BACKUP_PATH)
print("Backup verification:", RT_BACKUP_STATUS)
print("STATUS:", "PASS" if RT_FINAL_PASS else "FAIL")
if not RUN_TRAIN_SMOKE:
    print("Lý do chưa PASS toàn bước: RUN_TRAIN_SMOKE đang False.")
print("Không checkpoints, không ERC, không full training, không chuyển STEP 6.5C.")


=== STEP 6.5B-RETEST REPORT ===
Git commit: 665cb7b1cfbca4c6ed3507345b21008321129a2a
GPU: NVIDIA A100-SXM4-40GB
Dataset fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
vanilla_moe parameters: 52288000
vanilla_moe embedding std: 0.020000705495476723
vanilla_moe weight tying: True
vanilla_moe initial CE FP32/BF16: {'fp32': 9.851984977722168, 'bf16': 9.851974487304688}
vanilla_moe causal shift: PASS
vanilla_moe training smoke: PASS
vanilla_moe validation: {'ce': 9.289157390594482, 'perplexity': 10820.063169443585, 'perplexity_overflow': False, 'log_perplexity': 9.289157390594482, 'sequence_indices': [0, 1, 2, 3], 'unique_sequences': 4, 'input_token_positions': 2048, 'effective_next_token_targets': 2044, 'scope': 'fixed smoke subset, not full validation'}
aoe parameters: 52263424
aoe embedding std: 0.0200003981590271
aoe weight tying: True
aoe initial CE FP32/BF16: {'fp32': 9.

# Step 6.5C


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import ast
import hashlib
import inspect
import json
import linecache
import os
import shutil
import subprocess
import sys
import textwrap
import uuid

import numpy as np
import torch

def c65_require(condition, message):
    if not condition:
        raise RuntimeError(message)

def c65_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def c65_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def c65_write(path, value):
    with Path(path).open("x", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.write("\n")

C65_HELPERS = [
    "rt_require", "rt_json", "rt_sha", "rt_git", "RTConfig",
    "rt_seed", "rt_state_sha", "rt_new_cpu", "rt_loader",
    "RTLossTap", "rt_compute_loss", "rt_optimizer_scheduler",
    "rt_check_gradients", "rt_train_step",
]
C65_REQUIRED = C65_HELPERS + [
    "RT_PROJECT", "RT_DATA", "RT_TOKENIZER_PATH", "RT_COMMIT",
    "RT_TOKENIZER_SHA", "RT_DATA_FP", "RT_CFG", "RT_CONFIGS",
    "RT_CODE_GUARD", "RT_ARTIFACT_GUARD", "RT_INDICES",
    "RT_FINAL_PASS", "rt_verify_immutable",
]
missing = [name for name in C65_REQUIRED if name not in globals()]
c65_require(not missing, f"Thiếu notebook helpers/variables: {missing}")
c65_require(RT_FINAL_PASS is True, "STEP 6.5B-RETEST chưa có RT_FINAL_PASS=True.")

C65_COMMIT = "665cb7b1cfbca4c6ed3507345b21008321129a2a"
C65_TOKENIZER_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
C65_DATA_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"
C65_KINDS = ("vanilla_moe", "aoe")
C65_COUNTS = {"vanilla_moe": 52_288_000, "aoe": 52_263_424}

c65_require(
    Path(RT_PROJECT) == Path("/content/moe-erc")
    and RT_COMMIT == C65_COMMIT
    and RT_TOKENIZER_SHA == C65_TOKENIZER_SHA
    and RT_DATA_FP == C65_DATA_FP,
    "Notebook identity khác thiết kế được phê duyệt.",
)
rt_verify_immutable()
c65_require(torch.cuda.is_available(), "CUDA unavailable.")
c65_require(torch.cuda.device_count() == 1, "Test yêu cầu đúng một GPU.")
C65_GPU = torch.cuda.get_device_name(0)
c65_require("A100" in C65_GPU.upper(), f"GPU thực tế: {C65_GPU}")
c65_require(torch.cuda.is_bf16_supported(), "BF16 unavailable.")

# Lấy đúng source của các definitions đang tồn tại trong kernel.
# Fallback chỉ đọc history/source cache, không execute các cell cũ.
def c65_source(name):
    obj = globals()[name]
    try:
        return textwrap.dedent(inspect.getsource(obj))
    except (OSError, TypeError):
        pass

    method = (
        obj.validate if name == "RTConfig"
        else obj.__init__ if name == "RTLossTap"
        else obj
    )
    filename = method.__code__.co_filename
    candidates = ["".join(linecache.getlines(filename))]
    ip = get_ipython()
    candidates.extend(reversed(ip.history_manager.input_hist_raw))
    for source in candidates:
        if not source:
            continue
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name == name:
                start = min(
                    [node.lineno] + [d.lineno for d in node.decorator_list]
                ) - 1
                result = "\n".join(source.splitlines()[start:node.end_lineno]) + "\n"
                # Với fallback, source phải chứa đúng definition đang thực thi.
                if filename in ("<unknown>", ""):
                    continue
                if source == "".join(linecache.getlines(filename)):
                    return result
                # IPython history entry được đối chiếu với code filename cache.
                try:
                    cached = "".join(linecache.getlines(filename))
                    if result.strip() in cached:
                        return result
                except Exception:
                    pass
    raise RuntimeError(
        f"Không lấy được source của {name}. "
        f"Chỉ cần cung cấp riêng definition này; không cần upload notebook."
    )

C65_DEFS = {name: c65_source(name) for name in C65_HELPERS}

# Chỉ mở rộng giới hạn smoke 3 -> 4 trong bản sao helpers của subprocess.
# Không sửa helper trong notebook hoặc source repository.
for name, old, new in (
    ("RTConfig",
     "1 <= self.optimizer_steps <= 3",
     "1 <= self.optimizer_steps <= 4"),
    ("rt_train_step",
     "1 <= step <= RT_CFG.optimizer_steps <= 3",
     "1 <= step <= RT_CFG.optimizer_steps <= 4"),
):
    c65_require(C65_DEFS[name].count(old) == 1,
                f"Hard limit của {name} khác source đã kiểm tra.")
    C65_DEFS[name] = C65_DEFS[name].replace(old, new, 1)

C65_TRAINING = asdict(RT_CFG)
c65_require(
    C65_TRAINING["seed"] == 42
    and C65_TRAINING["micro_batch_size"] == 2
    and C65_TRAINING["accumulation_steps"] == 2
    and C65_TRAINING["lb_weight"] == 0.01,
    "Training semantics khác STEP 6.5B-RETEST đã kiểm tra.",
)
C65_TRAINING["optimizer_steps"] = 4

pilot_path = Path(RT_DATA) / "pilot_50m_manifest.json"
pilot = c65_json(pilot_path)
pilot_sha = c65_sha(pilot_path)
c65_require(
    pilot_sha == (Path(RT_DATA) / "pilot_50m_manifest.sha256").read_text().strip()
    and pilot["combined_source_fingerprint"] == C65_DATA_FP
    and pilot["tokenizer_sha256"] == C65_TOKENIZER_SHA
    and pilot["selected_input_tokens"] == 50_000_384
    and pilot["split"] == "train",
    "Pilot manifest identity mismatch.",
)
index_name = pilot["index_file"]
c65_require(
    not Path(index_name).is_absolute()
    and ".." not in Path(index_name).parts
    and "\\" not in index_name,
    "Unsafe pilot index path.",
)
index_path = Path(RT_DATA) / index_name
c65_require(c65_sha(index_path) == pilot["index_file_sha256"], "Pilot index SHA mismatch.")
disk_indices = np.load(index_path, allow_pickle=False)
c65_require(
    len(disk_indices) == 97657
    and np.array_equal(disk_indices, RT_INDICES)
    and len(np.unique(disk_indices)) == len(disk_indices),
    "Pilot indices trong kernel khác dữ liệu đã lưu.",
)
C65_PREFIX = [int(x) for x in disk_indices[:16]]
c65_require(len(set(C65_PREFIX)) == 16, "Pilot prefix không hợp lệ.")

drive_root = Path("/content/drive/MyDrive")
c65_require(drive_root.is_dir(), "Google Drive chưa mounted.")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:12]
C65_RUN = Path(RT_PROJECT) / "data/metadata/step65c_final" / run_id
C65_DRIVE = drive_root / "MoE_ERC/checkpoint_acceptance" / run_id
C65_RUN.mkdir(parents=True, exist_ok=False)
C65_DRIVE.mkdir(parents=True, exist_ok=False)

for directory in (C65_RUN, C65_DRIVE):
    free = shutil.disk_usage(directory).free
    print(directory, "free GiB:", round(free / 1024**3, 2))
    c65_require(free >= 4 * 1024**3, f"Cần ít nhất 4 GiB dự phòng: {directory}")
c65_write(C65_DRIVE / "preflight.json", {"git": C65_COMMIT, "run_id": run_id})

C65_GUARDS = {**RT_CODE_GUARD, **RT_ARTIFACT_GUARD}
for p in (
    pilot_path, index_path, Path(RT_TOKENIZER_PATH),
    Path(RT_DATA) / "dataset_manifest.json",
):
    C65_GUARDS[str(p)] = c65_sha(p)

C65_SETUP = {
    "project": str(RT_PROJECT), "data": str(RT_DATA),
    "tokenizer": str(RT_TOKENIZER_PATH), "commit": C65_COMMIT,
    "training": C65_TRAINING, "counts": C65_COUNTS,
    "run": str(C65_RUN), "drive": str(C65_DRIVE), "guards": C65_GUARDS,
    "identities": {
        kind: {
            "schema": "step65c-v1", "git": C65_COMMIT, "model_type": kind,
            "architecture": RT_CONFIGS[kind].to_dict(),
            "parameters": C65_COUNTS[kind], "training": C65_TRAINING,
            "dataset_fp": C65_DATA_FP,
            "dataset_manifest_sha256": c65_sha(Path(RT_DATA) / "dataset_manifest.json"),
            "tokenizer_sha256": C65_TOKENIZER_SHA,
            "pilot_manifest_sha256": pilot_sha,
            "pilot_index_sha256": pilot["index_file_sha256"],
            "prefix": C65_PREFIX,
            "torch": str(torch.__version__), "cuda": str(torch.version.cuda),
            "gpu": C65_GPU, "gpu_count": 1,
            "mixed_precision": "bf16_autocast_fp32_parameters", "scaler": None,
        }
        for kind in C65_KINDS
    },
}
C65_RESULT = {"status": "NOT RUN", "models": {}, "blocking_error": None}
print("PREFLIGHT: PASS")
print("Helpers lấy từ kernel:", sorted(C65_DEFS))
print("Run:", C65_RUN)
print("Drive:", C65_DRIVE)
print("Không training; không chạy lại STEP 6.5A/6.5B.")
print("Drive free-space qua mount không phải quota API của tài khoản.")

/content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae free GiB: 191.13
/content/drive/MyDrive/MoE_ERC/checkpoint_acceptance/20261025T181533Z-c4158a07f9ae free GiB: 181.57
PREFLIGHT: PASS
Helpers lấy từ kernel: ['RTConfig', 'RTLossTap', 'rt_check_gradients', 'rt_compute_loss', 'rt_git', 'rt_json', 'rt_loader', 'rt_new_cpu', 'rt_optimizer_scheduler', 'rt_require', 'rt_seed', 'rt_sha', 'rt_state_sha', 'rt_train_step']
Run: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae
Drive: /content/drive/MyDrive/MoE_ERC/checkpoint_acceptance/20261025T181533Z-c4158a07f9ae
Không training; không chạy lại STEP 6.5A/6.5B.
Drive free-space qua mount không phải quota API của tài khoản.


In [ ]:
C65_WORKER = C65_RUN / "resume_worker.py"

C65_IMPORTS = r'''
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
import copy
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
import traceback
import uuid
import numpy as np
import torch
from moe_erc.config import ModelConfig, AoEConfig
from moe_erc.model_factory import create_model
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.dataset import PackedTokenDataset
'''

C65_UTILITIES = r'''
def write_json(path, value):
    with Path(path).open("x", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.write("\n")

def frozen(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: frozen(v) for k, v in x.items()}
    if isinstance(x, tuple):
        return tuple(frozen(v) for v in x)
    if isinstance(x, list):
        return [frozen(v) for v in x]
    return copy.deepcopy(x)

def exact(a, b, path="state"):
    if isinstance(a, torch.Tensor):
        rt_require(isinstance(b, torch.Tensor), f"{path}: tensor type")
        a, b = a.detach().cpu(), b.detach().cpu()
        rt_require(a.dtype == b.dtype and a.shape == b.shape, f"{path}: dtype/shape")
        aa = a.contiguous().reshape(-1).view(torch.uint8)
        bb = b.contiguous().reshape(-1).view(torch.uint8)
        rt_require(torch.equal(aa, bb), f"{path}: tensor content differs")
    elif isinstance(a, dict):
        rt_require(type(a) is type(b) and a.keys() == b.keys(), f"{path}: keys/type")
        for k in a:
            exact(a[k], b[k], f"{path}.{k}")
    elif isinstance(a, (tuple, list)):
        rt_require(type(a) is type(b) and len(a) == len(b), f"{path}: length/type")
        for i, (x, y) in enumerate(zip(a, b)):
            exact(x, y, f"{path}[{i}]")
    else:
        rt_require(type(a) is type(b) and a == b, f"{path}: {a!r} != {b!r}")

def tensor_addresses(x):
    if isinstance(x, torch.Tensor):
        return {(str(x.device), x.untyped_storage().data_ptr())} if x.numel() else set()
    if isinstance(x, dict):
        values = x.values()
    elif isinstance(x, (tuple, list)):
        values = x
    else:
        return set()
    result = set()
    for v in values:
        result.update(tensor_addresses(v))
    return result

def no_alias(a, b):
    rt_require(not tensor_addresses(a).intersection(tensor_addresses(b)),
               "Comparison reference aliasing.")

def digest_tree(x):
    h = hashlib.sha256()
    def visit(v):
        h.update(type(v).__name__.encode() + b"\0")
        if isinstance(v, torch.Tensor):
            t = v.detach().cpu().contiguous()
            h.update(str(t.dtype).encode())
            h.update(str(tuple(t.shape)).encode())
            h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
        elif isinstance(v, dict):
            for k in sorted(v, key=repr):
                visit(k)
                visit(v[k])
        elif isinstance(v, (list, tuple)):
            h.update(str(len(v)).encode() + b"\0")
            for item in v:
                visit(item)
        else:
            h.update(repr(v).encode() + b"\0")
    visit(x)
    return h.hexdigest()

def rng_get():
    n = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": (n[0], n[1].tolist(), n[2], n[3], n[4]),
        "cpu": torch.get_rng_state().clone(),
        "cuda": [t.clone() for t in torch.cuda.get_rng_state_all()],
    }

def rng_set(r):
    random.setstate(r["python"])
    n = r["numpy"]
    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(r["cpu"])
    rt_require(len(r["cuda"]) == torch.cuda.device_count(), "CUDA RNG count.")
    torch.cuda.set_rng_state_all(r["cuda"])

def names_by_group(model, optimizer):
    names = {id(p): n for n, p in model.named_parameters()}
    groups = [[names[id(p)] for p in g["params"]] for g in optimizer.param_groups]
    flat = [n for group in groups for n in group]
    rt_require(len(flat) == len(set(flat)), "Repeated optimizer parameter.")
    return groups

def check_tying(model):
    a, b = model.token_embedding.weight, model.lm_head.weight
    rt_require(a is b and a.data_ptr() == b.data_ptr(), "Weight tying lost.")
    rt_require(sum(p.numel() for p in model.parameters()) == ID["parameters"],
               "Parameter count mismatch.")

def check_progress(p, sampler):
    step = p["step"]
    rt_require(0 <= step <= 4, "Acceptance step budget exceeded.")
    rt_require(sampler["cursor"] == step * 4, "Sampler cursor mismatch.")
    exact(ID["prefix"][:step * 4], sampler["trace"], "sampler.trace")
    rt_require(
        sampler["epoch"] == 0 and sampler["shuffle"] is False
        and sampler["pilot_index_sha256"] == ID["pilot_index_sha256"],
        "Sampler identity mismatch.",
    )
    rt_require(p["input_tokens"] == step * 2048, "Input token accounting.")
    rt_require(p["targets"] == step * 2044, "Target token accounting.")
    rt_require(len(p["rows"]) == step, "Trajectory length.")
    for i, row in enumerate(p["rows"], 1):
        rt_require(row["step"] == i, "Trajectory step number.")
        exact(ID["prefix"][(i-1)*4:i*4], row["indices"], "trajectory.indices")

def snapshot(model, optimizer, scheduler, progress, sampler):
    torch.cuda.synchronize()
    state = {
        "model": frozen(model.state_dict()),
        "buffers": frozen(dict(model.named_buffers())),
        "optimizer": frozen(optimizer.state_dict()),
        "parameter_names": names_by_group(model, optimizer),
        "scheduler": frozen(scheduler.state_dict()),
        "rng": frozen(rng_get()),
        "progress": copy.deepcopy(progress),
        "sampler": copy.deepcopy(sampler),
        "modes": {n: m.training for n, m in model.named_modules()},
        "scaler": None,
    }
    no_alias(model.state_dict(), state["model"])
    no_alias(optimizer.state_dict(), state["optimizer"])
    return state

STATE_FIELDS = {
    "model", "buffers", "optimizer", "parameter_names", "scheduler",
    "rng", "progress", "sampler", "modes", "scaler",
}

def validate_payload(payload, identity):
    rt_require(isinstance(payload, dict)
               and set(payload) == {"schema", "identity", "state"}, "PAYLOAD_FIELDS")
    rt_require(payload["schema"] == "step65c-v1", "PAYLOAD_SCHEMA")
    rt_require(payload["identity"] == identity, "IDENTITY_MISMATCH")
    rt_require(isinstance(payload["state"], dict)
               and set(payload["state"]) == STATE_FIELDS, "STATE_FIELDS")
    rt_require(payload["state"]["scaler"] is None, "Unexpected scaler.")

def file_check(path, expected_sha):
    p = Path(path)
    rt_require(p.is_file() and not p.is_symlink(), f"Missing/unsafe file: {p}")
    rt_require(rt_sha(p) == expected_sha, "CHECKSUM_MISMATCH")

def verify(root, identity, manifest_sha, unpublished=False):
    root = Path(root)
    rt_require(root.is_dir() and not root.is_symlink(), "Unsafe checkpoint directory.")
    marker = "INCOMPLETE.json" if unpublished else "COMPLETE.json"
    required = {"checkpoint.pt", "manifest.json", "manifest.sha256", marker}
    rt_require({p.name for p in root.iterdir()} == required, "CHECKPOINT_INVENTORY")
    rt_require(all(p.is_file() and not p.is_symlink() for p in root.iterdir()),
               "Unsafe checkpoint entry.")
    file_check(root / "manifest.json", manifest_sha)
    rt_require((root / "manifest.sha256").read_text().strip() == manifest_sha,
               "MANIFEST_SIDECAR")
    m = rt_json(root / "manifest.json")
    rt_require(set(m) == {"schema", "identity", "payload", "state_digest"}
               and m["schema"] == "step65c-v1", "MANIFEST_FIELDS")
    rt_require(m["identity"] == identity, "IDENTITY_MISMATCH")
    e = m["payload"]
    rt_require(set(e) == {"file", "bytes", "sha256"}
               and e["file"] == "checkpoint.pt", "PAYLOAD_ENTRY")
    rt_require((root / "checkpoint.pt").stat().st_size == e["bytes"], "PAYLOAD_SIZE")
    file_check(root / "checkpoint.pt", e["sha256"])
    if not unpublished:
        rt_require(rt_json(root / marker) == {
            "status": "COMPLETE", "manifest_sha256": manifest_sha,
        }, "COMPLETE_MARKER")
    return m

def save_checkpoint(root, model, optimizer, scheduler, progress, sampler):
    root = Path(root)
    rt_require(not root.exists(), "Refusing checkpoint overwrite.")
    check_progress(progress, sampler)
    check_tying(model)
    rt_require(progress["step"] == scheduler.last_epoch == 2, "Save boundary step.")
    rt_require(all(p.grad is None for p in model.parameters()), "Gradient boundary.")

    state = snapshot(model, optimizer, scheduler, progress, sampler)
    state_digest = digest_tree(state)
    payload = {"schema": "step65c-v1", "identity": ID, "state": state}
    validate_payload(payload, ID)
    stage = root.with_name(root.name + ".partial-" + uuid.uuid4().hex)
    stage.mkdir(exist_ok=False)
    write_json(stage / "INCOMPLETE.json", {"status": "INCOMPLETE"})
    with (stage / "checkpoint.pt.part").open("xb") as f:
        torch.save(payload, f)
        f.flush()
        os.fsync(f.fileno())
    (stage / "checkpoint.pt.part").rename(stage / "checkpoint.pt")

    manifest = {
        "schema": "step65c-v1", "identity": ID, "state_digest": state_digest,
        "payload": {
            "file": "checkpoint.pt",
            "bytes": (stage / "checkpoint.pt").stat().st_size,
            "sha256": rt_sha(stage / "checkpoint.pt"),
        },
    }
    write_json(stage / "manifest.json", manifest)
    ms = rt_sha(stage / "manifest.json")
    with (stage / "manifest.sha256").open("x") as f:
        f.write(ms + "\n")
    verify(stage, ID, ms, unpublished=True)

    # So sánh nội dung sau serialization với reference độc lập trước save.
    loaded = torch.load(stage / "checkpoint.pt", map_location="cpu", weights_only=True)
    validate_payload(loaded, ID)
    exact(state, loaded["state"], "serialization")
    rt_require(digest_tree(loaded["state"]) == state_digest, "Serialization content digest.")
    del loaded
    write_json(stage / "COMPLETE.json", {"status": "COMPLETE", "manifest_sha256": ms})
    (stage / "INCOMPLETE.json").unlink()  # Chỉ marker mới tạo trong run này.
    rt_require(not root.exists(), "Refusing checkpoint overwrite.")
    stage.rename(root)
    verify(root, ID, ms)
    exact(state["rng"], rng_get(), "save_rng_side_effect")
    return {
        "status": "PASS", "manifest_sha256": ms,
        "payload_sha256": manifest["payload"]["sha256"],
        "bytes": manifest["payload"]["bytes"], "state_digest": state_digest,
    }

def negative_tests(root, ms, manifest, payload):
    result = {}
    def reject(name, fn, message):
        try:
            fn()
        except RuntimeError as exc:
            rt_require(message in str(exc), f"{name}: unrelated rejection: {exc}")
            result[name] = "PASS"
        else:
            raise RuntimeError(f"Invalid checkpoint accepted: {name}")

    for name, key, value in (
        ("dataset", "dataset_fp", "0"*64),
        ("tokenizer", "tokenizer_sha256", "0"*64),
        ("git", "git", "0"*40),
        ("config", "architecture", {"invalid": True}),
    ):
        wrong = copy.deepcopy(ID)
        wrong[key] = value
        reject(name, lambda wrong=wrong: verify(root, wrong, ms), "IDENTITY_MISMATCH")

    wrong_ms = ("0" if ms[0] != "0" else "1") + ms[1:]
    reject("manifest_checksum", lambda: verify(root, ID, wrong_ms), "CHECKSUM_MISMATCH")
    ps = manifest["payload"]["sha256"]
    wrong_ps = ("0" if ps[0] != "0" else "1") + ps[1:]
    reject("payload_checksum",
           lambda: file_check(Path(root) / "checkpoint.pt", wrong_ps),
           "CHECKSUM_MISMATCH")

    missing = dict(payload)
    missing.pop("state")
    reject("required_field", lambda: validate_payload(missing, ID), "PAYLOAD_FIELDS")
    missing_rng = dict(payload)
    missing_rng["state"] = dict(payload["state"])
    missing_rng["state"].pop("rng")
    reject("required_rng", lambda: validate_payload(missing_rng, ID), "STATE_FIELDS")
    return result

def restore_checkpoint(root, ms, model, optimizer, scheduler):
    manifest = verify(root, ID, ms)
    payload = torch.load(Path(root) / "checkpoint.pt",
                         map_location="cpu", weights_only=True)
    validate_payload(payload, ID)
    rt_require(digest_tree(payload["state"]) == manifest["state_digest"],
               "Saved content differs from pre-save state.")

    # Clone trước load, tránh optimizer.load_state_dict giữ reference tới payload.
    reference = frozen(payload["state"])
    no_alias(reference, payload["state"])
    negative = negative_tests(root, ms, manifest, payload)
    exact(reference["parameter_names"], names_by_group(model, optimizer),
          "optimizer.parameter_mapping")

    model.load_state_dict(payload["state"]["model"], strict=True, assign=False)
    buffers = dict(model.named_buffers())
    rt_require(buffers.keys() == reference["buffers"].keys(), "Buffer keys.")
    with torch.no_grad():
        for name, value in reference["buffers"].items():
            buffers[name].copy_(value)
    optimizer.load_state_dict(payload["state"]["optimizer"])

    # AdamW không capturable/fused: moments trên parameter device; step trên CPU.
    for group in optimizer.param_groups:
        rt_require(not group.get("capturable", False)
                   and not group.get("fused", False), "Unexpected AdamW mode.")
        for parameter in group["params"]:
            for key, value in list(optimizer.state.get(parameter, {}).items()):
                if not isinstance(value, torch.Tensor):
                    continue
                if key == "step":
                    optimizer.state[parameter][key] = value.to("cpu")
                else:
                    optimizer.state[parameter][key] = value.to(parameter.device)

    scheduler.load_state_dict(payload["state"]["scheduler"])
    modules = dict(model.named_modules())
    rt_require(modules.keys() == reference["modes"].keys(), "Module mode keys.")
    for name, flag in reference["modes"].items():
        modules[name].training = flag
    progress = copy.deepcopy(reference["progress"])
    sampler = copy.deepcopy(reference["sampler"])
    check_progress(progress, sampler)
    check_tying(model)
    rng_set(reference["rng"])  # Sau initialization và toàn bộ load operations.

    actual = snapshot(model, optimizer, scheduler, progress, sampler)
    no_alias(reference, actual)
    equality = {}
    for key in sorted(STATE_FIELDS):
        exact(reference[key], actual[key], f"restore.{key}")
        equality[key] = "PASS"
    rt_require(digest_tree(actual) == manifest["state_digest"],
               "Immediate restored content differs.")
    rt_require(scheduler.last_epoch == progress["step"], "Scheduler restore step.")
    exact(optimizer.param_groups[0]["lr"], scheduler.get_last_lr()[0], "restored.lr")
    del payload, reference, actual
    return progress, sampler, equality, negative

def train_until(model, optimizer, scheduler, progress, sampler, stop):
    dataset = PackedTokenDataset(
        S["data"], "train", tokenizer=FrozenTokenizer(S["tokenizer"]),
    )
    try:
        while progress["step"] < stop:
            check_progress(progress, sampler)
            step, cursor = progress["step"] + 1, sampler["cursor"]
            indices = ID["prefix"][cursor:cursor+4]
            rt_require(len(indices) == 4 and step <= 4, "Step/data budget.")
            batches = list(rt_loader(dataset, indices, RT_CFG.micro_batch_size))
            rt_require(len(batches) == 2, "Microbatch count.")
            for micro, batch in enumerate(batches):
                for j in range(2):
                    record = dataset[indices[micro*2+j]]
                    exact(record["input_ids"], batch["input_ids"][j], "batch.input_ids")
                    exact(record["labels"], batch["labels"][j], "batch.labels")
                rt_require(torch.equal(batch["input_ids"], batch["labels"]),
                           "Labels must remain unshifted.")

            rt_require(scheduler.last_epoch == progress["step"], "Scheduler before update.")
            expected_lr = RT_CFG.peak_lr * scheduler.lr_lambdas[0](progress["step"])
            exact(expected_lr, optimizer.param_groups[0]["lr"], "lr_schedule")

            # Training/loss/gradient logic được lấy từ notebook đã kiểm thử.
            row = rt_train_step(model, KIND, optimizer, scheduler, batches, step)
            optimizer.zero_grad(set_to_none=True)
            rt_require(row["optimizer_parameter_updated"], "Missing optimizer update.")
            for key in ("ce", "lb_raw", "lb_weighted", "total_loss",
                        "gradient_norm_before_clip"):
                rt_require(math.isfinite(row[key]), f"Nonfinite metric: {key}")
            for name, parameter in model.named_parameters():
                rt_require(bool(torch.isfinite(parameter).all()), f"Nonfinite parameter: {name}")
            for state in optimizer.state.values():
                for key, value in state.items():
                    if isinstance(value, torch.Tensor):
                        rt_require(bool(torch.isfinite(value).all()), f"Nonfinite Adam {key}")

            row["indices"] = indices
            progress["rows"].append(row)
            progress["step"] = step
            progress["input_tokens"] += row["input_token_positions"]
            progress["targets"] += row["effective_next_token_targets"]
            sampler["cursor"] += 4
            sampler["trace"].extend(indices)
            check_progress(progress, sampler)
            check_tying(model)
            print(json.dumps({"model": KIND, "phase": PHASE, **row}), flush=True)
    finally:
        dataset.close()

def context_check():
    rt_require(rt_git("rev-parse", "HEAD").decode().strip() == S["commit"], "Git mismatch.")
    for path, expected in S["guards"].items():
        file_check(path, expected)
    file_check(__file__, ID["worker_sha256"])
    rt_require(
        str(torch.__version__) == ID["torch"]
        and str(torch.version.cuda) == ID["cuda"]
        and torch.cuda.device_count() == ID["gpu_count"]
        and torch.cuda.get_device_name(0) == ID["gpu"],
        "Runtime identity mismatch.",
    )

def main():
    global S, ID, KIND, PHASE
    global RT_PROJECT, RT_CFG, RT_CONFIGS, RT_EXPECTED_PARAMETERS, RT_DEVICE
    config_path, config_sha, KIND, PHASE = sys.argv[1:5]
    file_check(config_path, config_sha)
    S = rt_json(config_path)
    ID = S["identities"][KIND]
    RT_PROJECT = Path(S["project"])
    RT_CFG = RTConfig(**S["training"])
    RT_CFG.validate()
    RT_CONFIGS = {"vanilla_moe": ModelConfig(), "aoe": AoEConfig()}
    RT_EXPECTED_PARAMETERS = S["counts"]
    RT_DEVICE = torch.device("cuda:0")
    case = Path(S["run"]) / KIND
    result = {"status": "FAIL", "pid": os.getpid(), "identity": ID, "phase": PHASE}
    try:
        context_check()
        model = rt_new_cpu(KIND).to(RT_DEVICE)
        exact(ID["architecture"], model.config.to_dict(), "architecture")
        check_tying(model)
        result["initial_model"] = "PASS"
        optimizer, scheduler = rt_optimizer_scheduler(model)

        if PHASE == "save":
            progress = {"step": 0, "input_tokens": 0, "targets": 0, "rows": []}
            sampler = {
                "cursor": 0, "trace": [], "epoch": 0, "shuffle": False,
                "pilot_index_sha256": ID["pilot_index_sha256"],
            }
            train_until(model, optimizer, scheduler, progress, sampler, 2)
            result["checkpoint"] = save_checkpoint(
                case / "checkpoint-step2", model, optimizer, scheduler, progress, sampler,
            )
        elif PHASE == "resume":
            receipt = rt_json(case / "drive-receipt.json")
            progress, sampler, equality, negative = restore_checkpoint(
                receipt["path"], receipt["manifest_sha256"], model, optimizer, scheduler,
            )
            rt_require(progress["step"] == 2 and sampler["cursor"] == 8, "Restore position.")
            result.update(restore="PASS", equality=equality, negative_tests=negative)
            train_until(model, optimizer, scheduler, progress, sampler, 4)
        else:
            raise RuntimeError("Invalid phase.")
        context_check()
        result.update(
            status="PASS", progress=progress, sampler=sampler,
            scheduler_last_epoch=scheduler.last_epoch,
        )
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    finally:
        write_json(case / f"{PHASE}-report.json", result)
    if result["status"] != "PASS":
        raise SystemExit(1)

if __name__ == "__main__":
    main()
'''

source = (
    C65_IMPORTS
    + "\n\n".join(C65_DEFS[name] for name in C65_HELPERS)
    + C65_UTILITIES
)
compile(source, str(C65_WORKER), "exec")
with C65_WORKER.open("x", encoding="utf-8") as f:
    f.write(source)

for identity in C65_SETUP["identities"].values():
    identity["worker_sha256"] = c65_sha(C65_WORKER)
C65_CONFIG = C65_RUN / "config.json"
c65_write(C65_CONFIG, C65_SETUP)
C65_CONFIG_SHA = c65_sha(C65_CONFIG)

# Import chỉ các definitions; không chạy main() hoặc tạo model.
import importlib.util
spec = importlib.util.spec_from_file_location("step65c_worker", C65_WORKER)
C65_API = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = C65_API
spec.loader.exec_module(C65_API)

print("Checkpoint utilities/script:", C65_WORKER)
print("Worker SHA-256:", c65_sha(C65_WORKER))
print("Config SHA-256:", C65_CONFIG_SHA)
print("READY TO TEST — chưa training.")

Checkpoint utilities/script: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae/resume_worker.py
Worker SHA-256: 5839d70f5495b186b0cb1b57e1cd85ee8a6b45e8bdf2e3ca2168fa54098b7888
Config SHA-256: c226ec390f338a9abcbd298336e9b716bad4bbe8fd9bb8ed30af0dedf49e192d
READY TO TEST — chưa training.


In [ ]:
RUN_STEP65C = True

def c65_guard():
    rt_verify_immutable()
    c65_require(c65_sha(C65_CONFIG) == C65_CONFIG_SHA, "Config changed.")
    for path, expected in C65_GUARDS.items():
        c65_require(c65_sha(path) == expected, f"Protected artifact changed: {path}")
    c65_require(
        c65_sha(C65_WORKER) ==
        C65_SETUP["identities"]["vanilla_moe"]["worker_sha256"],
        "Worker changed.",
    )

def c65_child(kind, phase):
    case = C65_RUN / kind
    report_path = case / f"{phase}-report.json"
    log_path = case / f"{phase}.log"
    c65_require(not report_path.exists() and not log_path.exists(), "Phase already executed.")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(RT_PROJECT) + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONHASHSEED"] = "42"
    command = [
        sys.executable, "-u", str(C65_WORKER),
        str(C65_CONFIG), C65_CONFIG_SHA, kind, phase,
    ]
    with log_path.open("x", encoding="utf-8") as log:
        child = subprocess.Popen(
            command, cwd=str(RT_PROJECT), env=env,
            stdout=log, stderr=subprocess.STDOUT,
        )
        try:
            code = child.wait(timeout=900)
        except subprocess.TimeoutExpired:
            child.kill()
            child.wait()
            raise RuntimeError(f"{kind}/{phase}: timeout; log={log_path}")
    if code != 0 or not report_path.is_file():
        print(log_path.read_text(encoding="utf-8", errors="replace")[-10000:])
        detail = c65_json(report_path).get("error") if report_path.is_file() else None
        raise RuntimeError(f"{kind}/{phase}: {detail or 'subprocess failed'}")
    report = c65_json(report_path)
    c65_require(report["status"] == "PASS", report.get("error", "Child failed."))
    c65_require(report["pid"] != os.getpid(), "Not a fresh process.")
    c65_require(report["identity"] == C65_SETUP["identities"][kind], "Child identity.")
    print(kind, phase, "PASS; PID:", report["pid"], "; log:", log_path)
    return report

def c65_drive_copy(kind, saved):
    source = C65_RUN / kind / "checkpoint-step2"
    identity = C65_SETUP["identities"][kind]
    ms = saved["checkpoint"]["manifest_sha256"]
    manifest = C65_API.verify(source, identity, ms)
    size = sum(p.stat().st_size for p in source.iterdir())
    c65_require(
        shutil.disk_usage(C65_DRIVE).free >= size + 256 * 1024**2,
        "Insufficient Drive free-space estimate.",
    )
    destination = C65_DRIVE / f"{kind}-step000002"
    stage = C65_DRIVE / f".{kind}-INCOMPLETE-{uuid.uuid4().hex}"
    c65_require(not destination.exists(), "Refusing Drive checkpoint overwrite.")
    stage.mkdir(exist_ok=False)
    c65_write(stage / "INCOMPLETE.json", {"status": "INCOMPLETE"})
    for name in ("checkpoint.pt", "manifest.json", "manifest.sha256"):
        with (source / name).open("rb") as src, (stage / name).open("xb") as dst:
            shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)
        c65_require(
            (stage / name).stat().st_size == (source / name).stat().st_size
            and c65_sha(stage / name) == c65_sha(source / name),
            f"Drive copy mismatch: {name}",
        )
    C65_API.verify(stage, identity, ms, unpublished=True)
    c65_write(stage / "COMPLETE.json", {"status": "COMPLETE", "manifest_sha256": ms})
    (stage / "INCOMPLETE.json").unlink()  # Chỉ marker của bản copy mới.
    c65_require(not destination.exists(), "Refusing Drive overwrite.")
    stage.rename(destination)
    C65_API.verify(destination, identity, ms)
    receipt = {
        "status": "PASS", "path": str(destination),
        "manifest_sha256": ms, "payload_sha256": manifest["payload"]["sha256"],
        "bytes": manifest["payload"]["bytes"],
    }
    c65_write(C65_RUN / kind / "drive-receipt.json", receipt)
    print(kind, "Drive backup/checksum: PASS")
    return receipt

if not RUN_STEP65C:
    print("NOT RUN — chủ động đổi RUN_STEP65C=True để chạy.")
else:
    # Ngăn chạy lại cùng run và vượt quá 4 optimizer steps/model.
    with (C65_RUN / "STARTED").open("x") as f:
        f.write("Maximum four optimizer steps per model. No automatic retry.\n")
    C65_RESULT = {"status": "FAIL", "models": {}, "blocking_error": None}
    try:
        c65_guard()
        for kind in C65_KINDS:
            (C65_RUN / kind).mkdir(exist_ok=False)
            result = {"status": "FAIL"}
            C65_RESULT["models"][kind] = result

            result["save"] = c65_child(kind, "save")
            result["drive"] = c65_drive_copy(kind, result["save"])
            result["resume"] = c65_child(kind, "resume")
            saved, resumed = result["save"], result["resume"]

            c65_require(saved["progress"]["step"] == 2, "Save step mismatch.")
            c65_require(
                resumed["progress"]["step"] == resumed["scheduler_last_epoch"] == 4,
                "Final global/scheduler step mismatch.",
            )
            c65_require(
                resumed["sampler"]["cursor"] == 16
                and resumed["sampler"]["trace"] == C65_PREFIX,
                "Final data cursor/order mismatch.",
            )
            c65_require(
                resumed["progress"]["input_tokens"] == 8192
                and resumed["progress"]["targets"] == 8176,
                "Final token accounting mismatch.",
            )
            c65_require(
                resumed["progress"]["rows"][:2] == saved["progress"]["rows"],
                "Saved steps 1–2 trajectory changed.",
            )
            c65_require(
                all(v == "PASS" for v in resumed["equality"].values()),
                "Immediate state restoration mismatch.",
            )
            c65_require(
                all(v == "PASS" for v in resumed["negative_tests"].values()),
                "Checkpoint rejection tests failed.",
            )
            c65_require(
                all(row["optimizer_parameter_updated"]
                    and row["finite_checks"] == "PASS"
                    and row["gradient_checks"] == "PASS"
                    for row in resumed["progress"]["rows"]),
                "Training continuation checks failed.",
            )
            result["status"] = "PASS"
            c65_guard()

        C65_RESULT["status"] = "PASS"
    except Exception as exc:
        C65_RESULT["blocking_error"] = f"{type(exc).__name__}: {exc}"
        print("BLOCKING ERROR:", C65_RESULT["blocking_error"])
        print("Dừng; không chạy mô hình tiếp theo hoặc tự động retry.")

    c65_write(C65_RUN / "execution-result.json", C65_RESULT)
    print("Operational test:", C65_RESULT["status"])
    print("Exact deterministic replay: NOT VERIFIED")

vanilla_moe save PASS; PID: 46284 ; log: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae/vanilla_moe/save.log
vanilla_moe Drive backup/checksum: PASS
vanilla_moe resume PASS; PID: 46444 ; log: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae/vanilla_moe/resume.log
aoe save PASS; PID: 46544 ; log: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae/aoe/save.log
aoe Drive backup/checksum: PASS
aoe resume PASS; PID: 46712 ; log: /content/moe-erc/data/metadata/step65c_final/20261025T181533Z-c4158a07f9ae/aoe/resume.log
Operational test: PASS
Exact deterministic replay: NOT VERIFIED


In [ ]:
def c65_final_audit():
    executed = C65_RESULT.get("status") != "NOT RUN"
    operational_checks = C65_RESULT.get("status") == "PASS"
    protected = "NOT RUN"
    backup = "NOT RUN"
    blocking = C65_RESULT.get("blocking_error")
    audit_path = C65_RUN / "final-audit.json"

    try:
        c65_guard()
        protected = "PASS"
        for kind, result in C65_RESULT.get("models", {}).items():
            if "drive" not in result:
                continue
            receipt = result["drive"]
            identity = C65_SETUP["identities"][kind]
            local = C65_RUN / kind / "checkpoint-step2"
            remote = Path(receipt["path"])
            local_manifest = C65_API.verify(local, identity, receipt["manifest_sha256"])
            drive_manifest = C65_API.verify(remote, identity, receipt["manifest_sha256"])
            c65_require(local_manifest == drive_manifest, f"{kind}: manifest mismatch.")
            c65_require(
                c65_sha(local / "checkpoint.pt") == c65_sha(remote / "checkpoint.pt"),
                f"{kind}: final Drive checksum mismatch.",
            )
    except Exception as exc:
        operational_checks = False
        if blocking is None:
            blocking = f"{type(exc).__name__}: {exc}"

    negative_pass = all(
        C65_RESULT.get("models", {}).get(kind, {}).get("resume", {}).get("negative_tests")
        and all(v == "PASS" for v in
                C65_RESULT["models"][kind]["resume"]["negative_tests"].values())
        for kind in C65_KINDS
    )
    operational_checks = operational_checks and negative_pass and protected == "PASS"

    audit = {
        "schema": "step65c-audit-v1",
        "git": C65_COMMIT, "gpu": C65_GPU,
        "torch": str(torch.__version__), "cuda": str(torch.version.cuda),
        "dataset_fp": C65_DATA_FP, "tokenizer_sha256": C65_TOKENIZER_SHA,
        "pilot_manifest_sha256": pilot_sha,
        "worker_sha256": c65_sha(C65_WORKER),
        "config_sha256": C65_CONFIG_SHA,
        "result": C65_RESULT,
        "negative_tests": "PASS" if negative_pass else "NOT PASS",
        "protected_artifacts": protected,
        "operational_checks_passed": operational_checks,
        "exact_deterministic_replay": "NOT VERIFIED",
        "blocking_error": blocking,
        "audit_backup_verification": "See audit-receipt.json",
    }
    try:
        c65_write(audit_path, audit)
        evidence = [audit_path, C65_WORKER, C65_CONFIG]
        if (C65_RUN / "execution-result.json").is_file():
            evidence.append(C65_RUN / "execution-result.json")
        for kind in C65_KINDS:
            case = C65_RUN / kind
            if case.is_dir():
                evidence.extend(
                    p for p in case.iterdir()
                    if p.is_file() and p.suffix in (".json", ".log")
                )

        entries = []
        for source in evidence:
            relative = source.relative_to(C65_RUN)
            target = C65_DRIVE / "evidence" / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, target.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            c65_require(c65_sha(source) == c65_sha(target),
                        f"Audit/evidence backup mismatch: {relative}")
            entries.append({
                "path": str(target.relative_to(C65_DRIVE)),
                "bytes": target.stat().st_size, "sha256": c65_sha(target),
            })
        receipt = {"status": "PASS", "files": entries}
        c65_write(C65_RUN / "audit-receipt.json", receipt)
        c65_write(C65_DRIVE / "audit-receipt.json", receipt)
        c65_require(
            c65_sha(C65_RUN / "audit-receipt.json")
            == c65_sha(C65_DRIVE / "audit-receipt.json"),
            "Audit receipt checksum mismatch.",
        )
        backup = "PASS"
    except Exception as exc:
        backup = "FAIL"
        if blocking is None:
            blocking = f"{type(exc).__name__}: {exc}"

    final_pass = operational_checks and backup == "PASS"

    def show_model(kind, title):
        result = C65_RESULT.get("models", {}).get(kind, {})
        saved = result.get("save", {})
        resumed = result.get("resume", {})
        equality = resumed.get("equality", {})
        done = result.get("status") == "PASS"
        print(f"\n{title}:")
        print("- Initial model:", saved.get("initial_model", "NOT RUN"))
        print("- Steps 1–2:", saved.get("status", "NOT RUN"))
        print("- Checkpoint saved:", saved.get("checkpoint", {}).get("status", "NOT RUN"))
        print("- Drive backup:", result.get("drive", {}).get("status", "NOT RUN"))
        print("- SHA-256 verification:", result.get("drive", {}).get("status", "NOT RUN"))
        print("- Fresh-process restore:", resumed.get("restore", "NOT RUN"))
        for label, key in (
            ("Model state", "model"), ("Optimizer state", "optimizer"),
            ("Scheduler state", "scheduler"), ("RNG state", "rng"),
            ("Sampler cursor", "sampler"),
        ):
            status = equality.get(key, "NOT RUN")
            if key == "model" and equality.get("buffers") != "PASS":
                status = "NOT PASS"
            if key == "optimizer" and equality.get("parameter_names") != "PASS":
                status = "NOT PASS"
            print(f"- {label}:", status)
        print("- Steps 3–4:", "PASS" if done else "NOT PASS")
        print("- Token accounting:", "PASS" if done else "NOT PASS")
        print("- Training continuation:", "PASS" if done else "NOT PASS")
        print("- STATUS:", result.get("status", "NOT RUN"))

    print("\n=== STEP 6.5C FINAL REPORT ===")
    print("\nEnvironment:")
    print("- Git commit:", C65_COMMIT)
    print("- GPU:", C65_GPU)
    print("- Dataset fingerprint:", C65_DATA_FP)
    print("- Tokenizer SHA-256:", C65_TOKENIZER_SHA)
    print("- Pilot manifest identity:", pilot_sha)
    show_model("vanilla_moe", "Vanilla MoE")
    show_model("aoe", "AoE")
    print("\nGeneral:")
    print("- Negative tests:", "PASS" if negative_pass else "NOT PASS")
    print("- Protected artifacts unchanged:", protected)
    print("- Audit path:", audit_path)
    print("- Audit backup verification:", backup)
    print("- Exact deterministic replay: NOT VERIFIED")
    print("- Operational resume:", "PASS" if final_pass else "FAIL")
    print("- STEP 6.5C STATUS:",
          "PASS" if final_pass else ("FAIL" if executed else "BLOCKED"))
    print("- STEP 6 READINESS:", "READY" if final_pass else "NOT READY")
    if blocking:
        print("- Blocking error:", blocking)
    elif not executed:
        print("- Blocking error: RUN_STEP65C=False; acceptance test chưa chạy.")
    print(
        "\nDrive read-back SHA xác nhận nội dung đọc được tại thời điểm kiểm tra; "
        "không bảo đảm tuyệt đối dịch vụ Drive đã đồng bộ xong."
    )
    print("Dừng tại STEP 6.5C; không tự chuyển Step 7.")

c65_final_audit()


=== STEP 6.5C FINAL REPORT ===

Environment:
- Git commit: 665cb7b1cfbca4c6ed3507345b21008321129a2a
- GPU: NVIDIA A100-SXM4-40GB
- Dataset fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
- Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
- Pilot manifest identity: 812ea85c57344598adf4c18268fcca656da8661bdab400cdfd6542b9cc9fbe98

Vanilla MoE:
- Initial model: PASS
- Steps 1–2: PASS
- Checkpoint saved: PASS
- Drive backup: PASS
- SHA-256 verification: PASS
- Fresh-process restore: PASS
- Model state: PASS
- Optimizer state: PASS
- Scheduler state: PASS
- RNG state: PASS
- Sampler cursor: PASS
- Steps 3–4: PASS
- Token accounting: PASS
- Training continuation: PASS
- STATUS: PASS

AoE:
- Initial model: PASS
- Steps 1–2: PASS
- Checkpoint saved: PASS
- Drive backup: PASS
- SHA-256 verification: PASS
- Fresh-process restore: PASS
- Model state: PASS
- Optimizer state: PASS
- Scheduler state: PASS
- RNG state: PASS
- Sampler c

In [ ]:
from pathlib import Path
import json

audit = Path(
    "/content/moe-erc/data/metadata/step65c_final/"
    "20261025T181533Z-c4158a07f9ae/final-audit.json"
)

data = json.loads(audit.read_text())

def show_drive_paths(obj):
    if isinstance(obj, dict):
        for v in obj.values():
            show_drive_paths(v)
    elif isinstance(obj, list):
        for v in obj:
            show_drive_paths(v)
    elif isinstance(obj, str) and "/content/drive/" in obj:
        p = Path(obj)
        print("Drive path:", p)
        print("Exists:", p.exists())
        print()

show_drive_paths(data)

Drive path: /content/drive/MyDrive/MoE_ERC/checkpoint_acceptance/20261025T181533Z-c4158a07f9ae/vanilla_moe-step000002
Exists: True

Drive path: /content/drive/MyDrive/MoE_ERC/checkpoint_acceptance/20261025T181533Z-c4158a07f9ae/aoe-step000002
Exists: True



# Step 7


In [9]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import ast
import copy
import gc
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import uuid

import numpy as np
import torch
from google.colab import drive

# Remote lấy trực tiếp từ Git config của workspace local.
REPO_URL = "https://github.com/kptrong/MoE-ERC-Loss.git"
STEP6_AUDIT_DIR = None  # Optional: chỉ định checkpoint_acceptance/<run đã PASS>.

PROJECT = Path("/content/moe-erc")
DRIVE = Path("/content/drive/MyDrive/MoE_ERC")
COMMIT = "0539303d628cee21941ca3f28d5ef1da7839da50"
FIX_COMMIT = "665cb7b1cfbca4c6ed3507345b21008321129a2a"
CREATION_COMMIT = "2fd3478e478d3a20381e95b06890f8dcdf2ae013"
TOKEN_SHA = "bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797"
DATA_FP = "92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144"
PILOT_SHA = "812ea85c57344598adf4c18268fcca656da8661bdab400cdfd6542b9cc9fbe98"

DATA = PROJECT / "data/tokenized_datasets/dolma55m-mixed512-2b7fb5b510874ad9"
TOK = PROJECT / "data/tokenizers/dolma-bpe16k-2412fe19bc1a0d22"
DATA_SNAPSHOT = DRIVE / (
    "tokenized_dataset_snapshots/"
    "dataset-20261024T071437Z-92acf38d5017-201fa9ae45a4"
)
TOK_SNAPSHOT = DRIVE / (
    "tokenizer_snapshots/"
    "dolma-bpe16k-20261024T044101Z-2412fe19bc1a0d22-1a4739a56d17"
)

REPORT_A = {"status": "FAIL"}
REPORT_B = {"status": "NOT RUN"}
REPORT_C = {"status": "NOT RUN"}

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_new(path, value):
    with Path(path).open("x", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.write("\n")

def safe(root, name):
    require(isinstance(name, str) and name, "Empty artifact path.")
    rel = PurePosixPath(name)
    require(
        not rel.is_absolute() and ".." not in rel.parts
        and rel.as_posix() == name and "\\" not in name and ":" not in name
        and not any(ord(c) < 32 for c in name),
        f"Unsafe artifact path: {name!r}",
    )
    root = Path(root)
    require(not root.is_symlink(), f"Symlink root: {root}")
    p = root
    for part in rel.parts:
        p /= part
        require(not p.is_symlink(), f"Symlink: {p}")
    require(p.is_file() and p.resolve().is_relative_to(root.resolve()), f"Missing file: {p}")
    return p

def inventory(root):
    result = set()
    for p in Path(root).rglob("*"):
        require(not p.is_symlink(), f"Symlink: {p}")
        if p.is_file():
            result.add(p.relative_to(root).as_posix())
        else:
            require(p.is_dir(), f"Unsupported artifact: {p}")
    return result

def verify_entries(root, entries, exact=False):
    seen = set()
    for entry in entries:
        name = entry["path"]
        require(name not in seen, f"Duplicate entry: {name}")
        seen.add(name)
        p = safe(root, name)
        require(type(entry["bytes"]) is int and entry["bytes"] >= 0, "Invalid size.")
        require(
            p.stat().st_size == entry["bytes"] and sha(p) == entry["sha256"],
            f"SHA/size mismatch: {p}",
        )
    if exact:
        require(inventory(root) == seen, f"Inventory mismatch: {root}")

def git(*args):
    return subprocess.check_output(["git", "-C", str(PROJECT), *args])

def blob_sha(commit, name):
    return hashlib.sha256(git("show", f"{commit}:{name}")).hexdigest()

require(torch.cuda.is_available(), "CUDA unavailable.")
require(torch.cuda.device_count() == 1, "Yêu cầu đúng một GPU.")
GPU = torch.cuda.get_device_name(0)
require("A100" in GPU.upper(), f"GPU thực tế không phải A100: {GPU}")
require(torch.cuda.is_bf16_supported(), "BF16 unavailable.")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
require(DRIVE.is_dir(), "Google Drive unavailable.")

if not PROJECT.exists():
    # Không dùng --no-checkout.
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
require((PROJECT / ".git").exists(), "Project không phải Git checkout.")
require(not PROJECT.is_symlink(), "Project root là symlink.")

# Phục hồi riêng trường hợp clone --no-checkout chưa hoàn tất của cell cũ.
head = git("rev-parse", "HEAD").decode().strip()
if head == COMMIT and not git("ls-files", "--stage", "-z"):
    tracked = [
        os.fsdecode(p)
        for p in git("ls-tree", "-r", "--name-only", "-z", COMMIT).split(b"\0") if p
    ]
    require(tracked, "Empty Git tree.")
    for name in tracked:
        rel = Path(name)
        require(not rel.is_absolute() and ".." not in rel.parts, "Unsafe Git path.")
        target = PROJECT / rel
        for p in (target, *target.parents):
            require(not p.is_symlink(), f"Symlink destination: {p}")
        require(not os.path.lexists(target), f"Refusing overwrite: {target}")
    subprocess.run(
        ["git", "-C", str(PROJECT), "restore",
         "--source", COMMIT, "--staged", "--worktree", "--", "."],
        check=True,
    )

require(not git("status", "--porcelain").decode().strip(), "Working tree không sạch.")
for commit in (COMMIT, FIX_COMMIT, CREATION_COMMIT):
    try:
        git("cat-file", "-e", f"{commit}^{{commit}}")
    except subprocess.CalledProcessError:
        subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", commit], check=True)
if git("rev-parse", "HEAD").decode().strip() != COMMIT:
    subprocess.run(
        ["git", "-C", str(PROJECT), "checkout", "--detach", COMMIT], check=True,
    )
require(git("rev-parse", "HEAD").decode().strip() == COMMIT, "Approved Git HEAD mismatch.")
require(not git("status", "--porcelain").decode().strip(), "Dirty source after checkout.")

# Cài chỉ dependencies bị thiếu/không đạt version; không cài lại PyTorch.
from importlib.metadata import version, PackageNotFoundError
from pip._vendor.packaging.specifiers import SpecifierSet

torch_before = str(torch.__version__)
for package, specifier in (("numpy", ">=1.24"), ("tokenizers", ">=0.20,<1")):
    try:
        compatible = version(package) in SpecifierSet(specifier)
    except PackageNotFoundError:
        compatible = False
    if not compatible:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", package + specifier], check=True,
        )
require(str(torch.__version__) == torch_before, "PyTorch unexpectedly changed.")
require(version("numpy") == np.__version__,
        "NumPy vừa thay đổi; restart runtime rồi chạy lại Cell A để tránh import cũ.")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from moe_erc.config import ModelConfig, AoEConfig, ERCConfig
from moe_erc.model_factory import create_model
from moe_erc.tokenizer_utils import FrozenTokenizer
from moe_erc.data_processing import validate_dataset
from moe_erc.dataset import PackedTokenDataset
from moe_erc.erc_loss import generate_router_proxies, compute_activation_matrix
from moe_erc.erc_training import (
    compute_model_erc, finish_optimizer_step, validate_erc_checkpoint_config,
)
import inspect
require(Path(inspect.getfile(finish_optimizer_step)).resolve()
        == PROJECT / "moe_erc/erc_training.py", "Imported wrong package.")

# Pin source mới và chứng minh delta đối với baseline đã duyệt.
package_files = {
    n for n in git("ls-tree", "-r", "--name-only", COMMIT, "--", "moe_erc").decode().splitlines()
    if n.endswith(".py")
}
require(
    {p.relative_to(PROJECT).as_posix() for p in (PROJECT / "moe_erc").rglob("*.py")}
    == package_files,
    "Runtime source inventory mismatch.",
)
GUARDS = {}
for name in package_files:
    p = safe(PROJECT, name)
    require(sha(p) == blob_sha(COMMIT, name), f"Source mismatch: {name}")
    GUARDS[str(p)] = sha(p)

delta = set(git(
    "diff", "--name-only", FIX_COMMIT, COMMIT, "--", "moe_erc",
).decode().splitlines())
require(delta == {
    "moe_erc/config.py", "moe_erc/erc_loss.py", "moe_erc/erc_training.py",
}, f"Unapproved ERC source delta: {delta}")

old_cfg = ast.parse(git("show", f"{FIX_COMMIT}:moe_erc/config.py").decode())
new_cfg = ast.parse((PROJECT / "moe_erc/config.py").read_text())
new_cfg.body = [
    n for n in new_cfg.body if not (isinstance(n, ast.ClassDef) and n.name == "ERCConfig")
]
require(ast.dump(old_cfg) == ast.dump(new_cfg), "Architecture config changed.")

# Chứng minh model creation -> FIX chỉ thêm initializer đã duyệt.
old_model = ast.parse(git("show", f"{CREATION_COMMIT}:moe_erc/model.py").decode())
fix_model = ast.parse(git("show", f"{FIX_COMMIT}:moe_erc/model.py").decode())
cls = next(n for n in fix_model.body if isinstance(n, ast.ClassDef)
           and n.name == "MoELanguageModel")
ctor = next(n for n in cls.body if isinstance(n, ast.FunctionDef) and n.name == "__init__")
initializer = ast.parse(
    "nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)"
).body[0]
require(ast.dump(ctor.body[-1]) == ast.dump(initializer), "Initialization FIX mismatch.")
ctor.body.pop()
require(ast.dump(old_model) == ast.dump(fix_model), "Unexpected historical model changes.")

def verify_snapshot(root, kind):
    require(root.is_dir() and not root.is_symlink(), f"Missing snapshot: {root}")
    require(not (root / "INCOMPLETE.json").exists()
            and not (root / "COMPLETE.pending.json").exists(), "Incomplete snapshot.")
    ms = sha(safe(root, "backup_manifest.json"))
    require((root / "backup_manifest.sha256").read_text().strip() == ms, "Manifest SHA.")
    marker = read_json(safe(root, "COMPLETE.json"))
    require(marker["status"] == "COMPLETE" and marker["manifest_sha256"] == ms,
            "Snapshot COMPLETE marker.")
    m = read_json(root / "backup_manifest.json")
    require(m["format_version"] == 1 and m["tokenizer_sha256"] == TOKEN_SHA,
            "Snapshot format/tokenizer identity.")
    if kind == "dataset":
        require(m["kind"] == "dataset" and m["combined_source_fingerprint"] == DATA_FP,
                "Dataset snapshot identity.")
        require(all(e["path"].startswith("dataset/") for e in m["files"]),
                "Dataset payload path.")
    else:
        require(m["vocab_size"] == 16000 and m["eos_token_id"] == 2, "Tokenizer config.")
    verify_entries(root, m["files"])
    require(inventory(root) == {e["path"] for e in m["files"]} | {
        "backup_manifest.json", "backup_manifest.sha256", "COMPLETE.json",
    }, "Snapshot inventory mismatch.")
    return m

def restore_payload(snapshot, manifest, target, prefix=""):
    entries = [{**e, "path": e["path"][len(prefix):]} for e in manifest["files"]]
    for p in (target, *target.parents):
        require(not p.is_symlink(), f"Symlink destination: {p}")
    if target.exists():
        verify_entries(target, entries, exact=True)
        print("Existing local artifact verified:", target)
        return
    size = sum(e["bytes"] for e in entries)
    require(shutil.disk_usage(PROJECT).free >= size + 4 * 1024**3, "Insufficient local disk.")
    target.parent.mkdir(parents=True, exist_ok=True)
    stage = target.with_name(target.name + ".restore-" + uuid.uuid4().hex)
    stage.mkdir(exist_ok=False)
    for original, entry in zip(manifest["files"], entries):
        rel = PurePosixPath(entry["path"])
        require(entry["path"] and not rel.is_absolute() and ".." not in rel.parts
                and rel.as_posix() == entry["path"], "Unsafe restore path.")
        dest = stage.joinpath(*rel.parts)
        dest.parent.mkdir(parents=True, exist_ok=True)
        with safe(snapshot, original["path"]).open("rb") as src, dest.open("xb") as dst:
            shutil.copyfileobj(src, dst)
    verify_entries(stage, entries, exact=True)
    require(not target.exists(), "Refusing overwrite.")
    stage.rename(target)

restore_payload(DATA_SNAPSHOT, verify_snapshot(DATA_SNAPSHOT, "dataset"), DATA, "dataset/")
restore_payload(TOK_SNAPSHOT, verify_snapshot(TOK_SNAPSHOT, "tokenizer"), TOK)

require(sha(TOK / "tokenizer.json") == TOKEN_SHA, "Tokenizer checksum.")
tokenizer = FrozenTokenizer(TOK / "tokenizer.json")
require(tokenizer.vocab_size == 16000 and tokenizer.eos_id == 2, "Tokenizer config.")
core = read_json(DATA / "dataset_manifest.json")
complete = read_json(DATA / "COMPLETE.json")
require(
    complete["status"] == "COMPLETE" and complete["tests_status"] == "PASS"
    and sha(DATA / "dataset_manifest.json") == complete["dataset_manifest_sha256"]
    and core["combined_source_fingerprint"] == DATA_FP,
    "Dataset identity.",
)
verify_entries(DATA, core["files"])
design = read_json(DATA / "packing_design.json")
encoded_design = json.dumps(
    design, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
).encode()
require(hashlib.sha256(encoded_design).hexdigest() == core["design_sha256"],
        "Packing design fingerprint.")
recorded = design["pipeline_source_sha256"]
require(set(recorded) == {
    "data_processing.py", "dataset.py", "tokenizer_utils.py", "model.py",
}, "Protected pipeline source set changed.")
for name, recorded_sha in recorded.items():
    require(recorded_sha == blob_sha(CREATION_COMMIT, f"moe_erc/{name}"),
            f"Historical source identity: {name}")
    expected = blob_sha(FIX_COMMIT, f"moe_erc/{name}") if name == "model.py" else recorded_sha
    require(sha(PROJECT / "moe_erc" / name) == expected, f"Compatibility mismatch: {name}")

metadata = validate_dataset(DATA, tokenizer=tokenizer, checksums=True)
require(
    metadata["plan"]["provenance"]["git_commit"] == CREATION_COMMIT
    and metadata["sequence_length"] == 512
    and metadata["splits"]["train"]["sequences"] == 107478
    and metadata["splits"]["validation"]["sequences"] == 1006,
    "Dataset creation identity/counts.",
)
pilot = read_json(DATA / "pilot_50m_manifest.json")
require(
    sha(DATA / "pilot_50m_manifest.json") == PILOT_SHA
    == (DATA / "pilot_50m_manifest.sha256").read_text().strip()
    and pilot["combined_source_fingerprint"] == DATA_FP
    and pilot["tokenizer_sha256"] == TOKEN_SHA
    and pilot["dataset_manifest_sha256"] == sha(DATA / "dataset_manifest.json")
    and pilot["split"] == "train" and pilot["selected_input_tokens"] == 50_000_384,
    "Pilot identity.",
)
index_path = safe(DATA, pilot["index_file"])
require(sha(index_path) == pilot["index_file_sha256"], "Pilot indices checksum.")
indices = np.load(index_path, allow_pickle=False)
require(
    indices.ndim == 1 and len(indices) == 97657
    and len(np.unique(indices)) == len(indices)
    and int(indices.min()) >= 0 and int(indices.max()) < 107478,
    "Pilot indices invalid.",
)

# Tái sử dụng worker đã được backup cùng audit PASS của Step 6.5C.
def find_step6_worker():
    roots = (
        [Path(STEP6_AUDIT_DIR)] if STEP6_AUDIT_DIR else
        sorted((DRIVE / "checkpoint_acceptance").glob("*"), reverse=True)
    )
    errors = []
    needed = {
        "verify", "save_checkpoint", "restore_checkpoint", "snapshot",
        "rng_get", "rng_set", "train_until", "rt_new_cpu", "rt_compute_loss",
        "rt_optimizer_scheduler", "rt_check_gradients", "check_progress",
    }
    for root in roots:
        try:
            receipt = read_json(root / "audit-receipt.json")
            entries = receipt.get("files", receipt.get("evidence"))
            require(receipt["status"] == "PASS" and isinstance(entries, list), "Audit receipt.")
            verify_entries(root, entries)
            paths = {Path(e["path"]).name: e["path"] for e in entries}
            require({"resume_worker.py", "config.json", "final-audit.json"} <= paths.keys(),
                    "Not the verified Step 6 worker format.")
            audit = read_json(safe(root, paths["final-audit.json"]))
            require(
                audit["operational_checks_passed"] is True
                and audit["result"]["status"] == "PASS"
                and all(audit["result"]["models"][k]["status"] == "PASS"
                        for k in ("vanilla_moe", "aoe")),
                "Step 6 acceptance not PASS.",
            )
            worker = safe(root, paths["resume_worker.py"])
            tree = ast.parse(worker.read_text(encoding="utf-8"))
            functions = {n.name for n in tree.body if isinstance(n, ast.FunctionDef)}
            require(needed <= functions, f"Checkpoint worker API mismatch: {needed-functions}")
            return root, worker, read_json(safe(root, paths["config.json"]))
        except Exception as exc:
            errors.append(f"{root}: {exc}")
    raise RuntimeError(
        "Không tìm thấy worker Step 6 có verified audit. "
        "Chỉ định STEP6_AUDIT_DIR tới đúng run đã PASS; không tạo framework thay thế.\n"
        + "\n".join(errors[-3:])
    )

STEP6_ROOT, STEP6_SOURCE, STEP6_CONFIG = find_step6_worker()
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:12]
RUN = PROJECT / "data/metadata/step75_erc_smoke" / run_id
CHECKPOINT_DRIVE = DRIVE / "erc_checkpoint_acceptance" / run_id
for p in (RUN, CHECKPOINT_DRIVE):
    p.mkdir(parents=True, exist_ok=False)
    require(shutil.disk_usage(p).free >= 2 * 1024**3, f"Insufficient free space: {p}")
WORKER = RUN / "step6_worker.py"
with STEP6_SOURCE.open("rb") as src, WORKER.open("xb") as dst:
    shutil.copyfileobj(src, dst)
require(sha(WORKER) == sha(STEP6_SOURCE), "Worker copy checksum.")

for root in (DATA, TOK):
    for p in root.rglob("*"):
        if p.is_file():
            GUARDS[str(p)] = sha(p)
GUARDS[str(WORKER)] = sha(WORKER)

training = dict(STEP6_CONFIG["training"])
training["optimizer_steps"] = 3
require(
    training["seed"] == 42 and training["micro_batch_size"] == 2
    and training["accumulation_steps"] == 2 and training["lb_weight"] == 0.01,
    "Step 6 training config mismatch.",
)
ERC = ERCConfig(enabled=True, alpha=1.0, coefficient=1.0)
IDENTITY = {
    "schema": "step65c-v1", "git": COMMIT, "model_type": "vanilla_moe",
    "architecture": ModelConfig().to_dict(), "parameters": 52_288_000,
    "training": training, "erc_config": ERC.to_dict(),
    "dataset_fp": DATA_FP, "dataset_manifest_sha256": sha(DATA / "dataset_manifest.json"),
    "tokenizer_sha256": TOKEN_SHA, "pilot_manifest_sha256": PILOT_SHA,
    "pilot_index_sha256": pilot["index_file_sha256"],
    "prefix": [int(x) for x in indices[:16]],
    "torch": str(torch.__version__), "cuda": str(torch.version.cuda),
    "gpu": GPU, "gpu_count": 1, "scaler": None, "worker_sha256": sha(WORKER),
    "schedule_scope": "three-update smoke only; not Step 8",
}
SETUP = {
    "project": str(PROJECT), "data": str(DATA),
    "tokenizer": str(TOK / "tokenizer.json"), "commit": COMMIT,
    "training": training, "counts": {"vanilla_moe": 52_288_000, "aoe": 52_263_424},
    "run": str(RUN), "worker": str(WORKER), "guards": GUARDS,
    "step6_audit": str(STEP6_ROOT), "identities": {"vanilla_moe": IDENTITY},
}

def load_worker(setup, phase):
    spec = importlib.util.spec_from_file_location("_verified_step6", setup["worker"])
    w = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = w
    spec.loader.exec_module(w)
    require({"S", "ID", "rt_train_step"} <= set(w.train_until.__code__.co_names),
            "Unsupported Step 6 worker contract.")
    w.S, w.ID = setup, setup["identities"]["vanilla_moe"]
    w.KIND, w.PHASE = "vanilla_moe", phase
    w.RT_PROJECT = Path(setup["project"])
    w.RT_CFG = w.RTConfig(**setup["training"])
    w.RT_CFG.validate()
    w.RT_CONFIGS = {"vanilla_moe": ModelConfig(), "aoe": AoEConfig()}
    w.RT_EXPECTED_PARAMETERS = setup["counts"]
    w.RT_DEVICE = torch.device("cuda:0")
    return w

def guard():
    require(git("rev-parse", "HEAD").decode().strip() == COMMIT, "Git HEAD changed.")
    for path, expected in GUARDS.items():
        require(sha(path) == expected, f"Protected artifact changed: {path}")

def model_preflight():
    w = load_worker(SETUP, "preflight")
    model = w.rt_new_cpu("vanilla_moe")
    info = {
        "parameters": sum(p.numel() for p in model.parameters()),
        "embedding_std": float(model.token_embedding.weight.detach().std(unbiased=False)),
        "same_parameter": model.token_embedding.weight is model.lm_head.weight,
        "same_storage": model.token_embedding.weight.data_ptr() == model.lm_head.weight.data_ptr(),
    }
    require(info["parameters"] == 52_288_000, "Parameter count.")
    require(abs(info["embedding_std"] - 0.02) < 1e-4, "Initialization.")
    require(info["same_parameter"] and info["same_storage"], "Weight tying.")
    return info

MODEL_INFO = model_preflight()
w = load_worker(SETUP, "loader")
for split, selected in (("train", IDENTITY["prefix"][:2]), ("validation", [0, 1])):
    ds = PackedTokenDataset(DATA, split, tokenizer=tokenizer)
    try:
        batch = next(iter(w.rt_loader(ds, selected, 2)))
        require(batch["input_ids"].shape == (2, 512), f"{split} DataLoader shape.")
        require(torch.equal(batch["input_ids"], batch["labels"]), f"{split} labels shifted.")
    finally:
        ds.close()
gc.collect()
guard()
REPORT_A = {
    "status": "PASS", "git": COMMIT, "gpu": GPU, "dataset_fp": DATA_FP,
    "tokenizer_sha256": TOKEN_SHA, "pilot_manifest_sha256": PILOT_SHA,
    "dataset_integrity": "PASS", "source_compatibility": "PASS",
    "imports": "PASS", "dataloader": "PASS", "model": MODEL_INFO,
    "step6_audit": str(STEP6_ROOT),
}
write_new(RUN / "restore-report.json", REPORT_A)
print("\n=== STEP 7.5A RESTORE REPORT ===")
for name, value in (
    ("GPU", GPU), ("CUDA/BF16", (torch.cuda.is_available(), torch.cuda.is_bf16_supported())),
    ("Current Git commit", COMMIT), ("ERC imports", "PASS"),
    ("Dataset fingerprint", DATA_FP), ("Tokenizer SHA-256", TOKEN_SHA),
    ("Pilot manifest", PILOT_SHA), ("Dataset integrity", "PASS"),
    ("Source compatibility", "PASS"), ("Model parameters", MODEL_INFO["parameters"]),
    ("Initialization", MODEL_INFO["embedding_std"]),
    ("Weight tying", "PASS"), ("DataLoader", "PASS"), ("STATUS", "PASS"),
):
    print(f"- {name}:", value)
print("Step 6 worker:", STEP6_ROOT)
print("Run:", RUN)


=== STEP 7.5A RESTORE REPORT ===
- GPU: NVIDIA A100-SXM4-40GB
- CUDA/BF16: (True, True)
- Current Git commit: 0539303d628cee21941ca3f28d5ef1da7839da50
- ERC imports: PASS
- Dataset fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
- Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
- Pilot manifest: 812ea85c57344598adf4c18268fcca656da8661bdab400cdfd6542b9cc9fbe98
- Dataset integrity: PASS
- Source compatibility: PASS
- Model parameters: 52288000
- Initialization: 0.020000705495476723
- Weight tying: PASS
- DataLoader: PASS
- STATUS: PASS
Step 6 worker: /content/drive/MyDrive/MoE_ERC/checkpoint_acceptance/20261025T181533Z-c4158a07f9ae
Run: /content/moe-erc/data/metadata/step75_erc_smoke/20261027T155240Z-b001019ddd47


In [10]:
def diagnose_real_batch():
    w = load_worker(SETUP, "forward-backward")
    model = w.rt_new_cpu("vanilla_moe").cuda().train()
    ds = PackedTokenDataset(DATA, "train", tokenizer=FrozenTokenizer(TOK / "tokenizer.json"))
    try:
        cpu_batch = next(iter(w.rt_loader(ds, IDENTITY["prefix"][:2], 2)))
        batch = {k: v.cuda() for k, v in cpu_batch.items()}
        torch.cuda.reset_peak_memory_stats()
        data_metrics = {}
        for bf16 in (False, True):
            with torch.no_grad():
                losses = w.rt_compute_loss(model, "vanilla_moe", batch, bf16=bf16)
                logits = losses["output"].logits
                require(logits.shape == (2, 512, 16000), "Logits shape.")
                require(torch.isfinite(logits).all().item(), "Nonfinite logits.")
                reference_ce = torch.nn.functional.cross_entropy(
                    logits[:, :-1].float().reshape(-1, 16000),
                    batch["labels"][:, 1:].reshape(-1),
                )
                torch.testing.assert_close(losses["ce"], reference_ce, rtol=1e-5, atol=1e-4)
                require(0 < float(reference_ce) < 50, "Catastrophic initial CE scale.")
                data_metrics["bf16" if bf16 else "fp32"] = {
                    k: float(losses[k]) for k in ("ce", "lb_raw", "lb_weighted")
                }
                del losses, logits, reference_ce

        rng_before = w.rng_get()
        erc = compute_model_erc(model, ERC)
        erc_metrics = dict(erc.metrics)
        rng_after = w.rng_get()
        gates = [e.gate_proj.weight for layer in model.layers for e in layer.moe.experts]
        routers = [layer.moe.router.weight for layer in model.layers]
        other = [
            p for layer in model.layers for e in layer.moe.experts
            for p in (e.up_proj.weight, e.down_proj.weight)
        ]
        grads = torch.autograd.grad(
            erc.weighted_loss, gates + routers + other, allow_unused=True,
        )
        gate_grads = grads[:len(gates)]
        router_grads = grads[len(gates):len(gates)+len(routers)]
        other_grads = grads[len(gates)+len(routers):]
        require(all(g is not None and torch.isfinite(g).all().item() for g in gate_grads),
                "Invalid ERC gate gradients.")
        require(all(g is None for g in router_grads), "Direct ERC router gradient detected.")
        require(all(g is None for g in other_grads), "ERC reached up/down projections.")
        gate_norms = [float(g.norm()) for g in gate_grads]
        gradient_report = {
            "gate_norms": gate_norms,
            "nonzero_gate_gradients": sum(x > 0 for x in gate_norms),
            "direct_router_gradients": 0, "up_down_gradients": 0,
            "status": "PASS",
        }
        del grads, gate_grads, router_grads, other_grads

        # Cùng noise, kiểm tra ERC FP32 dưới BF16 autocast.
        w.rng_set(rng_before)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            bf16_erc = compute_model_erc(model, ERC)
        require(bf16_erc.raw_loss.dtype == torch.float32, "ERC not FP32.")
        torch.testing.assert_close(bf16_erc.raw_loss, erc.raw_loss, rtol=0, atol=0)
        del bf16_erc

        # Tái tạo đúng noise đã đo để thống kê M/epsilon.
        w.rng_set(rng_before)
        statistics = []
        with torch.no_grad():
            for li, layer in enumerate(model.layers):
                proxy, eps = generate_router_proxies(layer.moe.router.weight)
                matrix = compute_activation_matrix(proxy, layer.moe.get_gate_weights())
                require(matrix.shape == (16, 16), "M orientation/shape.")
                d = matrix.diagonal()
                off = ~torch.eye(16, dtype=torch.bool, device=matrix.device)
                norms = layer.moe.router.weight.float().norm(dim=1)
                statistics.append({
                    "layer": li, "matrix_shape": list(matrix.shape),
                    "matrix_min": float(matrix.min()), "matrix_mean": float(matrix.mean()),
                    "matrix_max": float(matrix.max()), "diagonal_mean": float(d.mean()),
                    "epsilon_min": float(eps.min()), "epsilon_mean": float(eps.mean()),
                    "epsilon_max": float(eps.max()),
                    "router_norm_min": float(norms.min()),
                    "router_norm_mean": float(norms.mean()),
                    "router_norm_max": float(norms.max()),
                    "row_violation_rate": float(
                        (matrix - ERC.alpha*d[:, None])[off].gt(0).float().mean()),
                    "column_violation_rate": float(
                        (matrix - ERC.alpha*d[None, :])[off].gt(0).float().mean()),
                })
        del erc

        # Backward tổng loss, không optimizer step; vẫn fresh weights.
        model.zero_grad(set_to_none=True)
        w.rng_set(rng_before)
        losses = w.rt_compute_loss(model, "vanilla_moe", batch, bf16=True)
        combined_erc = compute_model_erc(model, ERC)
        total = losses["total"] + combined_erc.weighted_loss
        require(torch.isfinite(total).item(), "Nonfinite total loss.")
        total.backward()
        active = [
            set(torch.where(info.diagnostics.token_counts > 0)[0].tolist())
            for info in losses["output"].routing_info
        ]
        present = w.rt_check_gradients(model, "vanilla_moe", active)
        w.rng_set(rng_after)
        torch.cuda.synchronize()
        return {
            "status": "PASS", "data": data_metrics, **erc_metrics,
            "total_loss": float(total.detach()),
            "gradient": gradient_report, "combined_gradient_tensors": present,
            "layer_statistics": statistics, "finite_checks": "PASS",
            "erc_fp32_under_bf16": "PASS",
            "fp32_bf16_ce_absolute_difference": abs(
                data_metrics["fp32"]["ce"] - data_metrics["bf16"]["ce"]),
            "peak_gpu_memory_gib": torch.cuda.max_memory_allocated()/1024**3,
        }
    finally:
        ds.close()

require(REPORT_A["status"] == "PASS", "Cell A chưa PASS.")
guard()
try:
    REPORT_B = diagnose_real_batch()
except Exception as exc:
    REPORT_B = {"status": "FAIL", "error": f"{type(exc).__name__}: {exc}"}
    raise
finally:
    gc.collect()
    torch.cuda.empty_cache()
    write_new(RUN / "forward-backward-report.json", REPORT_B)
guard()

print("\n=== STEP 7.5B ERC FORWARD/BACKWARD REPORT ===")
for name, value in (
    ("Model parameters", MODEL_INFO["parameters"]),
    ("CE", REPORT_B["data"]),
    ("LB raw/weighted", {
        k: REPORT_B["data"]["bf16"][k] for k in ("lb_raw", "lb_weighted")}),
    ("ERC raw/weighted", (REPORT_B["erc_raw"], REPORT_B["erc_weighted"])),
    ("ERC per-layer", REPORT_B["erc_per_layer"]),
    ("Total loss", REPORT_B["total_loss"]),
    ("Activation matrix / Epsilon / Constraint violations", REPORT_B["layer_statistics"]),
    ("Expert gate gradients", REPORT_B["gradient"]),
    ("Router gradient semantics", "PASS: isolated ERC has no router/up/down gradients"),
    ("FP32/BF16", {
        "ERC_same_noise": REPORT_B["erc_fp32_under_bf16"],
        "CE_absolute_difference": REPORT_B["fp32_bf16_ce_absolute_difference"]}),
    ("NaN/Inf", REPORT_B["finite_checks"]),
    ("Peak GPU memory GiB", REPORT_B["peak_gpu_memory_gib"]),
    ("STATUS", REPORT_B["status"]),
):
    print(f"- {name}:", value)
print("Optimizer steps: 0. Không yêu cầu mọi gate gradient khác zero.")


=== STEP 7.5B ERC FORWARD/BACKWARD REPORT ===
- Model parameters: 52288000
- CE: {'fp32': {'ce': 9.851984977722168, 'lb_raw': 1.5391662120819092, 'lb_weighted': 0.015391661785542965}, 'bf16': {'ce': 9.851974487304688, 'lb_raw': 1.5392942428588867, 'lb_weighted': 0.015392942354083061}}
- LB raw/weighted: {'lb_raw': 1.5392942428588867, 'lb_weighted': 0.015392942354083061}
- ERC raw/weighted: (0.013851640745997429, 0.013851640745997429)
- ERC per-layer: [0.019064251333475113, 0.009415516629815102, 0.014857140369713306, 0.01227931585162878, 0.01284609455615282, 0.014647524803876877]
- Total loss: 9.881218910217285
- Activation matrix / Epsilon / Constraint violations: [{'layer': 0, 'matrix_shape': [16, 16], 'matrix_min': 0.22234469652175903, 'matrix_mean': 0.25704020261764526, 'matrix_max': 0.2911694645881653, 'diagonal_mean': 0.25115570425987244, 'epsilon_min': 0.6498993635177612, 'epsilon_mean': 0.6729446053504944, 'epsilon_max': 0.68867027759552, 'router_norm_min': 0.5533933639526367, 

In [11]:
RUN_ERC_SMOKE = True

RUNNER_SOURCE = r'''
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import os
import sys
import time
import traceback
from unittest.mock import patch

import torch
from moe_erc.config import ModelConfig, AoEConfig, ERCConfig
from moe_erc.erc_loss import compute_erc_loss
from moe_erc.erc_training import (
    compute_model_erc, finish_optimizer_step, validate_erc_checkpoint_config,
)

config_path, config_sha, phase = sys.argv[1:4]
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

if sha(config_path) != config_sha:
    raise RuntimeError("Configuration checksum mismatch.")
S = json.loads(Path(config_path).read_text())
ID = S["identities"]["vanilla_moe"]
spec = importlib.util.spec_from_file_location("_verified_step6", S["worker"])
w = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = w
spec.loader.exec_module(w)
w.S, w.ID, w.KIND, w.PHASE = S, ID, "vanilla_moe", phase
w.RT_PROJECT = Path(S["project"])
w.RT_CFG = w.RTConfig(**S["training"])
w.RT_CFG.validate()
w.RT_CONFIGS = {"vanilla_moe": ModelConfig(), "aoe": AoEConfig()}
w.RT_EXPECTED_PARAMETERS = S["counts"]
w.RT_DEVICE = torch.device("cuda:0")
erc_config = ERCConfig.from_dict(ID["erc_config"])

def context_check():
    w.rt_require(w.rt_git("rev-parse", "HEAD").decode().strip() == ID["git"], "Git identity.")
    for path, expected in S["guards"].items():
        w.file_check(path, expected)
    w.file_check(__file__, ID["erc_runner_sha256"])
    w.rt_require(
        str(torch.__version__) == ID["torch"]
        and str(torch.version.cuda) == ID["cuda"]
        and torch.cuda.device_count() == 1
        and torch.cuda.get_device_name(0) == ID["gpu"],
        "Runtime identity.",
    )
    validate_erc_checkpoint_config(ID["erc_config"], erc_config)

def erc_update(model, kind, optimizer, scheduler, batches, step):
    w.rt_require(kind == "vanilla_moe" and 1 <= step <= 3, "Smoke budget exceeded.")
    counts = [int((b["labels"][:, 1:] != -100).sum()) for b in batches]
    w.rt_require(len(batches) == w.RT_CFG.accumulation_steps and all(n > 0 for n in counts),
                 "Invalid accumulation/targets.")
    model.train()
    optimizer.zero_grad(set_to_none=True)
    gate_before = model.layers[0].moe.get_gate_weights().detach().clone()
    totals = {k: 0.0 for k in ("ce", "lb_raw", "lb_weighted")}
    active = [set() for _ in model.layers]
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    for cpu_batch, count in zip(batches, counts):
        batch = {k: v.cuda() for k, v in cpu_batch.items()}
        losses = w.rt_compute_loss(model, kind, batch, bf16=True)
        for li, routing in enumerate(losses["output"].routing_info):
            active[li].update(torch.where(routing.diagnostics.token_counts > 0)[0].tolist())
        q = count / sum(counts)
        (q * losses["total"]).backward()
        for key in totals:
            totals[key] += q * float(losses[key].detach())
        del losses, batch

    present = w.rt_check_gradients(model, kind, active)
    events = ["data_backward_complete"]
    original_compute, original_clip = compute_model_erc, torch.nn.utils.clip_grad_norm_
    original_optimizer, original_scheduler = optimizer.step, scheduler.step

    def traced_erc(*args, **kwargs):
        events.append("erc_compute")
        result = original_compute(*args, **kwargs)
        w.rt_require(result.weighted_loss.requires_grad, "Detached ERC.")
        def record_backward(gradient):
            events.append("erc_backward")
        result.weighted_loss.register_hook(record_backward)
        return result

    def traced_clip(parameters, *args, **kwargs):
        events.append("clip")
        return original_clip(parameters, *args, **kwargs)

    def traced_optimizer(*args, **kwargs):
        events.append("optimizer_step")
        return original_optimizer(*args, **kwargs)

    def traced_scheduler(*args, **kwargs):
        events.append("scheduler_step")
        return original_scheduler(*args, **kwargs)

    with patch("moe_erc.erc_training.compute_model_erc", side_effect=traced_erc), \
         patch("moe_erc.erc_training.compute_erc_loss", wraps=compute_erc_loss) as core, \
         patch("torch.nn.utils.clip_grad_norm_", side_effect=traced_clip) as clip, \
         patch.object(optimizer, "step", side_effect=traced_optimizer) as opt, \
         patch.object(scheduler, "step", side_effect=traced_scheduler) as sched, \
         patch.object(optimizer, "zero_grad", wraps=optimizer.zero_grad) as zero:
        metrics = finish_optimizer_step(
            model, erc_config, optimizer, scheduler,
            **totals, global_step=step-1, effective_targets=sum(counts),
            clip_norm=w.RT_CFG.clip_norm,
        )
        w.rt_require(opt.call_count == sched.call_count == clip.call_count == 1,
                     "Double/missing optimizer/scheduler/clipping.")
        w.rt_require(core.call_count == len(model.layers), "Double/missing ERC computation.")
        zero.assert_not_called()

    w.rt_require(events == [
        "data_backward_complete", "erc_compute", "erc_backward",
        "clip", "optimizer_step", "scheduler_step",
    ], f"Incorrect update order: {events}")
    w.rt_require(metrics["global_optimizer_step"] == step, "Global step mismatch.")
    w.rt_require(metrics["total_loss"] ==
                 metrics["ce"] + metrics["lb_weighted"] + metrics["erc_weighted"],
                 "Loss decomposition.")
    for key in ("ce", "lb_raw", "lb_weighted", "erc_raw", "erc_weighted",
                "total_loss", "gradient_norm_before_clip"):
        w.rt_require(math.isfinite(metrics[key]), f"Nonfinite {key}")
    updated = not torch.equal(gate_before, model.layers[0].moe.get_gate_weights().detach())
    w.rt_require(updated, "Expert gates did not update.")
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    positions = sum(b["input_ids"].numel() for b in batches)
    return {
        **metrics, "step": step, "events": events,
        "input_token_positions": positions,
        "effective_next_token_targets": sum(counts),
        "gradient_tensors_present": present,
        "optimizer_parameter_updated": updated,
        "gradient_checks": "PASS", "finite_checks": "PASS",
        "single_optimizer_scheduler_clip": "PASS",
        "elapsed_seconds": elapsed,
        "input_tokens_per_second": positions/elapsed,
        "peak_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,
        "throughput_scope": "smoke update including checks; excludes setup/checkpoint/data loading",
    }

# Chỉ thay callback update; giữ nguyên checkpoint framework Step 6.
w.rt_train_step = erc_update
case = Path(S["run"]) / "vanilla_moe"
result = {"status": "FAIL", "phase": phase, "pid": os.getpid(), "identity": ID}
try:
    context_check()
    model = w.rt_new_cpu("vanilla_moe").cuda()
    optimizer, scheduler = w.rt_optimizer_scheduler(model)
    result["rng_settings"] = {
        "seed": S["training"]["seed"],
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    }
    if phase == "save":
        progress = {"step": 0, "input_tokens": 0, "targets": 0, "rows": []}
        sampler = {
            "cursor": 0, "trace": [], "epoch": 0, "shuffle": False,
            "pilot_index_sha256": ID["pilot_index_sha256"],
        }
        w.train_until(model, optimizer, scheduler, progress, sampler, 2)
        result["checkpoint"] = w.save_checkpoint(
            case/"checkpoint-step2", model, optimizer, scheduler, progress, sampler,
        )
    elif phase == "resume":
        receipt = w.rt_json(case/"drive-receipt.json")
        manifest = w.verify(receipt["path"], ID, receipt["manifest_sha256"])
        validate_erc_checkpoint_config(manifest["identity"]["erc_config"], erc_config)
        extra_negative = {}
        for key, bad in {
            "enabled": False, "alpha": 0.5, "coefficient": 0.5,
            "layer_reduction": "sum", "noise_timing": "microbatch",
            "algorithm_version": "invalid",
        }.items():
            wrong = dict(ID["erc_config"])
            wrong[key] = bad
            try:
                validate_erc_checkpoint_config(wrong, erc_config)
            except ValueError:
                extra_negative["erc_"+key] = "PASS"
            else:
                raise RuntimeError(f"Invalid ERC identity accepted: {key}")

        wrong_source = dict(ID)
        wrong_source["worker_sha256"] = "0"*64
        try:
            w.verify(receipt["path"], wrong_source, receipt["manifest_sha256"])
        except RuntimeError as exc:
            w.rt_require("IDENTITY_MISMATCH" in str(exc), "Unrelated source rejection.")
            extra_negative["source_identity"] = "PASS"
        else:
            raise RuntimeError("Invalid source identity accepted.")

        progress, sampler, equality, negative = w.restore_checkpoint(
            receipt["path"], receipt["manifest_sha256"], model, optimizer, scheduler,
        )
        w.rt_require(progress["step"] == 2 and sampler["cursor"] == 8, "Restore position.")
        w.rt_require(progress["input_tokens"] == 4096 and progress["targets"] == 4088,
                     "Restore counters.")
        w.check_tying(model)
        result.update(
            restore="PASS", equality=equality, negative_tests=negative,
            extra_negative_tests=extra_negative, erc_config_identity="PASS",
            weight_tying="PASS", immediate_global_step=2,
            immediate_input_tokens=4096, immediate_targets=4088,
        )
        w.train_until(model, optimizer, scheduler, progress, sampler, 3)
    else:
        raise RuntimeError("Invalid phase.")
    context_check()
    result.update(
        status="PASS", progress=progress, sampler=sampler,
        scheduler_last_epoch=scheduler.last_epoch,
    )
except Exception as exc:
    result["error"] = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()
finally:
    w.write_json(case/f"{phase}-report.json", result)
if result["status"] != "PASS":
    raise SystemExit(1)
'''

def run_child(phase):
    case = RUN / "vanilla_moe"
    log_path, report_path = case/f"{phase}.log", case/f"{phase}-report.json"
    require(not log_path.exists() and not report_path.exists(), "Phase already executed.")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT) + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONHASHSEED"] = "42"
    with log_path.open("x", encoding="utf-8") as log:
        child = subprocess.Popen(
            [sys.executable, "-u", str(RUNNER), str(CONFIG_PATH), CONFIG_SHA, phase],
            cwd=str(PROJECT), env=env, stdout=log, stderr=subprocess.STDOUT,
        )
        try:
            code = child.wait(timeout=900)
        except subprocess.TimeoutExpired:
            child.kill()
            child.wait()
            raise RuntimeError(f"{phase}: timeout; log={log_path}")
    if code != 0 or not report_path.is_file():
        print(log_path.read_text(encoding="utf-8", errors="replace")[-12000:])
        detail = read_json(report_path).get("error") if report_path.is_file() else None
        raise RuntimeError(detail or f"{phase}: subprocess failed.")
    report = read_json(report_path)
    require(report["status"] == "PASS" and report["pid"] != os.getpid(),
            "Invalid fresh-process result.")
    require(report["identity"] == IDENTITY, "Child identity mismatch.")
    print(phase, "PASS; PID:", report["pid"], "; log:", log_path)
    return report

if not RUN_ERC_SMOKE:
    print("=== STEP 7.5C ERC TRAINING & RESUME REPORT ===")
    print("- STATUS: NOT RUN; đổi RUN_ERC_SMOKE=True để cho phép chạy.")
else:
    require(REPORT_A["status"] == REPORT_B["status"] == "PASS", "A/B chưa PASS.")
    guard()
    with (RUN / "STARTED").open("x") as f:
        f.write("Three optimizer steps total; no automatic retry.\n")
    RUNNER = RUN / "erc_resume_entry.py"
    compile(RUNNER_SOURCE, str(RUNNER), "exec")
    with RUNNER.open("x", encoding="utf-8") as f:
        f.write(RUNNER_SOURCE)
    IDENTITY["erc_runner_sha256"] = sha(RUNNER)
    GUARDS[str(RUNNER)] = sha(RUNNER)
    CONFIG_PATH = RUN / "config.json"
    write_new(CONFIG_PATH, SETUP)
    CONFIG_SHA = sha(CONFIG_PATH)
    case = RUN / "vanilla_moe"
    case.mkdir(exist_ok=False)
    REPORT_C = {"status": "FAIL", "blocking_error": None}
    try:
        saved = run_child("save")
        REPORT_C["save"] = saved
        require(saved["progress"]["step"] == 2, "Save step mismatch.")
        w = load_worker(SETUP, "backup")
        source = case / "checkpoint-step2"
        ms = saved["checkpoint"]["manifest_sha256"]
        manifest = w.verify(source, IDENTITY, ms)
        size = sum(p.stat().st_size for p in source.iterdir())
        require(shutil.disk_usage(CHECKPOINT_DRIVE).free >= size + 256*1024**2,
                "Insufficient Drive free-space estimate.")
        stage = CHECKPOINT_DRIVE / (".INCOMPLETE-" + uuid.uuid4().hex)
        destination = CHECKPOINT_DRIVE / "vanilla_moe-erc-step000002"
        require(not destination.exists(), "Refusing checkpoint overwrite.")
        stage.mkdir(exist_ok=False)
        write_new(stage / "INCOMPLETE.json", {"status": "INCOMPLETE"})
        for name in ("checkpoint.pt", "manifest.json", "manifest.sha256"):
            with (source/name).open("rb") as src, (stage/name).open("xb") as dst:
                shutil.copyfileobj(src, dst, length=4*1024*1024)
            require((source/name).stat().st_size == (stage/name).stat().st_size
                    and sha(source/name) == sha(stage/name), f"Drive copy mismatch: {name}")
        w.verify(stage, IDENTITY, ms, unpublished=True)
        write_new(stage / "COMPLETE.json", {"status": "COMPLETE", "manifest_sha256": ms})
        (stage / "INCOMPLETE.json").unlink()  # Chỉ marker vừa tạo.
        require(not destination.exists(), "Refusing Drive overwrite.")
        stage.rename(destination)
        w.verify(destination, IDENTITY, ms)
        receipt = {
            "status": "PASS", "path": str(destination), "manifest_sha256": ms,
            "payload_sha256": manifest["payload"]["sha256"],
        }
        write_new(case / "drive-receipt.json", receipt)
        REPORT_C["drive"] = receipt

        resumed = run_child("resume")
        REPORT_C["resume"] = resumed
        p, s = resumed["progress"], resumed["sampler"]
        require(
            p["step"] == resumed["scheduler_last_epoch"] == 3
            and s["cursor"] == 12 and s["trace"] == IDENTITY["prefix"][:12]
            and p["input_tokens"] == 6144 and p["targets"] == 6132,
            "Final step/data continuation/token accounting mismatch.",
        )
        require(p["rows"][:2] == saved["progress"]["rows"], "Saved trajectory changed.")
        require(all(v == "PASS" for v in resumed["equality"].values()), "Restore state mismatch.")
        require(all(v == "PASS" for v in resumed["negative_tests"].values())
                and all(v == "PASS" for v in resumed["extra_negative_tests"].values()),
                "Negative identity tests failed.")
        require(all(r["single_optimizer_scheduler_clip"] == "PASS"
                    and r["optimizer_parameter_updated"] and r["finite_checks"] == "PASS"
                    for r in p["rows"]), "Training integration failed.")
        guard()
        REPORT_C["status"] = "PASS"
    except Exception as exc:
        REPORT_C["blocking_error"] = f"{type(exc).__name__}: {exc}"
    finally:
        write_new(RUN / "training-resume-report.json", REPORT_C)

    print("\n=== STEP 7.5C ERC TRAINING & RESUME REPORT ===")
    print("- Training steps 1–2:", REPORT_C.get("save", {}).get("status", "NOT RUN"))
    resumed = REPORT_C.get("resume", {})
    equality = resumed.get("equality", {})
    for name, key in (
        ("Model state equality", "model"), ("Optimizer state equality", "optimizer"),
        ("Scheduler state equality", "scheduler"), ("RNG equality", "rng"),
        ("Sampler equality", "sampler"),
    ):
        print(f"- {name}:", equality.get(key, "NOT RUN"))
    for name, value in (
        ("Checkpoint save", REPORT_C.get("save", {}).get("checkpoint", {}).get("status", "NOT RUN")),
        ("Drive backup / SHA-256 verification", REPORT_C.get("drive", {}).get("status", "NOT RUN")),
        ("Fresh-process restore", resumed.get("restore", "NOT RUN")),
        ("ERC configuration", resumed.get("erc_config_identity", "NOT RUN")),
        ("Resume step 3", resumed.get("progress", {}).get("step", "NOT RUN")),
        ("Operational ERC resume", REPORT_C["status"]),
        ("STATUS", REPORT_C["status"]),
    ):
        print(f"- {name}:", value)
    if REPORT_C["status"] != "PASS":
        raise RuntimeError("STOP: " + REPORT_C["blocking_error"])
    print(json.dumps(resumed["progress"]["rows"], indent=2))
    print("Gradient accumulation/clipping/single updates/data order/token accounting/negative tests: PASS")

save PASS; PID: 16593 ; log: /content/moe-erc/data/metadata/step75_erc_smoke/20261027T155240Z-b001019ddd47/vanilla_moe/save.log
resume PASS; PID: 16775 ; log: /content/moe-erc/data/metadata/step75_erc_smoke/20261027T155240Z-b001019ddd47/vanilla_moe/resume.log

=== STEP 7.5C ERC TRAINING & RESUME REPORT ===
- Training steps 1–2: PASS
- Model state equality: PASS
- Optimizer state equality: PASS
- Scheduler state equality: PASS
- RNG equality: PASS
- Sampler equality: PASS
- Checkpoint save: PASS
- Drive backup / SHA-256 verification: PASS
- Fresh-process restore: PASS
- ERC configuration: PASS
- Resume step 3: 3
- Operational ERC resume: PASS
- STATUS: PASS
[
  {
    "ce": 9.76702070236206,
    "lb_raw": 1.5448904037475586,
    "lb_weighted": 0.015448903664946556,
    "erc_applied": true,
    "erc_raw": 0.013851640745997429,
    "erc_weighted": 0.013851640745997429,
    "erc_per_layer": [
      0.019064251333475113,
      0.009415516629815102,
      0.014857140369713306,
      0.0122793

In [13]:
def final_audit():
    prerequisites = {
        "A": REPORT_A.get("status"), "B": REPORT_B.get("status"),
        "C": REPORT_C.get("status"),
    }
    if any(value != "PASS" for value in prerequisites.values()):
        print("=== STEP 7.5 FINAL ERC REPORT ===")
        print("Step statuses:", prerequisites)
        print("- STEP 7 STATUS: BLOCKED")
        print("- STEP 8 READINESS: NOT READY")
        print("Cell D không chạy final audit vì A/B/C chưa cùng PASS.")
        return

    status_d, blocking = "FAIL", None
    try:
        guard()
        require(sha(CONFIG_PATH) == CONFIG_SHA, "Training config file changed.")
        receipt = REPORT_C["drive"]
        w = load_worker(SETUP, "final-audit")
        local = RUN / "vanilla_moe/checkpoint-step2"
        remote = Path(receipt["path"])
        a = w.verify(local, IDENTITY, receipt["manifest_sha256"])
        b = w.verify(remote, IDENTITY, receipt["manifest_sha256"])
        require(a == b and sha(local/"checkpoint.pt") == sha(remote/"checkpoint.pt"),
                "Final local/Drive checkpoint verification failed.")
        validate_erc_checkpoint_config(b["identity"]["erc_config"], ERC)
        equality = REPORT_C["resume"]["equality"]
        require(all(value == "PASS" for value in equality.values()), "Incomplete exact equality.")
        rows = REPORT_C["resume"]["progress"]["rows"]
        require(len(rows) == 3, "Unexpected update count.")

        # Unique audit attempt; rerunning D không overwrite audit trước.
        audit_id = run_id + "-audit-" + uuid.uuid4().hex[:8]
        audit_dir = RUN / audit_id
        drive_audit = DRIVE / "erc_smoke_audits" / audit_id
        audit_dir.mkdir(exist_ok=False)
        drive_audit.mkdir(parents=True, exist_ok=False)
        audit_path = audit_dir / "final-audit.json"
        audit = {
            "schema": "step75-final-audit-v1",
            "identity": IDENTITY, "model": MODEL_INFO,
            "step75A": REPORT_A, "step75B": REPORT_B, "step75C": REPORT_C,
            "protected_artifacts": "PASS",
            "step6_checkpoint_implementation_audit": str(STEP6_ROOT),
            "checkpoint_local": str(local), "checkpoint_drive": str(remote),
            "exact_deterministic_replay": "NOT VERIFIED",
            "audit_backup_verification": "See backup-receipt.json",
        }
        write_new(audit_path, audit)

        evidence = [
            audit_path, WORKER, RUNNER, CONFIG_PATH,
            RUN / "restore-report.json",
            RUN / "forward-backward-report.json",
            RUN / "training-resume-report.json",
        ]
        evidence.extend(
            p for p in (RUN/"vanilla_moe").iterdir()
            if p.is_file() and p.suffix in (".json", ".log")
        )
        total_size = sum(p.stat().st_size for p in evidence)
        require(shutil.disk_usage(drive_audit).free >= total_size + 32*1024**2,
                "Insufficient audit backup free-space estimate.")
        entries = []
        for source in evidence:
            rel = source.relative_to(RUN)
            target = drive_audit / "evidence" / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, target.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            require(sha(source) == sha(target), f"Audit backup mismatch: {rel}")
            entries.append({
                "path": str(target.relative_to(drive_audit)),
                "bytes": target.stat().st_size, "sha256": sha(target),
            })
        backup_receipt = {
            "status": "PASS", "files": entries, "audit_sha256": sha(audit_path),
            "local_audit": str(audit_path), "drive_audit": str(drive_audit),
        }
        write_new(audit_dir / "backup-receipt.json", backup_receipt)
        write_new(drive_audit / "backup-receipt.json", backup_receipt)
        require(sha(audit_dir/"backup-receipt.json") == sha(drive_audit/"backup-receipt.json"),
                "Audit receipt checksum mismatch.")
        verify_entries(drive_audit, entries)
        guard()
        status_d = "PASS"
    except Exception as exc:
        blocking = f"{type(exc).__name__}: {exc}"

    passed = status_d == "PASS"
    resumed = REPORT_C["resume"]
    eq = resumed["equality"]
    rows = resumed["progress"]["rows"]
    print("\n=== STEP 7.5 FINAL ERC REPORT ===")
    print("\nEnvironment:")
    for name, value in (
        ("Git commit", COMMIT), ("GPU", GPU),
        ("PyTorch/CUDA", f"{torch.__version__}/{torch.version.cuda}"),
        ("Dataset fingerprint", DATA_FP), ("Tokenizer SHA-256", TOKEN_SHA),
        ("Pilot manifest", PILOT_SHA),
    ):
        print(f"- {name}:", value)

    print("\nModel:")
    print("- Parameters:", MODEL_INFO["parameters"])
    print("- Initialization:", MODEL_INFO["embedding_std"])
    print("- Weight tying:", MODEL_INFO["same_parameter"] and MODEL_INFO["same_storage"])

    print("\nERC:")
    for name, value in (
        ("Alpha", ERC.alpha), ("ERC coefficient", ERC.coefficient),
        ("LB coefficient", SETUP["training"]["lb_weight"]),
        ("Layer reduction", ERC.layer_reduction), ("Noise timing", ERC.noise_timing),
        ("Algorithm version", ERC.algorithm_version),
        ("ERC forward", REPORT_B["status"]), ("ERC backward", REPORT_B["status"]),
        ("Expert gradients", REPORT_B["gradient"]["status"]),
        ("Router gradient semantics", "PASS: no direct ERC gradient"),
        ("FP32/BF16", REPORT_B["erc_fp32_under_bf16"]),
    ):
        print(f"- {name}:", value)

    print("\nTraining:")
    print("- Steps 1–3:", [r["step"] for r in rows])
    print("- Loss decomposition / Gradient accumulation: PASS")
    print("- Optimizer update / Scheduler update: PASS, one each per update")
    print("- NaN/Inf:", all(r["finite_checks"] == "PASS" for r in rows))
    print("- Peak VRAM GiB:", max(r["peak_allocated_gib"] for r in rows))
    print("- Throughput tokens/s:", [r["input_tokens_per_second"] for r in rows])
    print("  Smoke-only measurements including checks; not a performance benchmark.")

    print("\nCheckpoint:")
    for name, value in (
        ("Save", REPORT_C["save"]["checkpoint"]["status"]),
        ("Drive backup", REPORT_C["drive"]["status"]),
        ("SHA-256 verification", "PASS" if passed else "NOT PASS"),
        ("Fresh-process restore", resumed["restore"]),
        ("Model state", eq["model"]), ("Optimizer state", eq["optimizer"]),
        ("Scheduler state", eq["scheduler"]), ("RNG", eq["rng"]),
        ("Sampler", eq["sampler"]), ("ERC config", resumed["erc_config_identity"]),
        ("Training continuation", REPORT_C["status"]),
        ("Operational resume", REPORT_C["status"]),
    ):
        print(f"- {name}:", value)

    print("\nFinal:")
    print("- Protected artifacts unchanged:", "PASS" if passed else "NOT PASS")
    print("- Audit path:", str(audit_path) if "audit_path" in locals() else "NOT CREATED")
    print("- Audit backup:", str(drive_audit) if passed else "FAIL")
    print("- Step 7.5A status:", REPORT_A["status"])
    print("- Step 7.5B status:", REPORT_B["status"])
    print("- Step 7.5C status:", REPORT_C["status"])
    print("- Step 7.5D status:", status_d)
    print("- Exact deterministic replay: NOT VERIFIED")
    print("- STEP 7 STATUS:", "PASS" if passed else "FAIL")
    print("- STEP 8 READINESS:", "READY" if passed else "NOT READY")
    if blocking:
        print("- Blocking error:", blocking)
    print("Drive read-back SHA xác nhận nội dung tại thời điểm đọc; không bảo đảm đồng bộ tuyệt đối.")
    print("Dừng tại Step 7.5; không tự chạy Step 8.")

final_audit()


=== STEP 7.5 FINAL ERC REPORT ===

Environment:
- Git commit: 0539303d628cee21941ca3f28d5ef1da7839da50
- GPU: NVIDIA A100-SXM4-40GB
- PyTorch/CUDA: 2.11.0+cu130/13.0
- Dataset fingerprint: 92acf38d5017658cd71df8449d8edc553d663cd32fcad4350bb55b7c8cc6e144
- Tokenizer SHA-256: bfdf3501c3037ef2692e039e11b593e9d89b9143caa213ab16c8381e86831797
- Pilot manifest: 812ea85c57344598adf4c18268fcca656da8661bdab400cdfd6542b9cc9fbe98

Model:
- Parameters: 52288000
- Initialization: 0.020000705495476723
- Weight tying: True

ERC:
- Alpha: 1.0
- ERC coefficient: 1.0
- LB coefficient: 0.01
- Layer reduction: mean
- Noise timing: once_per_optimizer_step
- Algorithm version: erc-figure8-stopgrad-v1
- ERC forward: PASS
- ERC backward: PASS
- Expert gradients: PASS
- Router gradient semantics: PASS: no direct ERC gradient
- FP32/BF16: PASS

Training:
- Steps 1–3: [1, 2, 3]
- Loss decomposition / Gradient accumulation: PASS
- Optimizer update / Scheduler update: PASS, one each per update
- NaN/Inf: True
- P

# Step 8
